# Model and numerical implementation overview

This repository implements a continuous-time, two-agent, two-asset planner model with a Poisson automation transition.

The model is designed to study optimal public finance when automation changes the production frontier, workers are hand-to-mouth, capital owners face incomplete markets, and the government can use taxes, transfers, debt, and public ownership of capital. The numerical implementation is organised around one central rule:

$$
\boxed{
\text{freeze private continuation objects, but evaluate current prices, fiscal objects, and drifts live.}
}
$$

This rule is the main organising principle of the codebase.

---

## Model overview

The economy has two groups of agents.

Workers are hand-to-mouth. They supply labour inelastically, do not trade assets, and consume current wages plus transfers. Capital owners hold financial wealth, choose consumption and portfolio exposure, and face idiosyncratic capital-income risk. The idiosyncratic risk washes out in the aggregate, so aggregate quantities are deterministic conditional on the automation regime.

The only aggregate uncertainty is a one-time Poisson arrival of an automation frontier jump. I write the automation regime as

$$
s\in\{0,1\},
$$

where $s=0$ is the pre-automation regime and $s=1$ is the post-automation regime. Regime $1$ is absorbing. The automation frontier is

$$
I_s =
\begin{cases}
I_0, & s=0,\\
I_0+\Delta I, & s=1.
\end{cases}
$$

Conditional on the regime, the production block constructs regime-specific schedules in efficiency units:

$$
Y_s(k),
\qquad
w_s(k),
\qquad
R_s^K(k),
\qquad
r_s^k(k),
\qquad
\sigma_s^K(k).
$$

I collect these schedules in a regime-primitives object,

$$
\mathcal G
=
\left\{
I_s,
\Phi(I_s),
Y_s(k),
w_s(k),
R_s^K(k),
r_s^k(k),
\sigma_s^K(k),
\ldots
\right\}_{s=0,1}.
$$

Downstream modules consume $\mathcal G$. They do not reconstruct the production formulas.

---

## Planner state, controls, and accounting

The planner state is

$$
x=(k,L),
$$

where $k$ is detrended installed capital and $L$ is government net liabilities.

The planner control is

$$
u=(\tau,T,H),
$$

where $\tau$ is the capital-income tax rate, $T$ is the worker transfer, and $H$ is the government holding of the diversified capital claim.

The key balance-sheet identities are

$$
B=L+H,
$$

$$
E^{priv}=k-H,
$$

and

$$
W^K=(k-H)+B=k+L.
$$

Here $B$ is gross government debt, $E^{priv}$ is privately held risky capital exposure, and $W^K$ is capital-owner financial wealth. The carried fiscal state is $L=B-H$, so $H$ changes the composition of the government balance sheet while $L$ is the net fiscal state.

At the automation switch, the continuous state variables do not jump:

$$
k_{\tau+}=k_{\tau-},
\qquad
L_{\tau+}=L_{\tau-}.
$$

Policy instruments may jump across regimes. In particular, $H$ may jump, so gross debt $B=L+H$ may jump, but the carried fiscal state $L$ does not jump.

The primitive feasible state set is

$$
S
=
\{(k,L): k\ge 0,\ k+L\ge 0\}.
$$

The two primitive state walls are

$$
k=0
$$

and

$$
k+L=0.
$$

The inequalities $k+L\ge 0$ and $L\ge -k$ describe the same geometric wall. In the implementation, I treat them as one primitive boundary, even if I attach several economic labels to that boundary in diagnostics.

---

## Current policy set

The full current policy set is

$$
U_s^{full}(k,L)
=
\left\{
(\tau,T,H):
\tau\in[0,\bar\tau),
\quad
T\in[\underline T_s(k),\infty),
\quad
H\in[\max\{0,-L\},k]
\right\}.
$$

The transfer control is semi-infinite. It has a lower bound, but no primitive upper bound. The lower bound ensures positive worker consumption,

$$
C_s^W=w_s(k)+T>0.
$$

Any finite upper transfer cap used in the code is a numerical compactification, not an economic primitive. I therefore treat binding at the artificial transfer cap as a diagnostic object, not as an economic optimum.

The government balance-sheet restriction

$$
H\in[\max\{0,-L\},k]
$$

implies

$$
E^{priv}=k-H\ge 0
$$

and

$$
B=L+H\ge 0.
$$

Since

$$
W^K=E^{priv}+B,
$$

the mechanically feasible risky-share range is

$$
\pi^{mc}\in[0,1],
$$

whenever $W^K=k+L>0$.

---

## Asset-market closure

The market-clearing risky share supplied to capital owners is

$$
\pi^{mc}(k,L,H)
=
\frac{k-H}{k+L}
=
\frac{E^{priv}}{W^K}.
$$

The baseline implementation uses portfolio bounds that strictly contain the mechanically feasible range after accounting for the numerical portfolio tolerance:

$$
\underline\pi+\varepsilon_\pi < 0,
\qquad
1 < \bar\pi-\varepsilon_\pi.
$$

Equivalently, ignoring the numerical tolerance, the economic condition is

$$
\underline\pi<0,
\qquad
\bar\pi>1.
$$

The first implementation may also use infinite bounds,

$$
\underline\pi=-\infty,
\qquad
\bar\pi=+\infty.
$$

This keeps the baseline equilibrium on the interior Merton branch. In that branch, the safe rate is single-valued:

$$
r_{f,s}(k,L;H,\tau)
=
r_s^k(k)
-
\gamma(1-\tau)
\left(\sigma_s^K(k)\right)^2
\pi^{mc}(k,L,H).
$$

The code still includes a portfolio-bound branch. In version 1, this branch is diagnostic: it marks the candidate as outside the implemented interior pricing branch rather than solving a full binding-portfolio complementarity problem.

---

## Numerical map

The numerical implementation is organised as a sequence of explicit blocks rather than as one monolithic nonlinear system. The working map is

$$
\boxed{
\mathcal G
\to
\hat u
\to
\mathcal C[\hat u]
\to
\mathcal O_s(x,u)
\to
(V_1^{\hat u},V_0^{\hat u})
\to
u^\star
\to
\hat u'
}
$$

where:

- $\mathcal G$ is the regime-primitives bundle from the automation and production block;
- $\hat u$ is the anticipated Markov planner rule;
- $\mathcal C[\hat u]$ is the frozen private continuation bundle induced by $\hat u$;
- $\mathcal O_s(x,u)$ is the live current-control oracle;
- $V_s^{\hat u}$ is the pure conditional viability set under the frozen continuation environment;
- $u^\star$ is the planner best response;
- $\hat u'$ is the updated anticipated rule.

The main numerical invariant is

$$
\boxed{
\text{freeze continuation objects, but evaluate current pricing, fiscal objects, and drifts live.}
}
$$

This means that private continuation objects such as $\Psi_s^{\hat u}$ and $\omega_s^{\hat u}$ are fixed inside a planner best-response problem, while current objects such as $\pi^{mc}$, $r_f$, $\dot k$, $\dot L$, tax bases, revenue, and boundary residuals are evaluated at the current candidate control $u=(\tau,T,H)$.

---

## Frozen private continuation block

Given an anticipated Markov rule

$$
\hat u_s(k,L)
=
(\hat\tau_s(k,L),\hat T_s(k,L),\hat H_s(k,L)),
$$

capital owners solve their private continuation problem. The continuation block returns

$$
\mathcal C[\hat u]
=
\left\{
\Psi_s^{\hat u},
\omega_s^{\hat u},
\chi^{\hat u},
\lambda^{Q,\hat u},
\text{validity masks}
\right\}_{s=0,1}.
$$

The owner value function has the homothetic form

$$
V_s^K(W;k,L)
=
\frac{W^{1-\gamma}}{1-\gamma}
\Psi_s^{\hat u}(k,L),
$$

so the owner consumption-wealth ratio is

$$
\omega_s^{\hat u}(k,L)
=
\left(\Psi_s^{\hat u}(k,L)\right)^{-1/\gamma}.
$$

Owner consumption is then

$$
C_s^K(k,L)
=
\omega_s^{\hat u}(k,L)(k+L).
$$

The continuation objects $\Psi_s^{\hat u}$ and $\omega_s^{\hat u}$ are frozen inside a planner best-response problem. They are not recomputed inside the live oracle, the viability witness search, the pointwise policy-improvement step, or Howard iteration.

The solve order is:

1. solve regime $s=1$ first, because it is absorbing;
2. solve regime $s=0$ second, using regime $1$ in the Poisson continuation term.

The continuation block may also compute the pricing-kernel jump factor

$$
\chi^{\hat u}(k,L)
=
\left(
\frac{
\omega_1^{\hat u}(k,L)
}{
\omega_0^{\hat u}(k,L)
}
\right)^{-\gamma},
$$

and the risk-neutral arrival intensity

$$
\lambda^{Q,\hat u}(k,L)
=
\lambda\chi^{\hat u}(k,L).
$$

These are pricing diagnostics. For hard physical viability, I use the physical support of the Poisson event rather than the risk-neutral intensity.

---

## Live current-control oracle

For each candidate state-control pair, the live oracle evaluates

$$
\mathcal O_s(x,u;\mathcal G,\mathcal C[\hat u]).
$$

It takes

$$
s,
\quad
x=(k,L),
\quad
u=(\tau,T,H),
\quad
\mathcal G,
\quad
\mathcal C[\hat u],
$$

and returns current equilibrium objects evaluated at the current candidate control.

The oracle computes

$$
W^K=k+L,
\qquad
B=L+H,
\qquad
E^{priv}=k-H.
$$

When $W^K>0$, it computes

$$
\pi^{mc}(k,L,H)=\frac{k-H}{k+L}.
$$

If the portfolio branch is interior, it computes

$$
r_{f,s}(k,L;H,\tau)
=
r_s^k(k)
-
\gamma(1-\tau)
\left(\sigma_s^K(k)\right)^2
\pi^{mc}(k,L,H).
$$

The oracle then returns current consumption, tax bases, revenues, and drifts. The core drift objects are

$$
\dot k_s^{\hat u}(x;u)
=
Y_s(k)
-
\bigl(w_s(k)+T\bigr)
-
\omega_s^{\hat u}(k,L)(k+L)
-
(\delta+g)k,
$$

and

$$
\dot L_s^{\hat u}(x;u)
=
r_{f,s}(k,L;H,\tau)(L+H)
+
T
-
Hr_s^k(k)
-
\tau
\left[
(k-H)r_s^k(k)
+
r_{f,s}(k,L;H,\tau)(L+H)
\right].
$$

The oracle also reports

$$
\dot W_s^{K,\hat u}(x;u)
=
\dot k_s^{\hat u}(x;u)
+
\dot L_s^{\hat u}(x;u).
$$

I use the notation

$$
f_s^{\hat u}(x;u)
=
\left(
\dot k_s^{\hat u}(x;u),
\dot L_s^{\hat u}(x;u)
\right).
$$

The superscript $\hat u$ means that the private continuation environment is frozen. The argument $u$ means that current policy is evaluated live.

This distinction prevents a common numerical error: reusing old arrays for $r_f$, $\dot k$, or $\dot L$ during policy improvement. Current $\tau$ and $H$ change current pricing through $\pi^{mc}$ and the short rate, so these channels must remain live.

On the exact diagonal wall,

$$
k+L=0,
$$

the oracle does not evaluate divided formulas such as

$$
\pi^{mc}=\frac{k-H}{k+L}.
$$

Instead, it uses boundary-specific logic and unsimplified accounting expressions.

---

## Pure viability sets

For a frozen anticipated rule $\hat u$, I define the current-control drift correspondence

$$
\mathcal F_s^{\hat u}(x)
=
\left\{
f_s^{\hat u}(x;u):
u\in U_s^{full}(x)
\right\}.
$$

The viability problem asks whether there exists an admissible current control that keeps the state feasible. It is not a Hamiltonian maximisation problem.

The post-switch viability set is

$$
V_1^{\hat u}
=
\operatorname{Viab}_{\mathcal F_1^{\hat u}}(S).
$$

The pre-switch viability set is

$$
V_0^{\hat u}
=
\operatorname{Viab}_{\mathcal F_0^{\hat u}}
\left(
S\cap V_1^{\hat u}
\right).
$$

This construction imposes that, before automation arrives, the state must remain inside the post-switch viable set at every date. It is stronger than merely checking that the initial pre-switch state lies in $V_1^{\hat u}$.

On the grid, I compute viability by peeling candidate masks to a greatest fixed point. I store witness controls as certificates of viability, but these witnesses are not planner policies.

The primitive inward checks are analytic. At the wall $k=0$, I require

$$
\dot k\ge 0.
$$

At the wall $k+L=0$, I require

$$
\dot k+\dot L\ge 0.
$$

I keep pure viability sets separate from Howard active masks:

$$
V_s^{\hat u}
=
\text{pure conditional viability set},
$$

while

$$
A_s
=
\text{Howard-only active mask}.
$$

Howard may update $A_s$, but it must not redefine $V_s^{\hat u}$.

---

## Planner pointwise active-set solver

Given costates

$$
p=(J_{s,k},J_{s,L}),
$$

the planner pointwise Hamiltonian is

$$
\mathcal H_s^{\hat u}(x,u;p)
=
U_s^{\hat u}(x;u)
+
p\cdot f_s^{\hat u}(x;u).
$$

The pointwise solver is an active-set solver. It considers interior candidates, control-bound candidates, primitive feasibility, oracle validity, inward feasibility, and branch comparison.

Newton steps are local tools inside smooth branches. The mathematical problem is an active-set problem.

The transfer control requires special care because

$$
T\in[\underline T_s(k),\infty).
$$

Using the baseline transfer derivative convention,

$$
\partial_T C_s^W=1,
$$

$$
\partial_T C_s^K=0,
$$

$$
\partial_T\dot k=-1,
$$

and

$$
\partial_T\dot L=1.
$$

Therefore the drift contribution to the Hamiltonian has linear $T$ coefficient

$$
J_{s,L}-J_{s,k}.
$$

The pointwise solver must distinguish:

- a finite interior transfer solution;
- a lower-bound transfer solution;
- a monotone branch with no finite maximiser;
- an asymptotic or ill-posed branch;
- artificial compactification-cap binding.

The artificial transfer cap is a numerical diagnostic, not an economic optimum.

---

## Howard inner planner solver

For a fixed anticipated environment, Howard iteration solves the planner HJB on fixed pure viability sets.

Within a Howard cycle, I:

1. freeze $V_1^{\hat u}$ and $V_0^{\hat u}$;
2. freeze the current Howard active masks during linear evaluation;
3. solve the linear HJB under the current policy;
4. improve policies node by node using the active-set solver and live oracle;
5. optionally update Howard active masks between sweeps;
6. damp policy updates if needed;
7. repeat until policy and active-mask diagnostics stabilise.

The hard rule is

$$
\boxed{
\text{Howard may update numerical active masks, but it must not redefine the pure viability sets.}
}
$$

Pure viability sets are economic domain objects. Howard active masks are numerical working domains.

---

## Outer Markov-perfect fixed point

The outer fixed point is over the anticipated planner rule $\hat u$.

At outer iteration $n$, I:

1. start from $\hat u^{(n)}$;
2. solve the private continuation block exactly:

$$
\mathcal C^{(n)}
=
\mathcal C[\hat u^{(n)}];
$$

3. build the live oracle from $\mathcal G$ and $\mathcal C^{(n)}$;
4. compute $V_1^{\hat u^{(n)}}$;
5. compute $V_0^{\hat u^{(n)}}$ inside $S\cap V_1^{\hat u^{(n)}}$;
6. run Howard on the frozen continuation environment and frozen viability sets;
7. obtain the planner best response $u^{\star,(n)}$;
8. update the anticipated rule by relaxed Picard iteration:

$$
\hat u^{(n+1)}
=
(1-\alpha_n)\hat u^{(n)}
+
\alpha_n u^{\star,(n)}.
$$

The baseline rule is

$$
\boxed{
\text{relax }\hat u\text{ only, then recompute }\mathcal C[\hat u]\text{ exactly next iteration.}
}
$$

If I later damp continuation objects directly, I treat that as a false-transient numerical device rather than as the baseline Markov-perfect equilibrium map.

---

## Block structure

### Block 0: automation and production primitives

I first construct the regime-primitives object $\mathcal G$. This block owns the automation and production formulas. It returns callable schedules for output, wages, capital rental rates, net returns, volatility, and derivatives.

It does not solve household HJBs, compute planner controls, evaluate the oracle, construct viability sets, or run Howard iteration.

### Block 1: planner state, controls, and accounting

I then define the canonical planner state and control objects,

$$
x=(k,L),
\qquad
u=(\tau,T,H),
$$

together with the primitive feasible set $S$, primitive control bounds, and the balance-sheet identities

$$
W^K=k+L,
\qquad
B=L+H,
\qquad
E^{priv}=k-H.
$$

This layer is deliberately small, strict, scalar-first, and algebraic.

### Block 2: full current policy set

The policy-set block defines the full admissible current policy set $U_s^{full}(k,L)$ and its numerical compactification $U_s^M(k,L)$.

It also reports binding diagnostics for $\tau$, $T$, and $H$. The compactified transfer cap is treated as a diagnostic object, not an economic primitive.

### Block 3: asset-market parameters and regularity

The asset-market block provides the canonical parameters

$$
\gamma,
\qquad
\underline\pi,
\qquad
\bar\pi,
\qquad
\varepsilon_\pi.
$$

It checks whether a supplied risky share lies on the interior Merton branch:

$$
\underline\pi+\varepsilon_\pi
<
\pi
<
\bar\pi-\varepsilon_\pi.
$$

Block 3 does not construct $\pi^{mc}$. The live oracle constructs $\pi^{mc}(k,L,H)$ later from the current state-control pair. Block 3 only supplies the asset-market regularity checks and diagnostics.

### Block 4: frozen private continuation

Given an anticipated Markov rule $\hat u$, I solve the private owner continuation problem and return $\mathcal C[\hat u]$. The post-automation regime is solved first because it is absorbing. The pre-automation regime is solved second, using the post-regime continuation value in the Poisson term.

### Block 5: automation-risk pricing diagnostics

The continuation block may compute $\chi^{\hat u}$ and $\lambda^{Q,\hat u}$. These are pricing diagnostics. For hard physical viability, I use the physical support of the Poisson event, not the risk-neutral intensity.

### Block 6: live current-control oracle

The oracle evaluates current equilibrium objects at a candidate $(s,x,u)$. It is cheap and algebraic. It does not call the continuation solver, the viability solver, Howard iteration, or an outer fixed point.

### Block 7: pure viability sets

The viability block computes $V_1^{\hat u}$ and $V_0^{\hat u}$ using full-policy-set witness search. Viability is an existence problem, not a planner maximisation problem.

### Block 8: planner pointwise active-set solver

The pointwise solver improves the planner policy at each node given costates. It solves an active-set problem, not a coarse global control-grid maximisation.

### Block 9: Howard inner planner solver

Howard solves the planner HJB for a fixed anticipated environment and fixed pure viability sets. It may update numerical active masks, but not pure viability sets.

### Block 10: outer fixed point

The outer loop updates the anticipated Markov rule $\hat u$ using the planner best response and relaxed Picard iteration.

---

## Implementation principles

I use a block-by-block implementation with explicit contracts. Each module states its inputs, outputs, forbidden responsibilities, diagnostics, and tests. No module should do hidden work outside its contract.

I keep notebooks thin. Notebooks set calibrations, build grids, call block-level routines, inspect diagnostics, plot outputs, and run smoke tests. Core solver logic belongs in importable modules.

I use one canonical evaluator for each economic object:

$$
Y_s,w_s,R_s^K,r_s^k,\sigma_s^K
\quad
\text{come from}
\quad
\mathcal G,
$$

$$
\Psi_s^{\hat u},\omega_s^{\hat u}
\quad
\text{come from}
\quad
\mathcal C[\hat u],
$$

and

$$
\pi^{mc},r_f,\dot k,\dot L
\quad
\text{come from}
\quad
\mathcal O_s.
$$

I avoid duplicating formulas across modules.

I fail fast on economic-domain violations. Core routines should raise explicit errors for invalid states, invalid controls, non-positive owner wealth, invalid production inputs, non-finite risky shares, or unsupported portfolio branches. Clipping and extrapolation are diagnostic-only unless explicitly enabled.

I keep boundary logic explicit. Interior formulas assume interior states. Primitive walls use analytic inward checks:

$$
k=0
\Rightarrow
\dot k\ge 0,
$$

and

$$
k+L=0
\Rightarrow
\dot k+\dot L\ge 0.
$$

I treat viability as an existence problem and planner improvement as an optimisation problem. These are different numerical objects and should not be merged.

I treat diagnostics as first-class outputs. Each block reports domain failures, algebraic identity failures, convergence failures, boundary failures, portfolio-branch failures, KKT failures, viability failures, interpolation failures, and support failures.

---

## Validation strategy

I validate the implementation in stages.

First, I validate the automation and production block by checking

$$
Y_s(k),
\qquad
w_s(k),
\qquad
R_s^K(k),
\qquad
r_s^k(k),
\qquad
\sigma_s^K(k),
$$

along with identities, shape preservation, finite values, and regime differences.

Second, I validate the planner accounting, policy-set, and asset-market blocks by checking

$$
W^K=k+L,
\qquad
B=L+H,
\qquad
E^{priv}=k-H,
$$

and the mechanical risky-share condition

$$
\pi^{mc}\in[0,1]
$$

on primitive-feasible interior points. I also check that the baseline portfolio bounds strictly contain the mechanical range after applying the numerical tolerance.

Third, I validate the continuation block on simple anticipated policies, checking

$$
\Psi_s>0,
\qquad
\omega_s>0,
\qquad
\omega_s=\Psi_s^{-1/\gamma}.
$$

Fourth, I validate the live oracle on interior and boundary states. The oracle should not call the continuation solver. It should report correct statuses, avoid divided formulas on the exact diagonal, and keep the portfolio-bound branch silent under infinite baseline bounds.

Fifth, I validate the viability solver on coarse grids. The post-switch set starts from the primitive grid. The pre-switch set starts from the intersection of the primitive grid and the post-switch viable set. The solver returns witness maps, and full restarts are used to check that states can re-enter after the outer operator changes.

Sixth, I validate the pointwise planner solver using active-set harnesses. I check interior candidates, lower-bound transfer solutions, no-finite-maximiser branches, primitive feasibility filters, oracle-validity filters, and inward-feasibility filters.

Seventh, I validate Howard iteration with fixed continuation objects and fixed pure viability sets. I track HJB residuals, policy residuals, KKT residuals, active-mask movement, and accidental mutation of pure viability sets.

Finally, I validate the full Markov-perfect loop:

$$
\hat u^{(n)}
\to
\mathcal C[\hat u^{(n)}]
\to
(V_1^{\hat u^{(n)}},V_0^{\hat u^{(n)}})
\to
u^{\star,(n)}
\to
\hat u^{(n+1)}.
$$

I begin with coarse grids and easier calibrations, then refine using grid continuation and parameter continuation.

---

## One-line summary

I solve the Markov-perfect planner by combining

$$
\boxed{
\text{microfounded regime primitives}
\to
\text{frozen private continuation}
\to
\text{live current-control oracle}
\to
\text{conditional full-policy-set viability}
\to
\text{Howard planner best response}
\to
\text{relaxed outer fixed point in }\hat u.
}
$$

The key implementation discipline is that the private continuation environment is frozen within a best-response problem, while current prices, fiscal objects, and drifts are always evaluated live at the current candidate control.

# Block 0 — automation / production microfoundations

This block constructs the **microfounded regime-primitives bundle** $\mathcal G$.

Its job is to take the task-based automation model and produce the regime-specific production schedules that all later blocks consume.

The output of this block is upstream of the continuation block, the live oracle, the viability solver, and Howard iteration.

The Plan 10 architecture is:

$$
\mathcal G
\to
\hat u
\to
\mathcal C[\hat u]
\to
\mathcal O_s(x,u)
\to
(V_1^{\hat u},V_0^{\hat u})
\to
u^\star
\to
\hat u'.
$$

Block 0 only constructs the first object in this chain:

$$
\mathcal G.
$$

It does not solve for $\mathcal C[\hat u]$, it does not evaluate $\mathcal O_s(x,u)$, and it does not construct viability sets.

---

## Economic role of Block 0

The economy has a Poisson automation shock.

The feasible automation frontier jumps once at a random arrival time $\tau$ with hazard $\lambda$.

Under the maintained frontier-adoption assumption, equilibrium adoption is always at the frontier:

$$
I_t^{eq}
=
I_t^{tech}.
$$

The automation frontier is

$$
I_t^{tech}
=
I_0
+
\Delta I \cdot 1\{t\ge \tau\}.
$$

Therefore there are two regimes:

- regime $s=0$: pre-automation, with automation share $I_0$;
- regime $s=1$: post-automation, with automation share $I_1=I_0+\Delta I$.

Conditional on the regime, the production side is deterministic.

So Block 0 converts the task-based automation model into regime-specific schedules indexed by $s\in\{0,1\}$.

---

## Technology and reduced form

Production is built from a continuum of tasks.

Labour tasks use effective labour $A_t\ell_t(i)$.

Automated tasks use capital $k_t(i)$.

With the Cobb-Douglas task aggregator and frontier adoption, aggregate output is

$$
Y_t
=
\Phi(I_t)
K_t^{I_t}
(A_tL_t)^{1-I_t}.
$$

The productivity term is

$$
\Phi(I)
=
I^{-I}(1-I)^{-(1-I)}.
$$

The factor-price formulas are

$$
w_t
=
(1-I_t)\frac{Y_t}{L_t},
$$

$$
R_t^K
=
I_t\frac{Y_t}{K_t},
$$

and

$$
r_t^k
=
R_t^K-\delta.
$$

Because the numerical model is solved in efficiency units, Block 0 should expose these objects as functions of detrended capital $k$.

The regime-specific efficiency-unit schedules are:

$$
Y_s(k)
=
\Phi(I_s)k^{I_s},
$$

$$
w_s(k)
=
(1-I_s)Y_s(k),
$$

$$
R_s^K(k)
=
I_s\frac{Y_s(k)}{k},
$$

and

$$
r_s^k(k)
=
R_s^K(k)-\delta.
$$

These are the canonical production schedules used downstream.

---

## Output of Block 0

Block 0 should return a callable regime-primitives bundle:

$$
\mathcal G
=
\left\{
I_s,
\Phi(I_s),
Y_s(k),
w_s(k),
R_s^K(k),
r_s^k(k),
\sigma_s^K(k),
\ldots
\right\}_{s=0,1}.
$$

At minimum, $\mathcal G$ should provide:

- $I_0$ and $I_1$;
- $\Phi(I_0)$ and $\Phi(I_1)$;
- $Y_0(k)$ and $Y_1(k)$;
- $w_0(k)$ and $w_1(k)$;
- $R_0^K(k)$ and $R_1^K(k)$;
- $r_0^k(k)$ and $r_1^k(k)$;
- $\sigma_0^K(k)$ and $\sigma_1^K(k)$ if volatility is exogenous or calibrated separately;
- derivatives with respect to $k$ when they are repeatedly needed later.

The key design rule is:

$$
\boxed{
\text{Downstream modules consume }\mathcal G.
\text{ They do not reconstruct production formulas.}
}
$$

---

## What Block 0 is allowed to depend on

Block 0 may depend on:

- automation parameters;
- production parameters;
- depreciation $\delta$;
- calibrated or callable volatility schedules $\sigma_s^K(k)$;
- numerical tolerances used only for validation.

Block 0 may store parameters such as $\lambda$, $A_0$, and $g$ for downstream consistency.

But the static efficiency-unit schedules $Y_s(k)$, $w_s(k)$, $R_s^K(k)$, and $r_s^k(k)$ are functions of $k$, $I_s$, and $\delta$.

The Poisson intensity $\lambda$ enters later through the continuation block.

Trend growth $g$ enters later through detrended laws of motion.

The level $A_0$ matters for level quantities, but Block 0 returns efficiency-unit schedules.

---

## What Block 0 must not do

Block 0 should not:

- solve the owner continuation problem;
- compute $\Psi_s$ or $\omega_s$;
- evaluate planner controls $(\tau,T,H)$;
- compute $\pi^{mc}$;
- compute $r_f$;
- compute $\dot k$ or $\dot L$;
- construct viability sets;
- run witness search;
- solve any HJB;
- run Howard iteration;
- assemble sparse matrices.

Those responsibilities belong to later blocks.

In particular, the Plan 10 wide-portfolio-bound assumption does **not** belong in Block 0.

Portfolio bounds are part of the asset-market/oracle layer, not the production block.

---

## Numerical conventions

Block 0 should be a small, pure module with a thin notebook wrapper.

The canonical module should be something like:

```text
automation_block.py

In [1]:
%%writefile automation_block.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable, Optional, Union
import warnings

import numpy as np


ArrayLike = Union[float, int, np.ndarray]
Schedule = Union[float, int, Callable[[ArrayLike], ArrayLike]]


# ============================================================
# Block 0 contract
# ============================================================
#
# Inputs:
#   - automation / production calibration;
#   - optional volatility schedules.
#
# Outputs:
#   - callable planner-unit regime primitives:
#       Y_s(k), w_s(k), R_s^K(k), r_s^k(k), sigma_s^K(k),
#       and useful derivatives.
#
# Forbidden responsibilities:
#   - no owner continuation solve;
#   - no live pricing oracle;
#   - no planner controls;
#   - no viability sets;
#   - no Howard iteration;
#   - no sparse linear algebra.
#
# Important convention:
#   The production schedules are interior formulas on k > 0.
#   Boundary logic at k = 0 belongs later in state_constraints.py
#   or named oracle boundary branches.


# ============================================================
# Shape and domain helpers
# ============================================================

def _is_scalar_like(x: Any) -> bool:
    return np.ndim(x) == 0


def _as_float_array(x: ArrayLike) -> np.ndarray:
    return np.asarray(x, dtype=float)


def _return_like_input(out: np.ndarray, like: ArrayLike) -> ArrayLike:
    """
    Scalar input -> Python float.
    Array input  -> ndarray.
    """
    out = np.asarray(out, dtype=float)

    if _is_scalar_like(like):
        if out.shape == ():
            return float(out)
        if out.size == 1:
            return float(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar input produced non-scalar output with shape {out.shape}."
        )

    return out


def _require_positive_k(k: ArrayLike, *, name: str = "k") -> np.ndarray:
    """
    Require strictly positive k for interior production schedules.

    This deliberately raises at k <= 0. Do not silently clip in core logic.
    """
    k_arr = _as_float_array(k)

    if not np.all(np.isfinite(k_arr)):
        raise ValueError(f"{name} contains non-finite values.")

    if np.any(k_arr <= 0.0):
        raise ValueError(
            f"{name} must be strictly positive for Block 0 schedules. "
            f"Got min({name})={float(np.min(k_arr))}."
        )

    return k_arr


def clip_k_for_plotting(k: ArrayLike, eps: float = 1.0e-12) -> ArrayLike:
    """
    Explicit diagnostic helper for plots only.

    Do not use this in solver logic.
    """
    k_arr = _as_float_array(k)
    out = np.maximum(k_arr, float(eps))
    return _return_like_input(out, k)


def _constant_schedule(c: float) -> Callable[[ArrayLike], ArrayLike]:
    c = float(c)

    def fn(k: ArrayLike) -> ArrayLike:
        k_arr = _as_float_array(k)
        out = np.full(k_arr.shape, c, dtype=float)
        return _return_like_input(out, k)

    return fn


def _coerce_schedule(obj: Schedule, *, name: str) -> Callable[[ArrayLike], ArrayLike]:
    """
    Convert a constant or callable into a shape-safe schedule.

    Contract:
      scalar input -> scalar output;
      array input  -> same-shape array output.

    This catches cases like lambda k: 0.15 returning a scalar for vector input.
    """
    if not callable(obj):
        return _constant_schedule(float(obj))

    def fn(k: ArrayLike) -> ArrayLike:
        k_arr = _as_float_array(k)
        call_arg: ArrayLike = float(k_arr) if _is_scalar_like(k) else k_arr

        raw = obj(call_arg)
        out = np.asarray(raw, dtype=float)

        if out.shape == ():
            out = np.full(k_arr.shape, float(out), dtype=float)
        else:
            try:
                out = np.broadcast_to(out, k_arr.shape).astype(float, copy=True)
            except ValueError as exc:
                raise ValueError(
                    f"Schedule '{name}' returned shape {out.shape}, "
                    f"but input k has shape {k_arr.shape}."
                ) from exc

        if not np.all(np.isfinite(out)):
            raise ValueError(f"Schedule '{name}' returned non-finite values.")

        return _return_like_input(out, k)

    return fn


def _eval_array(name: str, value: ArrayLike, expected_shape: tuple[int, ...]) -> np.ndarray:
    arr = np.asarray(value, dtype=float)

    if arr.shape != expected_shape:
        raise RuntimeError(
            f"{name} has shape {arr.shape}, expected {expected_shape}."
        )

    if not np.all(np.isfinite(arr)):
        raise RuntimeError(f"{name} contains non-finite values.")

    return arr


def _check_close(
    name: str,
    lhs: np.ndarray,
    rhs: np.ndarray,
    *,
    atol: float,
    rtol: float,
) -> float:
    lhs = np.asarray(lhs, dtype=float)
    rhs = np.asarray(rhs, dtype=float)

    scale = max(
        1.0,
        float(np.nanmax(np.abs(lhs))),
        float(np.nanmax(np.abs(rhs))),
    )
    err = float(np.nanmax(np.abs(lhs - rhs)))
    allowed = atol + rtol * scale

    if err > allowed:
        raise RuntimeError(
            f"{name} failed: max abs error {err:.3e}, allowed {allowed:.3e}."
        )

    return err


# ============================================================
# Automation primitives
# ============================================================

def log_phi(I: float) -> float:
    """
    log Phi(I), where Phi(I) = I^{-I} (1-I)^{-(1-I)}.
    """
    I = float(I)

    if not (0.0 < I < 1.0):
        raise ValueError(f"I must lie strictly in (0,1). Got I={I}.")

    return -I * np.log(I) - (1.0 - I) * np.log(1.0 - I)


def phi(I: float) -> float:
    """
    Phi(I) = I^{-I} (1-I)^{-(1-I)}.
    """
    return float(np.exp(log_phi(I)))


@dataclass(frozen=True)
class AutomationParams:
    """
    Primitive automation / production parameters.

    Block 0 returns planner-unit / efficiency-unit schedules:
        Y_s(k), w_s(k), R_s^K(k), r_s^k(k), sigma_s^K(k).

    lam, A0, and g are stored here for downstream consistency:
      - lam enters later through continuation and switching;
      - g enters later in detrended laws of motion and HJB reductions;
      - A0 matters for level variables, but this block returns efficiency-unit schedules.

    The deterministic task model pins down output, wages, and rental rates.
    sigma_s^K(k) is supplied as an exogenous or calibrated volatility schedule.
    """
    lam: float
    I0: float
    dI: float
    delta: float
    A0: float = 1.0
    g: float = 0.0
    sigma0: Schedule = 0.0
    sigma1: Optional[Schedule] = None

    def __post_init__(self) -> None:
        if self.lam <= 0.0:
            raise ValueError("lam must be positive.")

        if self.A0 <= 0.0:
            raise ValueError("A0 must be positive.")

        if self.delta < 0.0:
            raise ValueError("delta must be nonnegative.")

        if self.dI < 0.0:
            raise ValueError("dI should be nonnegative for an automation frontier jump.")

        if not (0.0 < self.I0 < 1.0):
            raise ValueError(f"I0 must lie strictly in (0,1). Got I0={self.I0}.")

        if not (0.0 < self.I1 < 1.0):
            raise ValueError(
                f"I1 = I0 + dI must lie strictly in (0,1). Got I1={self.I1}."
            )

    @property
    def I1(self) -> float:
        return self.I0 + self.dI


@dataclass(frozen=True)
class RegimePrimitives:
    """
    Callable microfounded regime-primitives bundle G.

    This object is intended to be immutable. Downstream modules should consume
    these schedules rather than re-deriving production formulas.
    """
    params: AutomationParams
    sigma0_fn: Callable[[ArrayLike], ArrayLike]
    sigma1_fn: Callable[[ArrayLike], ArrayLike]

    # ---------- regime metadata ----------

    def I(self, s: int) -> float:
        if s == 0:
            return self.params.I0
        if s == 1:
            return self.params.I1
        raise ValueError("Regime s must be 0 or 1.")

    def log_Phi(self, s: int) -> float:
        return log_phi(self.I(s))

    def Phi(self, s: int) -> float:
        return phi(self.I(s))

    # ---------- efficiency-unit production schedules ----------

    def Y(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Output in efficiency units:
            Y_s(k) = Phi(I_s) k^{I_s}.
        """
        I = self.I(s)
        k_arr = _require_positive_k(k)
        out = np.exp(log_phi(I)) * np.power(k_arr, I)
        return _return_like_input(out, k)

    def dY_dk(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        First derivative:
            dY_s/dk = I_s Phi(I_s) k^{I_s - 1}.
        """
        I = self.I(s)
        k_arr = _require_positive_k(k)
        out = I * np.exp(log_phi(I)) * np.power(k_arr, I - 1.0)
        return _return_like_input(out, k)

    def d2Y_dk2(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Second derivative:
            d2Y_s/dk2 = I_s (I_s - 1) Phi(I_s) k^{I_s - 2}.
        """
        I = self.I(s)
        k_arr = _require_positive_k(k)
        out = I * (I - 1.0) * np.exp(log_phi(I)) * np.power(k_arr, I - 2.0)
        return _return_like_input(out, k)

    def w(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Wage per unit raw labour in efficiency units:
            w_s(k) = (1 - I_s) Y_s(k).
        """
        I = self.I(s)
        out = (1.0 - I) * _as_float_array(self.Y(s, k))
        return _return_like_input(out, k)

    def dw_dk(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Derivative of wage:
            dw_s/dk = (1 - I_s) dY_s/dk.
        """
        I = self.I(s)
        out = (1.0 - I) * _as_float_array(self.dY_dk(s, k))
        return _return_like_input(out, k)

    def Rk(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Gross rental rate:
            R_s^K(k) = I_s Y_s(k) / k
                     = I_s Phi(I_s) k^{I_s - 1}.
        """
        return self.dY_dk(s, k)

    def dRk_dk(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Derivative of gross rental rate:
            dR_s^K/dk = d2Y_s/dk2.
        """
        return self.d2Y_dk2(s, k)

    def rk(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Net return on capital:
            r_s^k(k) = R_s^K(k) - delta.
        """
        out = _as_float_array(self.Rk(s, k)) - self.params.delta
        return _return_like_input(out, k)

    def drk_dk(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Derivative of net return:
            dr_s^k/dk = dR_s^K/dk.
        """
        return self.dRk_dk(s, k)

    # ---------- exogenous / calibrated volatility schedules ----------

    def sigmaK(self, s: int, k: ArrayLike) -> ArrayLike:
        """
        Idiosyncratic capital-return volatility sigma_s^K(k).

        This is supplied as a calibrated constant or callable schedule.
        It is not derived by the deterministic task block.
        """
        _require_positive_k(k)

        if s == 0:
            out = self.sigma0_fn(k)
        elif s == 1:
            out = self.sigma1_fn(k)
        else:
            raise ValueError("Regime s must be 0 or 1.")

        out_arr = _as_float_array(out)

        if not np.all(np.isfinite(out_arr)):
            raise ValueError(f"sigmaK_{s} returned non-finite values.")

        if np.any(out_arr < 0.0):
            raise ValueError(f"sigmaK_{s} returned negative volatility values.")

        return _return_like_input(out_arr, k)

    # ---------- convenience aliases ----------

    def Y0(self, k: ArrayLike) -> ArrayLike:
        return self.Y(0, k)

    def Y1(self, k: ArrayLike) -> ArrayLike:
        return self.Y(1, k)

    def w0(self, k: ArrayLike) -> ArrayLike:
        return self.w(0, k)

    def w1(self, k: ArrayLike) -> ArrayLike:
        return self.w(1, k)

    def Rk0(self, k: ArrayLike) -> ArrayLike:
        return self.Rk(0, k)

    def Rk1(self, k: ArrayLike) -> ArrayLike:
        return self.Rk(1, k)

    def rk0(self, k: ArrayLike) -> ArrayLike:
        return self.rk(0, k)

    def rk1(self, k: ArrayLike) -> ArrayLike:
        return self.rk(1, k)

    def sigmaK0(self, k: ArrayLike) -> ArrayLike:
        return self.sigmaK(0, k)

    def sigmaK1(self, k: ArrayLike) -> ArrayLike:
        return self.sigmaK(1, k)


def build_regime_primitives(params: AutomationParams) -> RegimePrimitives:
    """
    Build the immutable callable regime-primitives bundle G.
    """
    sigma0_fn = _coerce_schedule(params.sigma0, name="sigma0")
    sigma1_source = params.sigma0 if params.sigma1 is None else params.sigma1
    sigma1_fn = _coerce_schedule(sigma1_source, name="sigma1")

    return RegimePrimitives(
        params=params,
        sigma0_fn=sigma0_fn,
        sigma1_fn=sigma1_fn,
    )


# ============================================================
# Validation
# ============================================================

def validate_regime_primitives(
    G: RegimePrimitives,
    k_grid: ArrayLike,
    *,
    atol: float = 1.0e-10,
    rtol: float = 1.0e-8,
    warn_on_weak_regime_difference: bool = True,
) -> dict[str, float]:
    """
    Validate the Block 0 contract on a strictly positive k-grid.

    Checks:
      - k_grid is strictly positive and finite;
      - every schedule returns the same shape as k_grid;
      - all values are finite;
      - output, wages, and gross rental rates are positive;
      - volatility is nonnegative;
      - core identities hold:
            w_s = (1 - I_s) Y_s
            R_s^K = I_s Y_s / k
            dY_s/dk = R_s^K
            r_s^k = R_s^K - delta
            dr_s^k/dk = dR_s^K/dk
      - scalar input returns scalar output.
    """
    k_arr = _require_positive_k(k_grid, name="k_grid")
    expected_shape = k_arr.shape

    if k_arr.size == 0:
        raise ValueError("k_grid must be non-empty.")

    report: dict[str, float] = {
        "k_min": float(np.min(k_arr)),
        "k_max": float(np.max(k_arr)),
        "n_grid": float(k_arr.size),
    }

    max_identity_error = 0.0
    max_derivative_error = 0.0

    for s in (0, 1):
        I = G.I(s)

        Ys = _eval_array(f"Y_{s}", G.Y(s, k_arr), expected_shape)
        dYs = _eval_array(f"dY_dk_{s}", G.dY_dk(s, k_arr), expected_shape)
        d2Ys = _eval_array(f"d2Y_dk2_{s}", G.d2Y_dk2(s, k_arr), expected_shape)
        ws = _eval_array(f"w_{s}", G.w(s, k_arr), expected_shape)
        dws = _eval_array(f"dw_dk_{s}", G.dw_dk(s, k_arr), expected_shape)
        Rks = _eval_array(f"Rk_{s}", G.Rk(s, k_arr), expected_shape)
        dRks = _eval_array(f"dRk_dk_{s}", G.dRk_dk(s, k_arr), expected_shape)
        rks = _eval_array(f"rk_{s}", G.rk(s, k_arr), expected_shape)
        drks = _eval_array(f"drk_dk_{s}", G.drk_dk(s, k_arr), expected_shape)
        sigs = _eval_array(f"sigmaK_{s}", G.sigmaK(s, k_arr), expected_shape)

        if np.any(Ys <= 0.0):
            raise RuntimeError(f"Y_{s}(k) must be strictly positive.")

        if np.any(ws <= 0.0):
            raise RuntimeError(f"w_{s}(k) must be strictly positive.")

        if np.any(Rks <= 0.0):
            raise RuntimeError(f"Rk_{s}(k) must be strictly positive.")

        if np.any(sigs < -atol):
            raise RuntimeError(f"sigmaK_{s}(k) must be nonnegative.")

        if np.any(dYs <= 0.0):
            raise RuntimeError(f"dY_dk_{s}(k) should be strictly positive.")

        if np.any(dRks >= 0.0):
            raise RuntimeError(
                f"dRk_dk_{s}(k) should be strictly negative for I_s in (0,1)."
            )

        err_w = _check_close(
            f"w identity, regime {s}",
            ws,
            (1.0 - I) * Ys,
            atol=atol,
            rtol=rtol,
        )

        err_dw = _check_close(
            f"dw identity, regime {s}",
            dws,
            (1.0 - I) * dYs,
            atol=atol,
            rtol=rtol,
        )

        err_R = _check_close(
            f"Rk identity, regime {s}",
            Rks,
            I * Ys / k_arr,
            atol=atol,
            rtol=rtol,
        )

        err_dY = _check_close(
            f"dY identity, regime {s}",
            dYs,
            Rks,
            atol=atol,
            rtol=rtol,
        )

        err_dR = _check_close(
            f"dR identity, regime {s}",
            dRks,
            d2Ys,
            atol=atol,
            rtol=rtol,
        )

        err_r = _check_close(
            f"rk identity, regime {s}",
            rks,
            Rks - G.params.delta,
            atol=atol,
            rtol=rtol,
        )

        err_dr = _check_close(
            f"drk identity, regime {s}",
            drks,
            dRks,
            atol=atol,
            rtol=rtol,
        )

        max_identity_error = max(max_identity_error, err_w, err_R, err_r)
        max_derivative_error = max(
            max_derivative_error,
            err_dw,
            err_dY,
            err_dR,
            err_dr,
        )

        scalar_k = float(k_arr.reshape(-1)[k_arr.size // 2])
        scalar_methods = [
            G.Y,
            G.dY_dk,
            G.d2Y_dk2,
            G.w,
            G.dw_dk,
            G.Rk,
            G.dRk_dk,
            G.rk,
            G.drk_dk,
            G.sigmaK,
        ]

        for method in scalar_methods:
            val = method(s, scalar_k)
            if not np.isscalar(val):
                raise RuntimeError(
                    f"{method.__name__}(s, scalar_k) should return a scalar."
                )

    report["max_identity_error"] = float(max_identity_error)
    report["max_derivative_error"] = float(max_derivative_error)

    Y_diff = float(np.max(np.abs(_as_float_array(G.Y(1, k_arr)) - _as_float_array(G.Y(0, k_arr)))))
    w_diff = float(np.max(np.abs(_as_float_array(G.w(1, k_arr)) - _as_float_array(G.w(0, k_arr)))))
    R_diff = float(np.max(np.abs(_as_float_array(G.Rk(1, k_arr)) - _as_float_array(G.Rk(0, k_arr)))))
    r_diff = float(np.max(np.abs(_as_float_array(G.rk(1, k_arr)) - _as_float_array(G.rk(0, k_arr)))))

    report["max_abs_Y1_minus_Y0"] = Y_diff
    report["max_abs_w1_minus_w0"] = w_diff
    report["max_abs_Rk1_minus_Rk0"] = R_diff
    report["max_abs_rk1_minus_rk0"] = r_diff

    if warn_on_weak_regime_difference and G.params.dI != 0.0:
        max_diff = max(Y_diff, w_diff, R_diff, r_diff)
        if max_diff <= atol:
            warnings.warn(
                "dI is nonzero, but regime schedules are nearly identical "
                "on the supplied grid. Check calibration and grid.",
                RuntimeWarning,
            )

    return report


def module_smoke_test() -> dict[str, float]:
    """
    Minimal self-test for development.
    """
    params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    G = build_regime_primitives(params)
    k_grid = np.logspace(-3, 2, 200)

    report = validate_regime_primitives(G, k_grid)

    # Strict-domain tests.
    for bad_k in (0.0, -1.0):
        try:
            G.Y(0, bad_k)
        except ValueError:
            pass
        else:
            raise RuntimeError("Strict k-domain check failed.")

    # Shape-safe callable schedule test.
    sig = np.asarray(G.sigmaK(1, k_grid), dtype=float)
    if sig.shape != k_grid.shape:
        raise RuntimeError("Shape-safe callable schedule test failed.")

    # Immutability smoke test.
    try:
        G.params = params
    except Exception:
        pass
    else:
        raise RuntimeError("RegimePrimitives should be frozen / immutable.")

    return report


__all__ = [
    "ArrayLike",
    "Schedule",
    "AutomationParams",
    "RegimePrimitives",
    "build_regime_primitives",
    "validate_regime_primitives",
    "module_smoke_test",
    "log_phi",
    "phi",
    "clip_k_for_plotting",
]

Overwriting automation_block.py


In [2]:
import importlib
import numpy as np

import automation_block
importlib.reload(automation_block)

params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(params)

k_test = np.logspace(-3, 2, 200)
report = automation_block.validate_regime_primitives(G, k_test)

print("Block 0 validation passed.")
print(report)

Block 0 validation passed.
{'k_min': 0.001, 'k_max': 100.0, 'n_grid': 200.0, 'max_identity_error': 7.105427357601002e-15, 'max_derivative_error': 0.0, 'max_abs_Y1_minus_Y0': 7.632405050816173, 'max_abs_w1_minus_w0': 2.5794430304897045, 'max_abs_Rk1_minus_Rk0': 17.8476031950515, 'max_abs_rk1_minus_rk0': 17.847603195051498}


In [3]:
# Strict k-domain tests.
for bad_k in [0.0, -1.0]:
    try:
        G.Rk(0, bad_k)
    except ValueError as exc:
        print(f"Correctly rejected k={bad_k}: {exc}")
    else:
        raise AssertionError(f"G.Rk accepted invalid k={bad_k}.")

# Shape-safe callable schedule test.
k_vec = np.array([0.5, 1.0, 2.0])
sig_vec = G.sigmaK(1, k_vec)

assert isinstance(sig_vec, np.ndarray)
assert sig_vec.shape == k_vec.shape
assert np.allclose(sig_vec, 0.20)

sig_scalar = G.sigmaK(1, 1.0)
assert np.isscalar(sig_scalar)
assert np.isclose(sig_scalar, 0.20)

# Immutability test.
try:
    G.params = params
except Exception:
    print("RegimePrimitives is frozen as intended.")
else:
    raise AssertionError("RegimePrimitives should be immutable.")

print("Block 0 strictness tests passed.")

Correctly rejected k=0.0: k must be strictly positive for Block 0 schedules. Got min(k)=0.0.
Correctly rejected k=-1.0: k must be strictly positive for Block 0 schedules. Got min(k)=-1.0.
RegimePrimitives is frozen as intended.
Block 0 strictness tests passed.


# Block 1 — canonical planner economics layer

Block 1 defines the **planner-side state, controls, bookkeeping identities, primitive feasible set, and primitive control correspondence**.

It sits directly on top of the regime-primitives bundle $\mathcal G$ produced by Block 0.

Block 0 gives us:

$$
\mathcal G
=
\left\{
I_s,
\Phi(I_s),
Y_s(k),
w_s(k),
R_s^K(k),
r_s^k(k),
\sigma_s^K(k),
\ldots
\right\}_{s=0,1}.
$$

Block 1 does **not** reconstruct these production schedules. It consumes $\mathcal G$ as upstream input.

The role of Block 1 is to define the common economic objects that later blocks will use:

$$
x=(k,L),
\qquad
u=(\tau,T,H),
$$

the primitive state set $S$, the primitive control set $U_s(x)$, and the balance-sheet identities:

$$
W^K,
\qquad
B,
\qquad
E^{priv}.
$$

This block should be small, strict, scalar-first, and algebraic.

It should not solve any HJB, compute private continuation objects, evaluate the live pricing oracle, construct viability sets, or run Howard iteration.

---

## Where Block 1 sits in Plan 10

The Plan 10 architecture is:

$$
\mathcal G
\to
\hat u
\to
\mathcal C[\hat u]
\to
\mathcal O_s(x,u)
\to
(V_1^{\hat u},V_0^{\hat u})
\to
u^\star
\to
\hat u'.
$$

Block 1 defines the state/control/accounting layer used by the later objects in this chain.

It is downstream of $\mathcal G$.

It is upstream of:

- the frozen continuation bundle $\mathcal C[\hat u]$;
- the live oracle $\mathcal O_s(x,u)$;
- the viability solver;
- the planner pointwise solver;
- the Howard loop.

---

## State variable

The planner state is

$$
x=(k,L),
$$

where:

- $k$ is detrended installed capital;
- $L$ is government net liabilities.

The fiscal state $L$ is carried across the automation switch.

At the regime switch,

$$
k_{\tau+}=k_{\tau-},
$$

and

$$
L_{\tau+}=L_{\tau-}.
$$

Policy instruments may jump at the switch, but the carried state variables do not.

In particular, $H$ may jump, so gross debt $B=L+H$ may jump, but $L$ itself is continuous.

---

## Controls

The planner control is

$$
u=(\tau,T,H),
$$

where:

- $\tau$ is the capital-income tax rate;
- $T$ is the worker transfer;
- $H$ is the government holding of the diversified capital claim.

The primitive control correspondence is:

$$
\tau\in[0,\bar\tau),
$$

$$
T\in[\underline T_s(k),\infty),
$$

and

$$
H\in[\max\{0,-L\},k].
$$

The transfer control is **semi-infinite**.

It has a lower bound but no primitive upper bound.

The numerical compactification of $T$ is not a primitive economic restriction. It belongs later in the viability and planner pointwise solvers.

---

## Balance-sheet accounting

The balance-sheet identities are:

$$
B=L+H,
$$

$$
E^{priv}=k-H,
$$

and

$$
W^K=(k-H)+B=k+L.
$$

Here:

- $B$ is gross government debt;
- $E^{priv}$ is privately held risky capital exposure;
- $W^K$ is capital-owner financial wealth.

The carried fiscal state is

$$
L=B-H.
$$

So $H$ changes the composition of the government balance sheet, while $L$ is the net fiscal state.

The model allows ordinary government debt issuance through $B>0$.

The baseline restriction $B\ge 0$ means the government does not hold a net safe asset position in this model.

The baseline restriction $0\le H\le k$ means $H$ is public ownership of installed capital, not a short position or leveraged derivative position in the capital claim.

---

## Primitive feasible state set

The primitive feasible set is

$$
S
=
\{(k,L): k\ge 0,\ k+L\ge 0\},
$$

plus any extra validity restrictions required by later equilibrium mappings.

The two default primitive state walls are:

$$
k=0,
$$

and

$$
k+L=0.
$$

The conditions

$$
k+L\ge 0
$$

and

$$
L\ge -k
$$

describe the same geometric wall.

They should not be encoded as two separate primitive boundaries.

The code may attach multiple economic labels to the diagonal wall, but there should be one geometric boundary object.

---

## State status

Block 1 should classify primitive state geometry.

A useful state-status set is:

```text
StateStatus in {
    "interior",
    "k_wall",
    "diagonal_wall",
    "corner",
    "invalid"
}

In [4]:
from pathlib import Path

Path("model").mkdir(exist_ok=True)
Path("model/__init__.py").touch()

In [5]:
%%writefile model/economy.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal, Optional, Tuple
import math
import numpy as np

from automation_block import RegimePrimitives


# ============================================================
# Block 1 contract
# ============================================================
#
# Inputs:
#   - scalar planner states x = (k, L);
#   - scalar planner controls u = (tau, T, H);
#   - regime-primitives bundle G from Block 0;
#   - primitive planner-economy parameters.
#
# Outputs:
#   - canonical State and Control dataclasses;
#   - primitive state diagnostics;
#   - primitive control bounds and admissibility checks;
#   - balance-sheet bookkeeping;
#   - transfer lower bound;
#   - utility / flow-payoff helpers.
#
# Forbidden responsibilities:
#   - no pi^{mc};
#   - no r_f;
#   - no kdot or Ldot;
#   - no tax-base / revenue evaluation;
#   - no continuation solve;
#   - no viability sets;
#   - no Howard iteration.
#
# Important convention:
#   Block 1 is scalar by design. Grid/vector wrappers come later.
#   Exact wall flags and near-wall flags are separate:
#       exact_* flags are used for algebraic branch identities;
#       near_* flags are diagnostics only.


StateStatus = Literal[
    "interior",
    "k_wall",
    "diagonal_wall",
    "corner",
    "invalid",
]


# ============================================================
# Helpers
# ============================================================

def _as_scalar(x: float, *, name: str) -> float:
    arr = np.asarray(x, dtype=float)
    if arr.shape != ():
        raise TypeError(
            f"{name} must be scalar in Block 1. "
            "Use an explicit grid/vector wrapper later."
        )
    val = float(arr)
    if not math.isfinite(val):
        raise ValueError(f"{name} must be finite.")
    return val


def _strictly_positive(x: float, *, name: str) -> None:
    if not (x > 0.0):
        raise ValueError(f"{name} must be strictly positive. Got {name}={x}.")


def _nonnegative(x: float, *, name: str) -> None:
    if not (x >= 0.0):
        raise ValueError(f"{name} must be nonnegative. Got {name}={x}.")


# ============================================================
# Parameters
# ============================================================

@dataclass(frozen=True)
class PlannerEconomyParams:
    """
    Primitive parameters for the canonical planner economics layer.

    tau_upper:
        The strict upper bound in tau in [0, tau_upper).

    transfer_min:
        Primitive lower bound on transfers before worker-consumption adjustment.
        For nonnegative transfers, use transfer_min = 0.

    worker_consumption_eps:
        Small positive buffer used in the transfer floor
            T >= max{transfer_min, -w_s(k) + worker_consumption_eps}.

    state_tol:
        Diagnostic tolerance for near-wall flags only.
        It must not trigger exact-wall algebra.

    control_tol:
        Tolerance used for closed control bounds such as H lower/upper and T lower.
        The tau upper bound remains open.
    """
    tau_upper: float = 1.0
    transfer_min: float = 0.0
    worker_consumption_eps: float = 1.0e-8
    state_tol: float = 1.0e-10
    control_tol: float = 1.0e-12

    def __post_init__(self) -> None:
        if not (0.0 < self.tau_upper <= 1.0):
            raise ValueError("tau_upper must lie in (0, 1].")
        if not math.isfinite(self.transfer_min):
            raise ValueError("transfer_min must be finite.")
        _strictly_positive(self.worker_consumption_eps, name="worker_consumption_eps")
        _nonnegative(self.state_tol, name="state_tol")
        _nonnegative(self.control_tol, name="control_tol")


# ============================================================
# State and control containers
# ============================================================

@dataclass(frozen=True)
class State:
    """
    Planner state x = (k, L).

    This dataclass only enforces scalar finiteness.
    Primitive feasibility is checked by primitive_state_diagnostics / require_primitive_state.
    """
    k: float
    L: float

    def __post_init__(self) -> None:
        object.__setattr__(self, "k", _as_scalar(self.k, name="k"))
        object.__setattr__(self, "L", _as_scalar(self.L, name="L"))

    @property
    def W_K(self) -> float:
        return self.k + self.L


@dataclass(frozen=True)
class Control:
    """
    Planner control u = (tau, T, H).

    This dataclass only enforces scalar finiteness.
    Primitive admissibility is checked by primitive_control_diagnostics.
    """
    tau: float
    T: float
    H: float

    def __post_init__(self) -> None:
        object.__setattr__(self, "tau", _as_scalar(self.tau, name="tau"))
        object.__setattr__(self, "T", _as_scalar(self.T, name="T"))
        object.__setattr__(self, "H", _as_scalar(self.H, name="H"))


# ============================================================
# Primitive state diagnostics
# ============================================================

@dataclass(frozen=True)
class StateDiagnostics:
    k: float
    L: float
    W_K: float

    is_valid: bool
    status: StateStatus
    invalid_reason: Optional[str]

    exact_k_wall: bool
    exact_diagonal_wall: bool
    exact_corner: bool

    near_k_wall: bool
    near_diagonal_wall: bool
    near_corner: bool


def primitive_state_diagnostics(
    x: State,
    params: PlannerEconomyParams,
) -> StateDiagnostics:
    """
    Diagnose primitive state feasibility.

    Primitive closed set:
        k >= 0,
        k + L >= 0.

    Exact-wall flags use exact equality only. Near-wall flags are diagnostics only.
    This avoids sending near-boundary interior states into exact-wall oracle branches.
    """
    k = x.k
    L = x.L
    W_K = x.W_K

    exact_k_wall = (k == 0.0)
    exact_diagonal_wall = (W_K == 0.0)
    exact_corner = exact_k_wall and exact_diagonal_wall

    near_k_wall = abs(k) <= params.state_tol
    near_diagonal_wall = abs(W_K) <= params.state_tol
    near_corner = near_k_wall and near_diagonal_wall

    if k < 0.0:
        return StateDiagnostics(
            k=k,
            L=L,
            W_K=W_K,
            is_valid=False,
            status="invalid",
            invalid_reason="k < 0",
            exact_k_wall=exact_k_wall,
            exact_diagonal_wall=exact_diagonal_wall,
            exact_corner=exact_corner,
            near_k_wall=near_k_wall,
            near_diagonal_wall=near_diagonal_wall,
            near_corner=near_corner,
        )

    if W_K < 0.0:
        return StateDiagnostics(
            k=k,
            L=L,
            W_K=W_K,
            is_valid=False,
            status="invalid",
            invalid_reason="k + L < 0",
            exact_k_wall=exact_k_wall,
            exact_diagonal_wall=exact_diagonal_wall,
            exact_corner=exact_corner,
            near_k_wall=near_k_wall,
            near_diagonal_wall=near_diagonal_wall,
            near_corner=near_corner,
        )

    if exact_corner:
        status: StateStatus = "corner"
    elif exact_diagonal_wall:
        status = "diagonal_wall"
    elif exact_k_wall:
        status = "k_wall"
    else:
        status = "interior"

    return StateDiagnostics(
        k=k,
        L=L,
        W_K=W_K,
        is_valid=True,
        status=status,
        invalid_reason=None,
        exact_k_wall=exact_k_wall,
        exact_diagonal_wall=exact_diagonal_wall,
        exact_corner=exact_corner,
        near_k_wall=near_k_wall,
        near_diagonal_wall=near_diagonal_wall,
        near_corner=near_corner,
    )


def require_primitive_state(
    x: State,
    params: PlannerEconomyParams,
) -> StateDiagnostics:
    diag = primitive_state_diagnostics(x, params)
    if not diag.is_valid:
        raise ValueError(f"Invalid primitive state: {diag.invalid_reason}. State={x}.")
    return diag


# ============================================================
# Balance-sheet bookkeeping
# ============================================================

@dataclass(frozen=True)
class BalanceSheet:
    W_K: float
    B: float
    E_priv: float

    @property
    def identity_error(self) -> float:
        return self.W_K - (self.B + self.E_priv)


def balance_sheet(x: State, u: Control) -> BalanceSheet:
    """
    Balance-sheet identities.

    This function does not compute pi^{mc}. The risky-share ratio belongs to the live oracle.
    """
    W_K = x.k + x.L
    B = x.L + u.H
    E_priv = x.k - u.H

    return BalanceSheet(
        W_K=W_K,
        B=B,
        E_priv=E_priv,
    )


# ============================================================
# Transfer floor
# ============================================================

def wage_for_transfer_floor(
    s: int,
    x: State,
    primitives: RegimePrimitives,
    params: PlannerEconomyParams,
) -> float:
    """
    Wage used in the worker-consumption transfer floor.

    Exact boundary convention:
        if k == 0 exactly, use w_s(0) = 0 as the analytic boundary limit.

    For every strictly positive k, including small positive k, call the strict
    production schedule from Block 0.
    """
    require_primitive_state(x, params)

    if x.k == 0.0:
        return 0.0

    return float(primitives.w(s, x.k))


def transfer_lower_bound(
    s: int,
    x: State,
    primitives: RegimePrimitives,
    params: PlannerEconomyParams,
) -> float:
    """
    Transfer lower bound:
        underline T_s(k) = max{transfer_min, -w_s(k) + worker_consumption_eps}.
    """
    w = wage_for_transfer_floor(s, x, primitives, params)
    return max(params.transfer_min, -w + params.worker_consumption_eps)


# ============================================================
# Primitive control bounds and admissibility
# ============================================================

@dataclass(frozen=True)
class ControlBounds:
    tau_lower: float
    tau_upper: float
    tau_upper_is_open: bool

    T_lower: float
    T_upper: float

    H_lower: float
    H_upper: float

    @property
    def T_is_semi_infinite(self) -> bool:
        return math.isinf(self.T_upper) and self.T_upper > 0.0

    def H_pinned(self, tol: float = 0.0) -> bool:
        return abs(self.H_upper - self.H_lower) <= tol


def control_bounds(
    s: int,
    x: State,
    primitives: RegimePrimitives,
    params: PlannerEconomyParams,
) -> ControlBounds:
    """
    Primitive control correspondence:
        tau in [0, tau_upper),
        T in [underline T_s(k), infinity),
        H in [max{0, -L}, k].

    The numerical compactification T <= T_bar^M is not primitive and belongs
    in a later working-control-set module.
    """
    require_primitive_state(x, params)

    H_lower = max(0.0, -x.L)
    H_upper = x.k

    if H_lower > H_upper + params.control_tol:
        raise RuntimeError(
            "Primitive state passed feasibility but H bounds are inconsistent. "
            f"H_lower={H_lower}, H_upper={H_upper}, state={x}."
        )

    T_lower = transfer_lower_bound(s, x, primitives, params)

    return ControlBounds(
        tau_lower=0.0,
        tau_upper=params.tau_upper,
        tau_upper_is_open=True,
        T_lower=T_lower,
        T_upper=math.inf,
        H_lower=H_lower,
        H_upper=H_upper,
    )


@dataclass(frozen=True)
class ControlDiagnostics:
    bounds: ControlBounds

    tau_ok: bool
    T_ok: bool
    H_ok: bool

    is_admissible: bool
    violations: Tuple[str, ...]


def primitive_control_diagnostics(
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    params: PlannerEconomyParams,
) -> ControlDiagnostics:
    bounds = control_bounds(s, x, primitives, params)
    tol = params.control_tol

    violations: list[str] = []

    tau_ok = (u.tau >= bounds.tau_lower) and (u.tau < bounds.tau_upper)
    if not tau_ok:
        if u.tau < bounds.tau_lower:
            violations.append("tau below lower bound")
        else:
            violations.append("tau at/above strict upper bound")

    T_ok = u.T >= bounds.T_lower - tol
    if not T_ok:
        violations.append("T below transfer lower bound")

    H_ok = (u.H >= bounds.H_lower - tol) and (u.H <= bounds.H_upper + tol)
    if not H_ok:
        if u.H < bounds.H_lower - tol:
            violations.append("H below lower bound")
        if u.H > bounds.H_upper + tol:
            violations.append("H above upper bound")

    return ControlDiagnostics(
        bounds=bounds,
        tau_ok=tau_ok,
        T_ok=T_ok,
        H_ok=H_ok,
        is_admissible=(tau_ok and T_ok and H_ok),
        violations=tuple(violations),
    )


def require_admissible_control(
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    params: PlannerEconomyParams,
) -> ControlDiagnostics:
    diag = primitive_control_diagnostics(s, x, u, primitives, params)
    if not diag.is_admissible:
        raise ValueError(f"Invalid primitive control: {diag.violations}. Control={u}.")
    return diag


# ============================================================
# Utility / flow-payoff helpers
# ============================================================

def crra_utility(c: float, gamma: float) -> float:
    """
    CRRA utility. Uses log utility when gamma == 1.
    """
    c = _as_scalar(c, name="c")
    gamma = _as_scalar(gamma, name="gamma")

    if c <= 0.0:
        raise ValueError(f"Consumption must be positive. Got c={c}.")
    if gamma <= 0.0:
        raise ValueError(f"gamma must be positive. Got gamma={gamma}.")

    if gamma == 1.0:
        return math.log(c)

    return (c ** (1.0 - gamma)) / (1.0 - gamma)


def planner_flow_payoff(
    c_worker: float,
    c_owner: float,
    *,
    gamma_worker: float,
    gamma_owner: float,
    weight_worker: float = 1.0,
    weight_owner: float = 1.0,
) -> float:
    """
    Simple additively separable planner flow payoff helper.

    The live oracle will provide c_worker and c_owner later.
    """
    weight_worker = _as_scalar(weight_worker, name="weight_worker")
    weight_owner = _as_scalar(weight_owner, name="weight_owner")

    if weight_worker < 0.0 or weight_owner < 0.0:
        raise ValueError("Planner weights must be nonnegative.")

    return (
        weight_worker * crra_utility(c_worker, gamma_worker)
        + weight_owner * crra_utility(c_owner, gamma_owner)
    )


# ============================================================
# Validation / smoke test
# ============================================================

def validate_planner_economy_layer(
    primitives: RegimePrimitives,
    params: Optional[PlannerEconomyParams] = None,
) -> dict[str, float]:
    """
    Validate Block 1.

    Tests:
      - interior state and transfer floor;
      - exact diagonal H interval collapse;
      - exact corner transfer floor and H interval;
      - k-wall status away from corner;
      - invalid state rejection;
      - strict tau upper rejection;
      - primitive control-bound failures;
      - exact-vs-near wall separation;
      - small positive k uses interior wage schedule;
      - balance-sheet identity;
      - finite planner flow payoff.
    """
    if params is None:
        params = PlannerEconomyParams()

    report: dict[str, float] = {}

    # Interior state.
    s = 0
    x_int = State(k=1.0, L=0.5)
    st_int = primitive_state_diagnostics(x_int, params)
    if st_int.status != "interior":
        raise RuntimeError(f"Expected interior state, got {st_int.status}.")

    b_int = control_bounds(s, x_int, primitives, params)
    report["interior_T_lower"] = float(b_int.T_lower)

    if not b_int.T_is_semi_infinite:
        raise RuntimeError("Primitive transfer control should be semi-infinite.")

    # Balance-sheet identity.
    u_int = Control(tau=0.2, T=b_int.T_lower, H=0.25)
    require_admissible_control(s, x_int, u_int, primitives, params)
    bs_int = balance_sheet(x_int, u_int)

    if abs(bs_int.identity_error) > 1.0e-12:
        raise RuntimeError("Balance-sheet identity W_K = B + E_priv failed.")

    report["balance_sheet_identity_error"] = float(abs(bs_int.identity_error))

    # Exact diagonal wall: k=1, L=-1, so H in [1,1].
    x_diag = State(k=1.0, L=-1.0)
    st_diag = primitive_state_diagnostics(x_diag, params)

    if st_diag.status != "diagonal_wall":
        raise RuntimeError(f"Expected exact diagonal wall, got {st_diag.status}.")
    if not st_diag.exact_diagonal_wall:
        raise RuntimeError("Exact diagonal flag should be true.")

    b_diag = control_bounds(s, x_diag, primitives, params)
    report["diagonal_H_lower"] = float(b_diag.H_lower)
    report["diagonal_H_upper"] = float(b_diag.H_upper)

    if not b_diag.H_pinned(params.control_tol):
        raise RuntimeError("H should be pinned on exact diagonal wall.")

    u_bad_diag_H = Control(tau=0.2, T=b_diag.T_lower, H=0.0)
    diag_bad = primitive_control_diagnostics(s, x_diag, u_bad_diag_H, primitives, params)
    if diag_bad.is_admissible:
        raise RuntimeError("H below the exact diagonal lower bound should be rejected.")

    # Exact corner: k=0, L=0.
    x_corner = State(k=0.0, L=0.0)
    st_corner = primitive_state_diagnostics(x_corner, params)

    if st_corner.status != "corner":
        raise RuntimeError(f"Expected corner state, got {st_corner.status}.")

    b_corner = control_bounds(s, x_corner, primitives, params)
    report["corner_T_lower"] = float(b_corner.T_lower)
    report["corner_H_lower"] = float(b_corner.H_lower)
    report["corner_H_upper"] = float(b_corner.H_upper)

    if abs(b_corner.T_lower - max(params.transfer_min, params.worker_consumption_eps)) > 1.0e-14:
        raise RuntimeError("Corner transfer floor should use w_s(0)=0 boundary limit.")
    if not b_corner.H_pinned(params.control_tol):
        raise RuntimeError("H should be pinned at the corner.")

    # k-wall away from corner: k=0, L>0.
    x_kwall = State(k=0.0, L=1.0)
    st_kwall = primitive_state_diagnostics(x_kwall, params)

    if st_kwall.status != "k_wall":
        raise RuntimeError(f"Expected k_wall, got {st_kwall.status}.")

    b_kwall = control_bounds(s, x_kwall, primitives, params)
    report["k_wall_H_lower"] = float(b_kwall.H_lower)
    report["k_wall_H_upper"] = float(b_kwall.H_upper)

    if not b_kwall.H_pinned(params.control_tol):
        raise RuntimeError("H should be pinned at H=0 on the k-wall.")

    # Invalid state rejections.
    invalid_state_rejections = 0

    for x_bad in (State(k=-1.0e-12, L=1.0), State(k=1.0, L=-2.0)):
        try:
            require_primitive_state(x_bad, params)
        except ValueError:
            invalid_state_rejections += 1
        else:
            raise RuntimeError(f"Invalid state was not rejected: {x_bad}.")

    report["invalid_state_rejections"] = float(invalid_state_rejections)

    # Strict tau upper rejection.
    u_tau_upper = Control(tau=params.tau_upper, T=b_int.T_lower, H=0.25)
    tau_upper_diag = primitive_control_diagnostics(s, x_int, u_tau_upper, primitives, params)

    if tau_upper_diag.is_admissible:
        raise RuntimeError("tau at the strict upper bound should be rejected.")

    report["tau_upper_rejected"] = 1.0

    # Other primitive control failures.
    control_failure_count = 0

    bad_controls = [
        Control(tau=-1.0e-12, T=b_int.T_lower, H=0.25),
        Control(tau=0.2, T=b_int.T_lower - 1.0, H=0.25),
        Control(tau=0.2, T=b_int.T_lower, H=b_int.H_lower - 1.0),
        Control(tau=0.2, T=b_int.T_lower, H=b_int.H_upper + 1.0),
    ]

    for u_bad in bad_controls:
        d_bad = primitive_control_diagnostics(s, x_int, u_bad, primitives, params)
        if d_bad.is_admissible:
            raise RuntimeError(f"Bad control was not rejected: {u_bad}.")
        control_failure_count += 1

    report["primitive_control_failures_checked"] = float(control_failure_count)

    # Exact-vs-near wall separation.
    # This is intentionally very close to the diagonal but still strictly interior.
    if params.state_tol <= 0.0:
        raise RuntimeError("state_tol must be positive for this validation test.")

    x_near_diag = State(k=1.0, L=-1.0 + 0.1 * params.state_tol)
    st_near_diag = primitive_state_diagnostics(x_near_diag, params)

    if st_near_diag.status != "interior":
        raise RuntimeError(
            "Near-diagonal but positive-wealth state must remain interior. "
            f"Got status={st_near_diag.status}."
        )
    if not st_near_diag.near_diagonal_wall:
        raise RuntimeError("Near-diagonal diagnostic flag should be true.")

    report["near_diagonal_W_K"] = float(st_near_diag.W_K)

    # Small positive k should use the interior wage schedule, not the k=0 boundary limit.
    x_small_k = State(k=0.1 * params.state_tol, L=1.0)
    st_small_k = primitive_state_diagnostics(x_small_k, params)

    if st_small_k.status != "interior":
        raise RuntimeError("Small positive k should be interior, not an exact k_wall.")
    if not st_small_k.near_k_wall:
        raise RuntimeError("Small positive k should trigger near_k_wall diagnostic.")

    wage_small = wage_for_transfer_floor(s, x_small_k, primitives, params)
    wage_direct = float(primitives.w(s, x_small_k.k))

    if not math.isclose(wage_small, wage_direct, rel_tol=1.0e-12, abs_tol=1.0e-14):
        raise RuntimeError("Small positive k should use the strict interior wage schedule.")

    report["small_positive_k_wage"] = float(wage_small)

    # Utility / flow payoff.
    payoff = planner_flow_payoff(
        c_worker=1.25,
        c_owner=2.0,
        gamma_worker=1.0,
        gamma_owner=2.0,
        weight_worker=1.0,
        weight_owner=1.0,
    )

    if not math.isfinite(payoff):
        raise RuntimeError("Flow payoff test returned non-finite value.")

    report["flow_payoff_test"] = float(payoff)

    return report


__all__ = [
    "StateStatus",
    "PlannerEconomyParams",
    "State",
    "Control",
    "StateDiagnostics",
    "BalanceSheet",
    "ControlBounds",
    "ControlDiagnostics",
    "primitive_state_diagnostics",
    "require_primitive_state",
    "balance_sheet",
    "wage_for_transfer_floor",
    "transfer_lower_bound",
    "control_bounds",
    "primitive_control_diagnostics",
    "require_admissible_control",
    "crra_utility",
    "planner_flow_payoff",
    "validate_planner_economy_layer",
]

Overwriting model/economy.py


In [6]:
import importlib
import numpy as np

import automation_block
import model.economy as economy

importlib.reload(automation_block)
importlib.reload(economy)

# Rebuild Block 0 primitives.
auto_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(auto_params)

k_test = np.logspace(-3, 2, 200)
block0_report = automation_block.validate_regime_primitives(G, k_test)

print("Block 0 validation passed.")
print(block0_report)

# Validate Block 1.
econ_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

block1_report = economy.validate_planner_economy_layer(G, econ_params)

print("\nBlock 1 validation passed.")
print(block1_report)

Block 0 validation passed.
{'k_min': 0.001, 'k_max': 100.0, 'n_grid': 200.0, 'max_identity_error': 7.105427357601002e-15, 'max_derivative_error': 0.0, 'max_abs_Y1_minus_Y0': 7.632405050816173, 'max_abs_w1_minus_w0': 2.5794430304897045, 'max_abs_Rk1_minus_Rk0': 17.8476031950515, 'max_abs_rk1_minus_rk0': 17.847603195051498}

Block 1 validation passed.
{'interior_T_lower': 0.0, 'balance_sheet_identity_error': 0.0, 'diagonal_H_lower': 1.0, 'diagonal_H_upper': 1.0, 'corner_T_lower': 1e-08, 'corner_H_lower': 0.0, 'corner_H_upper': 0.0, 'k_wall_H_lower': 0.0, 'k_wall_H_upper': 0.0, 'invalid_state_rejections': 2.0, 'tau_upper_rejected': 1.0, 'primitive_control_failures_checked': 4.0, 'near_diagonal_W_K': 1.000000082740371e-11, 'small_positive_k_wage': 4.682054920046203e-05, 'flow_payoff_test': -0.27685644868579024}


from IPython.display import Markdown, display

BLOCK_2_MARKDOWN = r"""
# Block 2 — full current policy set

Block 2 defines the **full admissible current policy set** used by the viability solver and the planner pointwise improvement step.

It sits directly after Block 1.

Block 1 defines the primitive planner state and control objects:

$$
x=(k,L),
\qquad
u=(\tau,T,H),
$$

the primitive state set

$$
S=\{(k,L):k\ge 0,\ k+L\ge 0\},
$$

and the primitive control correspondence

$$
\tau\in[0,\bar\tau),
$$

$$
T\in[\underline T_s(k),\infty),
$$

$$
H\in[\max\{0,-L\},k].
$$

Block 2 takes those primitive bounds and builds the **working current-control set** used by the numerical algorithm.

The key distinction is:

$$
\boxed{
\text{Block 1 defines primitive economic feasibility; Block 2 defines the numerical policy set used for search.}
}
$$

In particular, Block 2 owns the numerical compactification of the semi-infinite transfer control. That compactification is not a primitive economic restriction.

---

## Economic role of Block 2

The planner chooses a current control

$$
u=(\tau,T,H)
$$

at a current state

$$
x=(k,L)
$$

and regime

$$
s\in\{0,1\}.
$$

The full current policy set is

$$
U_s^{full}(k,L)
=
\left\{
(\tau,T,H):
\tau\in[0,\bar\tau),
\quad
T\in[\underline T_s(k),\infty),
\quad
H\in[\max\{0,-L\},k]
\right\}.
$$

This set is used for two conceptually different tasks.

First, viability asks whether there exists some current control that keeps the state feasible:

$$
\exists u\in U_s^{full}(k,L)
\quad
\text{such that the induced drift is inward or tangent.}
$$

Second, planner improvement asks which admissible current control maximises the planner Hamiltonian:

$$
\max_{u\in U_s^{full}(k,L)}
\mathcal H_s^{\hat u}(x,u;p).
$$

Block 2 provides the common current-control domain for both tasks, but it does not solve either problem.

---

## Where Block 2 sits in the architecture

The working map is

$$
\mathcal G
\to
\hat u
\to
\mathcal C[\hat u]
\to
\mathcal O_s(x,u)
\to
(V_1^{\hat u},V_0^{\hat u})
\to
u^\star
\to
\hat u'.
$$

Block 2 is downstream of:

- Block 0, which provides the regime primitives $\mathcal G$;
- Block 1, which defines the state, controls, primitive state geometry, transfer floor, and primitive control bounds.

Block 2 is upstream of:

- the frozen continuation block;
- the live current-control oracle;
- the viability solver;
- the planner pointwise solver;
- the Howard loop;
- the outer fixed-point update.

The role of Block 2 is to expose a clean current-policy-set interface. It should not compute equilibrium prices, drifts, continuation values, viability sets, or planner optima.

---

## Transfer lower bound

Block 1 defines the worker-consumption transfer floor

$$
\underline T_s(k)
=
\max\{\underline T,\,-w_s(k)+\varepsilon_W\}.
$$

This ensures

$$
C_s^W=w_s(k)+T>0.
$$

Block 2 should consume this object from Block 1 rather than reimplementing the wage or transfer-floor formula.

The lower bound is regime-specific because the wage schedule is regime-specific:

$$
w_s(k)
\quad
\text{depends on}
\quad
s.
$$

---

## Full current policy set

The full current policy set is

$$
U_s^{full}(k,L)
=
\left\{
(\tau,T,H):
\tau\in[0,\bar\tau),
\quad
T\in[\underline T_s(k),\infty),
\quad
H\in[H_{\min}(k,L),H_{\max}(k,L)]
\right\},
$$

where

$$
H_{\min}(k,L)=\max\{0,-L\},
$$

and

$$
H_{\max}(k,L)=k.
$$

Thus

$$
H\in[\max\{0,-L\},k].
$$

This restriction implies

$$
B=L+H\ge 0,
$$

and

$$
E^{priv}=k-H\ge 0.
$$

On the exact diagonal wall

$$
k+L=0,
$$

we have

$$
L=-k.
$$

Therefore

$$
H_{\min}(k,L)=H_{\max}(k,L)=k.
$$

So $H$ is pinned on the exact diagonal wall:

$$
H=k.
$$

Block 2 may report this pinning as a diagnostic, but it should not evaluate risky-share ratios or pricing formulas. Those belong to the live oracle and Block 3.

---

## Numerical compactification

The transfer control is semi-infinite:

$$
T\in[\underline T_s(k),\infty).
$$

For numerical witness search and planner pointwise improvement, Block 2 defines a compact working approximation:

$$
U_s^M(k,L)
=
\left\{
(\tau,T,H):
\tau\in[0,\bar\tau_M],
\quad
T\in[\underline T_s(k),\bar T_s^M(k,L)],
\quad
H\in[\max\{0,-L\},k]
\right\}.
$$

The tax cap is

$$
\bar\tau_M=\bar\tau-\varepsilon_\tau,
$$

where

$$
\varepsilon_\tau>0.
$$

This converts the open primitive tax interval

$$
[0,\bar\tau)
$$

into a closed numerical interval

$$
[0,\bar\tau_M].
$$

The transfer cap

$$
\bar T_s^M(k,L)
$$

is a numerical device. It is not an economic primitive.

Therefore every solver using $U_s^M(k,L)$ must report whether the artificial cap binds.

---

## Transfer-cap discipline

The upper transfer cap should be treated as diagnostic.

If a viability witness or planner optimum satisfies

$$
T=\bar T_s^M(k,L),
$$

then the run should record a cap hit.

The key diagnostic is the cap-hit share:

$$
\text{cap-hit share}
=
\frac{
\#\{x:\ T^{chosen}(x)=\bar T_s^M(x)\}
}{
\#\{x:\ x\text{ evaluated}\}
}.
$$

If the cap-hit share is negligible, the compactification is probably harmless.

If the cap-hit share is non-negligible, enlarge

$$
\bar T_s^M(k,L)
$$

and rerun.

If the cap remains binding after enlargement, the model has a genuine semi-infinite-control issue rather than a simple numerical-box issue.

The baseline interpretation is:

$$
\boxed{
T=\bar T_s^M(k,L)
\text{ is a numerical warning, not an economic optimum.}
}
$$

---

## Suggested transfer-cap rule

Block 2 should allow the transfer cap to be supplied as either:

- a constant cap;
- a callable state-dependent cap;
- a calibrated rule with buffers.

A generic form is

$$
\bar T_s^M(k,L)
=
\underline T_s(k)+\Delta T_s^M(k,L),
$$

where

$$
\Delta T_s^M(k,L)>0.
$$

The first implementation can use a simple constant buffer:

$$
\bar T_s^M(k,L)
=
\underline T_s(k)+\Delta T,
$$

with

$$
\Delta T>0.
$$

Later versions can make the cap scale with output, wealth, or fiscal space, for example

$$
\bar T_s^M(k,L)
=
\underline T_s(k)
+
a_T
+
b_TY_s(k)
+
c_T(k+L)_+.
$$

The exact formula is numerical, not theoretical. The important requirement is that it is explicit, reported, and stress-tested.

---

## Binding diagnostics

Block 2 owns diagnostics for the bounds of the current policy set.

For a candidate control

$$
u=(\tau,T,H),
$$

the relevant bound diagnostics are:

$$
\tau=0,
$$

$$
\tau=\bar\tau_M,
$$

$$
T=\underline T_s(k),
$$

$$
T=\bar T_s^M(k,L),
$$

$$
H=H_{\min}(k,L),
$$

$$
H=H_{\max}(k,L).
$$

It should also report whether the government equity interval is pinned:

$$
H_{\min}(k,L)=H_{\max}(k,L).
$$

This occurs on the exact diagonal wall

$$
k+L=0,
$$

and at the capital wall

$$
k=0
$$

when primitive feasibility implies

$$
H=0.
$$

---

## Current-policy-set status labels

Block 2 should return policy-set diagnostics, not oracle statuses.

A useful policy-bound status set is:

```text
PolicyBoundStatus in {
    "interior_policy",
    "tau_lower_bind",
    "tau_upper_bind",
    "T_lower_bind",
    "T_upper_cap_bind",
    "H_lower_bind",
    "H_upper_bind",
    "H_pinned",
    "invalid_policy"
}
```

These are not state statuses.

They are also not asset-market statuses.

In particular, Block 2 should not produce:

```text
"portfolio_bind"
```

Portfolio binding belongs to Block 3 and the live oracle.

---

## Block 2 contract

### Inputs

Block 2 should take:

$$
s,
\qquad
x=(k,L),
$$

the regime-primitives bundle

$$
\mathcal G,
$$

the Block 1 economy object, and numerical policy-set parameters.

At the code level, the main inputs are:

```python
s: int
x: State
primitives: RegimePrimitives
economy_params: PlannerEconomyParams
policy_params: PolicySetParams
```

The object `PolicySetParams` should contain numerical compactification parameters such as:

```python
tau_margin
transfer_cap_buffer
transfer_cap_rule
bound_tol
```

---

### Outputs

Block 2 should output a working control-bound object for the current state and regime.

Conceptually:

$$
U_s^M(k,L)
=
[\tau_{\min},\tau_{\max}]
\times
[T_{\min},T_{\max}]
\times
[H_{\min},H_{\max}].
$$

A useful dataclass is:

```python
@dataclass(frozen=True)
class WorkingPolicyBounds:
    tau_lower: float
    tau_upper: float

    T_lower: float
    T_upper: float

    H_lower: float
    H_upper: float

    tau_upper_is_compactified: bool
    T_upper_is_artificial_cap: bool
    H_is_pinned: bool
```

The object should distinguish primitive lower bounds from artificial numerical caps.

---

## Suggested module

This block should live in:

```text
policy_sets.py
```

It should import from:

```text
model/economy.py
```

because Block 1 already owns:

- `State`;
- `Control`;
- `PlannerEconomyParams`;
- `control_bounds`;
- `transfer_lower_bound`;
- primitive state and control diagnostics.

It may import from:

```text
automation_block.py
```

only through the `RegimePrimitives` object required by Block 1 transfer-floor routines.

It should not import from:

```text
asset_market.py
continuation_block.py
equilibrium_oracle.py
viability_sets.py
planner_pointwise.py
planner_howard.py
outer_fixed_point.py
```

This keeps the dependency direction clean.

---

## Suggested interface

A useful interface is:

```python
@dataclass(frozen=True)
class PolicySetParams:
    tau_margin: float = 1.0e-8
    transfer_cap_buffer: float = 10.0
    bound_tol: float = 1.0e-10
```

with functions:

```python
working_policy_bounds(
    s: int,
    x: State,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_params: PolicySetParams,
) -> WorkingPolicyBounds
```

```python
is_within_working_policy_set(
    u: Control,
    bounds: WorkingPolicyBounds,
    policy_params: PolicySetParams,
) -> bool
```

```python
policy_bound_diagnostics(
    u: Control,
    bounds: WorkingPolicyBounds,
    policy_params: PolicySetParams,
) -> PolicyBoundDiagnostics
```

```python
sample_policy_box(
    bounds: WorkingPolicyBounds,
    *,
    n_tau: int,
    n_T: int,
    n_H: int,
) -> PolicyGrid
```

The sampling helper is allowed, but it should be used only for diagnostics, rescue search, or coarse smoke tests. It should not become the main planner optimiser.

---

## What belongs in Block 2

Block 2 should compute:

1. the compact numerical tax interval

$$
\tau\in[0,\bar\tau_M],
$$

where

$$
\bar\tau_M=\bar\tau-\varepsilon_\tau;
$$

2. the transfer lower bound

$$
T_{\min}=\underline T_s(k);
$$

3. the artificial transfer cap

$$
T_{\max}=\bar T_s^M(k,L);
$$

4. the government equity bounds

$$
H_{\min}=\max\{0,-L\},
$$

$$
H_{\max}=k;
$$

5. policy-bound diagnostics for candidate controls;

6. cap-hit diagnostics for viability witnesses and planner policies;

7. simple validation tests for the compactified control set.

---

## What Block 2 must not compute

Block 2 should **not** compute:

- production schedules;
- wages except through Block 1’s transfer-floor interface;
- primitive state status;
- balance-sheet objects beyond reading control bounds;
- market-clearing risky share;
- safe rate;
- tax bases;
- fiscal revenue;
- $\dot k$;
- $\dot L$;
- owner continuation objects;
- $\Psi_s$;
- $\omega_s$;
- viability sets;
- planner Hamiltonians;
- KKT conditions;
- Howard masks;
- outer fixed-point updates.

In particular, Block 2 should not compute

$$
\pi^{mc}(k,L,H)
=
\frac{k-H}{k+L}.
$$

That ratio belongs to the live oracle, because the oracle owns boundary-safe evaluation and must avoid divided formulas on

$$
k+L=0.
$$

Block 2 should also not compute

$$
r_{f,s}(k,L;H,\tau).
$$

The safe-rate formula belongs to the live oracle and uses Block 3 asset-market parameters.

---

## Relationship to Block 3

Block 3 owns asset-market parameters and regularity:

$$
\gamma,
\qquad
\underline\pi,
\qquad
\bar\pi,
\qquad
\varepsilon_\pi.
$$

Block 2 owns policy-set bounds:

$$
\bar\tau_M,
\qquad
\underline T_s(k),
\qquad
\bar T_s^M(k,L),
\qquad
H_{\min}(k,L),
\qquad
H_{\max}(k,L).
$$

The distinction is:

$$
\boxed{
\text{Block 2 says which fiscal controls may be tried.}
}
$$

$$
\boxed{
\text{Block 3 says whether the resulting asset-market branch is regular.}
}
$$

So Block 2 should not contain `AssetMarketParams`, and Block 3 should not contain transfer compactification logic.

---

## Relationship to viability

For a frozen anticipated rule

$$
\hat u,
$$

the continuation block later returns

$$
\mathcal C[\hat u].
$$

The live oracle then defines, for each current control,

$$
f_s^{\hat u}(x;u)
=
\left(
\dot k_s^{\hat u}(x;u),
\dot L_s^{\hat u}(x;u)
\right).
$$

Block 2 provides the current-control set over which viability searches:

$$
u\in U_s^M(x).
$$

The discrete viability problem is:

$$
x\in V_s^{\hat u}
\quad
\Longleftrightarrow
\quad
\exists u\in U_s^M(x)
\text{ such that }
f_s^{\hat u}(x;u)
\text{ is inward or tangent.}
$$

Block 2 does not decide whether the drift is inward. It only provides the admissible controls to test.

---

## Relationship to planner pointwise improvement

Given a costate

$$
p=(J_{s,k},J_{s,L}),
$$

the planner pointwise Hamiltonian is

$$
\mathcal H_s^{\hat u}(x,u;p)
=
U_s^{\hat u}(x;u)
+
p\cdot f_s^{\hat u}(x;u).
$$

The planner pointwise solver maximises this over the working set:

$$
u^\star(x)
\in
\arg\max_{u\in U_s^M(x)}
\mathcal H_s^{\hat u}(x,u;p).
$$

Block 2 does not evaluate the Hamiltonian. It only provides the bounds, compactification, and diagnostics needed by the pointwise solver.

---

## Validation tests for Block 2

Before moving on, Block 2 should pass the following tests.

### 1. Primitive-bound consistency

For representative states, check that the Block 2 lower bounds match Block 1:

$$
T_{\min}=\underline T_s(k),
$$

$$
H_{\min}=\max\{0,-L\},
$$

$$
H_{\max}=k.
$$

### 2. Tax compactification

Check that

$$
0\le \bar\tau_M<\bar\tau.
$$

Check that a control with

$$
\tau=\bar\tau
$$

is rejected, while a control with

$$
\tau=\bar\tau_M
$$

is accepted by the working numerical set and marked as upper-bound binding.

### 3. Transfer cap ordering

Check that

$$
\bar T_s^M(k,L)>\underline T_s(k)
$$

for every valid test state.

If this fails, the policy-set parameters are invalid.

### 4. Exact diagonal pinning

At

$$
k+L=0,
$$

check that

$$
H_{\min}=H_{\max}=k.
$$

The diagnostics should report:

```text
H_pinned = True
```

### 5. Capital wall pinning

At

$$
k=0,
$$

with primitive feasibility

$$
L\ge 0,
$$

check that

$$
H_{\min}=H_{\max}=0.
$$

### 6. Cap-hit diagnostics

Construct candidate controls with

$$
T=\bar T_s^M(k,L).
$$

Verify that they are accepted by the working set but flagged as artificial-cap binding.

### 7. No asset-market leakage

Verify that `policy_sets.py` does not compute:

$$
\pi^{mc},
$$

$$
r_f,
$$

$$
\dot k,
$$

or

$$
\dot L.
$$

Those objects belong later.

---

## Diagnostics to report

Block 2 should expose diagnostics that later solvers can aggregate.

At minimum, report:

- number of controls checked;
- number of invalid controls;
- number of lower-transfer-bound hits;
- number of artificial-transfer-cap hits;
- number of tax-upper-bound hits;
- number of $H$ lower-bound hits;
- number of $H$ upper-bound hits;
- number of pinned-$H$ states;
- minimum and maximum selected $T$;
- minimum and maximum selected $H$;
- cap-hit share for viability witnesses;
- cap-hit share for planner optima.

The most important diagnostic is:

$$
\#\{T=\bar T_s^M(k,L)\}.
$$

A nonzero value is not automatically a failure, but it must be visible.

---

## Summary

Block 2 is the policy-set layer.

It defines

$$
U_s^{full}(k,L)
$$

and the numerical compactification

$$
U_s^M(k,L).
$$

It owns:

$$
\bar\tau_M,
\qquad
\underline T_s(k),
\qquad
\bar T_s^M(k,L),
\qquad
H_{\min}(k,L),
\qquad
H_{\max}(k,L),
$$

plus policy-bound diagnostics.

It does not compute prices, drifts, continuation values, viability sets, or planner optima.

The one-line contract is:

$$
\boxed{
\text{Block 2 provides the admissible current-control box used by later existence and optimisation routines.}
}
$$
"""

display(Markdown(BLOCK_2_MARKDOWN))

In [7]:
%%writefile policy_sets.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Literal, Optional, Tuple, Union
import math
import numpy as np

from automation_block import RegimePrimitives
from model.economy import (
    State,
    Control,
    PlannerEconomyParams,
    ControlBounds,
    control_bounds,
)


# ============================================================
# Block 2 contract
# ============================================================
#
# Inputs:
#   - regime s in {0, 1};
#   - scalar planner state x = (k, L);
#   - scalar planner control u = (tau, T, H);
#   - regime-primitives bundle G from Block 0;
#   - primitive planner-economy parameters from Block 1;
#   - numerical compactification options.
#
# Outputs:
#   - full current policy-set bounds U_s^{full}(k,L);
#   - compact numerical policy-set bounds U_s^M(k,L);
#   - admissibility diagnostics for full and compact sets;
#   - binding diagnostics for tau, T, and H bounds;
#   - small helpers for safe initial controls / witnesses.
#
# Forbidden responsibilities:
#   - no continuation solve;
#   - no live oracle;
#   - no pi^{mc};
#   - no r_f;
#   - no kdot or Ldot;
#   - no tax-base / revenue evaluation;
#   - no viability set construction;
#   - no Howard iteration.
#
# Important convention:
#   The transfer upper cap in U_s^M is numerical only.
#   A binding T upper cap must be reported as a diagnostic, not interpreted
#   as an economic optimum.


PolicySetKind = Literal["full", "compact"]


BindingLabel = Literal[
    "tau_lower",
    "tau_upper_cap",
    "tau_near_strict_upper",
    "T_lower",
    "T_upper_cap",
    "H_lower",
    "H_upper",
]


# ============================================================
# Helpers
# ============================================================

def _as_scalar(x: float, *, name: str) -> float:
    arr = np.asarray(x, dtype=float)
    if arr.shape != ():
        raise TypeError(
            f"{name} must be scalar in Block 2. "
            "Use an explicit grid/vector wrapper later."
        )

    val = float(arr)
    if not math.isfinite(val):
        raise ValueError(f"{name} must be finite.")

    return val


def _require_regime(s: int) -> int:
    if s not in (0, 1):
        raise ValueError("Regime s must be 0 or 1.")
    return int(s)


def _strictly_positive(x: float, *, name: str) -> None:
    x = _as_scalar(x, name=name)
    if not (x > 0.0):
        raise ValueError(f"{name} must be strictly positive. Got {name}={x}.")


def _nonnegative(x: float, *, name: str) -> None:
    x = _as_scalar(x, name=name)
    if not (x >= 0.0):
        raise ValueError(f"{name} must be nonnegative. Got {name}={x}.")


def _finite_or_posinf(x: float, *, name: str) -> float:
    x = float(x)
    if math.isnan(x) or x == -math.inf:
        raise ValueError(f"{name} must be finite or +inf.")
    return x


# ============================================================
# Numerical compactification options
# ============================================================

TransferCapFn = Callable[
    [int, State, ControlBounds, PlannerEconomyParams],
    float,
]


@dataclass(frozen=True)
class PolicySetOptions:
    """
    Numerical options for the compact current policy set U_s^M(k,L).

    tau_upper_margin:
        The buffer used to convert the primitive open upper bound
            tau < tau_upper
        into a closed numerical cap
            tau <= tau_upper - tau_upper_margin.

    transfer_cap_base:
        Baseline width added above the transfer lower bound.

    transfer_cap_wealth_mult:
        Multiplier on max{k + L, 0} in the transfer-cap width.

    transfer_cap_capital_mult:
        Multiplier on max{k, 0} in the transfer-cap width.

    transfer_cap_floor_mult:
        Multiplier on abs(T_lower) in the transfer-cap width.

    transfer_cap_min_width:
        Strict minimum width of the compact transfer interval.

    transfer_cap_fn:
        Optional custom cap function. If supplied, it must return the absolute
        upper bound T_upper, not the width. This is useful for calibration-
        specific caps. The returned cap is still interpreted as numerical.

    bound_tol:
        Tolerance used for diagnostics and closed bound checks in Block 2.
    """
    tau_upper_margin: float = 1.0e-8

    transfer_cap_base: float = 10.0
    transfer_cap_wealth_mult: float = 10.0
    transfer_cap_capital_mult: float = 5.0
    transfer_cap_floor_mult: float = 1.0
    transfer_cap_min_width: float = 1.0e-8

    transfer_cap_fn: Optional[TransferCapFn] = None

    bound_tol: float = 1.0e-10

    def __post_init__(self) -> None:
        _strictly_positive(self.tau_upper_margin, name="tau_upper_margin")

        _nonnegative(self.transfer_cap_base, name="transfer_cap_base")
        _nonnegative(self.transfer_cap_wealth_mult, name="transfer_cap_wealth_mult")
        _nonnegative(self.transfer_cap_capital_mult, name="transfer_cap_capital_mult")
        _nonnegative(self.transfer_cap_floor_mult, name="transfer_cap_floor_mult")
        _strictly_positive(self.transfer_cap_min_width, name="transfer_cap_min_width")

        _nonnegative(self.bound_tol, name="bound_tol")

        if self.transfer_cap_fn is not None and not callable(self.transfer_cap_fn):
            raise TypeError("transfer_cap_fn must be callable or None.")


# ============================================================
# Compact control bounds
# ============================================================

@dataclass(frozen=True)
class CompactControlBounds:
    """
    Numerical compactification of the primitive current policy set.

    This object represents U_s^M(k,L), not the primitive full set.

    tau_upper is closed here:
        tau in [tau_lower, tau_upper].

    T_upper is finite here:
        T in [T_lower, T_upper].

    H bounds are inherited from the primitive correspondence:
        H in [max{0,-L}, k].
    """
    tau_lower: float
    tau_upper: float
    tau_upper_is_open: bool

    T_lower: float
    T_upper: float

    H_lower: float
    H_upper: float

    transfer_cap_width: float
    full_bounds: ControlBounds

    def __post_init__(self) -> None:
        for name in (
            "tau_lower",
            "tau_upper",
            "T_lower",
            "T_upper",
            "H_lower",
            "H_upper",
            "transfer_cap_width",
        ):
            val = _as_scalar(getattr(self, name), name=name)
            object.__setattr__(self, name, val)

        if self.tau_upper_is_open:
            raise ValueError("CompactControlBounds should have a closed tau upper cap.")

        if self.tau_lower > self.tau_upper:
            raise ValueError("Compact tau bounds are inconsistent.")

        if self.T_lower > self.T_upper:
            raise ValueError("Compact transfer bounds are inconsistent.")

        if self.H_lower > self.H_upper:
            raise ValueError("Compact H bounds are inconsistent.")

        if self.transfer_cap_width <= 0.0:
            raise ValueError("transfer_cap_width must be strictly positive.")

    @property
    def T_is_semi_infinite(self) -> bool:
        return False

    def H_pinned(self, tol: float = 0.0) -> bool:
        return abs(self.H_upper - self.H_lower) <= tol


BoundsLike = Union[ControlBounds, CompactControlBounds]


# ============================================================
# Full and compact policy-set bounds
# ============================================================

def full_policy_bounds(
    s: int,
    x: State,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
) -> ControlBounds:
    """
    Full admissible current policy set:

        tau in [0, tau_upper),
        T in [underline T_s(k), infinity),
        H in [max{0,-L}, k].

    This is the economic baseline set for viability and planner improvement.
    """
    s = _require_regime(s)
    return control_bounds(s, x, primitives, economy_params)


def compact_transfer_upper_bound(
    s: int,
    x: State,
    full_bounds: ControlBounds,
    economy_params: PlannerEconomyParams,
    options: PolicySetOptions,
) -> float:
    """
    Numerical transfer cap T_bar_s^M(k,L).

    By default, the cap is constructed as

        T_upper = T_lower + width,

    where width scales with owner wealth, capital, and the transfer floor.

    This cap is not a primitive economic restriction.
    """
    s = _require_regime(s)

    if options.transfer_cap_fn is not None:
        T_upper = _as_scalar(
            options.transfer_cap_fn(s, x, full_bounds, economy_params),
            name="custom T_upper",
        )

        if T_upper < full_bounds.T_lower:
            raise ValueError(
                "Custom transfer cap lies below the transfer lower bound: "
                f"T_upper={T_upper}, T_lower={full_bounds.T_lower}."
            )

        return T_upper

    W_K = max(x.W_K, 0.0)
    k_pos = max(x.k, 0.0)
    T_floor_scale = abs(full_bounds.T_lower)

    width = (
        options.transfer_cap_base
        + options.transfer_cap_wealth_mult * W_K
        + options.transfer_cap_capital_mult * k_pos
        + options.transfer_cap_floor_mult * T_floor_scale
    )

    width = max(width, options.transfer_cap_min_width)

    if not math.isfinite(width):
        raise ValueError("Computed transfer-cap width is not finite.")

    return full_bounds.T_lower + width


def compact_policy_bounds(
    s: int,
    x: State,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    options: Optional[PolicySetOptions] = None,
) -> CompactControlBounds:
    """
    Numerical compactification U_s^M(k,L):

        tau in [0, tau_upper - epsilon_tau],
        T in [underline T_s(k), T_bar_s^M(k,L)],
        H in [max{0,-L}, k].

    The transfer cap is numerical only and must be monitored through binding
    diagnostics.
    """
    s = _require_regime(s)

    if options is None:
        options = PolicySetOptions()

    full = full_policy_bounds(s, x, primitives, economy_params)

    tau_upper_M = full.tau_upper - options.tau_upper_margin

    if tau_upper_M < full.tau_lower:
        raise ValueError(
            "tau_upper_margin is too large for the primitive tau interval: "
            f"tau_lower={full.tau_lower}, tau_upper={full.tau_upper}, "
            f"tau_upper_margin={options.tau_upper_margin}."
        )

    T_upper = compact_transfer_upper_bound(
        s=s,
        x=x,
        full_bounds=full,
        economy_params=economy_params,
        options=options,
    )

    transfer_cap_width = T_upper - full.T_lower

    return CompactControlBounds(
        tau_lower=full.tau_lower,
        tau_upper=tau_upper_M,
        tau_upper_is_open=False,
        T_lower=full.T_lower,
        T_upper=T_upper,
        H_lower=full.H_lower,
        H_upper=full.H_upper,
        transfer_cap_width=transfer_cap_width,
        full_bounds=full,
    )


# ============================================================
# Control-set diagnostics
# ============================================================

@dataclass(frozen=True)
class PolicyControlDiagnostics:
    """
    Pointwise diagnostic for whether a current control lies in a policy set.

    For set_kind="full":
        T_upper is +inf and tau_upper is open.

    For set_kind="compact":
        T_upper is finite and tau_upper is closed.
    """
    set_kind: PolicySetKind
    bounds: BoundsLike

    tau_ok: bool
    T_ok: bool
    H_ok: bool
    is_admissible: bool

    violations: Tuple[str, ...]
    bindings: Tuple[BindingLabel, ...]

    tau_lower_margin: float
    tau_upper_margin: float

    T_lower_margin: float
    T_upper_margin: float

    H_lower_margin: float
    H_upper_margin: float

    @property
    def binds_transfer_cap(self) -> bool:
        return "T_upper_cap" in self.bindings

    @property
    def binds_tau_upper_cap(self) -> bool:
        return "tau_upper_cap" in self.bindings

    @property
    def binds_any_H_bound(self) -> bool:
        return ("H_lower" in self.bindings) or ("H_upper" in self.bindings)


def diagnose_control_against_bounds(
    u: Control,
    bounds: BoundsLike,
    *,
    set_kind: PolicySetKind,
    tol: float,
) -> PolicyControlDiagnostics:
    """
    Diagnose control admissibility and bound binding against supplied bounds.

    This function does not compute economic prices or drifts.
    """
    tol = _as_scalar(tol, name="tol")
    if tol < 0.0:
        raise ValueError("tol must be nonnegative.")

    violations: list[str] = []
    bindings: list[BindingLabel] = []

    tau_lower_margin = u.tau - bounds.tau_lower
    tau_upper_margin = bounds.tau_upper - u.tau

    T_lower_margin = u.T - bounds.T_lower
    if math.isinf(bounds.T_upper):
        T_upper_margin = math.inf
    else:
        T_upper_margin = bounds.T_upper - u.T

    H_lower_margin = u.H - bounds.H_lower
    H_upper_margin = bounds.H_upper - u.H

    # Tau check.
    if u.tau < bounds.tau_lower - tol:
        tau_ok = False
        violations.append("tau below lower bound")
    else:
        if bounds.tau_upper_is_open:
            tau_ok = u.tau < bounds.tau_upper
            if not tau_ok:
                violations.append("tau at/above strict upper bound")
        else:
            tau_ok = u.tau <= bounds.tau_upper + tol
            if not tau_ok:
                violations.append("tau above compact upper cap")

    # Transfer check.
    T_ok = u.T >= bounds.T_lower - tol
    if not T_ok:
        violations.append("T below transfer lower bound")

    if not math.isinf(bounds.T_upper):
        if u.T > bounds.T_upper + tol:
            T_ok = False
            violations.append("T above compact transfer cap")

    # H check.
    H_ok = True

    if u.H < bounds.H_lower - tol:
        H_ok = False
        violations.append("H below lower bound")

    if u.H > bounds.H_upper + tol:
        H_ok = False
        violations.append("H above upper bound")

    # Binding diagnostics.
    if abs(tau_lower_margin) <= tol:
        bindings.append("tau_lower")

    if bounds.tau_upper_is_open:
        if (u.tau < bounds.tau_upper) and (tau_upper_margin <= tol):
            bindings.append("tau_near_strict_upper")
    else:
        if abs(tau_upper_margin) <= tol:
            bindings.append("tau_upper_cap")

    if abs(T_lower_margin) <= tol:
        bindings.append("T_lower")

    if not math.isinf(bounds.T_upper):
        if abs(T_upper_margin) <= tol:
            bindings.append("T_upper_cap")

    if abs(H_lower_margin) <= tol:
        bindings.append("H_lower")

    if abs(H_upper_margin) <= tol:
        bindings.append("H_upper")

    is_admissible = tau_ok and T_ok and H_ok

    return PolicyControlDiagnostics(
        set_kind=set_kind,
        bounds=bounds,
        tau_ok=bool(tau_ok),
        T_ok=bool(T_ok),
        H_ok=bool(H_ok),
        is_admissible=bool(is_admissible),
        violations=tuple(violations),
        bindings=tuple(bindings),
        tau_lower_margin=float(tau_lower_margin),
        tau_upper_margin=float(tau_upper_margin),
        T_lower_margin=float(T_lower_margin),
        T_upper_margin=float(T_upper_margin),
        H_lower_margin=float(H_lower_margin),
        H_upper_margin=float(H_upper_margin),
    )


def full_policy_diagnostics(
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    options: Optional[PolicySetOptions] = None,
) -> PolicyControlDiagnostics:
    """
    Diagnose whether u lies in U_s^{full}(k,L).
    """
    s = _require_regime(s)

    if options is None:
        options = PolicySetOptions()

    bounds = full_policy_bounds(s, x, primitives, economy_params)

    return diagnose_control_against_bounds(
        u,
        bounds,
        set_kind="full",
        tol=options.bound_tol,
    )


def compact_policy_diagnostics(
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    options: Optional[PolicySetOptions] = None,
) -> PolicyControlDiagnostics:
    """
    Diagnose whether u lies in U_s^M(k,L).
    """
    s = _require_regime(s)

    if options is None:
        options = PolicySetOptions()

    bounds = compact_policy_bounds(s, x, primitives, economy_params, options)

    return diagnose_control_against_bounds(
        u,
        bounds,
        set_kind="compact",
        tol=options.bound_tol,
    )


def require_full_policy_control(
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    options: Optional[PolicySetOptions] = None,
) -> PolicyControlDiagnostics:
    """
    Require u in U_s^{full}(k,L).
    """
    diag = full_policy_diagnostics(
        s=s,
        x=x,
        u=u,
        primitives=primitives,
        economy_params=economy_params,
        options=options,
    )

    if not diag.is_admissible:
        raise ValueError(
            f"Control is outside U_s^full(k,L): {diag.violations}. "
            f"Control={u}, state={x}."
        )

    return diag


def require_compact_policy_control(
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    options: Optional[PolicySetOptions] = None,
) -> PolicyControlDiagnostics:
    """
    Require u in U_s^M(k,L).
    """
    diag = compact_policy_diagnostics(
        s=s,
        x=x,
        u=u,
        primitives=primitives,
        economy_params=economy_params,
        options=options,
    )

    if not diag.is_admissible:
        raise ValueError(
            f"Control is outside U_s^M(k,L): {diag.violations}. "
            f"Control={u}, state={x}."
        )

    return diag


# ============================================================
# Useful control constructors
# ============================================================

def lower_bound_control(bounds: BoundsLike) -> Control:
    """
    Deterministic lower-bound corner control.

    Useful as a smoke-test control or as one candidate in a witness search.
    This is not a planner optimum.
    """
    return Control(
        tau=bounds.tau_lower,
        T=bounds.T_lower,
        H=bounds.H_lower,
    )


def upper_bound_control(bounds: CompactControlBounds) -> Control:
    """
    Deterministic compact upper-bound corner control.

    This is only defined for compact bounds because full bounds have
    T_upper = +inf and open tau_upper.
    """
    return Control(
        tau=bounds.tau_upper,
        T=bounds.T_upper,
        H=bounds.H_upper,
    )


def midpoint_control(bounds: CompactControlBounds) -> Control:
    """
    Interior-ish midpoint control for the compact box.

    If H is pinned, the midpoint equals the pinned value.
    """
    return Control(
        tau=0.5 * (bounds.tau_lower + bounds.tau_upper),
        T=0.5 * (bounds.T_lower + bounds.T_upper),
        H=0.5 * (bounds.H_lower + bounds.H_upper),
    )


def clamp_control_to_compact_bounds(
    u: Control,
    bounds: CompactControlBounds,
) -> Control:
    """
    Explicit clipping helper for initialization / diagnostics only.

    Do not use this to hide infeasible controls inside core solver logic.
    """
    tau = min(max(u.tau, bounds.tau_lower), bounds.tau_upper)
    T = min(max(u.T, bounds.T_lower), bounds.T_upper)
    H = min(max(u.H, bounds.H_lower), bounds.H_upper)

    return Control(tau=tau, T=T, H=H)


# ============================================================
# Binding summaries
# ============================================================

@dataclass(frozen=True)
class BindingSummary:
    n_total: int
    n_admissible: int
    n_inadmissible: int

    n_tau_lower: int
    n_tau_upper_cap: int
    n_tau_near_strict_upper: int

    n_T_lower: int
    n_T_upper_cap: int

    n_H_lower: int
    n_H_upper: int

    share_admissible: float
    share_T_upper_cap: float
    share_tau_upper_cap: float


def summarize_policy_diagnostics(
    diagnostics: Tuple[PolicyControlDiagnostics, ...],
) -> BindingSummary:
    """
    Summarize pointwise control-set diagnostics.

    This is useful for reporting how often artificial compactification caps bind.
    """
    n_total = len(diagnostics)

    if n_total == 0:
        raise ValueError("diagnostics must be non-empty.")

    def count_binding(label: BindingLabel) -> int:
        return int(sum(label in d.bindings for d in diagnostics))

    n_admissible = int(sum(d.is_admissible for d in diagnostics))
    n_inadmissible = n_total - n_admissible

    n_tau_lower = count_binding("tau_lower")
    n_tau_upper_cap = count_binding("tau_upper_cap")
    n_tau_near_strict_upper = count_binding("tau_near_strict_upper")

    n_T_lower = count_binding("T_lower")
    n_T_upper_cap = count_binding("T_upper_cap")

    n_H_lower = count_binding("H_lower")
    n_H_upper = count_binding("H_upper")

    return BindingSummary(
        n_total=n_total,
        n_admissible=n_admissible,
        n_inadmissible=n_inadmissible,
        n_tau_lower=n_tau_lower,
        n_tau_upper_cap=n_tau_upper_cap,
        n_tau_near_strict_upper=n_tau_near_strict_upper,
        n_T_lower=n_T_lower,
        n_T_upper_cap=n_T_upper_cap,
        n_H_lower=n_H_lower,
        n_H_upper=n_H_upper,
        share_admissible=float(n_admissible / n_total),
        share_T_upper_cap=float(n_T_upper_cap / n_total),
        share_tau_upper_cap=float(n_tau_upper_cap / n_total),
    )


# ============================================================
# Validation / smoke test
# ============================================================

def validate_policy_set_layer(
    primitives: RegimePrimitives,
    economy_params: Optional[PlannerEconomyParams] = None,
    options: Optional[PolicySetOptions] = None,
) -> dict[str, float]:
    """
    Validate Block 2.

    Tests:
      - full set has semi-infinite T;
      - compact set has finite T upper cap;
      - compact tau cap lies below primitive open tau upper;
      - lower-bound and upper-bound compact controls are diagnosed correctly;
      - controls above compact T cap are rejected by U_s^M;
      - controls above compact T cap are still allowed by U_s^{full};
      - diagonal-wall H interval remains pinned;
      - corner state works without calling k>0 production formulas;
      - invalid regime is rejected even at k=0;
      - summary diagnostics count cap bindings.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if options is None:
        options = PolicySetOptions()

    report: dict[str, float] = {}

    s = 0
    x = State(k=1.0, L=0.5)

    full = full_policy_bounds(s, x, primitives, economy_params)
    compact = compact_policy_bounds(s, x, primitives, economy_params, options)

    if not full.T_is_semi_infinite:
        raise RuntimeError("Full policy set should have semi-infinite transfer control.")

    if compact.T_is_semi_infinite:
        raise RuntimeError("Compact policy set should have finite transfer cap.")

    if not math.isfinite(compact.T_upper):
        raise RuntimeError("Compact T upper bound should be finite.")

    if compact.T_upper <= compact.T_lower:
        raise RuntimeError("Compact transfer interval should have positive width.")

    if not (compact.tau_upper < full.tau_upper):
        raise RuntimeError("Compact tau upper cap should lie below primitive open upper bound.")

    report["compact_tau_upper"] = float(compact.tau_upper)
    report["compact_T_lower"] = float(compact.T_lower)
    report["compact_T_upper"] = float(compact.T_upper)
    report["compact_transfer_cap_width"] = float(compact.transfer_cap_width)

    # Lower-bound corner control.
    u_lower = lower_bound_control(compact)
    d_lower = compact_policy_diagnostics(
        s, x, u_lower, primitives, economy_params, options
    )

    if not d_lower.is_admissible:
        raise RuntimeError(f"Lower-bound compact control should be admissible: {d_lower}")

    for label in ("tau_lower", "T_lower", "H_lower"):
        if label not in d_lower.bindings:
            raise RuntimeError(f"Expected lower-bound binding label {label}.")

    # Upper-bound compact control.
    u_upper = upper_bound_control(compact)
    d_upper = compact_policy_diagnostics(
        s, x, u_upper, primitives, economy_params, options
    )

    if not d_upper.is_admissible:
        raise RuntimeError(f"Upper-bound compact control should be admissible: {d_upper}")

    for label in ("tau_upper_cap", "T_upper_cap", "H_upper"):
        if label not in d_upper.bindings:
            raise RuntimeError(f"Expected upper-bound binding label {label}.")

    # Compact rejects T above cap.
    u_T_too_high = Control(
        tau=0.5 * (compact.tau_lower + compact.tau_upper),
        T=compact.T_upper + 10.0 * options.bound_tol + 1.0e-6,
        H=0.5 * (compact.H_lower + compact.H_upper),
    )

    d_T_high_compact = compact_policy_diagnostics(
        s, x, u_T_too_high, primitives, economy_params, options
    )

    if d_T_high_compact.is_admissible:
        raise RuntimeError("Compact policy set should reject T above compact cap.")

    # Full set allows arbitrarily high T, subject to primitive lower bound.
    d_T_high_full = full_policy_diagnostics(
        s, x, u_T_too_high, primitives, economy_params, options
    )

    if not d_T_high_full.is_admissible:
        raise RuntimeError("Full policy set should not reject high T.")

    # Full set rejects tau at the primitive strict upper bound.
    u_tau_strict_bad = Control(
        tau=full.tau_upper,
        T=full.T_lower,
        H=full.H_lower,
    )

    d_tau_bad = full_policy_diagnostics(
        s, x, u_tau_strict_bad, primitives, economy_params, options
    )

    if d_tau_bad.is_admissible:
        raise RuntimeError("Full policy set should reject tau at strict primitive upper bound.")

    # Exact diagonal wall: k=1, L=-1 pins H=1.
    x_diag = State(k=1.0, L=-1.0)
    b_diag = compact_policy_bounds(s, x_diag, primitives, economy_params, options)

    if not b_diag.H_pinned(economy_params.control_tol):
        raise RuntimeError("Compact H bounds should remain pinned on exact diagonal wall.")

    report["diagonal_H_lower"] = float(b_diag.H_lower)
    report["diagonal_H_upper"] = float(b_diag.H_upper)

    # Exact corner: k=0, L=0 should be valid for Block 2 bounds.
    x_corner = State(k=0.0, L=0.0)
    b_corner = compact_policy_bounds(s, x_corner, primitives, economy_params, options)

    if not b_corner.H_pinned(economy_params.control_tol):
        raise RuntimeError("H should be pinned at the exact corner.")

    if b_corner.T_lower < economy_params.transfer_min:
        raise RuntimeError("Corner transfer lower bound should respect transfer_min.")

    report["corner_T_lower"] = float(b_corner.T_lower)
    report["corner_T_upper"] = float(b_corner.T_upper)

    # Invalid regime should be rejected even at k=0.
    try:
        full_policy_bounds(2, x_corner, primitives, economy_params)
    except ValueError:
        pass
    else:
        raise RuntimeError("Invalid regime was not rejected at k=0.")

    # Summary diagnostics.
    summary = summarize_policy_diagnostics((d_lower, d_upper, d_T_high_compact))

    if summary.n_total != 3:
        raise RuntimeError("Binding summary has wrong total count.")

    if summary.n_T_upper_cap != 1:
        raise RuntimeError("Binding summary should count one T upper-cap binding.")

    report["summary_share_T_upper_cap"] = float(summary.share_T_upper_cap)
    report["summary_share_tau_upper_cap"] = float(summary.share_tau_upper_cap)

    return report


def module_smoke_test() -> dict[str, float]:
    """
    Minimal self-test for development.
    """
    from automation_block import AutomationParams, build_regime_primitives

    automation_params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    primitives = build_regime_primitives(automation_params)

    economy_params = PlannerEconomyParams(
        tau_upper=1.0,
        transfer_min=0.0,
        worker_consumption_eps=1.0e-8,
        state_tol=1.0e-10,
        control_tol=1.0e-12,
    )

    options = PolicySetOptions()

    return validate_policy_set_layer(
        primitives=primitives,
        economy_params=economy_params,
        options=options,
    )


__all__ = [
    "PolicySetKind",
    "BindingLabel",
    "PolicySetOptions",
    "CompactControlBounds",
    "PolicyControlDiagnostics",
    "BindingSummary",
    "full_policy_bounds",
    "compact_transfer_upper_bound",
    "compact_policy_bounds",
    "diagnose_control_against_bounds",
    "full_policy_diagnostics",
    "compact_policy_diagnostics",
    "require_full_policy_control",
    "require_compact_policy_control",
    "lower_bound_control",
    "upper_bound_control",
    "midpoint_control",
    "clamp_control_to_compact_bounds",
    "summarize_policy_diagnostics",
    "validate_policy_set_layer",
    "module_smoke_test",
]

Overwriting policy_sets.py


In [8]:
import importlib

import automation_block
import model.economy as economy
import policy_sets

importlib.reload(automation_block)
importlib.reload(economy)
importlib.reload(policy_sets)


automation_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(automation_params)

economy_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

policy_options = policy_sets.PolicySetOptions(
    tau_upper_margin=1.0e-8,
    transfer_cap_base=10.0,
    transfer_cap_wealth_mult=10.0,
    transfer_cap_capital_mult=5.0,
    transfer_cap_floor_mult=1.0,
    transfer_cap_min_width=1.0e-8,
    bound_tol=1.0e-10,
)

report = policy_sets.validate_policy_set_layer(
    primitives=G,
    economy_params=economy_params,
    options=policy_options,
)

print("Block 2 validation passed.")
print(report)

Block 2 validation passed.
{'compact_tau_upper': 0.99999999, 'compact_T_lower': 0.0, 'compact_T_upper': 30.0, 'compact_transfer_cap_width': 30.0, 'diagonal_H_lower': 1.0, 'diagonal_H_upper': 1.0, 'corner_T_lower': 1e-08, 'corner_T_upper': 10.000000020000002, 'summary_share_T_upper_cap': 0.3333333333333333, 'summary_share_tau_upper_cap': 0.3333333333333333}


In [9]:
s = 0
x = economy.State(k=1.0, L=0.5)

full_bounds = policy_sets.full_policy_bounds(
    s=s,
    x=x,
    primitives=G,
    economy_params=economy_params,
)

compact_bounds = policy_sets.compact_policy_bounds(
    s=s,
    x=x,
    primitives=G,
    economy_params=economy_params,
    options=policy_options,
)

u_mid = policy_sets.midpoint_control(compact_bounds)

diag = policy_sets.compact_policy_diagnostics(
    s=s,
    x=x,
    u=u_mid,
    primitives=G,
    economy_params=economy_params,
    options=policy_options,
)

print("Full bounds:")
print(full_bounds)

print("\nCompact bounds:")
print(compact_bounds)

print("\nMidpoint control:")
print(u_mid)

print("\nCompact diagnostic:")
print(diag)

Full bounds:
ControlBounds(tau_lower=0.0, tau_upper=1.0, tau_upper_is_open=True, T_lower=0.0, T_upper=inf, H_lower=0.0, H_upper=1.0)

Compact bounds:
CompactControlBounds(tau_lower=0.0, tau_upper=0.99999999, tau_upper_is_open=False, T_lower=0.0, T_upper=30.0, H_lower=0.0, H_upper=1.0, transfer_cap_width=30.0, full_bounds=ControlBounds(tau_lower=0.0, tau_upper=1.0, tau_upper_is_open=True, T_lower=0.0, T_upper=inf, H_lower=0.0, H_upper=1.0))

Midpoint control:
Control(tau=0.499999995, T=15.0, H=0.5)

Compact diagnostic:
PolicyControlDiagnostics(set_kind='compact', bounds=CompactControlBounds(tau_lower=0.0, tau_upper=0.99999999, tau_upper_is_open=False, T_lower=0.0, T_upper=30.0, H_lower=0.0, H_upper=1.0, transfer_cap_width=30.0, full_bounds=ControlBounds(tau_lower=0.0, tau_upper=1.0, tau_upper_is_open=True, T_lower=0.0, T_upper=inf, H_lower=0.0, H_upper=1.0)), tau_ok=True, T_ok=True, H_ok=True, is_admissible=True, violations=(), bindings=(), tau_lower_margin=0.499999995, tau_upper_margin

# Block 3 — asset-market parameters and regularity closure

This block defines the asset-market parameter bundle and the baseline portfolio-interiority regularity closure.

It provides one source of truth for:

$$
\gamma,
\qquad
\underline\pi,
\qquad
\bar\pi,
\qquad
\varepsilon_\pi.
$$

The key current-control object that will later be computed by the live oracle is the market-clearing risky share

$$
\pi^{mc}(k,L,H)
=
\frac{k-H}{k+L}
=
\frac{E^{priv}}{W^K}.
$$

Block 1 already defines

$$
W^K=k+L,
\qquad
E^{priv}=k-H,
\qquad
B=L+H.
$$

Under the primitive government balance-sheet restriction

$$
H\in[\max\{0,-L\},k],
$$

we have

$$
E^{priv}=k-H\ge 0,
\qquad
B=L+H\ge 0.
$$

If

$$
W^K=k+L>0,
$$

then

$$
W^K=E^{priv}+B,
$$

so

$$
\pi^{mc}\in[0,1].
$$

The baseline asset-market closure is therefore:

$$
\underline\pi < 0
\qquad
\text{and}
\qquad
\bar\pi > 1.
$$

Equivalently, version 1 may use

$$
\underline\pi=-\infty,
\qquad
\bar\pi=+\infty.
$$

This ensures that every mechanically feasible market-clearing risky share lies strictly inside the capital owners' portfolio set. The interior Merton equation can then pin down the safe rate in the live oracle.

The strict interior branch is:

$$
\underline\pi+\varepsilon_\pi
<
\pi
<
\bar\pi-\varepsilon_\pi.
$$

This block does not compute the live short rate, current fiscal objects, or drifts. Those belong to the live oracle.

The portfolio-bound branch still exists. In version 1, a lower or upper portfolio bind is a diagnostic / invalid-status branch, not a complementarity solver.

In [10]:
%%writefile asset_market.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Literal, Optional, Tuple, Union
import math
import numpy as np


ArrayLike = Union[float, int, np.ndarray]


# ============================================================
# Block 3 contract
# ============================================================
#
# Inputs:
#   - asset-market calibration:
#       gamma, pi_lower, pi_upper, pi_tol;
#   - already-computed risky-share values pi, usually pi^{mc};
#   - diagnostic grids of candidate risky shares.
#
# Outputs:
#   - canonical immutable AssetMarketParams;
#   - pointwise portfolio-interiority checks;
#   - vectorized portfolio status arrays;
#   - vectorized portfolio diagnostics.
#
# Forbidden responsibilities:
#   - no continuation solve;
#   - no live oracle;
#   - no computation of r_f;
#   - no computation of kdot or Ldot;
#   - no computation of tax bases or revenue;
#   - no viability peeling;
#   - no Howard iteration.
#
# Important convention:
#   Block 3 checks whether a supplied risky share lies on the interior
#   Merton branch. The live oracle later computes pi^{mc}(k,L,H).
#
# Version-1 branch convention:
#   If a finite portfolio bound binds, mark the candidate as invalid for
#   the interior-Merton oracle. Do not solve complementarity conditions here.


PortfolioStatus = Literal[
    "interior",
    "portfolio_lower_bind",
    "portfolio_upper_bind",
    "invalid",
]

PortfolioCoarseStatus = Literal[
    "interior",
    "portfolio_bind",
    "invalid",
]


# ============================================================
# Helpers
# ============================================================

def _is_scalar_like(x: Any) -> bool:
    return np.ndim(x) == 0


def _as_float_array(x: ArrayLike) -> np.ndarray:
    return np.asarray(x, dtype=float)


def _return_like_input_float(out: np.ndarray, like: ArrayLike) -> ArrayLike:
    out = np.asarray(out, dtype=float)

    if _is_scalar_like(like):
        if out.shape == ():
            return float(out)
        if out.size == 1:
            return float(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar input produced non-scalar output with shape {out.shape}."
        )

    return out


def _return_like_input_bool(out: np.ndarray, like: ArrayLike) -> Union[bool, np.ndarray]:
    out = np.asarray(out, dtype=bool)

    if _is_scalar_like(like):
        if out.shape == ():
            return bool(out)
        if out.size == 1:
            return bool(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar input produced non-scalar output with shape {out.shape}."
        )

    return out


def _return_like_input_object(out: np.ndarray, like: ArrayLike) -> Union[str, np.ndarray]:
    out = np.asarray(out, dtype=object)

    if _is_scalar_like(like):
        if out.shape == ():
            return str(out.item())
        if out.size == 1:
            return str(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar input produced non-scalar output with shape {out.shape}."
        )

    return out


def _finite_float(x: float, *, name: str) -> float:
    x = float(x)
    if not math.isfinite(x):
        raise ValueError(f"{name} must be finite.")
    return x


def _allow_infinite_bound(x: float, *, name: str) -> float:
    x = float(x)
    if math.isnan(x):
        raise ValueError(f"{name} must not be NaN.")
    return x


# ============================================================
# Canonical asset-market parameter object
# ============================================================

@dataclass(frozen=True)
class AssetMarketParams:
    """
    Canonical asset-market parameters.

    gamma:
        CRRA coefficient used in the owner portfolio problem.

    pi_lower, pi_upper:
        Exogenous portfolio bounds.

        Baseline version-1 choice:
            pi_lower = -inf,
            pi_upper = +inf.

        Finite wide diagnostic choice:
            pi_lower < 0 and pi_upper > 1.

    pi_tol:
        Strict-interiority tolerance.

        A supplied risky share pi is on the interior Merton branch only if

            pi_lower + pi_tol < pi < pi_upper - pi_tol

        for finite bounds.
    """
    gamma: float
    pi_lower: float = -math.inf
    pi_upper: float = math.inf
    pi_tol: float = 1.0e-10

    def __post_init__(self) -> None:
        gamma = _finite_float(self.gamma, name="gamma")
        pi_lower = _allow_infinite_bound(self.pi_lower, name="pi_lower")
        pi_upper = _allow_infinite_bound(self.pi_upper, name="pi_upper")
        pi_tol = _finite_float(self.pi_tol, name="pi_tol")

        object.__setattr__(self, "gamma", gamma)
        object.__setattr__(self, "pi_lower", pi_lower)
        object.__setattr__(self, "pi_upper", pi_upper)
        object.__setattr__(self, "pi_tol", pi_tol)

        if gamma <= 0.0:
            raise ValueError("gamma must be strictly positive.")

        if not (pi_lower < pi_upper):
            raise ValueError(
                "Portfolio bounds must satisfy pi_lower < pi_upper. "
                f"Got pi_lower={pi_lower}, pi_upper={pi_upper}."
            )

        if pi_tol < 0.0:
            raise ValueError("pi_tol must be nonnegative.")

        # Finite finite bounds must leave a non-empty strict interior after tolerance.
        if math.isfinite(pi_lower) and math.isfinite(pi_upper):
            if not (pi_lower + pi_tol < pi_upper - pi_tol):
                raise ValueError(
                    "Finite portfolio bounds leave no strict interior after pi_tol: "
                    f"pi_lower + pi_tol = {pi_lower + pi_tol}, "
                    f"pi_upper - pi_tol = {pi_upper - pi_tol}."
                )

    @property
    def lower_is_infinite(self) -> bool:
        return self.pi_lower == -math.inf

    @property
    def upper_is_infinite(self) -> bool:
        return self.pi_upper == math.inf

    @property
    def has_infinite_bounds(self) -> bool:
        return self.lower_is_infinite and self.upper_is_infinite

    @property
    def has_finite_bounds(self) -> bool:
        return math.isfinite(self.pi_lower) and math.isfinite(self.pi_upper)


def make_infinite_asset_market_params(
    *,
    gamma: float,
    pi_tol: float = 1.0e-10,
) -> AssetMarketParams:
    """
    Baseline version-1 asset-market parameters.

    With infinite portfolio bounds, every finite supplied risky share is on the
    interior Merton branch.
    """
    return AssetMarketParams(
        gamma=gamma,
        pi_lower=-math.inf,
        pi_upper=math.inf,
        pi_tol=pi_tol,
    )


def make_finite_wide_asset_market_params(
    *,
    gamma: float,
    pi_lower: float = -0.25,
    pi_upper: float = 1.25,
    pi_tol: float = 1.0e-10,
) -> AssetMarketParams:
    """
    Finite wide diagnostic bounds.

    The defaults strictly contain the mechanically feasible range [0, 1].
    """
    return AssetMarketParams(
        gamma=gamma,
        pi_lower=pi_lower,
        pi_upper=pi_upper,
        pi_tol=pi_tol,
    )


# ============================================================
# Portfolio bounds and margins
# ============================================================

def portfolio_cutoffs(params: AssetMarketParams) -> Tuple[float, float]:
    """
    Return the effective strict-interiority cutoffs:

        lower_cutoff = pi_lower + pi_tol,
        upper_cutoff = pi_upper - pi_tol.

    Infinite bounds remain infinite.
    """
    lower_cutoff = (
        -math.inf
        if params.lower_is_infinite
        else params.pi_lower + params.pi_tol
    )

    upper_cutoff = (
        math.inf
        if params.upper_is_infinite
        else params.pi_upper - params.pi_tol
    )

    return float(lower_cutoff), float(upper_cutoff)


def portfolio_margins(
    pi: float,
    params: AssetMarketParams,
) -> Tuple[float, float, float]:
    """
    Return margins relative to the strict-interiority cutoffs.

    Positive margins mean pi is inside that side of the strict interior.
    Negative margins mean pi violates that side.

    The true strict interior condition is:

        lower_margin > 0 and upper_margin > 0.

    Infinite bounds produce infinite margins on that side.
    """
    pi = float(pi)

    if not math.isfinite(pi):
        raise ValueError("pi must be finite when computing scalar portfolio margins.")

    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)

    lower_margin = math.inf if params.lower_is_infinite else pi - lower_cutoff
    upper_margin = math.inf if params.upper_is_infinite else upper_cutoff - pi
    interior_margin = min(lower_margin, upper_margin)

    return float(lower_margin), float(upper_margin), float(interior_margin)


def vectorized_portfolio_margins(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> Tuple[ArrayLike, ArrayLike, ArrayLike]:
    """
    Vectorized margins relative to the strict-interiority cutoffs.

    Non-finite pi values receive NaN margins.
    """
    pi = _as_float_array(pi_values)
    finite = np.isfinite(pi)

    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)

    if params.lower_is_infinite:
        lower_margin = np.full(pi.shape, math.inf, dtype=float)
    else:
        lower_margin = pi - lower_cutoff

    if params.upper_is_infinite:
        upper_margin = np.full(pi.shape, math.inf, dtype=float)
    else:
        upper_margin = upper_cutoff - pi

    interior_margin = np.minimum(lower_margin, upper_margin)

    lower_margin = np.where(finite, lower_margin, math.nan)
    upper_margin = np.where(finite, upper_margin, math.nan)
    interior_margin = np.where(finite, interior_margin, math.nan)

    return (
        _return_like_input_float(lower_margin, pi_values),
        _return_like_input_float(upper_margin, pi_values),
        _return_like_input_float(interior_margin, pi_values),
    )


def mechanical_range_strictly_inside_bounds(params: AssetMarketParams) -> bool:
    """
    Check whether the mechanically feasible risky-share range [0, 1] lies
    strictly inside the effective portfolio interior.

    This checks:

        lower_cutoff < 0
        and
        1 < upper_cutoff.
    """
    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)
    return bool(lower_cutoff < 0.0 and 1.0 < upper_cutoff)


def mechanical_range_margin(params: AssetMarketParams) -> float:
    """
    Minimum strict-interiority margin of the mechanical range [0, 1].

    For finite wide bounds this is

        min(0 - lower_cutoff, upper_cutoff - 1).

    With infinite bounds on both sides, this returns +inf.
    """
    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)
    return float(min(0.0 - lower_cutoff, upper_cutoff - 1.0))


def require_mechanical_range_strictly_inside_bounds(
    params: AssetMarketParams,
) -> None:
    """
    Require the mechanical risky-share range [0, 1] to be strictly inside bounds.
    """
    if not mechanical_range_strictly_inside_bounds(params):
        lower_cutoff, upper_cutoff = portfolio_cutoffs(params)
        raise ValueError(
            "Mechanical risky-share range [0,1] is not strictly inside "
            "portfolio bounds after tolerance. "
            f"Effective cutoffs: lower={lower_cutoff}, upper={upper_cutoff}."
        )


# ============================================================
# Pointwise checks
# ============================================================

@dataclass(frozen=True)
class PortfolioCheck:
    """
    Pointwise check for a supplied risky share.

    valid is True only on the interior Merton branch.

    In version 1, portfolio_lower_bind and portfolio_upper_bind are rejected
    by the interior-Merton oracle. They are diagnostic statuses, not solved
    complementarity branches.
    """
    pi: float
    status: PortfolioStatus
    valid: bool
    reason: Optional[str]

    lower_margin: float
    upper_margin: float
    interior_margin: float

    @property
    def coarse_status(self) -> PortfolioCoarseStatus:
        if self.status == "interior":
            return "interior"
        if self.status in ("portfolio_lower_bind", "portfolio_upper_bind"):
            return "portfolio_bind"
        return "invalid"

    @property
    def is_lower_bind(self) -> bool:
        return self.status == "portfolio_lower_bind"

    @property
    def is_upper_bind(self) -> bool:
        return self.status == "portfolio_upper_bind"

    @property
    def is_portfolio_bind(self) -> bool:
        return self.status in ("portfolio_lower_bind", "portfolio_upper_bind")


def check_portfolio_share(
    pi: float,
    params: AssetMarketParams,
) -> PortfolioCheck:
    """
    Check whether a scalar risky share lies on the interior Merton branch.

    The branch is valid only if:

        pi_lower + pi_tol < pi < pi_upper - pi_tol

    for finite bounds. With infinite bounds, only finiteness of pi matters.
    """
    pi = float(pi)

    if not math.isfinite(pi):
        return PortfolioCheck(
            pi=pi,
            status="invalid",
            valid=False,
            reason="pi is non-finite",
            lower_margin=math.nan,
            upper_margin=math.nan,
            interior_margin=math.nan,
        )

    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)
    lower_margin, upper_margin, interior_margin = portfolio_margins(pi, params)

    lower_binds = math.isfinite(lower_cutoff) and pi <= lower_cutoff
    upper_binds = math.isfinite(upper_cutoff) and pi >= upper_cutoff

    if lower_binds:
        return PortfolioCheck(
            pi=pi,
            status="portfolio_lower_bind",
            valid=False,
            reason="pi at/below lower portfolio interiority cutoff",
            lower_margin=lower_margin,
            upper_margin=upper_margin,
            interior_margin=interior_margin,
        )

    if upper_binds:
        return PortfolioCheck(
            pi=pi,
            status="portfolio_upper_bind",
            valid=False,
            reason="pi at/above upper portfolio interiority cutoff",
            lower_margin=lower_margin,
            upper_margin=upper_margin,
            interior_margin=interior_margin,
        )

    return PortfolioCheck(
        pi=pi,
        status="interior",
        valid=True,
        reason=None,
        lower_margin=lower_margin,
        upper_margin=upper_margin,
        interior_margin=interior_margin,
    )


def require_portfolio_interior(
    pi: float,
    params: AssetMarketParams,
) -> PortfolioCheck:
    """
    Require a scalar risky share to be on the interior Merton branch.
    """
    check = check_portfolio_share(pi, params)

    if not check.valid:
        raise ValueError(
            "Risky share is not on the interior Merton branch: "
            f"pi={check.pi}, status={check.status}, reason={check.reason}."
        )

    return check


# ============================================================
# Vectorized status logic
# ============================================================

@dataclass(frozen=True)
class _PortfolioMasks:
    pi: np.ndarray
    finite: np.ndarray
    interior: np.ndarray
    lower_bind: np.ndarray
    upper_bind: np.ndarray
    invalid: np.ndarray
    lower_margin: np.ndarray
    upper_margin: np.ndarray
    interior_margin: np.ndarray


def _portfolio_masks(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> _PortfolioMasks:
    """
    Internal vectorized status and margin computation.
    """
    pi = _as_float_array(pi_values)
    finite = np.isfinite(pi)

    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)

    if params.lower_is_infinite:
        lower_margin = np.full(pi.shape, math.inf, dtype=float)
        lower_ok = np.ones(pi.shape, dtype=bool)
        lower_bind = np.zeros(pi.shape, dtype=bool)
    else:
        lower_margin = pi - lower_cutoff
        lower_ok = pi > lower_cutoff
        lower_bind = finite & (pi <= lower_cutoff)

    if params.upper_is_infinite:
        upper_margin = np.full(pi.shape, math.inf, dtype=float)
        upper_ok = np.ones(pi.shape, dtype=bool)
        upper_bind = np.zeros(pi.shape, dtype=bool)
    else:
        upper_margin = upper_cutoff - pi
        upper_ok = pi < upper_cutoff
        upper_bind = finite & (pi >= upper_cutoff)

    interior = finite & lower_ok & upper_ok
    invalid = ~finite

    interior_margin = np.minimum(lower_margin, upper_margin)

    lower_margin = np.where(finite, lower_margin, math.nan)
    upper_margin = np.where(finite, upper_margin, math.nan)
    interior_margin = np.where(finite, interior_margin, math.nan)

    return _PortfolioMasks(
        pi=pi,
        finite=finite,
        interior=interior,
        lower_bind=lower_bind,
        upper_bind=upper_bind,
        invalid=invalid,
        lower_margin=lower_margin,
        upper_margin=upper_margin,
        interior_margin=interior_margin,
    )


def portfolio_status_array(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> Union[str, np.ndarray]:
    """
    Return pointwise portfolio statuses using vectorized masks.

    Scalar input returns a string.
    Array input returns an object array of strings.
    """
    masks = _portfolio_masks(pi_values, params)

    status = np.full(masks.pi.shape, "invalid", dtype=object)
    status[masks.interior] = "interior"
    status[masks.lower_bind] = "portfolio_lower_bind"
    status[masks.upper_bind] = "portfolio_upper_bind"
    status[masks.invalid] = "invalid"

    return _return_like_input_object(status, pi_values)


def portfolio_coarse_status_array(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> Union[str, np.ndarray]:
    """
    Return coarse oracle-facing statuses:

        interior,
        portfolio_bind,
        invalid.

    This is useful if the live oracle wants a compact status flag while still
    allowing side-specific diagnostics to be inspected separately.
    """
    masks = _portfolio_masks(pi_values, params)

    status = np.full(masks.pi.shape, "invalid", dtype=object)
    portfolio_bind = masks.lower_bind | masks.upper_bind

    status[masks.interior] = "interior"
    status[portfolio_bind] = "portfolio_bind"
    status[masks.invalid] = "invalid"

    return _return_like_input_object(status, pi_values)


def portfolio_share_is_interior(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> Union[bool, np.ndarray]:
    """
    Vectorized check for the interior Merton branch.
    """
    masks = _portfolio_masks(pi_values, params)
    return _return_like_input_bool(masks.interior, pi_values)


def require_all_portfolio_interior(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> None:
    """
    Require every supplied risky share to be on the interior Merton branch.
    """
    masks = _portfolio_masks(pi_values, params)

    if not bool(np.all(masks.interior)):
        n_total = int(masks.pi.size)
        n_interior = int(np.sum(masks.interior))
        n_lower = int(np.sum(masks.lower_bind))
        n_upper = int(np.sum(masks.upper_bind))
        n_invalid = int(np.sum(masks.invalid))

        raise ValueError(
            "Not all risky shares are on the interior Merton branch. "
            f"n_total={n_total}, n_interior={n_interior}, "
            f"n_lower_bind={n_lower}, n_upper_bind={n_upper}, "
            f"n_invalid={n_invalid}."
        )


# ============================================================
# Vectorized diagnostics
# ============================================================

@dataclass(frozen=True)
class PortfolioDiagnostics:
    """
    Summary diagnostics for a scalar or array of supplied risky shares.
    """
    n_total: int
    n_interior: int
    n_portfolio_bind: int
    n_lower_bind: int
    n_upper_bind: int
    n_invalid: int
    n_rejected: int

    min_pi: float
    max_pi: float

    min_lower_margin: float
    min_upper_margin: float
    min_interior_margin: float

    lower_bound: float
    upper_bound: float
    lower_cutoff: float
    upper_cutoff: float
    pi_tol: float

    mechanical_range_inside_bounds: bool
    mechanical_range_margin: float

    share_interior: float
    share_portfolio_bind: float
    share_invalid: float
    share_rejected: float

    @property
    def all_interior(self) -> bool:
        return self.n_interior == self.n_total

    @property
    def any_portfolio_bind(self) -> bool:
        return self.n_portfolio_bind > 0

    @property
    def any_invalid(self) -> bool:
        return self.n_invalid > 0


def summarize_portfolio_shares(
    pi_values: ArrayLike,
    params: AssetMarketParams,
) -> PortfolioDiagnostics:
    """
    Summarise supplied risky shares using vectorized NumPy masks.

    This function intentionally does not compute pi^{mc} from state/control
    objects. The live oracle owns that current-control computation.
    """
    masks = _portfolio_masks(pi_values, params)

    if masks.pi.size == 0:
        raise ValueError("pi_values must be non-empty.")

    n_total = int(masks.pi.size)
    n_interior = int(np.sum(masks.interior))
    n_lower_bind = int(np.sum(masks.lower_bind))
    n_upper_bind = int(np.sum(masks.upper_bind))
    n_portfolio_bind = n_lower_bind + n_upper_bind
    n_invalid = int(np.sum(masks.invalid))
    n_rejected = n_portfolio_bind + n_invalid

    finite = masks.finite

    if bool(np.any(finite)):
        min_pi = float(np.min(masks.pi[finite]))
        max_pi = float(np.max(masks.pi[finite]))
        min_lower_margin = float(np.min(masks.lower_margin[finite]))
        min_upper_margin = float(np.min(masks.upper_margin[finite]))
        min_interior_margin = float(np.min(masks.interior_margin[finite]))
    else:
        min_pi = math.nan
        max_pi = math.nan
        min_lower_margin = math.nan
        min_upper_margin = math.nan
        min_interior_margin = math.nan

    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)

    return PortfolioDiagnostics(
        n_total=n_total,
        n_interior=n_interior,
        n_portfolio_bind=n_portfolio_bind,
        n_lower_bind=n_lower_bind,
        n_upper_bind=n_upper_bind,
        n_invalid=n_invalid,
        n_rejected=n_rejected,
        min_pi=min_pi,
        max_pi=max_pi,
        min_lower_margin=min_lower_margin,
        min_upper_margin=min_upper_margin,
        min_interior_margin=min_interior_margin,
        lower_bound=float(params.pi_lower),
        upper_bound=float(params.pi_upper),
        lower_cutoff=float(lower_cutoff),
        upper_cutoff=float(upper_cutoff),
        pi_tol=float(params.pi_tol),
        mechanical_range_inside_bounds=mechanical_range_strictly_inside_bounds(params),
        mechanical_range_margin=mechanical_range_margin(params),
        share_interior=float(n_interior / n_total),
        share_portfolio_bind=float(n_portfolio_bind / n_total),
        share_invalid=float(n_invalid / n_total),
        share_rejected=float(n_rejected / n_total),
    )


# ============================================================
# Parameter validation
# ============================================================

@dataclass(frozen=True)
class AssetMarketValidationReport:
    """
    Validation report for Block 3 parameters.
    """
    gamma: float
    pi_lower: float
    pi_upper: float
    pi_tol: float

    lower_cutoff: float
    upper_cutoff: float

    mechanical_range_inside_bounds: bool
    mechanical_range_margin: float

    finite_grid_n_total: int
    finite_grid_n_interior: int
    finite_grid_n_portfolio_bind: int
    finite_grid_n_invalid: int

    invalid_param_rejections: int


def validate_asset_market_params(
    params: AssetMarketParams,
    *,
    require_mechanical_range: bool = True,
) -> AssetMarketValidationReport:
    """
    Validate Block 3 parameters.

    Checks:
      - gamma > 0;
      - pi_lower < pi_upper;
      - pi_tol >= 0;
      - finite finite bounds leave positive strict interior after pi_tol;
      - optionally, the mechanical range [0, 1] lies strictly inside bounds;
      - a test grid over [0, 1] is interior under baseline/wide bounds;
      - representative invalid parameter cases are rejected.
    """
    # Construction of AssetMarketParams already validates primitive conditions.
    lower_cutoff, upper_cutoff = portfolio_cutoffs(params)

    inside = mechanical_range_strictly_inside_bounds(params)
    margin = mechanical_range_margin(params)

    if require_mechanical_range and not inside:
        raise ValueError(
            "Asset-market parameters do not strictly contain the mechanical "
            "risky-share range [0,1] after tolerance."
        )

    # Test mechanical range.
    pi_grid = np.linspace(0.0, 1.0, 1001)
    diag = summarize_portfolio_shares(pi_grid, params)

    if require_mechanical_range and not diag.all_interior:
        raise RuntimeError(
            "Mechanical risky-share grid [0,1] should be fully interior "
            "under baseline/wide Block 3 bounds."
        )

    invalid_cases = [
        dict(gamma=0.0, pi_lower=-math.inf, pi_upper=math.inf, pi_tol=1.0e-10),
        dict(gamma=-1.0, pi_lower=-math.inf, pi_upper=math.inf, pi_tol=1.0e-10),
        dict(gamma=math.nan, pi_lower=-math.inf, pi_upper=math.inf, pi_tol=1.0e-10),
        dict(gamma=5.0, pi_lower=1.0, pi_upper=0.0, pi_tol=1.0e-10),
        dict(gamma=5.0, pi_lower=0.0, pi_upper=1.0, pi_tol=-1.0e-10),
        dict(gamma=5.0, pi_lower=0.0, pi_upper=1.0e-4, pi_tol=1.0e-3),
        dict(gamma=5.0, pi_lower=math.nan, pi_upper=math.inf, pi_tol=1.0e-10),
        dict(gamma=5.0, pi_lower=-math.inf, pi_upper=math.nan, pi_tol=1.0e-10),
    ]

    invalid_param_rejections = 0

    for kwargs in invalid_cases:
        try:
            AssetMarketParams(**kwargs)
        except ValueError:
            invalid_param_rejections += 1
        else:
            raise RuntimeError(
                "Invalid AssetMarketParams case was not rejected: "
                f"{kwargs}."
            )

    return AssetMarketValidationReport(
        gamma=float(params.gamma),
        pi_lower=float(params.pi_lower),
        pi_upper=float(params.pi_upper),
        pi_tol=float(params.pi_tol),
        lower_cutoff=float(lower_cutoff),
        upper_cutoff=float(upper_cutoff),
        mechanical_range_inside_bounds=bool(inside),
        mechanical_range_margin=float(margin),
        finite_grid_n_total=diag.n_total,
        finite_grid_n_interior=diag.n_interior,
        finite_grid_n_portfolio_bind=diag.n_portfolio_bind,
        finite_grid_n_invalid=diag.n_invalid,
        invalid_param_rejections=int(invalid_param_rejections),
    )


# ============================================================
# Smoke test
# ============================================================

def module_smoke_test() -> dict[str, float]:
    """
    Minimal Block 3 self-test.

    This test checks:
      - infinite baseline bounds;
      - finite wide bounds;
      - tight portfolio-bound side labels;
      - cutoff-relative margins;
      - vectorized summaries;
      - scalar vs array status shape behavior;
      - invalid parameter rejection.
    """
    report: dict[str, float] = {}

    # Infinite baseline bounds.
    infinite_params = make_infinite_asset_market_params(gamma=5.0)

    val_inf = validate_asset_market_params(
        infinite_params,
        require_mechanical_range=True,
    )

    report["infinite_gamma"] = float(val_inf.gamma)
    report["infinite_mechanical_range_margin"] = float(val_inf.mechanical_range_margin)

    pi_grid = np.linspace(0.0, 1.0, 1001)
    diag_inf = summarize_portfolio_shares(pi_grid, infinite_params)

    if not diag_inf.all_interior:
        raise RuntimeError("Infinite baseline bounds should make all finite pi interior.")

    if diag_inf.n_portfolio_bind != 0:
        raise RuntimeError("Portfolio bind should not fire under infinite bounds.")

    if not portfolio_share_is_interior(0.0, infinite_params):
        raise RuntimeError("pi=0 should be interior under infinite bounds.")

    if not portfolio_share_is_interior(1.0, infinite_params):
        raise RuntimeError("pi=1 should be interior under infinite bounds.")

    # Finite wide bounds.
    finite_wide = make_finite_wide_asset_market_params(
        gamma=5.0,
        pi_lower=-0.25,
        pi_upper=1.25,
        pi_tol=1.0e-8,
    )

    val_wide = validate_asset_market_params(
        finite_wide,
        require_mechanical_range=True,
    )

    report["finite_wide_mechanical_range_margin"] = float(
        val_wide.mechanical_range_margin
    )

    diag_wide = summarize_portfolio_shares(pi_grid, finite_wide)

    if not diag_wide.all_interior:
        raise RuntimeError("Finite wide bounds should contain [0,1] in the interior.")

    # Tight bounds should show side-specific branch labels.
    tight = AssetMarketParams(
        gamma=5.0,
        pi_lower=0.0,
        pi_upper=1.0,
        pi_tol=1.0e-3,
    )

    chk_low = check_portfolio_share(0.0, tight)
    chk_mid = check_portfolio_share(0.5, tight)
    chk_high = check_portfolio_share(1.0, tight)

    if chk_low.status != "portfolio_lower_bind":
        raise RuntimeError("pi=0 should be labelled portfolio_lower_bind under tight bounds.")

    if chk_mid.status != "interior":
        raise RuntimeError("pi=0.5 should be interior under tight bounds.")

    if chk_high.status != "portfolio_upper_bind":
        raise RuntimeError("pi=1 should be labelled portfolio_upper_bind under tight bounds.")

    if not (chk_low.interior_margin < 0.0):
        raise RuntimeError(
            "Cutoff-relative interior margin should be negative at lower bind."
        )

    if not (chk_high.interior_margin < 0.0):
        raise RuntimeError(
            "Cutoff-relative interior margin should be negative at upper bind."
        )

    if chk_low.coarse_status != "portfolio_bind":
        raise RuntimeError("Lower bind should map to coarse portfolio_bind.")

    if chk_high.coarse_status != "portfolio_bind":
        raise RuntimeError("Upper bind should map to coarse portfolio_bind.")

    # Vectorized status and diagnostics.
    test_pi = np.array([-0.1, 0.0, 0.5, 1.0, 1.1, np.nan])
    statuses = portfolio_status_array(test_pi, tight)

    expected = np.array(
        [
            "portfolio_lower_bind",
            "portfolio_lower_bind",
            "interior",
            "portfolio_upper_bind",
            "portfolio_upper_bind",
            "invalid",
        ],
        dtype=object,
    )

    if not np.array_equal(statuses, expected):
        raise RuntimeError(
            f"Unexpected vectorized statuses: got {statuses}, expected {expected}."
        )

    diag_tight = summarize_portfolio_shares(test_pi, tight)

    if diag_tight.n_total != 6:
        raise RuntimeError("Tight diagnostic total count failed.")

    if diag_tight.n_interior != 1:
        raise RuntimeError("Tight diagnostic interior count failed.")

    if diag_tight.n_lower_bind != 2:
        raise RuntimeError("Tight diagnostic lower-bind count failed.")

    if diag_tight.n_upper_bind != 2:
        raise RuntimeError("Tight diagnostic upper-bind count failed.")

    if diag_tight.n_invalid != 1:
        raise RuntimeError("Tight diagnostic invalid count failed.")

    if diag_tight.n_rejected != 5:
        raise RuntimeError("Tight diagnostic rejected count failed.")

    # Scalar status should return a string, not an array.
    scalar_status = portfolio_status_array(0.5, tight)
    if not isinstance(scalar_status, str) or scalar_status != "interior":
        raise RuntimeError("Scalar portfolio_status_array should return status string.")

    scalar_bool = portfolio_share_is_interior(0.5, tight)
    if not isinstance(scalar_bool, bool) or scalar_bool is not True:
        raise RuntimeError("Scalar portfolio_share_is_interior should return bool.")

    # Requirement helpers.
    try:
        require_portfolio_interior(0.0, tight)
    except ValueError:
        pass
    else:
        raise RuntimeError("require_portfolio_interior should reject lower bind.")

    require_portfolio_interior(0.5, tight)

    try:
        require_all_portfolio_interior(test_pi, tight)
    except ValueError:
        pass
    else:
        raise RuntimeError("require_all_portfolio_interior should reject test_pi.")

    require_all_portfolio_interior(pi_grid, finite_wide)

    # Mechanical range check should fail for tight bounds with tolerance.
    if mechanical_range_strictly_inside_bounds(tight):
        raise RuntimeError(
            "Tight [0,1] bounds with positive tolerance should not strictly "
            "contain the mechanical range [0,1]."
        )

    # Invalid parameter rejection count.
    report["invalid_param_rejections"] = float(
        val_inf.invalid_param_rejections
    )

    report["tight_n_interior"] = float(diag_tight.n_interior)
    report["tight_n_lower_bind"] = float(diag_tight.n_lower_bind)
    report["tight_n_upper_bind"] = float(diag_tight.n_upper_bind)
    report["tight_n_invalid"] = float(diag_tight.n_invalid)
    report["tight_min_interior_margin"] = float(diag_tight.min_interior_margin)

    return report


__all__ = [
    "ArrayLike",
    "PortfolioStatus",
    "PortfolioCoarseStatus",
    "AssetMarketParams",
    "PortfolioCheck",
    "PortfolioDiagnostics",
    "AssetMarketValidationReport",
    "make_infinite_asset_market_params",
    "make_finite_wide_asset_market_params",
    "portfolio_cutoffs",
    "portfolio_margins",
    "vectorized_portfolio_margins",
    "mechanical_range_strictly_inside_bounds",
    "mechanical_range_margin",
    "require_mechanical_range_strictly_inside_bounds",
    "check_portfolio_share",
    "require_portfolio_interior",
    "portfolio_status_array",
    "portfolio_coarse_status_array",
    "portfolio_share_is_interior",
    "require_all_portfolio_interior",
    "summarize_portfolio_shares",
    "validate_asset_market_params",
    "module_smoke_test",
]

Overwriting asset_market.py


In [11]:
import importlib
import asset_market

importlib.reload(asset_market)

block3_report = asset_market.module_smoke_test()

print("Block 3 validation passed.")
print(block3_report)

Block 3 validation passed.
{'infinite_gamma': 5.0, 'infinite_mechanical_range_margin': inf, 'finite_wide_mechanical_range_margin': 0.24999999, 'invalid_param_rejections': 8.0, 'tight_n_interior': 1.0, 'tight_n_lower_bind': 2.0, 'tight_n_upper_bind': 2.0, 'tight_n_invalid': 1.0, 'tight_min_interior_margin': -0.10100000000000009}


In [12]:
import numpy as np
import asset_market

# Baseline: infinite bounds. Every finite supplied risky share is interior.
baseline_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

assert asset_market.check_portfolio_share(0.0, baseline_params).status == "interior"
assert asset_market.check_portfolio_share(0.5, baseline_params).status == "interior"
assert asset_market.check_portfolio_share(1.0, baseline_params).status == "interior"
assert asset_market.check_portfolio_share(np.nan, baseline_params).status == "invalid"

baseline_diag = asset_market.summarize_portfolio_shares(
    np.linspace(0.0, 1.0, 1001),
    baseline_params,
)

assert baseline_diag.n_interior == baseline_diag.n_total
assert baseline_diag.n_portfolio_bind == 0
assert baseline_diag.n_invalid == 0

print("Infinite-bound baseline tests passed.")
print(baseline_diag)


# Finite wide bounds. The mechanical range [0,1] remains strictly interior.
wide_params = asset_market.make_finite_wide_asset_market_params(
    gamma=5.0,
    pi_lower=-0.25,
    pi_upper=1.25,
    pi_tol=1.0e-8,
)

assert asset_market.mechanical_range_strictly_inside_bounds(wide_params)
asset_market.require_mechanical_range_strictly_inside_bounds(wide_params)

wide_diag = asset_market.summarize_portfolio_shares(
    np.linspace(0.0, 1.0, 1001),
    wide_params,
)

assert wide_diag.n_interior == wide_diag.n_total
assert wide_diag.n_portfolio_bind == 0

print("\nFinite-wide-bound tests passed.")
print(wide_diag)


# Tight diagnostic bounds. These deliberately make endpoints bind.
tight_params = asset_market.AssetMarketParams(
    gamma=5.0,
    pi_lower=0.0,
    pi_upper=1.0,
    pi_tol=1.0e-3,
)

assert asset_market.check_portfolio_share(0.0, tight_params).status == "portfolio_lower_bind"
assert asset_market.check_portfolio_share(0.5, tight_params).status == "interior"
assert asset_market.check_portfolio_share(1.0, tight_params).status == "portfolio_upper_bind"

test_pi = np.array([-0.1, 0.0, 0.5, 1.0, 1.1, np.nan])

print("\nTight-bound statuses:")
print(asset_market.portfolio_status_array(test_pi, tight_params))

print("\nTight-bound coarse statuses:")
print(asset_market.portfolio_coarse_status_array(test_pi, tight_params))

print("\nTight-bound diagnostics:")
print(asset_market.summarize_portfolio_shares(test_pi, tight_params))

print("\nPortfolio-bound branch tests passed.")

Infinite-bound baseline tests passed.
PortfolioDiagnostics(n_total=1001, n_interior=1001, n_portfolio_bind=0, n_lower_bind=0, n_upper_bind=0, n_invalid=0, n_rejected=0, min_pi=0.0, max_pi=1.0, min_lower_margin=inf, min_upper_margin=inf, min_interior_margin=inf, lower_bound=-inf, upper_bound=inf, lower_cutoff=-inf, upper_cutoff=inf, pi_tol=1e-10, mechanical_range_inside_bounds=True, mechanical_range_margin=inf, share_interior=1.0, share_portfolio_bind=0.0, share_invalid=0.0, share_rejected=0.0)

Finite-wide-bound tests passed.
PortfolioDiagnostics(n_total=1001, n_interior=1001, n_portfolio_bind=0, n_lower_bind=0, n_upper_bind=0, n_invalid=0, n_rejected=0, min_pi=0.0, max_pi=1.0, min_lower_margin=0.24999999, min_upper_margin=0.24999999000000006, min_interior_margin=0.24999999, lower_bound=-0.25, upper_bound=1.25, lower_cutoff=-0.24999999, upper_cutoff=1.24999999, pi_tol=1e-08, mechanical_range_inside_bounds=True, mechanical_range_margin=0.24999999, share_interior=1.0, share_portfolio_bin

In [13]:
import asset_market

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

# Later, the live oracle will compute this from current state/control:
# pi_mc = (k - H) / (k + L)
#
# Block 3 only checks the supplied value.
pi_mc = 0.4

portfolio_check = asset_market.check_portfolio_share(
    pi=pi_mc,
    params=asset_params,
)

if portfolio_check.valid:
    print("Interior Merton branch is valid.")
else:
    print("Reject / branch in oracle:", portfolio_check.status, portfolio_check.reason)

print(portfolio_check)

Interior Merton branch is valid.
PortfolioCheck(pi=0.4, status='interior', valid=True, reason=None, lower_margin=inf, upper_margin=inf, interior_margin=inf)


# Block 4 — frozen continuation block

Given an anticipated Markov planner rule

$$
\hat u_s(k,L)
=
(\hat\tau_s(k,L),\hat T_s(k,L),\hat H_s(k,L)),
$$

this block represents the frozen private owner continuation environment

$$
\mathcal C[\hat u].
$$

The owner value function has the homothetic form

$$
V_s^K(W;k,L)
=
\frac{W^{1-\gamma}}{1-\gamma}
\Psi_s^{\hat u}(k,L),
$$

where

$$
\Psi_s^{\hat u}(k,L)>0.
$$

The associated owner consumption–wealth ratio is

$$
\omega_s^{\hat u}(k,L)
=
\left(\Psi_s^{\hat u}(k,L)\right)^{-1/\gamma}.
$$

Owner consumption is therefore

$$
C_s^K(k,L)
=
\omega_s^{\hat u}(k,L)(k+L).
$$

The central invariant is:

$$
\boxed{
\text{freeze continuation objects, but evaluate current pricing, fiscal objects, and drifts live.}
}
$$

So this block owns frozen objects such as

$$
\Psi_s^{\hat u},
\qquad
\log\Psi_s^{\hat u},
\qquad
\omega_s^{\hat u},
\qquad
\log\omega_s^{\hat u},
\qquad
\text{support masks}.
$$

It does not compute current-control objects such as

$$
\pi^{mc},
\qquad
r_f,
\qquad
\dot k,
\qquad
\dot L,
\qquad
\text{tax bases},
\qquad
\text{current fiscal revenue}.
$$

Those belong to the live oracle.

The solve order for the eventual continuation PDE solver is:

1. solve regime $s=1$ first, because regime $1$ is absorbing;
2. solve regime $s=0$ second, using the frozen regime-$1$ continuation object in the Poisson continuation term.

This module implements the strict continuation-bundle interface and validation harness. The actual PDE / false-transient solver can later replace the `solve_continuation_bundle` stub while preserving the same output interface.

In [14]:
%%writefile continuation_block.py
from __future__ import annotations

from dataclasses import dataclass
from types import MappingProxyType
from typing import Any, Callable, Literal, Mapping, Optional, Tuple, Union
import math
import numpy as np

from asset_market import AssetMarketParams, make_infinite_asset_market_params


ArrayLike = Union[float, int, np.ndarray]

ContinuationVariable = Literal[
    "Psi",
    "log_Psi",
    "omega",
    "log_omega",
]

ContinuationSolverName = Literal[
    "interface_only",
    "false_transient",
    "policy_evaluation",
    "stable_manifold",
    "external",
]

ContinuationFn = Callable[[ArrayLike, ArrayLike], ArrayLike]
SupportFn = Callable[[ArrayLike, ArrayLike], Union[bool, np.ndarray]]
RegimeFnMap = Mapping[int, ContinuationFn]
RegimeSupportMap = Union[SupportFn, Mapping[int, SupportFn]]


# ============================================================
# Block 4 contract
# ============================================================
#
# Inputs:
#   - asset-market parameters from Block 3, especially gamma;
#   - an anticipated Markov planner rule u_hat, once the solver is added;
#   - a computational support / interpolation support;
#   - frozen continuation functions Psi_s, log_Psi_s, omega_s, log_omega_s.
#
# Outputs:
#   - frozen ContinuationBundle C[u_hat];
#   - strict accessors for Psi_s, log_Psi_s, omega_s, log_omega_s;
#   - support masks and support diagnostics;
#   - owner-consumption helper C^K_s = omega_s(k,L)(k+L);
#   - validation diagnostics.
#
# Forbidden responsibilities:
#   - no live current-control oracle;
#   - no computation of pi^{mc};
#   - no computation of r_f;
#   - no computation of kdot or Ldot;
#   - no computation of current tax bases or fiscal revenue;
#   - no viability peeling;
#   - no Howard iteration;
#   - no planner pointwise maximisation.
#
# Important convention:
#   This block freezes continuation objects. Later blocks must consume them
#   without recomputing omega_s inside the oracle, viability witness search,
#   planner pointwise solver, or Howard loop.
#
# Block 5 will compute chi and lambda^Q from the frozen log_omega objects.


# ============================================================
# Shape and domain helpers
# ============================================================

def _is_scalar_like(x: Any) -> bool:
    return np.ndim(x) == 0


def _is_scalar_state(k: ArrayLike, L: ArrayLike) -> bool:
    return _is_scalar_like(k) and _is_scalar_like(L)


def _broadcast_state(k: ArrayLike, L: ArrayLike) -> Tuple[np.ndarray, np.ndarray]:
    k_arr = np.asarray(k, dtype=float)
    L_arr = np.asarray(L, dtype=float)

    try:
        k_b, L_b = np.broadcast_arrays(k_arr, L_arr)
    except ValueError as exc:
        raise ValueError(
            f"k and L could not be broadcast together: "
            f"k.shape={k_arr.shape}, L.shape={L_arr.shape}."
        ) from exc

    if not np.all(np.isfinite(k_b)):
        raise ValueError("k contains non-finite values.")

    if not np.all(np.isfinite(L_b)):
        raise ValueError("L contains non-finite values.")

    return k_b.astype(float, copy=False), L_b.astype(float, copy=False)


def _coerce_to_shape(
    raw: Any,
    shape: tuple[int, ...],
    *,
    name: str,
    dtype: Any,
) -> np.ndarray:
    out = np.asarray(raw, dtype=dtype)

    if out.shape == ():
        return np.full(shape, out.item(), dtype=dtype)

    try:
        return np.broadcast_to(out, shape).astype(dtype, copy=True)
    except ValueError as exc:
        raise ValueError(
            f"{name} returned shape {out.shape}, but state has shape {shape}."
        ) from exc


def _return_float_like_state(
    out: np.ndarray,
    k: ArrayLike,
    L: ArrayLike,
) -> ArrayLike:
    out = np.asarray(out, dtype=float)

    if _is_scalar_state(k, L):
        if out.shape == ():
            return float(out)
        if out.size == 1:
            return float(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar state produced non-scalar output with shape {out.shape}."
        )

    return out


def _return_bool_like_state(
    out: np.ndarray,
    k: ArrayLike,
    L: ArrayLike,
) -> Union[bool, np.ndarray]:
    out = np.asarray(out, dtype=bool)

    if _is_scalar_state(k, L):
        if out.shape == ():
            return bool(out)
        if out.size == 1:
            return bool(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar state produced non-scalar output with shape {out.shape}."
        )

    return out


def _finite_float(x: float, *, name: str) -> float:
    x = float(x)
    if not math.isfinite(x):
        raise ValueError(f"{name} must be finite.")
    return x


def _positive_float(x: float, *, name: str) -> float:
    x = _finite_float(x, name=name)
    if x <= 0.0:
        raise ValueError(f"{name} must be strictly positive.")
    return x


def require_regime(s: int) -> int:
    if s not in (0, 1):
        raise ValueError("Regime s must be 0 or 1.")
    return int(s)


def _coerce_continuation_fn(
    fn: ContinuationFn,
    *,
    name: str,
) -> ContinuationFn:
    if not callable(fn):
        raise TypeError(f"{name} must be callable.")

    def wrapped(k: ArrayLike, L: ArrayLike) -> ArrayLike:
        k_b, L_b = _broadcast_state(k, L)

        if k_b.shape == ():
            call_k: ArrayLike = float(k_b)
            call_L: ArrayLike = float(L_b)
        else:
            call_k = k_b
            call_L = L_b

        raw = fn(call_k, call_L)
        out = _coerce_to_shape(raw, k_b.shape, name=name, dtype=float)

        if not np.all(np.isfinite(out)):
            raise ValueError(f"{name} returned non-finite values.")

        return _return_float_like_state(out, k, L)

    return wrapped


def _coerce_support_fn(
    fn: SupportFn,
    *,
    name: str,
) -> SupportFn:
    if not callable(fn):
        raise TypeError(f"{name} must be callable.")

    def wrapped(k: ArrayLike, L: ArrayLike) -> Union[bool, np.ndarray]:
        k_b, L_b = _broadcast_state(k, L)

        if k_b.shape == ():
            call_k: ArrayLike = float(k_b)
            call_L: ArrayLike = float(L_b)
        else:
            call_k = k_b
            call_L = L_b

        raw = fn(call_k, call_L)
        out = _coerce_to_shape(raw, k_b.shape, name=name, dtype=bool)

        return _return_bool_like_state(out, k, L)

    return wrapped


def _coerce_regime_fn_map(
    fns: RegimeFnMap,
    *,
    name: str,
) -> Mapping[int, ContinuationFn]:
    if set(fns.keys()) != {0, 1}:
        raise ValueError(f"{name} must have exactly regime keys {{0, 1}}.")

    coerced = {
        0: _coerce_continuation_fn(fns[0], name=f"{name}[0]"),
        1: _coerce_continuation_fn(fns[1], name=f"{name}[1]"),
    }

    return MappingProxyType(coerced)


def _coerce_support_map(
    support_fns: RegimeSupportMap,
    *,
    name: str,
) -> Mapping[int, SupportFn]:
    if callable(support_fns):
        coerced = {
            0: _coerce_support_fn(support_fns, name=f"{name}[0]"),
            1: _coerce_support_fn(support_fns, name=f"{name}[1]"),
        }
        return MappingProxyType(coerced)

    if set(support_fns.keys()) != {0, 1}:
        raise ValueError(f"{name} must be callable or have exactly regime keys {{0, 1}}.")

    coerced = {
        0: _coerce_support_fn(support_fns[0], name=f"{name}[0]"),
        1: _coerce_support_fn(support_fns[1], name=f"{name}[1]"),
    }

    return MappingProxyType(coerced)


def _safe_exp(x: np.ndarray, *, clip: float) -> np.ndarray:
    return np.exp(np.clip(np.asarray(x, dtype=float), -clip, clip))


# ============================================================
# Options and diagnostics
# ============================================================

@dataclass(frozen=True)
class ContinuationOptions:
    """
    Numerical options for the eventual continuation solve.

    This object is already useful before the PDE solver is implemented because
    it fixes conventions for positivity, strict support, no extrapolation, and
    log-variable choice.

    variable:
        Preferred numerical unknown for the continuation solver.

        Good defaults:
            "log_omega" or "log_Psi".

    no_extrapolate:
        If True, continuation evaluation outside support should raise rather
        than silently extrapolating.

    positivity_floor:
        Strict lower floor used in validation for positive objects.

    exp_clip:
        Clipping threshold used only when converting log objects to level
        objects for diagnostics / accessors.
    """
    variable: Literal["log_Psi", "log_omega"] = "log_omega"
    solver: ContinuationSolverName = "interface_only"

    max_iter: int = 1000
    tol: float = 1.0e-8
    cfl: float = 0.9
    damping: float = 1.0

    no_extrapolate: bool = True
    positivity_floor: float = 1.0e-12
    support_tol: float = 1.0e-12
    exp_clip: float = 700.0

    def __post_init__(self) -> None:
        if self.variable not in ("log_Psi", "log_omega"):
            raise ValueError("variable must be 'log_Psi' or 'log_omega'.")

        if self.solver not in (
            "interface_only",
            "false_transient",
            "policy_evaluation",
            "stable_manifold",
            "external",
        ):
            raise ValueError(f"Unknown continuation solver name: {self.solver}.")

        if int(self.max_iter) <= 0:
            raise ValueError("max_iter must be positive.")
        object.__setattr__(self, "max_iter", int(self.max_iter))

        _positive_float(self.tol, name="tol")
        _positive_float(self.cfl, name="cfl")
        _positive_float(self.damping, name="damping")
        _positive_float(self.positivity_floor, name="positivity_floor")
        _finite_float(self.support_tol, name="support_tol")
        _positive_float(self.exp_clip, name="exp_clip")

        if self.support_tol < 0.0:
            raise ValueError("support_tol must be nonnegative.")

        if self.damping > 1.0:
            raise ValueError("damping should lie in (0,1].")


@dataclass(frozen=True)
class ContinuationDiagnostics:
    """
    Diagnostics attached to a frozen continuation bundle.

    The default diagnostic describes an interface-only bundle. Once the PDE
    solver is implemented, the same object should report actual convergence,
    residual norms, and iteration counts.
    """
    solver_name: str = "interface_only"
    converged: bool = False
    implemented_solver: bool = False

    regime_order: Tuple[int, int] = (1, 0)

    iterations_1: int = 0
    iterations_0: int = 0

    residual_norm_1: float = math.nan
    residual_norm_0: float = math.nan

    min_Psi_1: float = math.nan
    min_Psi_0: float = math.nan
    min_omega_1: float = math.nan
    min_omega_0: float = math.nan

    support_status: str = "not_evaluated"
    warm_start_used: bool = False
    message: str = "Continuation interface constructed; PDE solver not run."


@dataclass(frozen=True)
class ContinuationGridSummary:
    """
    Summary of frozen continuation objects on a supplied grid.
    """
    n_total: int
    n_supported_0: int
    n_supported_1: int
    n_joint_supported: int
    n_joint_unsupported: int

    min_omega_0: float
    max_omega_0: float
    min_omega_1: float
    max_omega_1: float

    min_Psi_0: float
    max_Psi_0: float
    min_Psi_1: float
    max_Psi_1: float

    min_owner_consumption_0: float
    min_owner_consumption_1: float


# ============================================================
# Frozen continuation bundle
# ============================================================

@dataclass(frozen=True)
class ContinuationBundle:
    """
    Frozen private continuation bundle C[u_hat].

    This object is deliberately immutable and function-based. It may wrap
    interpolation objects, analytic test functions, or solver output arrays.

    The bundle exposes four equivalent continuation objects:

        Psi_s(k,L),
        log_Psi_s(k,L),
        omega_s(k,L),
        log_omega_s(k,L).

    They must satisfy

        omega_s = Psi_s^(-1/gamma),

    equivalently,

        log_omega_s = -log_Psi_s / gamma.

    The bundle does not compute current prices or drifts.
    """
    asset_params: AssetMarketParams

    support_mask_fns: RegimeSupportMap

    Psi_fns: RegimeFnMap
    log_Psi_fns: RegimeFnMap
    omega_fns: RegimeFnMap
    log_omega_fns: RegimeFnMap

    diagnostics: Optional[ContinuationDiagnostics] = None
    source_label: str = "unnamed_continuation_bundle"

    def __post_init__(self) -> None:
        if not isinstance(self.asset_params, AssetMarketParams):
            raise TypeError("asset_params must be an AssetMarketParams instance.")

        object.__setattr__(
            self,
            "support_mask_fns",
            _coerce_support_map(self.support_mask_fns, name="support_mask_fns"),
        )

        object.__setattr__(
            self,
            "Psi_fns",
            _coerce_regime_fn_map(self.Psi_fns, name="Psi_fns"),
        )

        object.__setattr__(
            self,
            "log_Psi_fns",
            _coerce_regime_fn_map(self.log_Psi_fns, name="log_Psi_fns"),
        )

        object.__setattr__(
            self,
            "omega_fns",
            _coerce_regime_fn_map(self.omega_fns, name="omega_fns"),
        )

        object.__setattr__(
            self,
            "log_omega_fns",
            _coerce_regime_fn_map(self.log_omega_fns, name="log_omega_fns"),
        )

        if self.diagnostics is None:
            object.__setattr__(self, "diagnostics", ContinuationDiagnostics())

        if not isinstance(self.source_label, str):
            raise TypeError("source_label must be a string.")

    @property
    def gamma(self) -> float:
        return self.asset_params.gamma

    def is_supported(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
    ) -> Union[bool, np.ndarray]:
        """
        Regime-specific continuation support.
        """
        s = require_regime(s)
        return self.support_mask_fns[s](k, L)

    def is_jointly_supported(
        self,
        k: ArrayLike,
        L: ArrayLike,
    ) -> Union[bool, np.ndarray]:
        """
        Joint support across regimes 0 and 1.
        """
        supp0 = np.asarray(self.is_supported(0, k, L), dtype=bool)
        supp1 = np.asarray(self.is_supported(1, k, L), dtype=bool)
        out = supp0 & supp1
        return _return_bool_like_state(out, k, L)

    def require_supported(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
    ) -> None:
        """
        Raise if any requested state is outside regime-specific support.
        """
        s = require_regime(s)
        supp = np.asarray(self.is_supported(s, k, L), dtype=bool)

        if not bool(np.all(supp)):
            n_total = int(supp.size)
            n_supported = int(np.sum(supp))
            raise ValueError(
                "Requested state lies outside the frozen continuation support: "
                f"regime={s}, n_supported={n_supported}, n_total={n_total}."
            )

    def require_jointly_supported(
        self,
        k: ArrayLike,
        L: ArrayLike,
    ) -> None:
        """
        Raise if any requested state is outside joint support.
        """
        supp = np.asarray(self.is_jointly_supported(k, L), dtype=bool)

        if not bool(np.all(supp)):
            n_total = int(supp.size)
            n_supported = int(np.sum(supp))
            raise ValueError(
                "Requested state lies outside the joint continuation support: "
                f"n_joint_supported={n_supported}, n_total={n_total}."
            )

    def _eval_positive(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        fns: Mapping[int, ContinuationFn],
        *,
        name: str,
        strict: bool,
    ) -> ArrayLike:
        s = require_regime(s)

        if strict:
            self.require_supported(s, k, L)

        out = fns[s](k, L)
        arr = np.asarray(out, dtype=float)

        if not np.all(np.isfinite(arr)):
            raise ValueError(f"{name}_{s} returned non-finite values.")

        if np.any(arr <= 0.0):
            raise ValueError(f"{name}_{s} must be strictly positive.")

        return _return_float_like_state(arr, k, L)

    def _eval_log(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        fns: Mapping[int, ContinuationFn],
        *,
        name: str,
        strict: bool,
    ) -> ArrayLike:
        s = require_regime(s)

        if strict:
            self.require_supported(s, k, L)

        out = fns[s](k, L)
        arr = np.asarray(out, dtype=float)

        if not np.all(np.isfinite(arr)):
            raise ValueError(f"{name}_{s} returned non-finite values.")

        return _return_float_like_state(arr, k, L)

    def Psi(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        *,
        strict: bool = True,
    ) -> ArrayLike:
        return self._eval_positive(
            s,
            k,
            L,
            self.Psi_fns,
            name="Psi",
            strict=strict,
        )

    def log_Psi(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        *,
        strict: bool = True,
    ) -> ArrayLike:
        return self._eval_log(
            s,
            k,
            L,
            self.log_Psi_fns,
            name="log_Psi",
            strict=strict,
        )

    def omega(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        *,
        strict: bool = True,
    ) -> ArrayLike:
        return self._eval_positive(
            s,
            k,
            L,
            self.omega_fns,
            name="omega",
            strict=strict,
        )

    def log_omega(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        *,
        strict: bool = True,
    ) -> ArrayLike:
        return self._eval_log(
            s,
            k,
            L,
            self.log_omega_fns,
            name="log_omega",
            strict=strict,
        )

    def owner_wealth(
        self,
        k: ArrayLike,
        L: ArrayLike,
    ) -> ArrayLike:
        """
        Owner financial wealth:

            W^K = k + L.
        """
        k_b, L_b = _broadcast_state(k, L)
        W = k_b + L_b
        return _return_float_like_state(W, k, L)

    def owner_consumption(
        self,
        s: int,
        k: ArrayLike,
        L: ArrayLike,
        *,
        strict: bool = True,
        require_positive_wealth: bool = True,
    ) -> ArrayLike:
        """
        Frozen owner consumption implied by the continuation bundle:

            C_s^K(k,L) = omega_s(k,L)(k+L).

        This is a continuation object, not a current-control pricing object.
        """
        s = require_regime(s)

        k_b, L_b = _broadcast_state(k, L)
        W = k_b + L_b

        if require_positive_wealth and np.any(W <= 0.0):
            raise ValueError(
                "Owner wealth W^K = k + L must be strictly positive "
                "for owner consumption evaluation."
            )

        omega = np.asarray(self.omega(s, k_b, L_b, strict=strict), dtype=float)
        C = omega * W

        if not np.all(np.isfinite(C)):
            raise ValueError("Owner consumption returned non-finite values.")

        if require_positive_wealth and np.any(C <= 0.0):
            raise ValueError("Owner consumption must be strictly positive.")

        return _return_float_like_state(C, k, L)

    def summary_on_grid(
        self,
        k: ArrayLike,
        L: ArrayLike,
        *,
        strict: bool = False,
    ) -> ContinuationGridSummary:
        """
        Summarize frozen continuation objects on a supplied state grid.
        """
        k_b, L_b = _broadcast_state(k, L)
        W = k_b + L_b

        supp0 = np.asarray(self.is_supported(0, k_b, L_b), dtype=bool)
        supp1 = np.asarray(self.is_supported(1, k_b, L_b), dtype=bool)
        joint = supp0 & supp1

        n_total = int(k_b.size)
        n_supported_0 = int(np.sum(supp0))
        n_supported_1 = int(np.sum(supp1))
        n_joint_supported = int(np.sum(joint))
        n_joint_unsupported = n_total - n_joint_supported

        def minmax_level(method: Callable[..., ArrayLike], s: int) -> Tuple[float, float]:
            vals = np.asarray(method(s, k_b, L_b, strict=strict), dtype=float).reshape(-1)
            mask = joint.reshape(-1) & np.isfinite(vals)

            if not bool(np.any(mask)):
                return math.nan, math.nan

            return float(np.min(vals[mask])), float(np.max(vals[mask]))

        min_omega_0, max_omega_0 = minmax_level(self.omega, 0)
        min_omega_1, max_omega_1 = minmax_level(self.omega, 1)

        min_Psi_0, max_Psi_0 = minmax_level(self.Psi, 0)
        min_Psi_1, max_Psi_1 = minmax_level(self.Psi, 1)

        def min_owner_consumption(s: int) -> float:
            omega = np.asarray(self.omega(s, k_b, L_b, strict=strict), dtype=float)
            C = (omega * W).reshape(-1)
            mask = joint.reshape(-1) & np.isfinite(C)

            if not bool(np.any(mask)):
                return math.nan

            return float(np.min(C[mask]))

        return ContinuationGridSummary(
            n_total=n_total,
            n_supported_0=n_supported_0,
            n_supported_1=n_supported_1,
            n_joint_supported=n_joint_supported,
            n_joint_unsupported=n_joint_unsupported,
            min_omega_0=min_omega_0,
            max_omega_0=max_omega_0,
            min_omega_1=min_omega_1,
            max_omega_1=max_omega_1,
            min_Psi_0=min_Psi_0,
            max_Psi_0=max_Psi_0,
            min_Psi_1=min_Psi_1,
            max_Psi_1=max_Psi_1,
            min_owner_consumption_0=min_owner_consumption(0),
            min_owner_consumption_1=min_owner_consumption(1),
        )


# ============================================================
# Bundle constructors
# ============================================================

def primitive_interior_support(
    k: ArrayLike,
    L: ArrayLike,
    *,
    k_min: float = 0.0,
    wealth_min: float = 0.0,
) -> Union[bool, np.ndarray]:
    """
    Simple default support mask:

        k > k_min,
        k + L > wealth_min.

    This is useful for smoke tests and early oracle harnesses. A real
    continuation solver may use a larger interpolation support D.
    """
    k_b, L_b = _broadcast_state(k, L)

    out = (k_b > float(k_min)) & ((k_b + L_b) > float(wealth_min))

    return _return_bool_like_state(out, k, L)


def build_continuation_bundle_from_log_omega(
    *,
    asset_params: AssetMarketParams,
    support_mask_fns: RegimeSupportMap,
    log_omega_fns: RegimeFnMap,
    diagnostics: Optional[ContinuationDiagnostics] = None,
    source_label: str = "from_log_omega",
    options: Optional[ContinuationOptions] = None,
) -> ContinuationBundle:
    """
    Build a continuation bundle from log omega functions.

    This is the preferred interface if the numerical solver uses log omega as
    the unknown.

    Implied identities:

        omega_s = exp(log_omega_s),
        log_Psi_s = -gamma log_omega_s,
        Psi_s = exp(log_Psi_s).
    """
    if options is None:
        options = ContinuationOptions(variable="log_omega")

    gamma = asset_params.gamma
    clip = options.exp_clip

    log_omega_map = _coerce_regime_fn_map(log_omega_fns, name="log_omega_fns_input")

    def make_log_omega(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            return log_omega_map[s](k, L)
        return fn

    def make_omega(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            log_omega = np.asarray(log_omega_map[s](k, L), dtype=float)
            out = _safe_exp(log_omega, clip=clip)
            return _return_float_like_state(out, k, L)
        return fn

    def make_log_Psi(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            log_omega = np.asarray(log_omega_map[s](k, L), dtype=float)
            out = -gamma * log_omega
            return _return_float_like_state(out, k, L)
        return fn

    def make_Psi(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            log_omega = np.asarray(log_omega_map[s](k, L), dtype=float)
            log_Psi = -gamma * log_omega
            out = _safe_exp(log_Psi, clip=clip)
            return _return_float_like_state(out, k, L)
        return fn

    return ContinuationBundle(
        asset_params=asset_params,
        support_mask_fns=support_mask_fns,
        Psi_fns={0: make_Psi(0), 1: make_Psi(1)},
        log_Psi_fns={0: make_log_Psi(0), 1: make_log_Psi(1)},
        omega_fns={0: make_omega(0), 1: make_omega(1)},
        log_omega_fns={0: make_log_omega(0), 1: make_log_omega(1)},
        diagnostics=diagnostics,
        source_label=source_label,
    )


def build_continuation_bundle_from_omega(
    *,
    asset_params: AssetMarketParams,
    support_mask_fns: RegimeSupportMap,
    omega_fns: RegimeFnMap,
    diagnostics: Optional[ContinuationDiagnostics] = None,
    source_label: str = "from_omega",
    options: Optional[ContinuationOptions] = None,
) -> ContinuationBundle:
    """
    Build a continuation bundle from omega functions.
    """
    if options is None:
        options = ContinuationOptions(variable="log_omega")

    omega_map = _coerce_regime_fn_map(omega_fns, name="omega_fns_input")

    def make_log_omega(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            omega = np.asarray(omega_map[s](k, L), dtype=float)

            if np.any(omega <= options.positivity_floor):
                raise ValueError(
                    f"omega_{s} must exceed positivity_floor="
                    f"{options.positivity_floor}."
                )

            out = np.log(omega)
            return _return_float_like_state(out, k, L)
        return fn

    return build_continuation_bundle_from_log_omega(
        asset_params=asset_params,
        support_mask_fns=support_mask_fns,
        log_omega_fns={0: make_log_omega(0), 1: make_log_omega(1)},
        diagnostics=diagnostics,
        source_label=source_label,
        options=options,
    )


def build_continuation_bundle_from_log_Psi(
    *,
    asset_params: AssetMarketParams,
    support_mask_fns: RegimeSupportMap,
    log_Psi_fns: RegimeFnMap,
    diagnostics: Optional[ContinuationDiagnostics] = None,
    source_label: str = "from_log_Psi",
    options: Optional[ContinuationOptions] = None,
) -> ContinuationBundle:
    """
    Build a continuation bundle from log Psi functions.

    Implied identity:

        log_omega_s = -log_Psi_s / gamma.
    """
    if options is None:
        options = ContinuationOptions(variable="log_Psi")

    gamma = asset_params.gamma
    log_Psi_map = _coerce_regime_fn_map(log_Psi_fns, name="log_Psi_fns_input")

    def make_log_omega(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            log_Psi = np.asarray(log_Psi_map[s](k, L), dtype=float)
            out = -log_Psi / gamma
            return _return_float_like_state(out, k, L)
        return fn

    return build_continuation_bundle_from_log_omega(
        asset_params=asset_params,
        support_mask_fns=support_mask_fns,
        log_omega_fns={0: make_log_omega(0), 1: make_log_omega(1)},
        diagnostics=diagnostics,
        source_label=source_label,
        options=options,
    )


def build_continuation_bundle_from_Psi(
    *,
    asset_params: AssetMarketParams,
    support_mask_fns: RegimeSupportMap,
    Psi_fns: RegimeFnMap,
    diagnostics: Optional[ContinuationDiagnostics] = None,
    source_label: str = "from_Psi",
    options: Optional[ContinuationOptions] = None,
) -> ContinuationBundle:
    """
    Build a continuation bundle from Psi functions.
    """
    if options is None:
        options = ContinuationOptions(variable="log_Psi")

    Psi_map = _coerce_regime_fn_map(Psi_fns, name="Psi_fns_input")

    def make_log_Psi(s: int) -> ContinuationFn:
        def fn(k: ArrayLike, L: ArrayLike) -> ArrayLike:
            Psi = np.asarray(Psi_map[s](k, L), dtype=float)

            if np.any(Psi <= options.positivity_floor):
                raise ValueError(
                    f"Psi_{s} must exceed positivity_floor="
                    f"{options.positivity_floor}."
                )

            out = np.log(Psi)
            return _return_float_like_state(out, k, L)
        return fn

    return build_continuation_bundle_from_log_Psi(
        asset_params=asset_params,
        support_mask_fns=support_mask_fns,
        log_Psi_fns={0: make_log_Psi(0), 1: make_log_Psi(1)},
        diagnostics=diagnostics,
        source_label=source_label,
        options=options,
    )


# ============================================================
# Solver stub
# ============================================================

def solve_continuation_bundle(*args: Any, **kwargs: Any) -> ContinuationBundle:
    """
    Placeholder for the actual Block 4 PDE / false-transient solver.

    The interface above is intentionally complete before the solver is added.
    The eventual solver should return a ContinuationBundle and should preserve:

        - regime 1 solved before regime 0;
        - frozen continuation objects;
        - same gamma as Block 3 AssetMarketParams;
        - no live current-control oracle work inside this block.
    """
    raise NotImplementedError(
        "Block 4 continuation-bundle interface is implemented, but the PDE / "
        "false-transient continuation solver has not been attached yet. "
        "Implement the solver here and return a ContinuationBundle."
    )


# ============================================================
# Test bundle and validation
# ============================================================

def make_test_continuation_bundle(
    *,
    asset_params: Optional[AssetMarketParams] = None,
) -> ContinuationBundle:
    """
    Construct a small analytic continuation bundle for smoke tests.

    This is not an economic solution. It is a deterministic test object that
    validates the frozen-bundle interface and downstream oracle wiring.
    """
    if asset_params is None:
        asset_params = make_infinite_asset_market_params(gamma=5.0)

    def support(k: ArrayLike, L: ArrayLike) -> Union[bool, np.ndarray]:
        return primitive_interior_support(k, L, k_min=0.0, wealth_min=0.0)

    def log_omega_0(k: ArrayLike, L: ArrayLike) -> ArrayLike:
        k_b, L_b = _broadcast_state(k, L)
        # Mild state dependence to test shape handling.
        out = math.log(0.050) + 0.005 * np.log1p(k_b) + 0.002 * np.tanh(L_b)
        return _return_float_like_state(out, k, L)

    def log_omega_1(k: ArrayLike, L: ArrayLike) -> ArrayLike:
        k_b, L_b = _broadcast_state(k, L)
        # Slightly higher post-regime consumption-wealth ratio.
        out = math.log(0.060) + 0.004 * np.log1p(k_b) + 0.001 * np.tanh(L_b)
        return _return_float_like_state(out, k, L)

    diagnostics = ContinuationDiagnostics(
        solver_name="analytic_test_bundle",
        converged=True,
        implemented_solver=False,
        regime_order=(1, 0),
        iterations_1=0,
        iterations_0=0,
        residual_norm_1=0.0,
        residual_norm_0=0.0,
        support_status="analytic_support",
        message="Analytic smoke-test bundle; not an economic continuation solve.",
    )

    return build_continuation_bundle_from_log_omega(
        asset_params=asset_params,
        support_mask_fns=support,
        log_omega_fns={0: log_omega_0, 1: log_omega_1},
        diagnostics=diagnostics,
        source_label="analytic_test_continuation_bundle",
    )


def validate_continuation_bundle(
    bundle: ContinuationBundle,
    *,
    atol: float = 1.0e-10,
    rtol: float = 1.0e-8,
) -> dict[str, float]:
    """
    Validate the Block 4 continuation-bundle contract.

    Checks:
      - asset_params supplies gamma > 0;
      - scalar accessors return scalars;
      - array accessors preserve shape;
      - support masks preserve shape;
      - invalid regime is rejected;
      - strict support rejects the diagonal wall k + L = 0;
      - Psi, log_Psi, omega, and log_omega identities hold;
      - owner consumption equals omega_s(k,L)(k+L);
      - bundle dataclass is frozen;
      - summary diagnostics are finite on supported states.
    """
    if not isinstance(bundle, ContinuationBundle):
        raise TypeError("bundle must be a ContinuationBundle.")

    gamma = bundle.gamma

    if gamma <= 0.0:
        raise RuntimeError("ContinuationBundle gamma must be strictly positive.")

    report: dict[str, float] = {
        "gamma": float(gamma),
    }

    # Scalar tests.
    k0 = 1.0
    L0 = 0.25
    W0 = k0 + L0

    for s in (0, 1):
        psi = bundle.Psi(s, k0, L0)
        log_psi = bundle.log_Psi(s, k0, L0)
        omega = bundle.omega(s, k0, L0)
        log_omega = bundle.log_omega(s, k0, L0)
        Ck = bundle.owner_consumption(s, k0, L0)

        if not np.isscalar(psi):
            raise RuntimeError("Psi scalar evaluation should return a scalar.")

        if not np.isscalar(log_psi):
            raise RuntimeError("log_Psi scalar evaluation should return a scalar.")

        if not np.isscalar(omega):
            raise RuntimeError("omega scalar evaluation should return a scalar.")

        if not np.isscalar(log_omega):
            raise RuntimeError("log_omega scalar evaluation should return a scalar.")

        if psi <= 0.0:
            raise RuntimeError("Psi must be positive.")

        if omega <= 0.0:
            raise RuntimeError("omega must be positive.")

        err_log_omega = abs(math.log(omega) - log_omega)
        err_log_psi = abs(math.log(psi) - log_psi)
        err_identity = abs(log_omega + log_psi / gamma)
        err_consumption = abs(Ck - omega * W0)

        scale = max(1.0, abs(log_omega), abs(log_psi), abs(Ck))
        allowed = atol + rtol * scale

        if err_log_omega > allowed:
            raise RuntimeError(
                f"log omega identity failed in regime {s}: "
                f"error={err_log_omega}, allowed={allowed}."
            )

        if err_log_psi > allowed:
            raise RuntimeError(
                f"log Psi identity failed in regime {s}: "
                f"error={err_log_psi}, allowed={allowed}."
            )

        if err_identity > allowed:
            raise RuntimeError(
                f"omega/Psi homothetic identity failed in regime {s}: "
                f"error={err_identity}, allowed={allowed}."
            )

        if err_consumption > allowed:
            raise RuntimeError(
                f"owner consumption identity failed in regime {s}: "
                f"error={err_consumption}, allowed={allowed}."
            )

        report[f"omega_{s}_scalar"] = float(omega)
        report[f"Psi_{s}_scalar"] = float(psi)
        report[f"Ck_{s}_scalar"] = float(Ck)

    # Array tests.
    k_grid = np.array([0.5, 1.0, 2.0, 4.0])
    L_grid = np.array([0.2, 0.1, 0.5, 1.0])

    for s in (0, 1):
        omega_arr = np.asarray(bundle.omega(s, k_grid, L_grid), dtype=float)
        psi_arr = np.asarray(bundle.Psi(s, k_grid, L_grid), dtype=float)
        supp_arr = np.asarray(bundle.is_supported(s, k_grid, L_grid), dtype=bool)

        if omega_arr.shape != k_grid.shape:
            raise RuntimeError("omega array evaluation did not preserve shape.")

        if psi_arr.shape != k_grid.shape:
            raise RuntimeError("Psi array evaluation did not preserve shape.")

        if supp_arr.shape != k_grid.shape:
            raise RuntimeError("support mask did not preserve shape.")

        if not np.all(supp_arr):
            raise RuntimeError("Positive smoke-test grid should be supported.")

        if np.any(omega_arr <= 0.0):
            raise RuntimeError("omega array contains non-positive values.")

        if np.any(psi_arr <= 0.0):
            raise RuntimeError("Psi array contains non-positive values.")

    # Joint support.
    joint = np.asarray(bundle.is_jointly_supported(k_grid, L_grid), dtype=bool)

    if joint.shape != k_grid.shape:
        raise RuntimeError("joint support mask did not preserve shape.")

    if not np.all(joint):
        raise RuntimeError("Positive smoke-test grid should be jointly supported.")

    # Invalid regime.
    try:
        bundle.omega(2, k0, L0)
    except ValueError:
        pass
    else:
        raise RuntimeError("Invalid regime was not rejected.")

    # Strict support should reject the exact diagonal wall W^K = k + L = 0.
    try:
        bundle.omega(0, 1.0, -1.0, strict=True)
    except ValueError:
        pass
    else:
        raise RuntimeError("Strict continuation support did not reject k+L=0.")

    # Non-strict access can still evaluate test interpolants, useful for diagnostics.
    _ = bundle.omega(0, 1.0, -1.0, strict=False)

    # Owner consumption should reject non-positive wealth by default.
    try:
        bundle.owner_consumption(0, 1.0, -1.0, strict=False)
    except ValueError:
        pass
    else:
        raise RuntimeError("Owner consumption should reject W^K <= 0 by default.")

    # Immutability test.
    try:
        bundle.source_label = "mutated"
    except Exception:
        pass
    else:
        raise RuntimeError("ContinuationBundle should be frozen / immutable.")

    # Summary diagnostics.
    summary = bundle.summary_on_grid(k_grid, L_grid)

    if summary.n_total != k_grid.size:
        raise RuntimeError("ContinuationGridSummary has wrong total count.")

    if summary.n_joint_supported != k_grid.size:
        raise RuntimeError("ContinuationGridSummary has wrong joint-support count.")

    for name in (
        "min_omega_0",
        "min_omega_1",
        "min_Psi_0",
        "min_Psi_1",
        "min_owner_consumption_0",
        "min_owner_consumption_1",
    ):
        val = float(getattr(summary, name))
        if not math.isfinite(val):
            raise RuntimeError(f"ContinuationGridSummary {name} is not finite.")
        report[name] = val

    report["n_joint_supported"] = float(summary.n_joint_supported)
    report["n_total"] = float(summary.n_total)

    return report


def module_smoke_test() -> dict[str, float]:
    """
    Minimal Block 4 self-test.
    """
    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    bundle = make_test_continuation_bundle(asset_params=asset_params)

    return validate_continuation_bundle(bundle)


__all__ = [
    "ArrayLike",
    "ContinuationVariable",
    "ContinuationSolverName",
    "ContinuationFn",
    "SupportFn",
    "RegimeFnMap",
    "RegimeSupportMap",
    "ContinuationOptions",
    "ContinuationDiagnostics",
    "ContinuationGridSummary",
    "ContinuationBundle",
    "require_regime",
    "primitive_interior_support",
    "build_continuation_bundle_from_log_omega",
    "build_continuation_bundle_from_omega",
    "build_continuation_bundle_from_log_Psi",
    "build_continuation_bundle_from_Psi",
    "solve_continuation_bundle",
    "make_test_continuation_bundle",
    "validate_continuation_bundle",
    "module_smoke_test",
]

Overwriting continuation_block.py


In [15]:
import importlib

import asset_market
import continuation_block

importlib.reload(asset_market)
importlib.reload(continuation_block)

block4_report = continuation_block.module_smoke_test()

print("Block 4 validation passed.")
print(block4_report)

Block 4 validation passed.
{'gamma': 5.0, 'omega_0_scalar': 0.05019817034163904, 'Psi_0_scalar': 3137332.5849873037, 'Ck_0_scalar': 0.0627477129270488, 'omega_1_scalar': 0.060181323878565125, 'Psi_1_scalar': 1266751.169893326, 'Ck_1_scalar': 0.07522665484820641, 'min_omega_0': 0.050121250588855235, 'min_omega_1': 0.06010925349395833, 'min_Psi_0': 3050479.8936884548, 'min_Psi_1': 1240539.4146080625, 'min_owner_consumption_0': 0.03508487541219866, 'min_owner_consumption_1': 0.042076477445770824, 'n_joint_supported': 4.0, 'n_total': 4.0}


In [16]:
import numpy as np
import asset_market
import continuation_block

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

C_hat = continuation_block.make_test_continuation_bundle(
    asset_params=asset_params,
)

s = 0
k = 1.0
L = 0.25

omega = C_hat.omega(s, k, L)
Psi = C_hat.Psi(s, k, L)
C_K = C_hat.owner_consumption(s, k, L)

print("omega:", omega)
print("Psi:", Psi)
print("owner consumption:", C_K)

k_grid = np.array([0.5, 1.0, 2.0])
L_grid = np.array([0.1, 0.2, 0.3])

print("\nJoint support:")
print(C_hat.is_jointly_supported(k_grid, L_grid))

print("\nContinuation summary:")
print(C_hat.summary_on_grid(k_grid, L_grid))

omega: 0.05019817034163904
Psi: 3137332.5849873037
owner consumption: 0.0627477129270488

Joint support:
[ True  True  True]

Continuation summary:
ContinuationGridSummary(n_total=3, n_supported_0=3, n_supported_1=3, n_joint_supported=3, n_joint_unsupported=0, min_omega_0=0.05011145711907862, max_omega_0=0.05030470905444763, min_omega_1=0.06010338066646942, max_omega_1=0.06028180542695682, min_Psi_0=3104250.73306763, max_Psi_0=3164571.002440964, min_Psi_1=1256228.7999237746, max_Psi_1=1274986.236564332, min_owner_consumption_0=0.03006687427144717, min_owner_consumption_1=0.03606202839988165)


# Block 5 — price of automation risk

This block computes the price of automation-arrival risk from the frozen continuation bundle

$$
\mathcal C[\hat u].
$$

It consumes the frozen objects

$$
\omega_0^{\hat u}(k,L),
\qquad
\omega_1^{\hat u}(k,L),
\qquad
\log \omega_0^{\hat u}(k,L),
\qquad
\log \omega_1^{\hat u}(k,L),
$$

from Block 4.

The pricing-kernel jump factor is

$$
\chi^{\hat u}(k,L)
=
\left(
\frac{
\omega_1^{\hat u}(k,L)
}{
\omega_0^{\hat u}(k,L)
}
\right)^{-\gamma}.
$$

Equivalently, in log form,

$$
\log \chi^{\hat u}(k,L)
=
-\gamma
\left(
\log \omega_1^{\hat u}(k,L)
-
\log \omega_0^{\hat u}(k,L)
\right).
$$

The risk-neutral automation-arrival intensity is

$$
\lambda^{Q,\hat u}(k,L)
=
\lambda \chi^{\hat u}(k,L).
$$

These objects are continuation / pricing diagnostics.

For hard physical viability,

$$
\boxed{
\chi^{\hat u}
\text{ and }
\lambda^{Q,\hat u}
\text{ do not directly enter the no-default constraint.}
}
$$

The no-default constraint is pathwise. Since automation can arrive at any time, pre-switch feasibility requires remaining inside the post-switch viable set. This uses the physical support of the Poisson event, not the risk-neutral arrival intensity.

This block does not compute:

$$
\pi^{mc},
\qquad
r_f,
\qquad
\dot k,
\qquad
\dot L,
\qquad
\text{tax bases},
\qquad
\text{current fiscal revenue}.
$$

Those belong to the live oracle.

In [17]:
%%writefile automation_risk.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Literal, Optional, Tuple, Union
import math
import numpy as np

from continuation_block import ContinuationBundle, make_test_continuation_bundle
from asset_market import make_infinite_asset_market_params


ArrayLike = Union[float, int, np.ndarray]


# ============================================================
# Block 5 contract
# ============================================================
#
# Inputs:
#   - frozen ContinuationBundle C[u_hat] from Block 4;
#   - physical Poisson arrival intensity lambda > 0;
#   - state locations (k,L).
#
# Outputs:
#   - log pricing-kernel jump factor:
#         log_chi = -gamma * (log_omega_1 - log_omega_0);
#   - pricing-kernel jump factor:
#         chi = exp(log_chi);
#   - risk-neutral arrival intensity:
#         lambda_Q = lambda * chi;
#   - support / overflow / underflow diagnostics.
#
# Forbidden responsibilities:
#   - no continuation solve;
#   - no live current-control oracle;
#   - no pi^{mc};
#   - no r_f;
#   - no kdot or Ldot;
#   - no tax-base or revenue calculation;
#   - no viability peeling;
#   - no Howard iteration;
#   - no physical no-default decision.
#
# Important convention:
#   chi and lambda_Q are pricing / continuation diagnostics. They do not
#   directly enter hard physical no-default constraints.


AutomationRiskStatus = Literal[
    "valid",
    "unsupported_state",
    "nonfinite_log_omega",
    "nonfinite_log_jump_factor",
    "overflow",
    "underflow",
]


# ============================================================
# Shape and scalar helpers
# ============================================================

def _is_scalar_like(x: Any) -> bool:
    return np.ndim(x) == 0


def _is_scalar_state(k: ArrayLike, L: ArrayLike) -> bool:
    return _is_scalar_like(k) and _is_scalar_like(L)


def _broadcast_state(k: ArrayLike, L: ArrayLike) -> Tuple[np.ndarray, np.ndarray]:
    k_arr = np.asarray(k, dtype=float)
    L_arr = np.asarray(L, dtype=float)

    try:
        k_b, L_b = np.broadcast_arrays(k_arr, L_arr)
    except ValueError as exc:
        raise ValueError(
            f"k and L could not be broadcast together: "
            f"k.shape={k_arr.shape}, L.shape={L_arr.shape}."
        ) from exc

    if not np.all(np.isfinite(k_b)):
        raise ValueError("k contains non-finite values.")

    if not np.all(np.isfinite(L_b)):
        raise ValueError("L contains non-finite values.")

    return k_b.astype(float, copy=False), L_b.astype(float, copy=False)


def _return_float_like_state(
    out: np.ndarray,
    k: ArrayLike,
    L: ArrayLike,
) -> ArrayLike:
    out = np.asarray(out, dtype=float)

    if _is_scalar_state(k, L):
        if out.shape == ():
            return float(out)
        if out.size == 1:
            return float(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar state produced non-scalar output with shape {out.shape}."
        )

    return out


def _return_status_like_state(
    out: np.ndarray,
    k: ArrayLike,
    L: ArrayLike,
) -> Union[str, np.ndarray]:
    out = np.asarray(out, dtype=object)

    if _is_scalar_state(k, L):
        if out.shape == ():
            return str(out.item())
        if out.size == 1:
            return str(out.reshape(-1)[0])
        raise ValueError(
            f"Scalar state produced non-scalar status output with shape {out.shape}."
        )

    return out


def _finite_positive_float(x: float, *, name: str) -> float:
    x = float(x)

    if not math.isfinite(x):
        raise ValueError(f"{name} must be finite.")

    if x <= 0.0:
        raise ValueError(f"{name} must be strictly positive.")

    return x


def _finite_float(x: float, *, name: str) -> float:
    x = float(x)

    if not math.isfinite(x):
        raise ValueError(f"{name} must be finite.")

    return x


# ============================================================
# Options and diagnostics
# ============================================================

@dataclass(frozen=True)
class AutomationRiskOptions:
    """
    Numerical options for Block 5.

    strict_support:
        If True, raise when any requested state is outside the joint
        continuation support of regimes 0 and 1.

        If False, unsupported states are returned with status
        "unsupported_state" and NaN numeric diagnostics.

    overflow_clip / underflow_clip:
        Bounds used only for exponentiating log_chi.

        The raw log jump factor is always stored in log_jump_factor.
        The clipped value used for exp is stored separately in
        clipped_log_jump_factor.

    store_clipped_levels:
        If True, chi and lambda_Q are finite even when raw log_chi overflows
        or underflows; the status records "overflow" or "underflow".

        If False, chi and lambda_Q are set to inf or 0 on overflow/underflow.
    """
    strict_support: bool = True

    overflow_clip: float = 700.0
    underflow_clip: float = -700.0

    store_clipped_levels: bool = True

    def __post_init__(self) -> None:
        overflow_clip = _finite_float(self.overflow_clip, name="overflow_clip")
        underflow_clip = _finite_float(self.underflow_clip, name="underflow_clip")

        object.__setattr__(self, "overflow_clip", overflow_clip)
        object.__setattr__(self, "underflow_clip", underflow_clip)

        if not (underflow_clip < overflow_clip):
            raise ValueError("underflow_clip must be strictly below overflow_clip.")


@dataclass(frozen=True)
class AutomationRiskDiagnostics:
    """
    Block 5 output object.

    jump_factor:
        chi^{hat u}(k,L).

    log_jump_factor:
        Raw log chi. This is not clipped.

    clipped_log_jump_factor:
        The value used for exponentiation.

    lambda_Q:
        lambda * chi.

    status:
        Pointwise status.

    The count and min/max fields summarize valid/computable entries.
    """
    jump_factor: ArrayLike
    log_jump_factor: ArrayLike
    clipped_log_jump_factor: ArrayLike
    lambda_Q: ArrayLike
    status: Union[str, np.ndarray]

    physical_lambda: float
    gamma: float

    n_total: int
    n_supported: int
    n_unsupported: int

    n_valid: int
    n_nonfinite_log_omega: int
    n_nonfinite_log_jump_factor: int
    n_overflow: int
    n_underflow: int

    n_computable: int

    min_log_chi: float
    max_log_chi: float
    min_chi: float
    max_chi: float
    min_lambda_Q: float
    max_lambda_Q: float

    share_supported: float
    share_valid: float
    share_overflow: float
    share_underflow: float
    share_unsupported: float

    @property
    def all_valid(self) -> bool:
        return self.n_valid == self.n_total

    @property
    def any_unsupported(self) -> bool:
        return self.n_unsupported > 0

    @property
    def any_overflow_or_underflow(self) -> bool:
        return (self.n_overflow + self.n_underflow) > 0

    @property
    def any_nonfinite(self) -> bool:
        return (self.n_nonfinite_log_omega + self.n_nonfinite_log_jump_factor) > 0


# ============================================================
# Core log-safe computation
# ============================================================

def log_jump_factor_from_log_omega(
    *,
    log_omega_0: ArrayLike,
    log_omega_1: ArrayLike,
    gamma: float,
) -> ArrayLike:
    """
    Compute

        log chi = -gamma * (log_omega_1 - log_omega_0)

    using supplied log omega objects.

    This helper does not evaluate support masks. It is useful for direct unit
    tests and for comparing solver outputs.
    """
    gamma = _finite_positive_float(gamma, name="gamma")

    lw0 = np.asarray(log_omega_0, dtype=float)
    lw1 = np.asarray(log_omega_1, dtype=float)

    try:
        lw0_b, lw1_b = np.broadcast_arrays(lw0, lw1)
    except ValueError as exc:
        raise ValueError(
            "log_omega_0 and log_omega_1 could not be broadcast together."
        ) from exc

    out = -gamma * (lw1_b - lw0_b)

    if _is_scalar_like(log_omega_0) and _is_scalar_like(log_omega_1):
        return float(np.asarray(out).reshape(-1)[0])

    return out


def jump_factor_from_log_jump_factor(
    log_chi: ArrayLike,
    options: Optional[AutomationRiskOptions] = None,
) -> ArrayLike:
    """
    Exponentiate log chi using Block 5 clipping conventions.

    This helper returns the clipped-level chi, not a diagnostic object.
    """
    if options is None:
        options = AutomationRiskOptions()

    log_chi_arr = np.asarray(log_chi, dtype=float)
    clipped = np.clip(
        log_chi_arr,
        options.underflow_clip,
        options.overflow_clip,
    )
    chi = np.exp(clipped)

    if _is_scalar_like(log_chi):
        return float(np.asarray(chi).reshape(-1)[0])

    return chi


def automation_risk_diagnostics(
    continuation: ContinuationBundle,
    *,
    lam: float,
    k: ArrayLike,
    L: ArrayLike,
    options: Optional[AutomationRiskOptions] = None,
) -> AutomationRiskDiagnostics:
    """
    Compute Block 5 automation-risk diagnostics from a frozen continuation bundle.

    The core formulas are:

        log_chi = -gamma * (log_omega_1 - log_omega_0),

        chi = exp(log_chi),

        lambda_Q = lambda * chi.

    The function evaluates only frozen Block 4 continuation objects. It does
    not compute live prices, controls, drifts, or viability.
    """
    if not isinstance(continuation, ContinuationBundle):
        raise TypeError("continuation must be a ContinuationBundle.")

    lam = _finite_positive_float(lam, name="lam")

    if options is None:
        options = AutomationRiskOptions()

    gamma = continuation.gamma
    gamma = _finite_positive_float(gamma, name="continuation.gamma")

    k_b, L_b = _broadcast_state(k, L)
    shape = k_b.shape
    n_total = int(k_b.size)

    # Joint continuation support.
    supp0 = np.asarray(continuation.is_supported(0, k_b, L_b), dtype=bool)
    supp1 = np.asarray(continuation.is_supported(1, k_b, L_b), dtype=bool)

    try:
        supp0, supp1 = np.broadcast_arrays(supp0, supp1)
    except ValueError as exc:
        raise ValueError("Continuation support masks could not be broadcast.") from exc

    joint_supported = supp0 & supp1

    if options.strict_support and not bool(np.all(joint_supported)):
        n_supported = int(np.sum(joint_supported))
        raise ValueError(
            "Requested states fall outside the joint continuation support: "
            f"n_supported={n_supported}, n_total={n_total}."
        )

    status = np.full(shape, "unsupported_state", dtype=object)

    log_chi = np.full(shape, math.nan, dtype=float)
    clipped_log_chi = np.full(shape, math.nan, dtype=float)
    chi = np.full(shape, math.nan, dtype=float)
    lambda_Q = np.full(shape, math.nan, dtype=float)

    k_flat = k_b.reshape(-1)
    L_flat = L_b.reshape(-1)

    joint_flat = joint_supported.reshape(-1)
    status_flat = status.reshape(-1)

    log_chi_flat = log_chi.reshape(-1)
    clipped_log_chi_flat = clipped_log_chi.reshape(-1)
    chi_flat = chi.reshape(-1)
    lambda_Q_flat = lambda_Q.reshape(-1)

    if bool(np.any(joint_flat)):
        k_eval = k_flat[joint_flat]
        L_eval = L_flat[joint_flat]

        # Evaluate only on supported states. This avoids accidental
        # extrapolation by interpolation objects outside their support.
        lw0 = np.asarray(
            continuation.log_omega(0, k_eval, L_eval, strict=False),
            dtype=float,
        ).reshape(-1)

        lw1 = np.asarray(
            continuation.log_omega(1, k_eval, L_eval, strict=False),
            dtype=float,
        ).reshape(-1)

        if lw0.size != k_eval.size or lw1.size != k_eval.size:
            raise ValueError(
                "Continuation log_omega evaluations did not return the "
                "expected flattened size."
            )

        eval_indices = np.flatnonzero(joint_flat)

        log_omega_finite = np.isfinite(lw0) & np.isfinite(lw1)

        nonfinite_log_omega_indices = eval_indices[~log_omega_finite]
        status_flat[nonfinite_log_omega_indices] = "nonfinite_log_omega"

        good_indices = eval_indices[log_omega_finite]

        if good_indices.size > 0:
            raw_log_chi_good = -gamma * (
                lw1[log_omega_finite] - lw0[log_omega_finite]
            )

            finite_log_chi = np.isfinite(raw_log_chi_good)

            nonfinite_log_chi_indices = good_indices[~finite_log_chi]
            status_flat[nonfinite_log_chi_indices] = "nonfinite_log_jump_factor"

            finite_indices = good_indices[finite_log_chi]
            raw_log_chi_finite = raw_log_chi_good[finite_log_chi]

            if finite_indices.size > 0:
                overflow = raw_log_chi_finite > options.overflow_clip
                underflow = raw_log_chi_finite < options.underflow_clip
                valid = ~(overflow | underflow)

                overflow_indices = finite_indices[overflow]
                underflow_indices = finite_indices[underflow]
                valid_indices = finite_indices[valid]

                status_flat[overflow_indices] = "overflow"
                status_flat[underflow_indices] = "underflow"
                status_flat[valid_indices] = "valid"

                log_chi_flat[finite_indices] = raw_log_chi_finite

                clipped_vals = np.clip(
                    raw_log_chi_finite,
                    options.underflow_clip,
                    options.overflow_clip,
                )
                clipped_log_chi_flat[finite_indices] = clipped_vals

                if options.store_clipped_levels:
                    chi_vals = np.exp(clipped_vals)
                else:
                    chi_vals = np.empty_like(raw_log_chi_finite)
                    chi_vals[valid] = np.exp(raw_log_chi_finite[valid])
                    chi_vals[overflow] = math.inf
                    chi_vals[underflow] = 0.0

                chi_flat[finite_indices] = chi_vals
                lambda_Q_flat[finite_indices] = lam * chi_vals

    # Counts.
    status_arr = status.reshape(shape)

    n_supported = int(np.sum(joint_supported))
    n_unsupported = int(np.sum(status_arr == "unsupported_state"))

    n_valid = int(np.sum(status_arr == "valid"))
    n_nonfinite_log_omega = int(np.sum(status_arr == "nonfinite_log_omega"))
    n_nonfinite_log_jump_factor = int(np.sum(status_arr == "nonfinite_log_jump_factor"))
    n_overflow = int(np.sum(status_arr == "overflow"))
    n_underflow = int(np.sum(status_arr == "underflow"))

    computable_mask = (
        (status_arr == "valid")
        | (status_arr == "overflow")
        | (status_arr == "underflow")
    )
    n_computable = int(np.sum(computable_mask))

    if n_computable > 0:
        log_vals = log_chi[computable_mask]
        chi_vals = chi[computable_mask]
        lq_vals = lambda_Q[computable_mask]

        min_log_chi = float(np.nanmin(log_vals))
        max_log_chi = float(np.nanmax(log_vals))
        min_chi = float(np.nanmin(chi_vals))
        max_chi = float(np.nanmax(chi_vals))
        min_lambda_Q = float(np.nanmin(lq_vals))
        max_lambda_Q = float(np.nanmax(lq_vals))
    else:
        min_log_chi = math.nan
        max_log_chi = math.nan
        min_chi = math.nan
        max_chi = math.nan
        min_lambda_Q = math.nan
        max_lambda_Q = math.nan

    return AutomationRiskDiagnostics(
        jump_factor=_return_float_like_state(chi, k, L),
        log_jump_factor=_return_float_like_state(log_chi, k, L),
        clipped_log_jump_factor=_return_float_like_state(clipped_log_chi, k, L),
        lambda_Q=_return_float_like_state(lambda_Q, k, L),
        status=_return_status_like_state(status_arr, k, L),
        physical_lambda=float(lam),
        gamma=float(gamma),
        n_total=n_total,
        n_supported=n_supported,
        n_unsupported=n_unsupported,
        n_valid=n_valid,
        n_nonfinite_log_omega=n_nonfinite_log_omega,
        n_nonfinite_log_jump_factor=n_nonfinite_log_jump_factor,
        n_overflow=n_overflow,
        n_underflow=n_underflow,
        n_computable=n_computable,
        min_log_chi=min_log_chi,
        max_log_chi=max_log_chi,
        min_chi=min_chi,
        max_chi=max_chi,
        min_lambda_Q=min_lambda_Q,
        max_lambda_Q=max_lambda_Q,
        share_supported=float(n_supported / n_total),
        share_valid=float(n_valid / n_total),
        share_overflow=float(n_overflow / n_total),
        share_underflow=float(n_underflow / n_total),
        share_unsupported=float(n_unsupported / n_total),
    )


def require_valid_automation_risk(
    continuation: ContinuationBundle,
    *,
    lam: float,
    k: ArrayLike,
    L: ArrayLike,
    options: Optional[AutomationRiskOptions] = None,
) -> AutomationRiskDiagnostics:
    """
    Require every requested state to have status "valid".

    This is useful for tests or for code paths that deliberately reject
    overflow/underflow rather than treating them as clipped diagnostics.
    """
    diag = automation_risk_diagnostics(
        continuation=continuation,
        lam=lam,
        k=k,
        L=L,
        options=options,
    )

    if not diag.all_valid:
        raise ValueError(
            "Automation-risk diagnostics are not all valid: "
            f"n_total={diag.n_total}, n_valid={diag.n_valid}, "
            f"n_unsupported={diag.n_unsupported}, "
            f"n_overflow={diag.n_overflow}, n_underflow={diag.n_underflow}, "
            f"n_nonfinite_log_omega={diag.n_nonfinite_log_omega}, "
            f"n_nonfinite_log_jump_factor={diag.n_nonfinite_log_jump_factor}."
        )

    return diag


# ============================================================
# Validation
# ============================================================

def validate_automation_risk_layer(
    continuation: ContinuationBundle,
    *,
    lam: float,
    atol: float = 1.0e-10,
    rtol: float = 1.0e-8,
) -> dict[str, float]:
    """
    Validate Block 5.

    Checks:
      - scalar output types;
      - array shape preservation;
      - log formula matches direct computation;
      - lambda_Q = lambda * chi;
      - strict support rejects unsupported states;
      - non-strict support reports unsupported states without evaluation;
      - overflow / underflow masks are full-size and correct;
      - raw log_chi is stored separately from clipped_log_chi;
      - invalid lambda is rejected.
    """
    lam = _finite_positive_float(lam, name="lam")

    if not isinstance(continuation, ContinuationBundle):
        raise TypeError("continuation must be a ContinuationBundle.")

    report: dict[str, float] = {
        "gamma": float(continuation.gamma),
        "lambda": float(lam),
    }

    # --------------------------------------------------------
    # Scalar test.
    # --------------------------------------------------------
    k0 = 1.0
    L0 = 0.25

    diag_scalar = automation_risk_diagnostics(
        continuation=continuation,
        lam=lam,
        k=k0,
        L=L0,
    )

    if not isinstance(diag_scalar.jump_factor, float):
        raise RuntimeError("Scalar jump_factor should be a Python float.")

    if not isinstance(diag_scalar.log_jump_factor, float):
        raise RuntimeError("Scalar log_jump_factor should be a Python float.")

    if not isinstance(diag_scalar.lambda_Q, float):
        raise RuntimeError("Scalar lambda_Q should be a Python float.")

    if not isinstance(diag_scalar.status, str):
        raise RuntimeError("Scalar status should be a string.")

    if diag_scalar.status != "valid":
        raise RuntimeError(f"Scalar diagnostic should be valid, got {diag_scalar.status}.")

    log_omega_0 = continuation.log_omega(0, k0, L0)
    log_omega_1 = continuation.log_omega(1, k0, L0)

    expected_log_chi = -continuation.gamma * (log_omega_1 - log_omega_0)
    expected_chi = math.exp(
        min(
            max(expected_log_chi, AutomationRiskOptions().underflow_clip),
            AutomationRiskOptions().overflow_clip,
        )
    )
    expected_lambda_Q = lam * expected_chi

    if abs(diag_scalar.log_jump_factor - expected_log_chi) > atol + rtol * max(1.0, abs(expected_log_chi)):
        raise RuntimeError("Scalar log jump factor formula failed.")

    if abs(diag_scalar.jump_factor - expected_chi) > atol + rtol * max(1.0, abs(expected_chi)):
        raise RuntimeError("Scalar jump factor formula failed.")

    if abs(diag_scalar.lambda_Q - expected_lambda_Q) > atol + rtol * max(1.0, abs(expected_lambda_Q)):
        raise RuntimeError("Scalar lambda_Q formula failed.")

    report["scalar_log_chi"] = float(diag_scalar.log_jump_factor)
    report["scalar_chi"] = float(diag_scalar.jump_factor)
    report["scalar_lambda_Q"] = float(diag_scalar.lambda_Q)

    # --------------------------------------------------------
    # Array shape test.
    # --------------------------------------------------------
    k_grid = np.array([0.5, 1.0, 2.0, 4.0])
    L_grid = np.array([0.2, 0.1, 0.5, 1.0])

    diag_grid = automation_risk_diagnostics(
        continuation=continuation,
        lam=lam,
        k=k_grid,
        L=L_grid,
    )

    for name in ("jump_factor", "log_jump_factor", "clipped_log_jump_factor", "lambda_Q"):
        arr = np.asarray(getattr(diag_grid, name), dtype=float)
        if arr.shape != k_grid.shape:
            raise RuntimeError(f"{name} did not preserve array shape.")

    status_arr = np.asarray(diag_grid.status, dtype=object)

    if status_arr.shape != k_grid.shape:
        raise RuntimeError("status did not preserve array shape.")

    if not diag_grid.all_valid:
        raise RuntimeError("Positive smoke-test grid should be valid.")

    if not np.allclose(
        np.asarray(diag_grid.lambda_Q, dtype=float),
        lam * np.asarray(diag_grid.jump_factor, dtype=float),
        atol=atol,
        rtol=rtol,
    ):
        raise RuntimeError("Grid lambda_Q = lambda * chi identity failed.")

    report["grid_min_chi"] = float(diag_grid.min_chi)
    report["grid_max_chi"] = float(diag_grid.max_chi)
    report["grid_min_lambda_Q"] = float(diag_grid.min_lambda_Q)
    report["grid_max_lambda_Q"] = float(diag_grid.max_lambda_Q)

    # --------------------------------------------------------
    # Strict support should reject the diagonal wall.
    # --------------------------------------------------------
    try:
        automation_risk_diagnostics(
            continuation=continuation,
            lam=lam,
            k=1.0,
            L=-1.0,
            options=AutomationRiskOptions(strict_support=True),
        )
    except ValueError:
        pass
    else:
        raise RuntimeError("Strict support should reject k + L = 0.")

    # Non-strict support should report unsupported state.
    diag_unsupported = automation_risk_diagnostics(
        continuation=continuation,
        lam=lam,
        k=np.array([1.0, 1.0]),
        L=np.array([0.1, -1.0]),
        options=AutomationRiskOptions(strict_support=False),
    )

    status_unsupported = np.asarray(diag_unsupported.status, dtype=object)

    if status_unsupported[0] != "valid":
        raise RuntimeError("First non-strict support test point should be valid.")

    if status_unsupported[1] != "unsupported_state":
        raise RuntimeError("Second non-strict support test point should be unsupported.")

    if not math.isnan(np.asarray(diag_unsupported.jump_factor, dtype=float)[1]):
        raise RuntimeError("Unsupported state should have NaN jump_factor.")

    report["unsupported_count"] = float(diag_unsupported.n_unsupported)

    # --------------------------------------------------------
    # Overflow / underflow branch test with vectorized full-size masks.
    # --------------------------------------------------------
    asset_params = make_infinite_asset_market_params(gamma=5.0)

    def support_all(k: ArrayLike, L: ArrayLike) -> Union[bool, np.ndarray]:
        k_b, L_b = _broadcast_state(k, L)
        out = np.ones(k_b.shape, dtype=bool)
        if _is_scalar_state(k, L):
            return bool(out.reshape(-1)[0])
        return out

    def log_omega_0_extreme(k: ArrayLike, L: ArrayLike) -> ArrayLike:
        k_b, _ = _broadcast_state(k, L)
        out = np.zeros(k_b.shape, dtype=float)
        return _return_float_like_state(out, k, L)

    def log_omega_1_extreme(k: ArrayLike, L: ArrayLike) -> ArrayLike:
        k_b, _ = _broadcast_state(k, L)
        # k < 1.5 gives log_chi = -gamma * (-200) = +1000 -> overflow.
        # k >= 1.5 gives log_chi = -gamma * (+200) = -1000 -> underflow.
        out = np.where(k_b < 1.5, -200.0, 200.0)
        return _return_float_like_state(out, k, L)

    from continuation_block import build_continuation_bundle_from_log_omega

    extreme_bundle = build_continuation_bundle_from_log_omega(
        asset_params=asset_params,
        support_mask_fns=support_all,
        log_omega_fns={
            0: log_omega_0_extreme,
            1: log_omega_1_extreme,
        },
        source_label="automation_risk_extreme_test_bundle",
    )

    extreme_diag = automation_risk_diagnostics(
        continuation=extreme_bundle,
        lam=lam,
        k=np.array([1.0, 2.0]),
        L=np.array([0.0, 0.0]),
        options=AutomationRiskOptions(
            strict_support=True,
            overflow_clip=700.0,
            underflow_clip=-700.0,
            store_clipped_levels=True,
        ),
    )

    extreme_status = np.asarray(extreme_diag.status, dtype=object)

    if extreme_status[0] != "overflow":
        raise RuntimeError(f"Expected overflow status, got {extreme_status[0]}.")

    if extreme_status[1] != "underflow":
        raise RuntimeError(f"Expected underflow status, got {extreme_status[1]}.")

    extreme_log = np.asarray(extreme_diag.log_jump_factor, dtype=float)
    extreme_clipped_log = np.asarray(extreme_diag.clipped_log_jump_factor, dtype=float)

    if not np.allclose(extreme_log, np.array([1000.0, -1000.0])):
        raise RuntimeError("Raw log_chi should store unclipped extreme values.")

    if not np.allclose(extreme_clipped_log, np.array([700.0, -700.0])):
        raise RuntimeError("clipped_log_jump_factor should store clipped values.")

    if extreme_diag.n_overflow != 1 or extreme_diag.n_underflow != 1:
        raise RuntimeError("Overflow/underflow counts failed.")

    report["extreme_n_overflow"] = float(extreme_diag.n_overflow)
    report["extreme_n_underflow"] = float(extreme_diag.n_underflow)
    report["extreme_max_log_chi"] = float(extreme_diag.max_log_chi)
    report["extreme_min_log_chi"] = float(extreme_diag.min_log_chi)

    # --------------------------------------------------------
    # Invalid lambda should be rejected.
    # --------------------------------------------------------
    for bad_lam in (0.0, -1.0, math.nan, math.inf):
        try:
            automation_risk_diagnostics(
                continuation=continuation,
                lam=bad_lam,
                k=k0,
                L=L0,
            )
        except ValueError:
            pass
        else:
            raise RuntimeError(f"Invalid lambda was not rejected: {bad_lam}.")

    report["invalid_lambda_rejections"] = 4.0

    return report


def module_smoke_test() -> dict[str, float]:
    """
    Minimal Block 5 smoke test.
    """
    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    continuation = make_test_continuation_bundle(
        asset_params=asset_params,
    )

    return validate_automation_risk_layer(
        continuation=continuation,
        lam=0.10,
    )


__all__ = [
    "ArrayLike",
    "AutomationRiskStatus",
    "AutomationRiskOptions",
    "AutomationRiskDiagnostics",
    "log_jump_factor_from_log_omega",
    "jump_factor_from_log_jump_factor",
    "automation_risk_diagnostics",
    "require_valid_automation_risk",
    "validate_automation_risk_layer",
    "module_smoke_test",
]

Overwriting automation_risk.py


In [18]:
import importlib

import asset_market
import continuation_block
import automation_risk

importlib.reload(asset_market)
importlib.reload(continuation_block)
importlib.reload(automation_risk)

block5_report = automation_risk.module_smoke_test()

print("Block 5 validation passed.")
print(block5_report)

Block 5 validation passed.
{'gamma': 5.0, 'lambda': 0.1, 'scalar_log_chi': -0.9069174547549541, 'scalar_chi': 0.4037669375427252, 'scalar_lambda_Q': 0.040376693754272525, 'grid_min_chi': 0.4030907397025282, 'grid_max_chi': 0.4066702479091181, 'grid_min_lambda_Q': 0.04030907397025282, 'grid_max_lambda_Q': 0.04066702479091181, 'unsupported_count': 1.0, 'extreme_n_overflow': 1.0, 'extreme_n_underflow': 1.0, 'extreme_max_log_chi': 1000.0, 'extreme_min_log_chi': -1000.0, 'invalid_lambda_rejections': 4.0}


In [19]:
import numpy as np

import asset_market
import continuation_block
import automation_risk

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

C_hat = continuation_block.make_test_continuation_bundle(
    asset_params=asset_params,
)

risk_diag = automation_risk.automation_risk_diagnostics(
    continuation=C_hat,
    lam=0.10,
    k=1.0,
    L=0.25,
)

print("Scalar automation-risk diagnostic:")
print(risk_diag)


k_grid = np.array([0.5, 1.0, 2.0, 4.0])
L_grid = np.array([0.2, 0.1, 0.5, 1.0])

risk_grid_diag = automation_risk.automation_risk_diagnostics(
    continuation=C_hat,
    lam=0.10,
    k=k_grid,
    L=L_grid,
)

print("\nGrid jump factor:")
print(risk_grid_diag.jump_factor)

print("\nGrid lambda_Q:")
print(risk_grid_diag.lambda_Q)

print("\nGrid status:")
print(risk_grid_diag.status)

print("\nGrid summary:")
print(risk_grid_diag)

Scalar automation-risk diagnostic:
AutomationRiskDiagnostics(jump_factor=0.4037669375427252, log_jump_factor=-0.9069174547549541, clipped_log_jump_factor=-0.9069174547549541, lambda_Q=0.040376693754272525, status='valid', physical_lambda=0.1, gamma=5.0, n_total=1, n_supported=1, n_unsupported=0, n_valid=1, n_nonfinite_log_omega=0, n_nonfinite_log_jump_factor=0, n_overflow=0, n_underflow=0, n_computable=1, min_log_chi=-0.9069174547549541, max_log_chi=-0.9069174547549541, min_chi=0.4037669375427252, max_chi=0.4037669375427252, min_lambda_Q=0.040376693754272525, max_lambda_Q=0.040376693754272525, share_supported=1.0, share_valid=1.0, share_overflow=0.0, share_underflow=0.0, share_unsupported=0.0)

Grid jump factor:
[0.40309074 0.40347381 0.40502595 0.40667025]

Grid lambda_Q:
[0.04030907 0.04034738 0.0405026  0.04066702]

Grid status:
['valid' 'valid' 'valid' 'valid']

Grid summary:
AutomationRiskDiagnostics(jump_factor=array([0.40309074, 0.40347381, 0.40502595, 0.40667025]), log_jump_fac

# Block 6 — live current-control oracle

The live oracle is

$$
\mathcal O_s(x,u;\mathcal G,\mathcal C[\hat u]).
$$

It takes

$$
s,
\qquad
x=(k,L),
\qquad
u=(\tau,T,H),
\qquad
\mathcal G,
\qquad
\mathcal C[\hat u],
$$

and returns current objects evaluated at the current candidate control.

The central invariant is:

$$
\boxed{
\text{freeze continuation objects, but evaluate current pricing, fiscal objects, and drifts live.}
}
$$

So the oracle consumes frozen continuation objects such as

$$
\omega_s^{\hat u}(k,L)
$$

from Block 4, but it computes current objects such as

$$
\pi^{mc},
\qquad
r_f,
\qquad
\dot k,
\qquad
\dot L,
\qquad
\text{tax bases},
\qquad
\text{revenue}
$$

live at the current candidate control

$$
u=(\tau,T,H).
$$

The balance-sheet identities are

$$
W^K=k+L,
$$

$$
B=L+H,
$$

and

$$
E^{priv}=k-H.
$$

When

$$
W^K>0,
$$

the market-clearing risky share is

$$
\pi^{mc}(k,L,H)
=
\frac{k-H}{k+L}
=
\frac{E^{priv}}{W^K}.
$$

If the portfolio-interiority check passes, the interior Merton branch gives the safe rate

$$
r_{f,s}(k,L;H,\tau)
=
r_s^k(k)
-
\gamma(1-\tau)
\left(\sigma_s^K(k)\right)^2
\pi^{mc}(k,L,H).
$$

The oracle then returns worker consumption

$$
C_s^W=w_s(k)+T,
$$

owner consumption

$$
C_s^K
=
\omega_s^{\hat u}(k,L)(k+L),
$$

the capital drift

$$
\dot k_s^{\hat u}(x;u)
=
Y_s(k)
-
\bigl(w_s(k)+T\bigr)
-
\omega_s^{\hat u}(k,L)(k+L)
-
(\delta+g)k,
$$

and the fiscal-state drift

$$
\dot L_s^{\hat u}(x;u)
=
r_{f,s}(k,L;H,\tau)(L+H)
+
T
-
Hr_s^k(k)
-
\tau
\left[
(k-H)r_s^k(k)
+
r_{f,s}(k,L;H,\tau)(L+H)
\right].
$$

It also returns

$$
\dot W_s^{K,\hat u}(x;u)
=
\dot k_s^{\hat u}(x;u)
+
\dot L_s^{\hat u}(x;u).
$$

The oracle status set is

```text
status in {
    "interior",
    "k_wall",
    "diagonal_wall",
    "corner",
    "portfolio_bind",
    "invalid"
}

In [20]:
%%writefile equilibrium_oracle.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Literal, Optional, Tuple, Union
import math
import numpy as np

from automation_block import (
    RegimePrimitives,
    AutomationParams,
    build_regime_primitives,
)
from model.economy import (
    State,
    Control,
    PlannerEconomyParams,
    StateDiagnostics,
    BalanceSheet,
    primitive_state_diagnostics,
    balance_sheet,
)
import policy_sets
from policy_sets import PolicySetOptions
from asset_market import (
    AssetMarketParams,
    PortfolioCheck,
    check_portfolio_share,
    make_infinite_asset_market_params,
)
from continuation_block import (
    ContinuationBundle,
    make_test_continuation_bundle,
)


ArrayLike = Union[float, int, np.ndarray]


# ============================================================
# Block 6 contract
# ============================================================
#
# Inputs:
#   - regime s in {0, 1};
#   - scalar planner state x = (k, L);
#   - scalar current control u = (tau, T, H);
#   - regime-primitives bundle G from Block 0;
#   - frozen continuation bundle C[u_hat] from Block 4;
#   - asset-market parameters from Block 3;
#   - primitive economy parameters from Block 1;
#   - policy-set options from Block 2.
#
# Outputs:
#   - live current-control accounting:
#       W_K, B, E_priv;
#   - live market-clearing risky share pi_mc;
#   - live Merton safe rate r_f on the interior branch;
#   - worker consumption C_W;
#   - owner consumption C_K from frozen omega_s^{hat u};
#   - live tax bases, revenue, and fiscal objects;
#   - live drifts k_dot, L_dot, W_K_dot;
#   - oracle status and diagnostics.
#
# Forbidden responsibilities:
#   - no continuation solve;
#   - no recomputation of omega_s;
#   - no automation-risk recomputation;
#   - no viability peeling;
#   - no planner KKT solve;
#   - no Howard iteration;
#   - no outer fixed point.
#
# Important convention:
#   The continuation environment is frozen, but current prices and drifts are
#   live at the current candidate control u = (tau, T, H).


OracleStatus = Literal[
    "interior",
    "k_wall",
    "diagonal_wall",
    "corner",
    "portfolio_bind",
    "invalid",
]

OracleControlSet = Literal[
    "full",
    "compact",
    "none",
]


# ============================================================
# Helpers
# ============================================================

def _require_regime(s: int) -> int:
    if s not in (0, 1):
        raise ValueError("Regime s must be 0 or 1.")
    return int(s)


def _finite_float(x: float, *, name: str) -> float:
    x = float(x)
    if not math.isfinite(x):
        raise ValueError(f"{name} must be finite.")
    return x


def _positive_float(x: float, *, name: str) -> float:
    x = _finite_float(x, name=name)
    if x <= 0.0:
        raise ValueError(f"{name} must be strictly positive.")
    return x


def _nonnegative_float(x: float, *, name: str) -> float:
    x = _finite_float(x, name=name)
    if x < 0.0:
        raise ValueError(f"{name} must be nonnegative.")
    return x


def _all_finite(values: Tuple[float, ...]) -> bool:
    return all(math.isfinite(float(v)) for v in values)


def _safe_tuple(x: Any) -> Tuple[str, ...]:
    if x is None:
        return tuple()
    return tuple(str(v) for v in x)


# ============================================================
# Options
# ============================================================

@dataclass(frozen=True)
class OracleOptions:
    """
    Numerical and branch options for the live oracle.

    control_set:
        "full":
            Check u against U_s^{full}(x).

        "compact":
            Check u against U_s^M(x).

        "none":
            Do not check policy-set admissibility inside the oracle.
            This is useful only for low-level debugging.

    strict_continuation_support:
        If True, interior owner-consumption evaluation requires continuation
        support in Block 4.

    raise_on_invalid_state / raise_on_invalid_control:
        If True, domain failures raise instead of returning status="invalid".

    allow_diagonal_boundary_drift:
        If True, the oracle computes unsimplified boundary drifts on the exact
        diagonal wall k + L = 0 without evaluating pi_mc or r_f.

    allow_corner_boundary_drift:
        If True, the oracle computes the analytic corner convention:
            Y = w = C_K = 0,
            k_dot = -T,
            L_dot = T.

    require_worker_consumption_positive:
        If True, C_W must be strictly positive whenever it is evaluated.

    finite_tol:
        Tolerance for identity diagnostics only. It must not create artificial
        boundary branches.
    """
    control_set: OracleControlSet = "full"
    strict_continuation_support: bool = True

    raise_on_invalid_state: bool = False
    raise_on_invalid_control: bool = False

    allow_diagonal_boundary_drift: bool = True
    allow_corner_boundary_drift: bool = True

    require_worker_consumption_positive: bool = True

    finite_tol: float = 1.0e-10

    def __post_init__(self) -> None:
        if self.control_set not in ("full", "compact", "none"):
            raise ValueError("control_set must be 'full', 'compact', or 'none'.")
        _nonnegative_float(self.finite_tol, name="finite_tol")


# ============================================================
# Current primitive objects
# ============================================================

@dataclass(frozen=True)
class PrimitiveCurrentObjects:
    """
    Regime-primitives evaluated at the current state.

    These come only from Block 0. The oracle does not reconstruct production.
    """
    Y: float
    w: float
    r_k: float
    sigma_K: float
    delta: float
    g: float


def evaluate_current_primitives(
    s: int,
    k: float,
    primitives: RegimePrimitives,
) -> PrimitiveCurrentObjects:
    """
    Evaluate Block 0 primitives at k > 0.

    This deliberately calls strict Block 0 schedules. Boundary conventions at
    k = 0 are handled by named oracle branches, not by clipping k here.
    """
    s = _require_regime(s)
    k = _positive_float(k, name="k")

    Y = float(primitives.Y(s, k))
    w = float(primitives.w(s, k))
    r_k = float(primitives.rk(s, k))
    sigma_K = float(primitives.sigmaK(s, k))

    delta = float(primitives.params.delta)
    g = float(primitives.params.g)

    if not _all_finite((Y, w, r_k, sigma_K, delta, g)):
        raise ValueError("Current primitive evaluation returned non-finite values.")

    if Y <= 0.0:
        raise ValueError("Y_s(k) must be positive for k > 0.")

    if w <= 0.0:
        raise ValueError("w_s(k) must be positive for k > 0.")

    if sigma_K < 0.0:
        raise ValueError("sigma_K must be nonnegative.")

    return PrimitiveCurrentObjects(
        Y=Y,
        w=w,
        r_k=r_k,
        sigma_K=sigma_K,
        delta=delta,
        g=g,
    )


# ============================================================
# Market clearing and pricing
# ============================================================

def market_clearing_risky_share(
    x: State,
    u: Control,
) -> float:
    """
    Live market-clearing risky share:

        pi_mc = (k - H) / (k + L).

    This function requires W_K = k + L > 0.
    It must not be called on the exact diagonal wall.
    """
    W_K = x.k + x.L

    if W_K <= 0.0:
        raise ValueError(
            "Cannot compute pi_mc unless W_K = k + L is strictly positive."
        )

    pi = (x.k - u.H) / W_K

    if not math.isfinite(pi):
        raise ValueError("pi_mc is non-finite.")

    return float(pi)


def merton_safe_rate(
    *,
    r_k: float,
    sigma_K: float,
    pi_mc: float,
    tau: float,
    asset_params: AssetMarketParams,
) -> float:
    """
    Interior Merton safe rate:

        r_f = r_k - gamma (1 - tau) sigma_K^2 pi_mc.
    """
    r_k = _finite_float(r_k, name="r_k")
    sigma_K = _finite_float(sigma_K, name="sigma_K")
    pi_mc = _finite_float(pi_mc, name="pi_mc")
    tau = _finite_float(tau, name="tau")

    if sigma_K < 0.0:
        raise ValueError("sigma_K must be nonnegative.")

    if not (0.0 <= tau < 1.0):
        raise ValueError("tau must lie in [0,1) for the Merton safe-rate formula.")

    return float(
        r_k
        - asset_params.gamma * (1.0 - tau) * (sigma_K ** 2) * pi_mc
    )


# ============================================================
# Main oracle output
# ============================================================

@dataclass(frozen=True)
class OracleEval:
    """
    Live oracle evaluation at one scalar state-control pair.

    valid_for_pricing:
        True only for the interior Merton branch.

    valid_for_drift:
        True when k_dot, L_dot, and W_K_dot are finite under the branch.

    status:
        Main branch flag.

    portfolio_status:
        Side-specific portfolio status from Block 3 when relevant.
    """
    regime: int
    state: State
    control: Control

    status: OracleStatus
    valid_for_pricing: bool
    valid_for_drift: bool
    reason: Optional[str]

    state_status: str
    state_is_valid: bool
    state_invalid_reason: Optional[str]

    control_set: OracleControlSet
    control_is_admissible: bool
    control_violations: Tuple[str, ...]
    control_bindings: Tuple[str, ...]

    W_K: float
    B: float
    E_priv: float
    balance_sheet_identity_error: float

    pi_mc: float
    portfolio_status: Optional[str]
    portfolio_reason: Optional[str]
    portfolio_interior_margin: float

    r_f: float

    Y: float
    w: float
    r_k: float
    sigma_K: float

    omega: float
    C_W: float
    C_K: float

    debt_service: float
    public_capital_income: float
    private_capital_tax_base: float
    safe_bond_tax_base: float
    total_tax_base: float
    tax_revenue: float

    k_dot: float
    L_dot: float
    W_K_dot: float

    @property
    def is_interior(self) -> bool:
        return self.status == "interior"

    @property
    def is_boundary(self) -> bool:
        return self.status in ("k_wall", "diagonal_wall", "corner")

    @property
    def is_invalid(self) -> bool:
        return self.status == "invalid"

    @property
    def has_portfolio_bind(self) -> bool:
        return self.status == "portfolio_bind"


def _make_oracle_eval(
    *,
    s: int,
    x: State,
    u: Control,
    status: OracleStatus,
    valid_for_pricing: bool,
    valid_for_drift: bool,
    reason: Optional[str],
    state_diag: StateDiagnostics,
    control_set: OracleControlSet,
    control_diag: Optional[Any],
    bs: Optional[BalanceSheet] = None,
    pi_mc: float = math.nan,
    portfolio_check: Optional[PortfolioCheck] = None,
    r_f: float = math.nan,
    Y: float = math.nan,
    w: float = math.nan,
    r_k: float = math.nan,
    sigma_K: float = math.nan,
    omega: float = math.nan,
    C_W: float = math.nan,
    C_K: float = math.nan,
    debt_service: float = math.nan,
    public_capital_income: float = math.nan,
    private_capital_tax_base: float = math.nan,
    safe_bond_tax_base: float = math.nan,
    total_tax_base: float = math.nan,
    tax_revenue: float = math.nan,
    k_dot: float = math.nan,
    L_dot: float = math.nan,
    W_K_dot: float = math.nan,
) -> OracleEval:
    if bs is None:
        bs = balance_sheet(x, u)

    if control_diag is None:
        control_is_admissible = True
        control_violations: Tuple[str, ...] = tuple()
        control_bindings: Tuple[str, ...] = tuple()
    else:
        control_is_admissible = bool(control_diag.is_admissible)
        control_violations = _safe_tuple(control_diag.violations)
        control_bindings = _safe_tuple(control_diag.bindings)

    if portfolio_check is None:
        portfolio_status = None
        portfolio_reason = None
        portfolio_interior_margin = math.nan
    else:
        portfolio_status = str(portfolio_check.status)
        portfolio_reason = portfolio_check.reason
        portfolio_interior_margin = float(portfolio_check.interior_margin)

    return OracleEval(
        regime=int(s),
        state=x,
        control=u,
        status=status,
        valid_for_pricing=bool(valid_for_pricing),
        valid_for_drift=bool(valid_for_drift),
        reason=reason,
        state_status=str(state_diag.status),
        state_is_valid=bool(state_diag.is_valid),
        state_invalid_reason=state_diag.invalid_reason,
        control_set=control_set,
        control_is_admissible=control_is_admissible,
        control_violations=control_violations,
        control_bindings=control_bindings,
        W_K=float(bs.W_K),
        B=float(bs.B),
        E_priv=float(bs.E_priv),
        balance_sheet_identity_error=float(bs.identity_error),
        pi_mc=float(pi_mc),
        portfolio_status=portfolio_status,
        portfolio_reason=portfolio_reason,
        portfolio_interior_margin=float(portfolio_interior_margin),
        r_f=float(r_f),
        Y=float(Y),
        w=float(w),
        r_k=float(r_k),
        sigma_K=float(sigma_K),
        omega=float(omega),
        C_W=float(C_W),
        C_K=float(C_K),
        debt_service=float(debt_service),
        public_capital_income=float(public_capital_income),
        private_capital_tax_base=float(private_capital_tax_base),
        safe_bond_tax_base=float(safe_bond_tax_base),
        total_tax_base=float(total_tax_base),
        tax_revenue=float(tax_revenue),
        k_dot=float(k_dot),
        L_dot=float(L_dot),
        W_K_dot=float(W_K_dot),
    )


def _resolve_asset_params(
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
) -> AssetMarketParams:
    if not isinstance(continuation, ContinuationBundle):
        raise TypeError("continuation must be a ContinuationBundle.")

    if asset_params is None:
        asset_params = continuation.asset_params

    if not isinstance(asset_params, AssetMarketParams):
        raise TypeError("asset_params must be an AssetMarketParams instance.")

    if abs(asset_params.gamma - continuation.gamma) > 1.0e-12:
        raise ValueError(
            "AssetMarketParams gamma and ContinuationBundle gamma disagree: "
            f"asset gamma={asset_params.gamma}, continuation gamma={continuation.gamma}."
        )

    return asset_params


def _control_diagnostics(
    *,
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
) -> Optional[Any]:
    if oracle_options.control_set == "none":
        return None

    if oracle_options.control_set == "full":
        return policy_sets.full_policy_diagnostics(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            economy_params=economy_params,
            options=policy_options,
        )

    if oracle_options.control_set == "compact":
        return policy_sets.compact_policy_diagnostics(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            economy_params=economy_params,
            options=policy_options,
        )

    raise ValueError(f"Unknown control_set={oracle_options.control_set}.")


# ============================================================
# Boundary branches
# ============================================================

def _corner_oracle_eval(
    *,
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    state_diag: StateDiagnostics,
    control_diag: Optional[Any],
    oracle_options: OracleOptions,
) -> OracleEval:
    """
    Exact corner convention:

        k = 0,
        W_K = k + L = 0,
        H = 0,
        B = 0,
        E_priv = 0.

    Do not call Block 0 production formulas at k = 0.
    """
    bs = balance_sheet(x, u)

    if not oracle_options.allow_corner_boundary_drift:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="corner",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="corner branch reached; boundary drift disabled",
            state_diag=state_diag,
            control_set=oracle_options.control_set,
            control_diag=control_diag,
            bs=bs,
        )

    Y = 0.0
    w = 0.0
    C_W = u.T
    C_K = 0.0

    if oracle_options.require_worker_consumption_positive and not (C_W > 0.0):
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="corner worker consumption is not strictly positive",
            state_diag=state_diag,
            control_set=oracle_options.control_set,
            control_diag=control_diag,
            bs=bs,
            Y=Y,
            w=w,
            C_W=C_W,
            C_K=C_K,
        )

    k_dot = -C_W
    L_dot = u.T
    W_K_dot = k_dot + L_dot

    valid_for_drift = _all_finite((k_dot, L_dot, W_K_dot))

    return _make_oracle_eval(
        s=s,
        x=x,
        u=u,
        status="corner",
        valid_for_pricing=False,
        valid_for_drift=valid_for_drift,
        reason="exact corner; pi_mc and r_f are undefined",
        state_diag=state_diag,
        control_set=oracle_options.control_set,
        control_diag=control_diag,
        bs=bs,
        Y=Y,
        w=w,
        r_k=math.nan,
        sigma_K=math.nan,
        omega=math.nan,
        C_W=C_W,
        C_K=C_K,
        debt_service=0.0,
        public_capital_income=0.0,
        private_capital_tax_base=0.0,
        safe_bond_tax_base=0.0,
        total_tax_base=0.0,
        tax_revenue=0.0,
        k_dot=k_dot,
        L_dot=L_dot,
        W_K_dot=W_K_dot,
    )


def _diagonal_wall_oracle_eval(
    *,
    s: int,
    x: State,
    u: Control,
    primitives: RegimePrimitives,
    state_diag: StateDiagnostics,
    control_diag: Optional[Any],
    oracle_options: OracleOptions,
) -> OracleEval:
    """
    Exact diagonal-wall branch k + L = 0, k > 0.

    On this wall, admissibility pins H = k, so

        B = 0,
        E_priv = 0.

    Do not evaluate pi_mc or r_f.
    Use unsimplified drift expressions where r_f B and taxable private-capital
    terms are zero by accounting.
    """
    bs = balance_sheet(x, u)

    if not oracle_options.allow_diagonal_boundary_drift:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="diagonal_wall",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="diagonal-wall branch reached; boundary drift disabled",
            state_diag=state_diag,
            control_set=oracle_options.control_set,
            control_diag=control_diag,
            bs=bs,
        )

    try:
        prim = evaluate_current_primitives(s, x.k, primitives)
    except Exception as exc:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"failed to evaluate diagonal-wall current primitives: {exc}",
            state_diag=state_diag,
            control_set=oracle_options.control_set,
            control_diag=control_diag,
            bs=bs,
        )

    C_W = prim.w + u.T
    C_K = 0.0

    if oracle_options.require_worker_consumption_positive and not (C_W > 0.0):
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="diagonal-wall worker consumption is not strictly positive",
            state_diag=state_diag,
            control_set=oracle_options.control_set,
            control_diag=control_diag,
            bs=bs,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
            C_W=C_W,
            C_K=C_K,
        )

    debt_service = 0.0
    public_capital_income = u.H * prim.r_k
    private_capital_tax_base = 0.0
    safe_bond_tax_base = 0.0
    total_tax_base = 0.0
    tax_revenue = 0.0

    k_dot = (
        prim.Y
        - C_W
        - C_K
        - (prim.delta + prim.g) * x.k
    )

    L_dot = (
        u.T
        - public_capital_income
    )

    W_K_dot = k_dot + L_dot

    valid_for_drift = _all_finite((k_dot, L_dot, W_K_dot))

    return _make_oracle_eval(
        s=s,
        x=x,
        u=u,
        status="diagonal_wall",
        valid_for_pricing=False,
        valid_for_drift=valid_for_drift,
        reason="exact diagonal wall; pi_mc and r_f are undefined",
        state_diag=state_diag,
        control_set=oracle_options.control_set,
        control_diag=control_diag,
        bs=bs,
        Y=prim.Y,
        w=prim.w,
        r_k=prim.r_k,
        sigma_K=prim.sigma_K,
        omega=math.nan,
        C_W=C_W,
        C_K=C_K,
        debt_service=debt_service,
        public_capital_income=public_capital_income,
        private_capital_tax_base=private_capital_tax_base,
        safe_bond_tax_base=safe_bond_tax_base,
        total_tax_base=total_tax_base,
        tax_revenue=tax_revenue,
        k_dot=k_dot,
        L_dot=L_dot,
        W_K_dot=W_K_dot,
    )


def _k_wall_oracle_eval(
    *,
    s: int,
    x: State,
    u: Control,
    state_diag: StateDiagnostics,
    control_diag: Optional[Any],
    oracle_options: OracleOptions,
) -> OracleEval:
    """
    k = 0, W_K > 0 branch.

    The strict production block does not define r_k or sigma_K at k = 0.
    This branch therefore marks interior pricing and full drift evaluation as
    boundary-degenerate. Block 7 should own analytic inward checks at k = 0.
    """
    bs = balance_sheet(x, u)

    return _make_oracle_eval(
        s=s,
        x=x,
        u=u,
        status="k_wall",
        valid_for_pricing=False,
        valid_for_drift=False,
        reason=(
            "k_wall branch; production return and Merton pricing are "
            "boundary-degenerate at k=0"
        ),
        state_diag=state_diag,
        control_set=oracle_options.control_set,
        control_diag=control_diag,
        bs=bs,
    )


# ============================================================
# Live current-control oracle
# ============================================================

def live_oracle(
    s: int,
    x: State,
    u: Control,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    options: Optional[OracleOptions] = None,
) -> OracleEval:
    """
    Evaluate the live current-control oracle:

        O_s(x,u; G, C[u_hat]).

    This function is scalar by design. Grid wrappers should call it pointwise or
    implement a deliberately vectorized analogue later.
    """
    s = _require_regime(s)

    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if options is None:
        options = OracleOptions()

    asset_params = _resolve_asset_params(continuation, asset_params)

    state_diag = primitive_state_diagnostics(x, economy_params)

    # 1. Primitive state status first.
    if not state_diag.is_valid:
        if options.raise_on_invalid_state:
            raise ValueError(
                f"Invalid primitive state: {state_diag.invalid_reason}. State={x}."
            )

        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"invalid primitive state: {state_diag.invalid_reason}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=None,
            bs=balance_sheet(x, u),
        )

    # 2. Primitive current-control admissibility.
    try:
        control_diag = _control_diagnostics(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=options,
        )
    except Exception as exc:
        if options.raise_on_invalid_control:
            raise

        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"control diagnostic failed: {exc}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=None,
            bs=balance_sheet(x, u),
        )

    if control_diag is not None and not control_diag.is_admissible:
        if options.raise_on_invalid_control:
            raise ValueError(
                f"Control is outside {options.control_set} policy set: "
                f"{control_diag.violations}. Control={u}, state={x}."
            )

        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"control outside {options.control_set} policy set",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=balance_sheet(x, u),
        )

    bs = balance_sheet(x, u)

    if abs(bs.identity_error) > options.finite_tol:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="balance-sheet identity failed",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
        )

    # 3. Boundary branches before divided ratios.
    if state_diag.status == "corner":
        return _corner_oracle_eval(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            state_diag=state_diag,
            control_diag=control_diag,
            oracle_options=options,
        )

    if state_diag.status == "diagonal_wall":
        return _diagonal_wall_oracle_eval(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            state_diag=state_diag,
            control_diag=control_diag,
            oracle_options=options,
        )

    if state_diag.status == "k_wall":
        return _k_wall_oracle_eval(
            s=s,
            x=x,
            u=u,
            state_diag=state_diag,
            control_diag=control_diag,
            oracle_options=options,
        )

    if state_diag.status != "interior":
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"unexpected primitive state status: {state_diag.status}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
        )

    # 4. Interior primitive and market objects.
    try:
        prim = evaluate_current_primitives(s, x.k, primitives)
    except Exception as exc:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"failed to evaluate current primitives: {exc}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
        )

    try:
        pi_mc = market_clearing_risky_share(x, u)
    except Exception as exc:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"failed to compute pi_mc: {exc}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
        )

    portfolio_check = check_portfolio_share(pi_mc, asset_params)

    if not portfolio_check.valid:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="portfolio_bind",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="portfolio share is outside the interior Merton branch",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
        )

    try:
        r_f = merton_safe_rate(
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
            pi_mc=pi_mc,
            tau=u.tau,
            asset_params=asset_params,
        )
    except Exception as exc:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"failed to compute Merton safe rate: {exc}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
        )

    # 5. Frozen continuation consumption object.
    try:
        omega = float(
            continuation.omega(
                s,
                x.k,
                x.L,
                strict=options.strict_continuation_support,
            )
        )
    except Exception as exc:
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason=f"failed to evaluate frozen omega_{s}: {exc}",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            r_f=r_f,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
        )

    if not (math.isfinite(omega) and omega > 0.0):
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="frozen omega is non-finite or non-positive",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            r_f=r_f,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
            omega=omega,
        )

    C_W = prim.w + u.T
    C_K = omega * bs.W_K

    if options.require_worker_consumption_positive and not (C_W > 0.0):
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="worker consumption is not strictly positive",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            r_f=r_f,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
            omega=omega,
            C_W=C_W,
            C_K=C_K,
        )

    if not (math.isfinite(C_K) and C_K > 0.0):
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="owner consumption is non-finite or non-positive",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            r_f=r_f,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
            omega=omega,
            C_W=C_W,
            C_K=C_K,
        )

    # 6. Live fiscal objects and drifts.
    debt_service = r_f * bs.B
    public_capital_income = u.H * prim.r_k

    private_capital_tax_base = bs.E_priv * prim.r_k
    safe_bond_tax_base = debt_service
    total_tax_base = private_capital_tax_base + safe_bond_tax_base
    tax_revenue = u.tau * total_tax_base

    k_dot = (
        prim.Y
        - C_W
        - C_K
        - (prim.delta + prim.g) * x.k
    )

    L_dot = (
        debt_service
        + u.T
        - public_capital_income
        - tax_revenue
    )

    W_K_dot = k_dot + L_dot

    finite_required = (
        pi_mc,
        r_f,
        prim.Y,
        prim.w,
        prim.r_k,
        prim.sigma_K,
        omega,
        C_W,
        C_K,
        debt_service,
        public_capital_income,
        private_capital_tax_base,
        safe_bond_tax_base,
        total_tax_base,
        tax_revenue,
        k_dot,
        L_dot,
        W_K_dot,
    )

    if not _all_finite(finite_required):
        return _make_oracle_eval(
            s=s,
            x=x,
            u=u,
            status="invalid",
            valid_for_pricing=False,
            valid_for_drift=False,
            reason="one or more live oracle objects are non-finite",
            state_diag=state_diag,
            control_set=options.control_set,
            control_diag=control_diag,
            bs=bs,
            pi_mc=pi_mc,
            portfolio_check=portfolio_check,
            r_f=r_f,
            Y=prim.Y,
            w=prim.w,
            r_k=prim.r_k,
            sigma_K=prim.sigma_K,
            omega=omega,
            C_W=C_W,
            C_K=C_K,
            debt_service=debt_service,
            public_capital_income=public_capital_income,
            private_capital_tax_base=private_capital_tax_base,
            safe_bond_tax_base=safe_bond_tax_base,
            total_tax_base=total_tax_base,
            tax_revenue=tax_revenue,
            k_dot=k_dot,
            L_dot=L_dot,
            W_K_dot=W_K_dot,
        )

    return _make_oracle_eval(
        s=s,
        x=x,
        u=u,
        status="interior",
        valid_for_pricing=True,
        valid_for_drift=True,
        reason=None,
        state_diag=state_diag,
        control_set=options.control_set,
        control_diag=control_diag,
        bs=bs,
        pi_mc=pi_mc,
        portfolio_check=portfolio_check,
        r_f=r_f,
        Y=prim.Y,
        w=prim.w,
        r_k=prim.r_k,
        sigma_K=prim.sigma_K,
        omega=omega,
        C_W=C_W,
        C_K=C_K,
        debt_service=debt_service,
        public_capital_income=public_capital_income,
        private_capital_tax_base=private_capital_tax_base,
        safe_bond_tax_base=safe_bond_tax_base,
        total_tax_base=total_tax_base,
        tax_revenue=tax_revenue,
        k_dot=k_dot,
        L_dot=L_dot,
        W_K_dot=W_K_dot,
    )


def require_oracle_interior(
    s: int,
    x: State,
    u: Control,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    options: Optional[OracleOptions] = None,
) -> OracleEval:
    """
    Require the current state-control pair to be on the interior pricing branch.
    """
    ev = live_oracle(
        s=s,
        x=x,
        u=u,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev.status != "interior":
        raise ValueError(
            "Oracle evaluation is not on the interior branch: "
            f"status={ev.status}, reason={ev.reason}, "
            f"portfolio_status={ev.portfolio_status}."
        )

    return ev


# ============================================================
# Transfer derivative diagnostics
# ============================================================

@dataclass(frozen=True)
class ModeATransferDerivatives:
    """
    Mode-A transfer derivative convention.

    Current transfer T changes worker consumption and fiscal flows, but not
    frozen owner consumption or current pricing.
    """
    dC_W_dT: float
    dC_K_dT: float
    dk_dot_dT: float
    dL_dot_dT: float
    dW_K_dot_dT: float


def analytical_mode_a_transfer_derivatives() -> ModeATransferDerivatives:
    """
    Analytical Mode-A transfer derivatives:

        d C_W / dT = 1,
        d C_K / dT = 0,
        d k_dot / dT = -1,
        d L_dot / dT = 1,
        d W_K_dot / dT = 0.
    """
    return ModeATransferDerivatives(
        dC_W_dT=1.0,
        dC_K_dT=0.0,
        dk_dot_dT=-1.0,
        dL_dot_dT=1.0,
        dW_K_dot_dT=0.0,
    )


def finite_difference_transfer_derivatives(
    s: int,
    x: State,
    u: Control,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    options: Optional[OracleOptions] = None,
    h: float = 1.0e-6,
) -> ModeATransferDerivatives:
    """
    Finite-difference check for the Mode-A transfer derivatives.
    """
    h = _positive_float(h, name="h")

    ev0 = require_oracle_interior(
        s=s,
        x=x,
        u=u,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    u_plus = Control(
        tau=u.tau,
        T=u.T + h,
        H=u.H,
    )

    ev1 = require_oracle_interior(
        s=s,
        x=x,
        u=u_plus,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    return ModeATransferDerivatives(
        dC_W_dT=(ev1.C_W - ev0.C_W) / h,
        dC_K_dT=(ev1.C_K - ev0.C_K) / h,
        dk_dot_dT=(ev1.k_dot - ev0.k_dot) / h,
        dL_dot_dT=(ev1.L_dot - ev0.L_dot) / h,
        dW_K_dot_dT=(ev1.W_K_dot - ev0.W_K_dot) / h,
    )


# ============================================================
# Validation
# ============================================================

def _check_close(
    name: str,
    lhs: float,
    rhs: float,
    *,
    atol: float,
    rtol: float,
) -> float:
    lhs = float(lhs)
    rhs = float(rhs)

    scale = max(1.0, abs(lhs), abs(rhs))
    err = abs(lhs - rhs)
    allowed = atol + rtol * scale

    if err > allowed:
        raise RuntimeError(
            f"{name} failed: error={err:.3e}, allowed={allowed:.3e}, "
            f"lhs={lhs}, rhs={rhs}."
        )

    return float(err)


def validate_oracle_layer(
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    options: Optional[OracleOptions] = None,
    atol: float = 1.0e-8,
    rtol: float = 1.0e-7,
) -> dict[str, float]:
    """
    Validate Block 6.

    Checks:
      - interior oracle status;
      - balance-sheet identities;
      - pi_mc identity;
      - safe-rate formula;
      - worker and owner consumption identities;
      - k_dot, L_dot, and W_K_dot formulas;
      - Mode-A transfer derivatives;
      - portfolio_bind branch under tight finite bounds;
      - infinite baseline portfolio bounds do not bind at pi=0 or pi in (0,1);
      - diagonal wall avoids divided pricing ratios and returns finite drifts;
      - corner branch avoids production formulas at k=0;
      - invalid controls return status="invalid";
      - invalid primitive states return status="invalid".
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if options is None:
        options = OracleOptions(control_set="full")

    asset_params = _resolve_asset_params(continuation, asset_params)

    report: dict[str, float] = {
        "gamma": float(asset_params.gamma),
    }

    # --------------------------------------------------------
    # Interior branch.
    # --------------------------------------------------------
    s = 0
    x = State(k=1.0, L=0.5)

    compact_bounds = policy_sets.compact_policy_bounds(
        s=s,
        x=x,
        primitives=primitives,
        economy_params=economy_params,
        options=policy_options,
    )

    u = policy_sets.midpoint_control(compact_bounds)

    ev = live_oracle(
        s=s,
        x=x,
        u=u,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev.status != "interior":
        raise RuntimeError(f"Interior smoke test failed: {ev}")

    if not ev.valid_for_pricing:
        raise RuntimeError("Interior branch should be valid_for_pricing.")

    if not ev.valid_for_drift:
        raise RuntimeError("Interior branch should be valid_for_drift.")

    _check_close(
        "balance-sheet identity",
        ev.W_K,
        ev.B + ev.E_priv,
        atol=atol,
        rtol=rtol,
    )

    expected_pi = (x.k - u.H) / (x.k + x.L)

    _check_close(
        "pi_mc identity",
        ev.pi_mc,
        expected_pi,
        atol=atol,
        rtol=rtol,
    )

    expected_r_f = (
        ev.r_k
        - asset_params.gamma
        * (1.0 - u.tau)
        * (ev.sigma_K ** 2)
        * ev.pi_mc
    )

    _check_close(
        "Merton safe-rate formula",
        ev.r_f,
        expected_r_f,
        atol=atol,
        rtol=rtol,
    )

    _check_close(
        "worker consumption identity",
        ev.C_W,
        ev.w + u.T,
        atol=atol,
        rtol=rtol,
    )

    _check_close(
        "owner consumption identity",
        ev.C_K,
        ev.omega * ev.W_K,
        atol=atol,
        rtol=rtol,
    )

    expected_tax_revenue = u.tau * (
        ev.E_priv * ev.r_k
        + ev.r_f * ev.B
    )

    _check_close(
        "tax revenue identity",
        ev.tax_revenue,
        expected_tax_revenue,
        atol=atol,
        rtol=rtol,
    )

    expected_k_dot = (
        ev.Y
        - ev.C_W
        - ev.C_K
        - (primitives.params.delta + primitives.params.g) * x.k
    )

    _check_close(
        "k_dot identity",
        ev.k_dot,
        expected_k_dot,
        atol=atol,
        rtol=rtol,
    )

    expected_L_dot = (
        ev.r_f * ev.B
        + u.T
        - u.H * ev.r_k
        - u.tau * (
            (x.k - u.H) * ev.r_k
            + ev.r_f * ev.B
        )
    )

    _check_close(
        "L_dot identity",
        ev.L_dot,
        expected_L_dot,
        atol=atol,
        rtol=rtol,
    )

    _check_close(
        "W_K_dot identity",
        ev.W_K_dot,
        ev.k_dot + ev.L_dot,
        atol=atol,
        rtol=rtol,
    )

    report["interior_pi_mc"] = float(ev.pi_mc)
    report["interior_r_f"] = float(ev.r_f)
    report["interior_k_dot"] = float(ev.k_dot)
    report["interior_L_dot"] = float(ev.L_dot)

    # --------------------------------------------------------
    # Mode-A transfer derivative check.
    # --------------------------------------------------------
    fd = finite_difference_transfer_derivatives(
        s=s,
        x=x,
        u=u,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
        h=1.0e-6,
    )

    analytical = analytical_mode_a_transfer_derivatives()

    for name in (
        "dC_W_dT",
        "dC_K_dT",
        "dk_dot_dT",
        "dL_dot_dT",
        "dW_K_dot_dT",
    ):
        _check_close(
            f"Mode-A transfer derivative {name}",
            getattr(fd, name),
            getattr(analytical, name),
            atol=1.0e-5,
            rtol=1.0e-5,
        )

    report["fd_dC_W_dT"] = float(fd.dC_W_dT)
    report["fd_dC_K_dT"] = float(fd.dC_K_dT)
    report["fd_dk_dot_dT"] = float(fd.dk_dot_dT)
    report["fd_dL_dot_dT"] = float(fd.dL_dot_dT)

    # --------------------------------------------------------
    # Infinite baseline portfolio bounds should not bind on feasible shares.
    # --------------------------------------------------------
    for H in (0.0, x.k):
        u_H = Control(
            tau=u.tau,
            T=u.T,
            H=H,
        )

        ev_H = live_oracle(
            s=s,
            x=x,
            u=u_H,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            options=options,
        )

        if ev_H.status != "interior":
            raise RuntimeError(
                "Infinite baseline portfolio bounds should not bind for "
                f"mechanically feasible H={H}. Got {ev_H.status}."
            )

    # --------------------------------------------------------
    # Tight finite bounds should trigger portfolio_bind.
    # --------------------------------------------------------
    tight_asset_params = AssetMarketParams(
        gamma=asset_params.gamma,
        pi_lower=0.20,
        pi_upper=0.80,
        pi_tol=1.0e-8,
    )

    u_low_pi = Control(
        tau=u.tau,
        T=u.T,
        H=x.k,
    )

    ev_bind = live_oracle(
        s=s,
        x=x,
        u=u_low_pi,
        primitives=primitives,
        continuation=continuation,
        asset_params=tight_asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev_bind.status != "portfolio_bind":
        raise RuntimeError("Tight finite portfolio bounds should trigger portfolio_bind.")

    if ev_bind.portfolio_status != "portfolio_lower_bind":
        raise RuntimeError(
            "Tight lower test should report side-specific portfolio_lower_bind."
        )

    report["portfolio_bind_pi_mc"] = float(ev_bind.pi_mc)

    # --------------------------------------------------------
    # Exact diagonal wall.
    # --------------------------------------------------------
    x_diag = State(k=1.0, L=-1.0)

    diag_bounds = policy_sets.compact_policy_bounds(
        s=s,
        x=x_diag,
        primitives=primitives,
        economy_params=economy_params,
        options=policy_options,
    )

    u_diag = Control(
        tau=0.20,
        T=max(diag_bounds.T_lower + 1.0e-3, 1.0e-3),
        H=x_diag.k,
    )

    ev_diag = live_oracle(
        s=s,
        x=x_diag,
        u=u_diag,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev_diag.status != "diagonal_wall":
        raise RuntimeError(f"Diagonal-wall test failed: {ev_diag}")

    if not math.isnan(ev_diag.pi_mc):
        raise RuntimeError("pi_mc should be undefined on the exact diagonal wall.")

    if not math.isnan(ev_diag.r_f):
        raise RuntimeError("r_f should be undefined on the exact diagonal wall.")

    if not ev_diag.valid_for_drift:
        raise RuntimeError("Diagonal-wall branch should return finite unsimplified drifts.")

    report["diagonal_k_dot"] = float(ev_diag.k_dot)
    report["diagonal_L_dot"] = float(ev_diag.L_dot)

    # --------------------------------------------------------
    # Exact corner should not call production at k=0.
    # --------------------------------------------------------
    x_corner = State(k=0.0, L=0.0)

    corner_bounds = policy_sets.compact_policy_bounds(
        s=s,
        x=x_corner,
        primitives=primitives,
        economy_params=economy_params,
        options=policy_options,
    )

    u_corner = Control(
        tau=0.20,
        T=corner_bounds.T_lower,
        H=0.0,
    )

    ev_corner = live_oracle(
        s=s,
        x=x_corner,
        u=u_corner,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev_corner.status != "corner":
        raise RuntimeError(f"Corner test failed: {ev_corner}")

    if not ev_corner.valid_for_drift:
        raise RuntimeError("Corner branch should return finite analytic drifts.")

    _check_close(
        "corner k_dot",
        ev_corner.k_dot,
        -u_corner.T,
        atol=atol,
        rtol=rtol,
    )

    _check_close(
        "corner L_dot",
        ev_corner.L_dot,
        u_corner.T,
        atol=atol,
        rtol=rtol,
    )

    report["corner_k_dot"] = float(ev_corner.k_dot)
    report["corner_L_dot"] = float(ev_corner.L_dot)

    # --------------------------------------------------------
    # Invalid control: tau at strict primitive upper bound.
    # --------------------------------------------------------
    u_bad_tau = Control(
        tau=economy_params.tau_upper,
        T=u.T,
        H=u.H,
    )

    ev_bad_tau = live_oracle(
        s=s,
        x=x,
        u=u_bad_tau,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev_bad_tau.status != "invalid":
        raise RuntimeError("tau at strict upper bound should return status='invalid'.")

    # --------------------------------------------------------
    # Invalid primitive state.
    # --------------------------------------------------------
    x_bad = State(k=1.0, L=-2.0)

    ev_bad_state = live_oracle(
        s=s,
        x=x_bad,
        u=u,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=options,
    )

    if ev_bad_state.status != "invalid":
        raise RuntimeError("Invalid primitive state should return status='invalid'.")

    report["invalid_status_tests"] = 2.0

    return report


def module_smoke_test() -> dict[str, float]:
    """
    Minimal Block 6 smoke test.
    """
    automation_params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    primitives = build_regime_primitives(automation_params)

    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    continuation = make_test_continuation_bundle(
        asset_params=asset_params,
    )

    economy_params = PlannerEconomyParams(
        tau_upper=1.0,
        transfer_min=0.0,
        worker_consumption_eps=1.0e-8,
        state_tol=1.0e-10,
        control_tol=1.0e-12,
    )

    policy_options = PolicySetOptions()

    oracle_options = OracleOptions(
        control_set="full",
        strict_continuation_support=True,
    )

    return validate_oracle_layer(
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=oracle_options,
    )


__all__ = [
    "ArrayLike",
    "OracleStatus",
    "OracleControlSet",
    "OracleOptions",
    "PrimitiveCurrentObjects",
    "OracleEval",
    "ModeATransferDerivatives",
    "evaluate_current_primitives",
    "market_clearing_risky_share",
    "merton_safe_rate",
    "live_oracle",
    "require_oracle_interior",
    "analytical_mode_a_transfer_derivatives",
    "finite_difference_transfer_derivatives",
    "validate_oracle_layer",
    "module_smoke_test",
]

Overwriting equilibrium_oracle.py


In [21]:
import importlib

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import automation_risk
import equilibrium_oracle

importlib.reload(automation_block)
importlib.reload(economy)
importlib.reload(policy_sets)
importlib.reload(asset_market)
importlib.reload(continuation_block)
importlib.reload(automation_risk)
importlib.reload(equilibrium_oracle)

block6_report = equilibrium_oracle.module_smoke_test()

print("Block 6 validation passed.")
print(block6_report)

Block 6 validation passed.
{'gamma': 5.0, 'interior_pi_mc': 0.3333333333333333, 'interior_r_f': 0.7053026814956159, 'interior_k_dot': -14.371277289835826, 'interior_L_dot': 14.809611834822116, 'fd_dC_W_dT': 0.9999999974752427, 'fd_dC_K_dT': 0.0, 'fd_dk_dot_dT': -0.9999999974752427, 'fd_dL_dot_dT': 0.9999999992515995, 'portfolio_bind_pi_mc': 0.0, 'diagonal_k_dot': 0.7030526816831161, 'diagonal_L_dot': -0.7230526816831159, 'corner_k_dot': -1e-08, 'corner_L_dot': 1e-08, 'invalid_status_tests': 2.0}


In [22]:
import importlib

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle

importlib.reload(equilibrium_oracle)

automation_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(automation_params)

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

C_hat = continuation_block.make_test_continuation_bundle(
    asset_params=asset_params,
)

economy_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

policy_options = policy_sets.PolicySetOptions()

s = 0
x = economy.State(k=1.0, L=0.5)

bounds = policy_sets.compact_policy_bounds(
    s=s,
    x=x,
    primitives=G,
    economy_params=economy_params,
    options=policy_options,
)

u = policy_sets.midpoint_control(bounds)

oracle_eval = equilibrium_oracle.live_oracle(
    s=s,
    x=x,
    u=u,
    primitives=G,
    continuation=C_hat,
    asset_params=asset_params,
    economy_params=economy_params,
    policy_options=policy_options,
    options=equilibrium_oracle.OracleOptions(control_set="full"),
)

print(oracle_eval)

OracleEval(regime=0, state=State(k=1.0, L=0.5), control=Control(tau=0.499999995, T=15.0, H=0.5), status='interior', valid_for_pricing=True, valid_for_drift=True, reason=None, state_status='interior', state_is_valid=True, state_invalid_reason=None, control_set='full', control_is_admissible=True, control_violations=(), control_bindings=(), W_K=1.5, B=1.0, E_priv=0.5, balance_sheet_identity_error=0.0, pi_mc=0.3333333333333333, portfolio_status='interior', portfolio_reason=None, portfolio_interior_margin=inf, r_f=0.7053026814956159, Y=1.9601317042077895, w=1.1760790225246736, r_k=0.7240526816831159, sigma_K=0.15, omega=0.05021998101262648, C_W=16.176079022524675, C_K=0.07532997151893972, debt_service=0.7053026814956159, public_capital_income=0.36202634084155794, private_capital_tax_base=0.36202634084155794, safe_bond_tax_base=0.7053026814956159, total_tax_base=1.0673290223371739, tax_revenue=0.5336645058319418, k_dot=-14.371277289835826, L_dot=14.809611834822116, W_K_dot=0.4383345449862901

# Block 7 — state constraints and pure viability sets

Block 7 constructs the **pure conditional viability sets** for the planner problem.

It is the first block that turns the live current-control oracle into an admissibility object. The live oracle from Block 6 evaluates prices, tax bases, revenues, and drifts at a candidate current control. Block 7 then asks whether there exists an admissible current control that keeps the state inside the relevant feasible set.

The central distinction is:

$$
\boxed{
\text{Block 6 produces drifts. Block 7 decides whether those drifts are viable.}
}
$$

A finite drift is not the same thing as a viable drift. In particular,

$$
\texttt{valid\_for\_drift=True}
\not\Rightarrow
\text{inward feasible}.
$$

Block 7 owns the viability logic.

---

## Objects owned by Block 7

Block 7 distinguishes three different domain objects:

$$
S
=
\text{primitive feasible state set},
$$

$$
V_s^{\hat u}
=
\text{pure conditional viability set},
$$

and

$$
A_s
=
\text{Howard-only active mask}.
$$

The primitive feasible state set is

$$
S
=
\{(k,L): k\ge 0,\ k+L\ge 0\}.
$$

The conditional viability sets are economic domain objects. They depend on the frozen private continuation environment induced by the anticipated planner rule $\hat u$.

The Howard active masks are later numerical working domains. Howard iteration may update $A_s$, but it must not redefine the pure viability sets $V_s^{\hat u}$.

The hard separation is:

$$
\boxed{
V_s^{\hat u}
\text{ is a viability object, while }
A_s
\text{ is a Howard bookkeeping object.}
}
$$

---

## Drift correspondence

For a frozen anticipated rule $\hat u$, the private continuation block returns

$$
\mathcal C[\hat u]
=
\left\{
\Psi_s^{\hat u},
\omega_s^{\hat u},
\chi^{\hat u},
\lambda^{Q,\hat u},
\text{validity masks}
\right\}_{s=0,1}.
$$

The continuation objects are frozen. Current policy still varies.

Given the live oracle

$$
\mathcal O_s(x,u;\mathcal G,\mathcal C[\hat u]),
$$

define the current-control drift as

$$
f_s^{\hat u}(x;u)
=
\left(
\dot k_s^{\hat u}(x;u),
\dot L_s^{\hat u}(x;u)
\right).
$$

The current-control drift correspondence is

$$
\mathcal F_s^{\hat u}(x)
=
\left\{
f_s^{\hat u}(x;u):
u\in U_s^{full}(x)
\right\}.
$$

Here $U_s^{full}(x)$ is the full admissible current policy set:

$$
U_s^{full}(k,L)
=
\left\{
(\tau,T,H):
\tau\in[0,\bar\tau),
\quad
T\in[\underline T_s(k),\infty),
\quad
H\in[\max\{0,-L\},k]
\right\}.
$$

Thus Block 7 implements the model object:

$$
\boxed{
\text{current controls vary over the full policy set, while private continuation objects remain frozen.}
}
$$

---

## Full-policy-set viability

The post-switch viability set is

$$
V_1^{\hat u}
=
\operatorname{Viab}_{\mathcal F_1^{\hat u}}(S).
$$

The pre-switch viability set is

$$
V_0^{\hat u}
=
\operatorname{Viab}_{\mathcal F_0^{\hat u}}
\left(
S\cap V_1^{\hat u}
\right).
$$

The second expression is important. Before automation arrives, the state must remain inside the post-switch viable set at every date, because the automation event can arrive at any time.

Therefore the main computational object is **not**

$$
\operatorname{Viab}_{\mathcal F_0^{\hat u}}(S)
\cap
V_1^{\hat u}.
$$

That expression only checks that the initial pre-switch state is post-switch viable. It does not ensure that the regime-0 path remains post-switch viable while the economy waits for automation.

The intended object is instead:

$$
\boxed{
V_0^{\hat u}
=
\operatorname{Viab}_{\mathcal F_0^{\hat u}}
\left(
S\cap V_1^{\hat u}
\right).
}
$$

---

## Primitive inward conditions

Primitive state-constraint checks are analytic.

At the wall

$$
k=0,
$$

the inward condition is

$$
\dot k\ge 0.
$$

At the wall

$$
k+L=0,
$$

the inward condition is

$$
\dot k+\dot L\ge 0.
$$

At the corner

$$
k=0,
\qquad
k+L=0,
$$

both conditions must hold:

$$
\dot k\ge 0,
\qquad
\dot k+\dot L\ge 0.
$$

These conditions are not optional diagnostics. They are part of the viability definition.

This is why Block 7 must not confuse finite boundary drifts with inward-feasible boundary drifts. A boundary oracle evaluation can be algebraically finite and still fail viability.

---

## Boundary semantics

On the exact diagonal wall,

$$
k+L=0,
$$

we have

$$
L=-k.
$$

The admissible public-capital interval collapses:

$$
H\in[\max\{0,-L\},k]=[k,k].
$$

Therefore,

$$
H=k,
\qquad
B=L+H=0,
\qquad
E^{priv}=k-H=0.
$$

So $H$ is not a free rescue variable on the exact primitive diagonal. It may matter near the diagonal or on endogenous peeled boundaries, but on the exact primitive diagonal it is pinned by the balance sheet.

Likewise, at

$$
k=0,
$$

the transfer direction is important. Under the Mode-A transfer convention,

$$
\partial_T \dot k=-1,
\qquad
\partial_T \dot L=1.
$$

Therefore increasing transfers worsens the $k=0$ inward condition. At the $k$ wall, the rescue direction is toward lower transfers, not larger transfers.

This matters because the transfer control is semi-infinite above:

$$
T\in[\underline T_s(k),\infty).
$$

A large transfer can help some diagonal or endogenous-boundary geometry, but it cannot rescue a negative $\dot k$ at the exact $k=0$ wall.

---

## Treatment of the semi-infinite transfer control

The transfer control has no primitive upper bound:

$$
T\in[\underline T_s(k),\infty).
$$

Block 7 therefore must not treat the artificial compactification cap as an economic restriction.

For each fixed pair $(\tau,H)$, the live drift is affine in $T$ under the Mode-A convention. Let

$$
T
=
\underline T_s(k)+dT,
\qquad
dT\ge 0.
$$

Then

$$
f_s^{\hat u}(x;\tau,T,H)
=
f_s^{\hat u}(x;\tau,\underline T_s(k),H)
+
dT
\begin{pmatrix}
-1\\
1
\end{pmatrix}.
$$

Equivalently,

$$
\frac{\partial f_s^{\hat u}}{\partial T}
=
\begin{pmatrix}
-1\\
1
\end{pmatrix}.
$$

The viability witness search uses this exact transfer ray. It does not grid or cap $T$ as the baseline viability definition.

The numerical compactification $U_s^M(x)$ may still be useful later for diagnostics or for planner optimisation routines, but Block 7’s baseline viability kernel should be accurate with respect to the model’s full semi-infinite transfer set.

The guiding rule is:

$$
\boxed{
\text{The artificial transfer cap is a diagnostic object, not a primitive admissibility restriction.}
}
$$

---

## Discrete tangent-cone condition

On the grid, a candidate mask $A$ approximates a closed set.

At a grid node $x_{ij}$, Block 7 constructs a local discrete tangent cone $T_A(x_{ij})$ using neighbouring grid nodes that remain inside $A$.

A candidate control is viable at $x_{ij}$ if

$$
x_{ij}\in A
$$

and there exists

$$
u\in U_s^{full}(x_{ij})
$$

such that

$$
f_s^{\hat u}(x_{ij};u)\in T_A(x_{ij}),
$$

subject also to the primitive inward conditions on the walls of $S$.

Because $T$ is semi-infinite, the implementation does not search a box in $(\tau,T,H)$. Instead, it searches over $(\tau,H)$ and solves exactly along the transfer ray

$$
f_{\mathrm{floor}}
+
dT
\begin{pmatrix}
-1\\
1
\end{pmatrix},
\qquad
dT\ge 0.
$$

Thus the witness problem is:

$$
\exists
\ \tau\in[0,\bar\tau),
\quad
H\in[\max\{0,-L\},k],
\quad
dT\ge 0
$$

such that

$$
f_{\mathrm{floor}}(\tau,H)
+
dT
\begin{pmatrix}
-1\\
1
\end{pmatrix}
\in
T_A(x),
$$

with the primitive inward inequalities enforced at primitive walls.

---

## Regime-1 peeling operator

For the post-automation regime, initialise from the primitive feasible grid:

$$
A_1^{(0)}
=
S_{\mathrm{grid}}.
$$

Define the peeling operator

$$
A_1^{(m+1)}
=
P_1^{\hat u}(A_1^{(m)}),
$$

where $x\in P_1^{\hat u}(A)$ if and only if:

1. $x\in A$;
2. there exists a current control $u\in U_1^{full}(x)$;
3. the oracle returns a valid drift for that control;
4. the drift satisfies the analytic primitive inward conditions;
5. the drift lies in the discrete tangent cone $T_A(x)$.

The post-switch viability set is the greatest fixed point:

$$
V_1^{\hat u}
=
\operatorname{gfp}(P_1^{\hat u}).
$$

---

## Regime-0 conditional peeling operator

For the pre-automation regime, initialise inside the post-switch viable set:

$$
A_0^{(0)}
=
S_{\mathrm{grid}}
\cap
V_1^{\hat u}.
$$

Define

$$
A_0^{(m+1)}
=
Q_0^{\hat u}(A_0^{(m)};V_1^{\hat u}),
$$

where $x\in Q_0^{\hat u}(A;V_1^{\hat u})$ if and only if:

1. $x\in A$;
2. $x\in V_1^{\hat u}$;
3. there exists a current control $u\in U_0^{full}(x)$;
4. the oracle returns a valid drift for that control;
5. the drift satisfies the analytic primitive inward conditions;
6. the drift lies in the discrete tangent cone $T_A(x)$.

The pre-switch viability set is the greatest fixed point:

$$
V_0^{\hat u}
=
\operatorname{gfp}(Q_0^{\hat u}).
$$

This construction means the economy cannot drift out of the post-switch viable region while waiting for the Poisson event.

---

## Witness controls

Block 7 stores witness maps:

$$
w_1^{wit}(x),
\qquad
w_0^{wit}(x).
$$

A witness is a certificate that the state is viable. It is not a planner policy.

The planner later solves a separate optimisation problem on the viable domain. A witness merely proves existence of at least one admissible current-control selection that keeps the state feasible.

Thus:

$$
\boxed{
\text{viability witnesses are feasibility certificates, not optimal policies.}
}
$$

---

## Witness-search hierarchy

The witness-search hierarchy is:

$$
\text{previous witness}
\to
\text{neighbouring witness}
\to
\text{analytic candidates}
\to
\text{local feasibility solve}
\to
\text{tiny rescue grid}.
$$

The hierarchy is only a computational strategy. The mathematical criterion remains the same: existence of a control in the full current policy set whose drift lies in the tangent cone and satisfies primitive inward conditions.

The tiny rescue grid is not the baseline definition of viability. It is a fallback diagnostic.

---

## Module split

Block 7 is split into two modules.

### `state_constraints.py`

This module owns state-constraint and tangent-cone diagnostics.

It provides:

```text
StateConstraintOptions
PrimitiveInwardDiagnostics
DiscreteTangentDiagnostics
StateConstraintDiagnostics
primitive_grid_mask
primitive_inward_diagnostics
local_tangent_generators
discrete_tangent_diagnostics
state_constraint_diagnostics
```

This module does not solve viability kernels. It only classifies primitive feasibility, primitive inwardness, and local discrete tangent-cone membership.

### `viability_sets.py`

This module owns the actual pure viability computation.

It provides:

```text
ViabilityGrid
TauHBounds
TransferRaySolve
ViabilityOptions
ViabilityWitness
ViabilityKernel
ConditionalViabilityResult
analytic_tau_H_candidates
evaluate_tau_H_witness
find_viability_witness
peel_viability_kernel
compute_conditional_viability_sets
validate_viability_layer
module_smoke_test
```

This module runs the peeling fixed-point operators and returns the viability masks and witness maps.

---

## Inputs

Block 7 takes:

```text
grid
primitives
continuation
asset_params
economy_params
policy_options
oracle_options
state_options
viability_options
previous_V1
previous_V0
```

Economically, the key inputs are:

$$
\mathcal G,
\qquad
\mathcal C[\hat u],
\qquad
U_s^{full}(x),
\qquad
\mathcal O_s(x,u).
$$

The continuation bundle $\mathcal C[\hat u]$ is frozen. The oracle remains live in the current control $u$.

---

## Outputs

Block 7 returns:

$$
V_1^{\hat u},
\qquad
V_0^{\hat u},
$$

together with witness controls and diagnostics.

For each regime $s$, the viability kernel stores:

```text
mask
tau
T
H
k_dot
L_dot
converged
n_iter
n_initial
n_viable
n_removed
diagnostics
```

The witness arrays are defined only on viable nodes. Non-viable nodes remain masked out.

---

## Diagnostics

Block 7 reports diagnostics such as:

```text
n_primitive
n_V1
n_V0
share_V1
share_V0_of_V1
V1_converged
V0_converged
uses_full_transfer_halfline
n_tau_upper_cap_witnesses
share_tau_upper_cap_witnesses
n_H_bound_witnesses
share_H_bound_witnesses
n_large_transfer_witnesses
max_witness_T
```

The most important diagnostic is:

$$
\texttt{uses\_full\_transfer\_halfline}=1.
$$

This confirms that the baseline viability computation did not impose an artificial upper transfer cap.

Large-transfer witnesses should still be reported. They do not invalidate the viability kernel, but they identify nodes where the semi-infinite nature of the transfer control matters strongly.

---

## What Block 7 must not do

Block 7 should not:

- solve the private continuation problem;
- recompute $\Psi_s^{\hat u}$ or $\omega_s^{\hat u}$;
- freeze old arrays for $r_f$, $\dot k$, or $\dot L$;
- maximise the planner Hamiltonian;
- run Howard iteration;
- update the anticipated rule $\hat u$;
- redefine pure viability sets inside Howard;
- treat the artificial transfer cap as an economic primitive;
- treat witness controls as planner optima;
- evaluate divided pricing formulas on the exact diagonal wall.

The key forbidden confusion is:

$$
\boxed{
\text{Do not confuse viability with planner optimality.}
}
$$

Viability is an existence problem. Planner improvement is an optimisation problem.

---

## Validation checks

The Block 7 validation harness should check:

1. regime 1 starts from the primitive grid $S_{\mathrm{grid}}$;
2. regime 0 starts from $S_{\mathrm{grid}}\cap V_1^{\hat u}$;
3. $V_0^{\hat u}\subseteq V_1^{\hat u}$;
4. $V_1^{\hat u}\subseteq S_{\mathrm{grid}}$;
5. $V_0^{\hat u}\subseteq S_{\mathrm{grid}}$;
6. witness maps are returned on viable nodes;
7. the transfer half-line is used rather than an artificial transfer cap;
8. the corner with negative $\dot k$ fails the $k=0$ inward condition;
9. the diagonal wall with negative $\dot k+\dot L$ fails the diagonal inward condition;
10. viability peeling converges to a fixed point.

The validation should also check that a full restart from the true candidate superset is possible after the outer operator changes. Warm starts are useful for speed, but a shrink-only warm peel must not be the final solver after $\hat u$ changes, because the viability operator itself changes and states may re-enter.

---

## One-line summary

Block 7 computes

$$
\boxed{
V_1^{\hat u}
=
\operatorname{Viab}_{\mathcal F_1^{\hat u}}(S),
\qquad
V_0^{\hat u}
=
\operatorname{Viab}_{\mathcal F_0^{\hat u}}
\left(
S\cap V_1^{\hat u}
\right).
}
$$

It does this by peeling candidate masks to a greatest fixed point, using live oracle drifts, frozen private continuation objects, analytic primitive inward checks, discrete tangent cones, and the full semi-infinite transfer half-line.

In [23]:
%%writefile state_constraints.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Sequence
import math
import numpy as np

from model.economy import State, PlannerEconomyParams, primitive_state_diagnostics
from equilibrium_oracle import OracleEval


# ============================================================
# Block 7A contract: state constraints and tangent diagnostics
# ============================================================
#
# This module owns state-constraint admissibility diagnostics.
#
# It distinguishes:
#   S                  primitive closed feasible set;
#   V_s^{hat u}        pure conditional viability set;
#   A_s                Howard-only active mask, owned later by Howard.
#
# It does not solve viability kernels, does not optimise Hamiltonians,
# and does not run Howard. The viability peeling operator lives in
# viability_sets.py and calls the diagnostics here.


Array2Bool = np.ndarray


@dataclass(frozen=True)
class StateConstraintOptions:
    """
    Options for primitive and discrete tangent-cone checks.

    primitive_wall_tol:
        Tolerance for recognising numerical primitive walls when applying
        analytic inward inequalities.

    inward_tol:
        Allowed violation in analytic primitive inward inequalities.

    cone_residual_tol:
        Allowed relative residual for discrete tangent-cone membership.

    stencil_radius:
        Radius, in grid nodes, used to form local discrete tangent generators.
        The default 1 is the Moore neighbourhood.
    """
    primitive_wall_tol: float = 1.0e-10
    inward_tol: float = 1.0e-9
    cone_residual_tol: float = 1.0e-7
    stencil_radius: int = 1

    def __post_init__(self) -> None:
        if self.primitive_wall_tol < 0.0:
            raise ValueError("primitive_wall_tol must be nonnegative.")
        if self.inward_tol < 0.0:
            raise ValueError("inward_tol must be nonnegative.")
        if self.cone_residual_tol < 0.0:
            raise ValueError("cone_residual_tol must be nonnegative.")
        if self.stencil_radius < 1:
            raise ValueError("stencil_radius must be at least 1.")


@dataclass(frozen=True)
class PrimitiveInwardDiagnostics:
    """
    Analytic inward diagnostics for the primitive closed set

        S = {(k,L): k >= 0, k + L >= 0}.

    At k = 0, require k_dot >= 0.
    At k + L = 0, require k_dot + L_dot >= 0.
    """
    state_status: str
    active_k_wall: bool
    active_diagonal_wall: bool
    k_wall_deficit: float
    diagonal_wall_deficit: float
    accepted: bool
    reason: Optional[str]

    @property
    def max_deficit(self) -> float:
        return max(float(self.k_wall_deficit), float(self.diagonal_wall_deficit))


@dataclass(frozen=True)
class DiscreteTangentDiagnostics:
    """
    Discrete approximation to f in T_A(x) for a candidate grid mask A.
    """
    in_candidate_mask: bool
    accepted: bool
    reason: Optional[str]
    cone_residual: float
    drift_norm: float
    n_generators: int


@dataclass(frozen=True)
class StateConstraintDiagnostics:
    primitive: PrimitiveInwardDiagnostics
    tangent: DiscreteTangentDiagnostics

    @property
    def accepted(self) -> bool:
        return bool(self.primitive.accepted and self.tangent.accepted)

    @property
    def max_violation(self) -> float:
        tangent_violation = 0.0 if self.tangent.accepted else self.tangent.cone_residual
        return max(self.primitive.max_deficit, float(tangent_violation))


def primitive_grid_mask(
    k_grid: Sequence[float],
    L_grid: Sequence[float],
    economy_params: Optional[PlannerEconomyParams] = None,
) -> np.ndarray:
    """
    Return the primitive closed-set mask S_grid on a rectangular grid.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    k_arr = np.asarray(k_grid, dtype=float)
    L_arr = np.asarray(L_grid, dtype=float)

    if k_arr.ndim != 1 or L_arr.ndim != 1:
        raise ValueError("k_grid and L_grid must be one-dimensional.")
    if k_arr.size == 0 or L_arr.size == 0:
        raise ValueError("k_grid and L_grid must be non-empty.")

    out = np.zeros((k_arr.size, L_arr.size), dtype=bool)

    for i, k in enumerate(k_arr):
        for j, L in enumerate(L_arr):
            out[i, j] = primitive_state_diagnostics(
                State(float(k), float(L)),
                economy_params,
            ).is_valid

    return out


def _finite_pair(a: float, b: float) -> bool:
    return math.isfinite(float(a)) and math.isfinite(float(b))


def primitive_inward_diagnostics(
    x: State,
    k_dot: float,
    L_dot: float,
    *,
    economy_params: Optional[PlannerEconomyParams] = None,
    options: Optional[StateConstraintOptions] = None,
) -> PrimitiveInwardDiagnostics:
    """
    Check analytic inward inequalities for primitive walls.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()
    if options is None:
        options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    diag = primitive_state_diagnostics(x, economy_params)

    if not diag.is_valid:
        return PrimitiveInwardDiagnostics(
            state_status=diag.status,
            active_k_wall=False,
            active_diagonal_wall=False,
            k_wall_deficit=math.inf,
            diagonal_wall_deficit=math.inf,
            accepted=False,
            reason=f"invalid primitive state: {diag.invalid_reason}",
        )

    if not _finite_pair(k_dot, L_dot):
        return PrimitiveInwardDiagnostics(
            state_status=diag.status,
            active_k_wall=False,
            active_diagonal_wall=False,
            k_wall_deficit=math.inf,
            diagonal_wall_deficit=math.inf,
            accepted=False,
            reason="non-finite drift",
        )

    active_k_wall = x.k <= options.primitive_wall_tol
    active_diagonal_wall = x.W_K <= options.primitive_wall_tol

    k_wall_deficit = 0.0
    diagonal_wall_deficit = 0.0

    if active_k_wall:
        k_wall_deficit = max(0.0, -float(k_dot))

    if active_diagonal_wall:
        diagonal_wall_deficit = max(0.0, -float(k_dot + L_dot))

    accepted = (
        k_wall_deficit <= options.inward_tol
        and diagonal_wall_deficit <= options.inward_tol
    )

    return PrimitiveInwardDiagnostics(
        state_status=diag.status,
        active_k_wall=bool(active_k_wall),
        active_diagonal_wall=bool(active_diagonal_wall),
        k_wall_deficit=float(k_wall_deficit),
        diagonal_wall_deficit=float(diagonal_wall_deficit),
        accepted=bool(accepted),
        reason=None if accepted else "primitive inward inequality failed",
    )


def _cone_residual_2d(v: np.ndarray, generators: np.ndarray) -> float:
    """
    Relative least-squares residual for v in cone{generators} in R^2.

    By Caratheodory's theorem for cones in R^2, it is enough to test single
    rays and pairs of rays.
    """
    v = np.asarray(v, dtype=float).reshape(2)
    generators = np.asarray(generators, dtype=float)

    v_norm = float(np.linalg.norm(v))
    if v_norm == 0.0:
        return 0.0

    if generators.size == 0:
        return math.inf

    if generators.ndim != 2 or generators.shape[1] != 2:
        raise ValueError("generators must have shape (n, 2).")

    best = math.inf
    n = generators.shape[0]

    # Single-ray projections.
    for g in generators:
        gg = float(np.dot(g, g))
        if gg <= 0.0 or not math.isfinite(gg):
            continue
        a = max(0.0, float(np.dot(v, g)) / gg)
        res = float(np.linalg.norm(v - a * g)) / v_norm
        best = min(best, res)

    # Pair cones.
    for a_idx in range(n):
        for b_idx in range(a_idx + 1, n):
            G = np.column_stack((generators[a_idx], generators[b_idx]))
            det = float(np.linalg.det(G))
            if abs(det) <= 1.0e-14:
                continue
            coeff = np.linalg.solve(G, v)
            if np.all(coeff >= -1.0e-12):
                coeff = np.maximum(coeff, 0.0)
                res = float(np.linalg.norm(v - G @ coeff)) / v_norm
                best = min(best, res)

    return float(best)


def local_tangent_generators(
    candidate_mask: np.ndarray,
    i: int,
    j: int,
    k_grid: Sequence[float],
    L_grid: Sequence[float],
    *,
    radius: int = 1,
) -> np.ndarray:
    """
    Build local displacement generators from active neighbours of (i,j).
    """
    mask = np.asarray(candidate_mask, dtype=bool)
    k_arr = np.asarray(k_grid, dtype=float)
    L_arr = np.asarray(L_grid, dtype=float)

    if mask.shape != (k_arr.size, L_arr.size):
        raise ValueError("candidate_mask shape must equal (len(k_grid), len(L_grid)).")

    if not (0 <= i < k_arr.size and 0 <= j < L_arr.size):
        raise IndexError("grid index out of range.")

    gens: list[tuple[float, float]] = []

    for ii in range(max(0, i - radius), min(k_arr.size, i + radius + 1)):
        for jj in range(max(0, j - radius), min(L_arr.size, j + radius + 1)):
            if ii == i and jj == j:
                continue
            if bool(mask[ii, jj]):
                dk = float(k_arr[ii] - k_arr[i])
                dL = float(L_arr[jj] - L_arr[j])
                if dk != 0.0 or dL != 0.0:
                    gens.append((dk, dL))

    if not gens:
        return np.zeros((0, 2), dtype=float)

    return np.asarray(gens, dtype=float)


def discrete_tangent_diagnostics(
    candidate_mask: np.ndarray,
    i: int,
    j: int,
    k_dot: float,
    L_dot: float,
    k_grid: Sequence[float],
    L_grid: Sequence[float],
    *,
    options: Optional[StateConstraintOptions] = None,
) -> DiscreteTangentDiagnostics:
    """
    Check whether the drift belongs to the local discrete tangent cone of a mask.
    """
    if options is None:
        options = StateConstraintOptions()

    mask = np.asarray(candidate_mask, dtype=bool)
    k_arr = np.asarray(k_grid, dtype=float)
    L_arr = np.asarray(L_grid, dtype=float)

    if mask.shape != (k_arr.size, L_arr.size):
        raise ValueError("candidate_mask shape must equal (len(k_grid), len(L_grid)).")

    if not bool(mask[i, j]):
        return DiscreteTangentDiagnostics(
            in_candidate_mask=False,
            accepted=False,
            reason="node is not in candidate mask",
            cone_residual=math.inf,
            drift_norm=math.inf,
            n_generators=0,
        )

    if not _finite_pair(k_dot, L_dot):
        return DiscreteTangentDiagnostics(
            in_candidate_mask=True,
            accepted=False,
            reason="non-finite drift",
            cone_residual=math.inf,
            drift_norm=math.inf,
            n_generators=0,
        )

    v = np.asarray([float(k_dot), float(L_dot)], dtype=float)
    drift_norm = float(np.linalg.norm(v))

    if drift_norm <= options.inward_tol:
        return DiscreteTangentDiagnostics(
            in_candidate_mask=True,
            accepted=True,
            reason=None,
            cone_residual=0.0,
            drift_norm=drift_norm,
            n_generators=0,
        )

    gens = local_tangent_generators(
        mask,
        i,
        j,
        k_arr,
        L_arr,
        radius=options.stencil_radius,
    )

    residual = _cone_residual_2d(v, gens)
    accepted = residual <= options.cone_residual_tol

    return DiscreteTangentDiagnostics(
        in_candidate_mask=True,
        accepted=bool(accepted),
        reason=None if accepted else "drift is outside local discrete tangent cone",
        cone_residual=float(residual),
        drift_norm=drift_norm,
        n_generators=int(gens.shape[0]),
    )


def state_constraint_diagnostics(
    ev: OracleEval,
    candidate_mask: np.ndarray,
    i: int,
    j: int,
    k_grid: Sequence[float],
    L_grid: Sequence[float],
    *,
    economy_params: Optional[PlannerEconomyParams] = None,
    options: Optional[StateConstraintOptions] = None,
) -> StateConstraintDiagnostics:
    """
    Joint primitive-wall and candidate-mask tangent diagnostics for one oracle eval.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()
    if options is None:
        options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    primitive = primitive_inward_diagnostics(
        ev.state,
        ev.k_dot,
        ev.L_dot,
        economy_params=economy_params,
        options=options,
    )

    tangent = discrete_tangent_diagnostics(
        candidate_mask,
        i,
        j,
        ev.k_dot,
        ev.L_dot,
        k_grid,
        L_grid,
        options=options,
    )

    return StateConstraintDiagnostics(
        primitive=primitive,
        tangent=tangent,
    )


__all__ = [
    "StateConstraintOptions",
    "PrimitiveInwardDiagnostics",
    "DiscreteTangentDiagnostics",
    "StateConstraintDiagnostics",
    "primitive_grid_mask",
    "primitive_inward_diagnostics",
    "local_tangent_generators",
    "discrete_tangent_diagnostics",
    "state_constraint_diagnostics",
]

Overwriting state_constraints.py


In [24]:
%%writefile viability_sets.py
from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Iterable, Optional, Sequence
import math
import numpy as np

try:
    from scipy.optimize import minimize
except Exception:  # pragma: no cover
    minimize = None

from automation_block import AutomationParams, RegimePrimitives, build_regime_primitives
from model.economy import State, Control, PlannerEconomyParams, primitive_state_diagnostics
import policy_sets
from policy_sets import PolicySetOptions
from asset_market import AssetMarketParams, make_infinite_asset_market_params
from continuation_block import ContinuationBundle, make_test_continuation_bundle
from equilibrium_oracle import OracleOptions, live_oracle
from state_constraints import (
    StateConstraintOptions,
    local_tangent_generators,
    primitive_grid_mask,
    primitive_inward_diagnostics,
)


# ============================================================
# Block 7B contract: pure conditional viability sets
# ============================================================
#
# This module computes the Plan-11 viability objects
#
#     V_1^{hat u} = Viab_{F_1^{hat u}}(S),
#     V_0^{hat u} = Viab_{F_0^{hat u}}(S ∩ V_1^{hat u}),
#
# for a frozen continuation bundle C[hat u].
#
# Viability is an existence problem over current controls. It is not a
# Hamiltonian maximisation problem.
#
# The transfer control is handled as semi-infinite:
#
#     T >= T_lower.
#
# The solver does not impose the artificial compact transfer cap when deciding
# viability. Instead, for each (tau,H) candidate it uses the exact Mode-A affine
# transfer direction
#
#     d f / dT = (-1, +1),
#
# and checks whether some T >= T_lower puts the drift in the discrete tangent
# cone.
#
# Forbidden responsibilities:
#   - no private continuation solve;
#   - no planner Hamiltonian maximisation;
#   - no Howard iteration;
#   - no outer fixed point.


TRANSFER_DRIFT_DIRECTION = np.asarray([-1.0, 1.0], dtype=float)


@dataclass(frozen=True)
class ViabilityGrid:
    k_grid: np.ndarray
    L_grid: np.ndarray

    def __post_init__(self) -> None:
        k = np.asarray(self.k_grid, dtype=float)
        L = np.asarray(self.L_grid, dtype=float)

        if k.ndim != 1 or L.ndim != 1:
            raise ValueError("k_grid and L_grid must be one-dimensional.")
        if k.size == 0 or L.size == 0:
            raise ValueError("k_grid and L_grid must be non-empty.")
        if not np.all(np.isfinite(k)) or not np.all(np.isfinite(L)):
            raise ValueError("k_grid and L_grid must be finite.")
        if np.any(np.diff(k) <= 0.0) or np.any(np.diff(L) <= 0.0):
            raise ValueError("k_grid and L_grid must be strictly increasing.")

        object.__setattr__(self, "k_grid", k)
        object.__setattr__(self, "L_grid", L)

    @property
    def shape(self) -> tuple[int, int]:
        return (self.k_grid.size, self.L_grid.size)

    def state(self, i: int, j: int) -> State:
        return State(float(self.k_grid[i]), float(self.L_grid[j]))


@dataclass(frozen=True)
class TauHBounds:
    """
    Bounds used by the full-policy-set viability search.

    T_lower is the true lower transfer bound. There is no T_upper here.
    tau_upper_closed is the numerical closed representation of the primitive
    open upper bound tau < tau_upper.
    """
    tau_lower: float
    tau_upper_closed: float
    H_lower: float
    H_upper: float
    T_lower: float

    def __post_init__(self) -> None:
        for name in ("tau_lower", "tau_upper_closed", "H_lower", "H_upper", "T_lower"):
            val = float(getattr(self, name))
            if not math.isfinite(val):
                raise ValueError(f"{name} must be finite.")
            object.__setattr__(self, name, val)

        if self.tau_lower > self.tau_upper_closed:
            raise ValueError("tau bounds are inconsistent.")
        if self.H_lower > self.H_upper:
            raise ValueError("H bounds are inconsistent.")

    def tau_width(self) -> float:
        return max(0.0, self.tau_upper_closed - self.tau_lower)

    def H_width(self) -> float:
        return max(0.0, self.H_upper - self.H_lower)


@dataclass(frozen=True)
class TransferRaySolve:
    feasible: bool
    dT: float
    T: float
    k_dot: float
    L_dot: float
    primitive_deficit: float
    cone_residual: float
    objective: float
    reason: str
    n_generators: int


@dataclass(frozen=True)
class ViabilityOptions:
    """
    Options for the Block 7 peeling and witness search.

    The viability decision uses the full transfer half-line T >= T_lower. The
    optional tiny grid is over (tau,H) only and is a fallback diagnostic, not the
    definition of the kernel.
    """
    max_peel_iter: int = 200
    objective_tol: float = 1.0e-8
    local_solver_maxiter: int = 120
    use_local_solver: bool = True
    tiny_tau_H_grid_size: int = 0

    accept_portfolio_bound_as_viable: bool = False
    verbose: bool = False

    def __post_init__(self) -> None:
        if self.max_peel_iter < 1:
            raise ValueError("max_peel_iter must be at least 1.")
        if self.objective_tol < 0.0:
            raise ValueError("objective_tol must be nonnegative.")
        if self.local_solver_maxiter < 1:
            raise ValueError("local_solver_maxiter must be at least 1.")
        if self.tiny_tau_H_grid_size < 0:
            raise ValueError("tiny_tau_H_grid_size must be nonnegative.")


@dataclass(frozen=True)
class ViabilityWitness:
    feasible: bool
    control: Optional[Control]
    reason: str

    oracle_status: Optional[str]
    oracle_reason: Optional[str]

    k_dot: float = math.nan
    L_dot: float = math.nan
    W_K_dot: float = math.nan

    objective: float = math.inf
    primitive_deficit: float = math.inf
    cone_residual: float = math.inf
    dT_from_floor: float = math.nan
    n_tangent_generators: int = 0

    binds_tau_upper_cap: bool = False
    binds_H_bound: bool = False
    source: str = "none"


@dataclass(frozen=True)
class ViabilityKernel:
    regime: int
    grid: ViabilityGrid
    initial_mask: np.ndarray
    mask: np.ndarray

    tau: np.ndarray
    T: np.ndarray
    H: np.ndarray
    k_dot: np.ndarray
    L_dot: np.ndarray

    converged: bool
    n_iter: int
    n_initial: int
    n_viable: int
    n_removed: int

    uses_full_transfer_halfline: bool
    diagnostics: dict[str, float]

    def witness_control(self, i: int, j: int) -> Optional[Control]:
        if not bool(self.mask[i, j]):
            return None
        return Control(
            tau=float(self.tau[i, j]),
            T=float(self.T[i, j]),
            H=float(self.H[i, j]),
        )


@dataclass(frozen=True)
class ConditionalViabilityResult:
    grid: ViabilityGrid
    primitive_mask: np.ndarray
    V1: ViabilityKernel
    V0: ViabilityKernel
    diagnostics: dict[str, float]


# ============================================================
# Helpers
# ============================================================

def _require_regime(s: int) -> int:
    if s not in (0, 1):
        raise ValueError("regime s must be 0 or 1.")
    return int(s)


def _empty_float(shape: tuple[int, int]) -> np.ndarray:
    out = np.empty(shape, dtype=float)
    out.fill(np.nan)
    return out


def _clamp(v: float, lo: float, hi: float) -> float:
    return min(max(float(v), float(lo)), float(hi))


def _dedupe_pairs(
    pairs: Iterable[tuple[float, float]],
    *,
    tol: float = 1.0e-10,
) -> list[tuple[float, float]]:
    seen: set[tuple[int, int]] = set()
    out: list[tuple[float, float]] = []
    scale = 1.0 / max(tol, 1.0e-16)

    for tau, H in pairs:
        key = (
            int(round(float(tau) * scale)),
            int(round(float(H) * scale)),
        )
        if key not in seen:
            seen.add(key)
            out.append((float(tau), float(H)))

    return out


def _tau_H_bounds(
    s: int,
    x: State,
    *,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
) -> TauHBounds:
    """
    Bounds for the true viability controls.

    T uses the full lower bound only. Tau uses the standard closed numerical
    approximation to the primitive open upper bound tau < tau_upper.
    """
    full = policy_sets.full_policy_bounds(
        s,
        x,
        primitives,
        economy_params,
    )

    tau_upper_closed = full.tau_upper - policy_options.tau_upper_margin

    if tau_upper_closed < full.tau_lower:
        raise ValueError("tau_upper_margin is too large for the tau interval.")

    return TauHBounds(
        tau_lower=full.tau_lower,
        tau_upper_closed=tau_upper_closed,
        H_lower=full.H_lower,
        H_upper=full.H_upper,
        T_lower=full.T_lower,
    )


def _z_to_tau_H(z: Sequence[float], bounds: TauHBounds) -> tuple[float, float]:
    z = np.clip(np.asarray(z, dtype=float), 0.0, 1.0)

    tau = bounds.tau_lower + z[0] * (
        bounds.tau_upper_closed - bounds.tau_lower
    )
    H = bounds.H_lower + z[1] * (
        bounds.H_upper - bounds.H_lower
    )

    return float(tau), float(H)


def _tau_H_to_z(tau: float, H: float, bounds: TauHBounds) -> np.ndarray:
    z = np.asarray([0.5, 0.5], dtype=float)

    if bounds.tau_width() > 0.0:
        z[0] = (tau - bounds.tau_lower) / bounds.tau_width()

    if bounds.H_width() > 0.0:
        z[1] = (H - bounds.H_lower) / bounds.H_width()

    return np.clip(z, 0.0, 1.0)


def _same_node_witness_tau_H(
    kernel: Optional[ViabilityKernel],
    i: int,
    j: int,
) -> list[tuple[float, float]]:
    if kernel is None:
        return []

    u = kernel.witness_control(i, j)
    return [] if u is None else [(u.tau, u.H)]


def _neighbour_witness_tau_H(
    kernel: Optional[ViabilityKernel],
    i: int,
    j: int,
    *,
    radius: int = 1,
) -> list[tuple[float, float]]:
    if kernel is None:
        return []

    out: list[tuple[float, float]] = []
    n_k, n_L = kernel.mask.shape

    for ii in range(max(0, i - radius), min(n_k, i + radius + 1)):
        for jj in range(max(0, j - radius), min(n_L, j + radius + 1)):
            if ii == i and jj == j:
                continue
            u = kernel.witness_control(ii, jj)
            if u is not None:
                out.append((u.tau, u.H))

    return out


# ============================================================
# Full-transfer-halfline tangent solve
# ============================================================

def _primitive_transfer_upper_bound(
    x: State,
    k_dot_floor: float,
    L_dot_floor: float,
    *,
    economy_params: PlannerEconomyParams,
    state_options: StateConstraintOptions,
) -> tuple[bool, float, float, str]:
    """
    Return the admissible interval dT in [0, dT_max] implied by primitive
    inward constraints, where

        f(T_lower + dT) = f(T_lower) + dT * (-1, +1).
    """
    diag = primitive_state_diagnostics(x, economy_params)

    if not diag.is_valid:
        return False, math.nan, math.inf, f"invalid primitive state: {diag.invalid_reason}"

    if not (math.isfinite(k_dot_floor) and math.isfinite(L_dot_floor)):
        return False, math.nan, math.inf, "non-finite floor drift"

    active_k_wall = x.k <= state_options.primitive_wall_tol
    active_diag_wall = x.W_K <= state_options.primitive_wall_tol

    dT_max = math.inf

    if active_diag_wall:
        W_dot_floor = k_dot_floor + L_dot_floor
        if W_dot_floor < -state_options.inward_tol:
            deficit = max(0.0, -W_dot_floor)
            return False, math.nan, deficit, "diagonal inward inequality failed"

    if active_k_wall:
        # k_dot(T) = k_dot_floor - dT, so transfers can only worsen
        # k-wall inwardness.
        dT_max = k_dot_floor + state_options.inward_tol

        if dT_max < 0.0:
            deficit = max(0.0, -k_dot_floor)
            return False, math.nan, deficit, "k-wall inward inequality failed"

    return True, float(dT_max), 0.0, "primitive inward interval nonempty"


def _relative_residual(v: np.ndarray, approx: np.ndarray) -> float:
    scale = max(1.0, float(np.linalg.norm(v)))
    return float(np.linalg.norm(v - approx) / scale)


def _single_ray_solution(
    target: np.ndarray,
    ray: np.ndarray,
) -> Optional[tuple[float, float]]:
    rr = float(np.dot(ray, ray))
    if rr <= 0.0 or not math.isfinite(rr):
        return None

    a = float(np.dot(target, ray) / rr)
    approx = a * ray

    return a, _relative_residual(target, approx)


def _cone_residual_for_diagnostic(
    v: np.ndarray,
    generators: np.ndarray,
) -> float:
    v = np.asarray(v, dtype=float).reshape(2)
    generators = np.asarray(generators, dtype=float)

    v_norm = float(np.linalg.norm(v))

    if v_norm == 0.0:
        return 0.0

    if generators.size == 0:
        return math.inf

    if generators.ndim != 2 or generators.shape[1] != 2:
        raise ValueError("generators must have shape (n, 2).")

    best = math.inf
    n = generators.shape[0]

    for g in generators:
        gg = float(np.dot(g, g))
        if gg <= 0.0 or not math.isfinite(gg):
            continue
        a = max(0.0, float(np.dot(v, g)) / gg)
        best = min(best, float(np.linalg.norm(v - a * g)) / v_norm)

    for a_idx in range(n):
        for b_idx in range(a_idx + 1, n):
            G = np.column_stack((generators[a_idx], generators[b_idx]))
            det = float(np.linalg.det(G))
            if abs(det) <= 1.0e-14:
                continue

            coeff = np.linalg.solve(G, v)

            if np.all(coeff >= -1.0e-12):
                coeff = np.maximum(coeff, 0.0)
                best = min(
                    best,
                    float(np.linalg.norm(v - G @ coeff)) / v_norm,
                )

    return float(best)


def _bounded_transfer_ray_tangent_solve(
    f_floor: np.ndarray,
    generators: np.ndarray,
    *,
    T_lower: float,
    dT_upper: float,
    options: StateConstraintOptions,
) -> TransferRaySolve:
    """
    Decide whether there exists dT >= 0 such that

        f_floor + dT * (-1,+1) in cone(generators).

    This is solved exactly in R^2 by writing

        f_floor = cone(generators) + dT * (1,-1),

    and enumerating one- and two-ray conic representations. The last augmented
    ray is -TRANSFER_DRIFT_DIRECTION = (1,-1), whose coefficient is dT.
    """
    f_floor = np.asarray(f_floor, dtype=float).reshape(2)
    generators = np.asarray(generators, dtype=float)

    if generators.size == 0:
        generators = np.zeros((0, 2), dtype=float)

    if generators.ndim != 2 or generators.shape[1] != 2:
        raise ValueError("generators must have shape (n, 2).")

    q = TRANSFER_DRIFT_DIRECTION
    minus_q = -q
    target = f_floor
    target_norm = float(np.linalg.norm(target))

    if target_norm <= options.inward_tol:
        return TransferRaySolve(
            feasible=True,
            dT=0.0,
            T=float(T_lower),
            k_dot=float(f_floor[0]),
            L_dot=float(f_floor[1]),
            primitive_deficit=0.0,
            cone_residual=0.0,
            objective=0.0,
            reason="zero drift",
            n_generators=int(generators.shape[0]),
        )

    rays = list(generators) + [minus_q]
    transfer_ray_idx = len(rays) - 1

    best_residual = math.inf
    best_dT = 0.0
    best_v = f_floor.copy()

    def update_best(dT: float, residual: float) -> None:
        nonlocal best_residual, best_dT, best_v

        if dT < -options.inward_tol:
            return

        if math.isfinite(dT_upper) and dT > dT_upper + options.inward_tol:
            return

        dT_clip = max(0.0, float(dT))

        if residual < best_residual:
            best_residual = float(residual)
            best_dT = dT_clip
            best_v = f_floor + dT_clip * q

    # Single-ray conic representations of f_floor.
    for idx, ray in enumerate(rays):
        sol = _single_ray_solution(target, np.asarray(ray, dtype=float))
        if sol is None:
            continue

        coeff, residual = sol

        if coeff >= -options.cone_residual_tol:
            dT = coeff if idx == transfer_ray_idx else 0.0
            update_best(dT, residual)

    # Two-ray conic representations of f_floor.
    n = len(rays)

    for a_idx in range(n):
        for b_idx in range(a_idx + 1, n):
            G = np.column_stack((rays[a_idx], rays[b_idx]))
            det = float(np.linalg.det(G))

            if abs(det) <= 1.0e-14:
                continue

            coeff = np.linalg.solve(G, target)
            approx = G @ coeff
            residual = _relative_residual(target, approx)

            if np.all(coeff >= -options.cone_residual_tol):
                dT = 0.0

                if a_idx == transfer_ray_idx:
                    dT = float(coeff[0])
                elif b_idx == transfer_ray_idx:
                    dT = float(coeff[1])

                update_best(dT, residual)

    feasible = best_residual <= options.cone_residual_tol

    if feasible:
        return TransferRaySolve(
            feasible=True,
            dT=float(best_dT),
            T=float(T_lower + best_dT),
            k_dot=float(best_v[0]),
            L_dot=float(best_v[1]),
            primitive_deficit=0.0,
            cone_residual=float(best_residual),
            objective=float(best_residual),
            reason="transfer ray intersects tangent cone",
            n_generators=int(generators.shape[0]),
        )

    # Diagnostic residual from floor / endpoint candidates.
    candidates = [0.0]

    if math.isfinite(dT_upper):
        candidates.append(max(0.0, dT_upper))

    if best_residual < math.inf:
        candidates.append(best_dT)

    for d in candidates:
        if d < 0.0:
            continue
        if math.isfinite(dT_upper) and d > dT_upper:
            continue

        v = f_floor + d * q
        residual = _cone_residual_for_diagnostic(v, generators)
        update_best(d, residual)

    return TransferRaySolve(
        feasible=False,
        dT=float(best_dT),
        T=float(T_lower + best_dT),
        k_dot=float(best_v[0]),
        L_dot=float(best_v[1]),
        primitive_deficit=0.0,
        cone_residual=float(best_residual),
        objective=float(best_residual),
        reason="transfer ray does not intersect tangent cone",
        n_generators=int(generators.shape[0]),
    )


# ============================================================
# Witness evaluation
# ============================================================

def _oracle_full_options(base: Optional[OracleOptions]) -> OracleOptions:
    if base is None:
        return OracleOptions(control_set="full")
    return replace(base, control_set="full")


def evaluate_tau_H_witness(
    s: int,
    x: State,
    tau: float,
    H: float,
    i: int,
    j: int,
    candidate_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    viability_options: ViabilityOptions,
    source: str,
) -> ViabilityWitness:
    """
    Test whether some T >= T_lower makes (tau,T,H) a viability witness.
    """
    try:
        bounds = _tau_H_bounds(
            s=s,
            x=x,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    except Exception as exc:
        return ViabilityWitness(
            feasible=False,
            control=None,
            reason=f"failed to construct full bounds: {exc}",
            oracle_status=None,
            oracle_reason=str(exc),
            source=source,
        )

    tau = _clamp(tau, bounds.tau_lower, bounds.tau_upper_closed)
    H = _clamp(H, bounds.H_lower, bounds.H_upper)

    u_floor = Control(
        tau=tau,
        T=bounds.T_lower,
        H=H,
    )

    try:
        ev_floor = live_oracle(
            s=s,
            x=x,
            u=u_floor,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            options=oracle_options,
        )
    except Exception as exc:
        return ViabilityWitness(
            feasible=False,
            control=u_floor,
            reason=f"oracle exception at transfer floor: {exc}",
            oracle_status=None,
            oracle_reason=str(exc),
            source=source,
        )

    if ev_floor.status == "portfolio_bind" and not viability_options.accept_portfolio_bound_as_viable:
        return ViabilityWitness(
            feasible=False,
            control=u_floor,
            reason="portfolio branch is not implemented as a viable complementarity branch",
            oracle_status=ev_floor.status,
            oracle_reason=ev_floor.reason,
            k_dot=ev_floor.k_dot,
            L_dot=ev_floor.L_dot,
            W_K_dot=ev_floor.W_K_dot,
            source=source,
        )

    if not ev_floor.valid_for_drift:
        return ViabilityWitness(
            feasible=False,
            control=u_floor,
            reason="oracle did not return a valid drift at the transfer floor",
            oracle_status=ev_floor.status,
            oracle_reason=ev_floor.reason,
            k_dot=ev_floor.k_dot,
            L_dot=ev_floor.L_dot,
            W_K_dot=ev_floor.W_K_dot,
            source=source,
        )

    ok_interval, dT_upper, primitive_deficit, primitive_reason = (
        _primitive_transfer_upper_bound(
            x,
            ev_floor.k_dot,
            ev_floor.L_dot,
            economy_params=economy_params,
            state_options=state_options,
        )
    )

    if not ok_interval:
        return ViabilityWitness(
            feasible=False,
            control=u_floor,
            reason=primitive_reason,
            oracle_status=ev_floor.status,
            oracle_reason=ev_floor.reason,
            k_dot=ev_floor.k_dot,
            L_dot=ev_floor.L_dot,
            W_K_dot=ev_floor.W_K_dot,
            objective=float(primitive_deficit),
            primitive_deficit=float(primitive_deficit),
            cone_residual=math.inf,
            dT_from_floor=0.0,
            source=source,
        )

    generators = local_tangent_generators(
        candidate_mask,
        i,
        j,
        grid.k_grid,
        grid.L_grid,
        radius=state_options.stencil_radius,
    )

    ray = _bounded_transfer_ray_tangent_solve(
        np.asarray([ev_floor.k_dot, ev_floor.L_dot], dtype=float),
        generators,
        T_lower=bounds.T_lower,
        dT_upper=dT_upper,
        options=state_options,
    )

    u_star = Control(
        tau=tau,
        T=ray.T,
        H=H,
    )

    if not ray.feasible:
        return ViabilityWitness(
            feasible=False,
            control=u_star,
            reason=ray.reason,
            oracle_status=ev_floor.status,
            oracle_reason=ev_floor.reason,
            k_dot=ray.k_dot,
            L_dot=ray.L_dot,
            W_K_dot=ray.k_dot + ray.L_dot,
            objective=ray.objective,
            primitive_deficit=ray.primitive_deficit,
            cone_residual=ray.cone_residual,
            dT_from_floor=ray.dT,
            n_tangent_generators=ray.n_generators,
            binds_tau_upper_cap=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
            binds_H_bound=(
                abs(H - bounds.H_lower) <= policy_options.bound_tol
                or abs(H - bounds.H_upper) <= policy_options.bound_tol
            ),
            source=source,
        )

    # Re-evaluate at accepted T to preserve OracleEval status semantics.
    ev_star = live_oracle(
        s=s,
        x=x,
        u=u_star,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=oracle_options,
    )

    if not ev_star.valid_for_drift:
        return ViabilityWitness(
            feasible=False,
            control=u_star,
            reason="accepted transfer ray failed final oracle drift validation",
            oracle_status=ev_star.status,
            oracle_reason=ev_star.reason,
            objective=math.inf,
            source=source,
        )

    primitive_final = primitive_inward_diagnostics(
        x,
        ev_star.k_dot,
        ev_star.L_dot,
        economy_params=economy_params,
        options=state_options,
    )

    if not primitive_final.accepted:
        return ViabilityWitness(
            feasible=False,
            control=u_star,
            reason="accepted transfer ray failed final primitive inward validation",
            oracle_status=ev_star.status,
            oracle_reason=ev_star.reason,
            k_dot=ev_star.k_dot,
            L_dot=ev_star.L_dot,
            W_K_dot=ev_star.W_K_dot,
            objective=primitive_final.max_deficit,
            primitive_deficit=primitive_final.max_deficit,
            cone_residual=ray.cone_residual,
            dT_from_floor=ray.dT,
            source=source,
        )

    return ViabilityWitness(
        feasible=True,
        control=u_star,
        reason="accepted",
        oracle_status=ev_star.status,
        oracle_reason=ev_star.reason,
        k_dot=ev_star.k_dot,
        L_dot=ev_star.L_dot,
        W_K_dot=ev_star.W_K_dot,
        objective=0.0,
        primitive_deficit=primitive_final.max_deficit,
        cone_residual=ray.cone_residual,
        dT_from_floor=ray.dT,
        n_tangent_generators=ray.n_generators,
        binds_tau_upper_cap=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
        binds_H_bound=(
            abs(H - bounds.H_lower) <= policy_options.bound_tol
            or abs(H - bounds.H_upper) <= policy_options.bound_tol
        ),
        source=source,
    )


# ============================================================
# Candidate generation and search
# ============================================================

def analytic_tau_H_candidates(
    s: int,
    x: State,
    *,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
) -> list[tuple[float, float]]:
    s = _require_regime(s)

    bounds = _tau_H_bounds(
        s=s,
        x=x,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    )

    tau_vals = [
        bounds.tau_lower,
        0.5 * (bounds.tau_lower + bounds.tau_upper_closed),
        bounds.tau_upper_closed,
    ]

    H_vals = [
        bounds.H_lower,
        0.5 * (bounds.H_lower + bounds.H_upper),
        bounds.H_upper,
    ]

    if x.W_K > 0.0:
        # Maximises (k-H)(L+H)/(k+L) subject to H bounds.
        H_xi = _clamp(
            0.5 * (x.k - x.L),
            bounds.H_lower,
            bounds.H_upper,
        )
        H_vals.append(H_xi)

    return _dedupe_pairs(
        [(tau, H) for tau in tau_vals for H in H_vals],
        tol=policy_options.bound_tol,
    )


def _tiny_tau_H_grid(
    bounds: TauHBounds,
    n: int,
) -> list[tuple[float, float]]:
    if n <= 0:
        return []

    if n == 1:
        return [_z_to_tau_H((0.5, 0.5), bounds)]

    zs = np.linspace(0.0, 1.0, n)
    return [
        _z_to_tau_H((z_tau, z_H), bounds)
        for z_tau in zs
        for z_H in zs
    ]


def find_viability_witness(
    s: int,
    i: int,
    j: int,
    candidate_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    viability_options: ViabilityOptions,
    previous_kernel: Optional[ViabilityKernel] = None,
    current_warm_kernel: Optional[ViabilityKernel] = None,
) -> ViabilityWitness:
    """
    Search for a current-control witness at one node.

    Hierarchy:
      previous same-node witness
      -> neighbouring witnesses
      -> analytic tau/H candidates
      -> continuous local feasibility solve over tau/H
      -> optional tiny tau/H rescue grid.

    The transfer T is not gridded or capped. For each tau/H, the full half-line
    T >= T_lower is handled by the transfer-ray tangent solver.
    """
    s = _require_regime(s)
    x = grid.state(i, j)

    if not bool(candidate_mask[i, j]):
        return ViabilityWitness(
            feasible=False,
            control=None,
            reason="node not in candidate mask",
            oracle_status=None,
            oracle_reason=None,
        )

    try:
        bounds = _tau_H_bounds(
            s=s,
            x=x,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    except Exception as exc:
        return ViabilityWitness(
            feasible=False,
            control=None,
            reason=f"failed to construct full bounds: {exc}",
            oracle_status=None,
            oracle_reason=str(exc),
        )

    candidates: list[tuple[str, float, float]] = []

    for tau, H in _same_node_witness_tau_H(previous_kernel, i, j):
        candidates.append(("previous_same_node", tau, H))

    for tau, H in _same_node_witness_tau_H(current_warm_kernel, i, j):
        candidates.append(("warm_same_node", tau, H))

    for tau, H in _neighbour_witness_tau_H(current_warm_kernel, i, j, radius=1):
        candidates.append(("neighbour_witness", tau, H))

    for tau, H in analytic_tau_H_candidates(
        s,
        x,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    ):
        candidates.append(("analytic", tau, H))

    # Clamp and de-duplicate.
    scale = 1.0 / max(policy_options.bound_tol, 1.0e-16)
    seen: set[tuple[int, int]] = set()
    deduped: list[tuple[str, float, float]] = []

    for source, tau, H in candidates:
        tau = _clamp(tau, bounds.tau_lower, bounds.tau_upper_closed)
        H = _clamp(H, bounds.H_lower, bounds.H_upper)
        key = (
            int(round(tau * scale)),
            int(round(H * scale)),
        )
        if key not in seen:
            seen.add(key)
            deduped.append((source, tau, H))

    candidates = deduped

    best = ViabilityWitness(
        feasible=False,
        control=None,
        reason="no candidate tested",
        oracle_status=None,
        oracle_reason=None,
    )

    for source, tau, H in candidates:
        w = evaluate_tau_H_witness(
            s=s,
            x=x,
            tau=tau,
            H=H,
            i=i,
            j=j,
            candidate_mask=candidate_mask,
            grid=grid,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            viability_options=viability_options,
            source=source,
        )

        if w.feasible:
            return w

        if w.objective < best.objective:
            best = w

    # Continuous local feasibility solve over tau/H.
    if viability_options.use_local_solver and minimize is not None:
        starts = [
            _tau_H_to_z(tau, H, bounds)
            for _, tau, H in candidates
        ]
        starts.append(np.asarray([0.5, 0.5], dtype=float))

        unique_starts: list[np.ndarray] = []

        for z in starts:
            if not any(np.allclose(z, old) for old in unique_starts):
                unique_starts.append(z)

        def obj(z: Sequence[float]) -> float:
            tau, H = _z_to_tau_H(z, bounds)

            w = evaluate_tau_H_witness(
                s=s,
                x=x,
                tau=tau,
                H=H,
                i=i,
                j=j,
                candidate_mask=candidate_mask,
                grid=grid,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                viability_options=viability_options,
                source="local_solver",
            )

            if w.feasible:
                return 0.0

            if math.isfinite(w.objective):
                return float(w.objective)

            return 1.0e6

        for z0 in unique_starts:
            try:
                res = minimize(
                    obj,
                    z0,
                    method="Nelder-Mead",
                    options={
                        "maxiter": viability_options.local_solver_maxiter,
                        "xatol": viability_options.objective_tol,
                        "fatol": viability_options.objective_tol,
                        "disp": False,
                    },
                )

                tau, H = _z_to_tau_H(res.x, bounds)

                w = evaluate_tau_H_witness(
                    s=s,
                    x=x,
                    tau=tau,
                    H=H,
                    i=i,
                    j=j,
                    candidate_mask=candidate_mask,
                    grid=grid,
                    primitives=primitives,
                    continuation=continuation,
                    asset_params=asset_params,
                    economy_params=economy_params,
                    policy_options=policy_options,
                    oracle_options=oracle_options,
                    state_options=state_options,
                    viability_options=viability_options,
                    source="local_solver",
                )

                if w.feasible:
                    return w

                if w.objective < best.objective:
                    best = w

            except Exception:
                continue

    # Optional tiny tau/H grid for diagnostics.
    if viability_options.tiny_tau_H_grid_size > 0:
        for tau, H in _tiny_tau_H_grid(
            bounds,
            viability_options.tiny_tau_H_grid_size,
        ):
            w = evaluate_tau_H_witness(
                s=s,
                x=x,
                tau=tau,
                H=H,
                i=i,
                j=j,
                candidate_mask=candidate_mask,
                grid=grid,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                viability_options=viability_options,
                source="tiny_tau_H_grid",
            )

            if w.feasible:
                return w

            if w.objective < best.objective:
                best = w

    if best.control is None:
        return ViabilityWitness(
            feasible=False,
            control=None,
            reason="no viable witness found",
            oracle_status=None,
            oracle_reason=None,
        )

    return replace(best, reason="no viable witness found")


# ============================================================
# Peeling operator
# ============================================================

def peel_viability_kernel(
    s: int,
    initial_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: Optional[OracleOptions] = None,
    state_options: Optional[StateConstraintOptions] = None,
    viability_options: Optional[ViabilityOptions] = None,
    previous_kernel: Optional[ViabilityKernel] = None,
) -> ViabilityKernel:
    """
    Compute a single-regime pure viability kernel as a greatest fixed point.
    """
    s = _require_regime(s)
    oracle_options = _oracle_full_options(oracle_options)

    if state_options is None:
        state_options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    if viability_options is None:
        viability_options = ViabilityOptions()

    mask0 = np.asarray(initial_mask, dtype=bool)

    if mask0.shape != grid.shape:
        raise ValueError("initial_mask shape must match grid.shape.")

    current = mask0.copy()
    shape = grid.shape

    tau = _empty_float(shape)
    T = _empty_float(shape)
    H = _empty_float(shape)
    k_dot = _empty_float(shape)
    L_dot = _empty_float(shape)

    current_kernel: Optional[ViabilityKernel] = previous_kernel
    converged = False
    n_iter = 0
    last_removed = math.nan

    for it in range(1, viability_options.max_peel_iter + 1):
        n_iter = it

        next_mask = current.copy()

        next_tau = _empty_float(shape)
        next_T = _empty_float(shape)
        next_H = _empty_float(shape)
        next_k_dot = _empty_float(shape)
        next_L_dot = _empty_float(shape)

        removed_this_iter = 0

        for i in range(shape[0]):
            for j in range(shape[1]):
                if not bool(current[i, j]):
                    continue

                witness = find_viability_witness(
                    s=s,
                    i=i,
                    j=j,
                    candidate_mask=current,
                    grid=grid,
                    primitives=primitives,
                    continuation=continuation,
                    asset_params=asset_params,
                    economy_params=economy_params,
                    policy_options=policy_options,
                    oracle_options=oracle_options,
                    state_options=state_options,
                    viability_options=viability_options,
                    previous_kernel=previous_kernel,
                    current_warm_kernel=current_kernel,
                )

                if not witness.feasible or witness.control is None:
                    next_mask[i, j] = False
                    removed_this_iter += 1
                    continue

                next_tau[i, j] = witness.control.tau
                next_T[i, j] = witness.control.T
                next_H[i, j] = witness.control.H
                next_k_dot[i, j] = witness.k_dot
                next_L_dot[i, j] = witness.L_dot

        tau, T, H, k_dot, L_dot = (
            next_tau,
            next_T,
            next_H,
            next_k_dot,
            next_L_dot,
        )

        current = next_mask
        last_removed = float(removed_this_iter)

        if viability_options.verbose:
            print(
                f"regime {s} peel {it}: "
                f"kept={int(current.sum())}, "
                f"removed={removed_this_iter}"
            )

        current_kernel = ViabilityKernel(
            regime=s,
            grid=grid,
            initial_mask=mask0,
            mask=current,
            tau=tau,
            T=T,
            H=H,
            k_dot=k_dot,
            L_dot=L_dot,
            converged=False,
            n_iter=it,
            n_initial=int(mask0.sum()),
            n_viable=int(current.sum()),
            n_removed=int(mask0.sum() - current.sum()),
            uses_full_transfer_halfline=True,
            diagnostics={},
        )

        if removed_this_iter == 0:
            converged = True
            break

    # Final witness diagnostics.
    n_tau_cap = 0
    n_H_bound = 0
    n_large_T = 0
    n_wit = 0
    max_T = -math.inf

    for i in range(shape[0]):
        for j in range(shape[1]):
            if not bool(current[i, j]):
                continue

            n_wit += 1
            x = grid.state(i, j)

            try:
                b = _tau_H_bounds(
                    s=s,
                    x=x,
                    primitives=primitives,
                    economy_params=economy_params,
                    policy_options=policy_options,
                )

                if abs(tau[i, j] - b.tau_upper_closed) <= policy_options.bound_tol:
                    n_tau_cap += 1

                if (
                    abs(H[i, j] - b.H_lower) <= policy_options.bound_tol
                    or abs(H[i, j] - b.H_upper) <= policy_options.bound_tol
                ):
                    n_H_bound += 1

                large_T_threshold = (
                    b.T_lower
                    + 10.0
                    * (
                        1.0
                        + abs(b.T_lower)
                        + max(x.k, 0.0)
                        + max(x.W_K, 0.0)
                    )
                )

                if T[i, j] > large_T_threshold:
                    n_large_T += 1

                max_T = max(max_T, float(T[i, j]))

            except Exception:
                pass

    return ViabilityKernel(
        regime=s,
        grid=grid,
        initial_mask=mask0,
        mask=current,
        tau=tau,
        T=T,
        H=H,
        k_dot=k_dot,
        L_dot=L_dot,
        converged=bool(converged),
        n_iter=int(n_iter),
        n_initial=int(mask0.sum()),
        n_viable=int(current.sum()),
        n_removed=int(mask0.sum() - current.sum()),
        uses_full_transfer_halfline=True,
        diagnostics={
            "last_removed": float(last_removed),
            "n_witnesses": float(n_wit),
            "n_tau_upper_cap_witnesses": float(n_tau_cap),
            "share_tau_upper_cap_witnesses": (
                0.0 if n_wit == 0 else float(n_tau_cap / n_wit)
            ),
            "n_H_bound_witnesses": float(n_H_bound),
            "share_H_bound_witnesses": (
                0.0 if n_wit == 0 else float(n_H_bound / n_wit)
            ),
            "n_large_transfer_witnesses": float(n_large_T),
            "max_witness_T": 0.0 if max_T == -math.inf else float(max_T),
        },
    )


# ============================================================
# Conditional two-regime viability
# ============================================================

def compute_conditional_viability_sets(
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    oracle_options: Optional[OracleOptions] = None,
    state_options: Optional[StateConstraintOptions] = None,
    viability_options: Optional[ViabilityOptions] = None,
    previous_V1: Optional[ViabilityKernel] = None,
    previous_V0: Optional[ViabilityKernel] = None,
) -> ConditionalViabilityResult:
    """
    Compute Plan-11 conditional viability sets:

        V1 = Viab_{F1}(S),
        V0 = Viab_{F0}(S ∩ V1).
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if state_options is None:
        state_options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    if viability_options is None:
        viability_options = ViabilityOptions()

    oracle_options = _oracle_full_options(oracle_options)

    S_mask = primitive_grid_mask(
        grid.k_grid,
        grid.L_grid,
        economy_params=economy_params,
    )

    V1 = peel_viability_kernel(
        s=1,
        initial_mask=S_mask,
        grid=grid,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        viability_options=viability_options,
        previous_kernel=previous_V1,
    )

    pre_initial = S_mask & V1.mask

    V0 = peel_viability_kernel(
        s=0,
        initial_mask=pre_initial,
        grid=grid,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        viability_options=viability_options,
        previous_kernel=previous_V0,
    )

    return ConditionalViabilityResult(
        grid=grid,
        primitive_mask=S_mask,
        V1=V1,
        V0=V0,
        diagnostics={
            "n_primitive": float(S_mask.sum()),
            "n_V1": float(V1.n_viable),
            "n_V0": float(V0.n_viable),
            "share_V1": float(V1.n_viable / max(1, int(S_mask.sum()))),
            "share_V0_of_V1": float(V0.n_viable / max(1, int(V1.n_viable))),
            "V1_converged": float(V1.converged),
            "V0_converged": float(V0.converged),
            "uses_full_transfer_halfline": 1.0,
        },
    )


# ============================================================
# Validation
# ============================================================

def validate_viability_layer(
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    viability_options: Optional[ViabilityOptions] = None,
) -> dict[str, float]:
    """
    Small Block 7 validation harness.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if viability_options is None:
        viability_options = ViabilityOptions(
            max_peel_iter=30,
            use_local_solver=True,
            tiny_tau_H_grid_size=0,
        )

    grid = ViabilityGrid(
        k_grid=np.linspace(0.50, 1.50, 5),
        L_grid=np.linspace(-0.40, 1.00, 6),
    )

    result = compute_conditional_viability_sets(
        grid,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        viability_options=viability_options,
    )

    if not result.V1.converged:
        raise RuntimeError("V1 viability peeling did not converge.")

    if not result.V0.converged:
        raise RuntimeError("V0 viability peeling did not converge.")

    if np.any(result.V0.mask & ~result.V1.mask):
        raise RuntimeError("Pre-switch mask must be a subset of post-switch mask.")

    if np.any(result.V1.mask & ~result.primitive_mask):
        raise RuntimeError("V1 must be a subset of primitive feasibility mask.")

    if np.any(result.V0.mask & ~result.primitive_mask):
        raise RuntimeError("V0 must be a subset of primitive feasibility mask.")

    # Exact primitive-wall transfer direction checks.
    state_options = StateConstraintOptions(
        primitive_wall_tol=economy_params.state_tol,
    )

    corner = State(0.0, 0.0)

    ok, _, _, _ = _primitive_transfer_upper_bound(
        corner,
        k_dot_floor=-1.0e-8,
        L_dot_floor=1.0e-8,
        economy_params=economy_params,
        state_options=state_options,
    )

    if ok:
        raise RuntimeError("Corner with negative k_dot should fail k-wall inwardness.")

    diagonal = State(1.0, -1.0)

    ok, _, _, _ = _primitive_transfer_upper_bound(
        diagonal,
        k_dot_floor=1.0,
        L_dot_floor=-1.1,
        economy_params=economy_params,
        state_options=state_options,
    )

    if ok:
        raise RuntimeError("Diagonal wall with negative W_K_dot should fail inwardness.")

    return {
        "n_primitive": float(result.primitive_mask.sum()),
        "n_V1": float(result.V1.n_viable),
        "n_V0": float(result.V0.n_viable),
        "V0_subset_V1": 1.0,
        "V1_converged": float(result.V1.converged),
        "V0_converged": float(result.V0.converged),
        "uses_full_transfer_halfline": 1.0,
        "V1_share_tau_upper_cap": float(
            result.V1.diagnostics["share_tau_upper_cap_witnesses"]
        ),
        "V0_share_tau_upper_cap": float(
            result.V0.diagnostics["share_tau_upper_cap_witnesses"]
        ),
    }


def module_smoke_test() -> dict[str, float]:
    automation_params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    primitives = build_regime_primitives(automation_params)

    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    continuation = make_test_continuation_bundle(
        asset_params=asset_params,
    )

    economy_params = PlannerEconomyParams(
        tau_upper=1.0,
        transfer_min=0.0,
        worker_consumption_eps=1.0e-8,
        state_tol=1.0e-10,
        control_tol=1.0e-12,
    )

    return validate_viability_layer(
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
    )


__all__ = [
    "ViabilityGrid",
    "TauHBounds",
    "TransferRaySolve",
    "ViabilityOptions",
    "ViabilityWitness",
    "ViabilityKernel",
    "ConditionalViabilityResult",
    "analytic_tau_H_candidates",
    "evaluate_tau_H_witness",
    "find_viability_witness",
    "peel_viability_kernel",
    "compute_conditional_viability_sets",
    "validate_viability_layer",
    "module_smoke_test",
]

Overwriting viability_sets.py


In [25]:
import importlib

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import state_constraints
import viability_sets

importlib.reload(automation_block)
importlib.reload(economy)
importlib.reload(policy_sets)
importlib.reload(asset_market)
importlib.reload(continuation_block)
importlib.reload(equilibrium_oracle)
importlib.reload(state_constraints)
importlib.reload(viability_sets)

block7_report = viability_sets.module_smoke_test()

print("Block 7 validation passed.")
print(block7_report)

Block 7 validation passed.
{'n_primitive': 30.0, 'n_V1': 30.0, 'n_V0': 30.0, 'V0_subset_V1': 1.0, 'V1_converged': 1.0, 'V0_converged': 1.0, 'uses_full_transfer_halfline': 1.0, 'V1_share_tau_upper_cap': 0.1, 'V0_share_tau_upper_cap': 0.1}


In [26]:
import importlib
import numpy as np

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import state_constraints
import viability_sets

importlib.reload(viability_sets)

automation_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(automation_params)

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

C_hat = continuation_block.make_test_continuation_bundle(
    asset_params=asset_params,
)

economy_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

policy_options = policy_sets.PolicySetOptions()

grid = viability_sets.ViabilityGrid(
    k_grid=np.linspace(0.50, 1.50, 5),
    L_grid=np.linspace(-0.40, 1.00, 6),
)

result = viability_sets.compute_conditional_viability_sets(
    grid,
    primitives=G,
    continuation=C_hat,
    asset_params=asset_params,
    economy_params=economy_params,
    policy_options=policy_options,
    viability_options=viability_sets.ViabilityOptions(
        max_peel_iter=30,
        use_local_solver=True,
        tiny_tau_H_grid_size=0,
        verbose=False,
    ),
)

print("Conditional viability diagnostics:")
print(result.diagnostics)

print("\nV1 mask:")
print(result.V1.mask.astype(int))

print("\nV0 mask:")
print(result.V0.mask.astype(int))

Conditional viability diagnostics:
{'n_primitive': 30.0, 'n_V1': 30.0, 'n_V0': 30.0, 'share_V1': 1.0, 'share_V0_of_V1': 1.0, 'V1_converged': 1.0, 'V0_converged': 1.0, 'uses_full_transfer_halfline': 1.0}

V1 mask:
[[1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]]

V0 mask:
[[1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]]


# Block 8 — planner pointwise active-set / KKT solver

Block 8 solves the planner’s **pointwise policy-improvement problem** at a single state node.

It takes the value-function costates as given and chooses the current planner control

$$
u=(\tau,T,H)
$$

to maximise the current Hamiltonian, subject to primitive feasibility, oracle validity, inward-feasibility restrictions, and the pure viability domain inherited from Block 7.

The central rule is:

$$
\boxed{
\text{Block 8 optimises current controls, but it does not recompute private continuation objects.}
}
$$

The private continuation environment remains frozen at $\mathcal C[\hat u]$. Current prices, tax bases, revenue, and drifts are evaluated live through the Block 6 oracle at each candidate control.

---

## Role of Block 8

For a fixed anticipated planner rule $\hat u$, the private continuation block gives

$$
\mathcal C[\hat u]
=
\left\{
\Psi_s^{\hat u},
\omega_s^{\hat u},
\chi^{\hat u},
\lambda^{Q,\hat u},
\text{validity masks}
\right\}_{s=0,1}.
$$

Block 7 gives the pure conditional viability sets

$$
V_1^{\hat u},
\qquad
V_0^{\hat u}.
$$

Block 8 then solves the **nodewise planner improvement problem** on those fixed viability sets.

At each state $x=(k,L)$ and regime $s$, given costates

$$
p=(J_{s,k},J_{s,L}),
$$

Block 8 chooses a current control

$$
u^\star_s(x)
=
(\tau^\star_s(x),T^\star_s(x),H^\star_s(x)).
$$

The output is a candidate planner best response at that node, together with active-set and KKT diagnostics.

---

## Pointwise Hamiltonian

The current-control drift is

$$
f_s^{\hat u}(x;u)
=
\left(
\dot k_s^{\hat u}(x;u),
\dot L_s^{\hat u}(x;u)
\right).
$$

The superscript $\hat u$ means that private continuation objects are frozen. The argument $u$ means that current policy is evaluated live.

The pointwise planner Hamiltonian is

$$
\mathcal H_s^{\hat u}(x,u;p)
=
\mathcal U_s^{\hat u}(x,u)
+
p\cdot f_s^{\hat u}(x;u),
$$

or explicitly,

$$
\mathcal H_s^{\hat u}(x,u;p)
=
\mathcal U_s^{\hat u}(x,u)
+
J_{s,k}\dot k_s^{\hat u}(x;u)
+
J_{s,L}\dot L_s^{\hat u}(x;u).
$$

The flow payoff can be written as

$$
\mathcal U_s^{\hat u}(x,u)
=
\alpha_W u_W(C_s^W)
+
\alpha_K u_K(C_s^K),
$$

where

$$
C_s^W=w_s(k)+T,
$$

and

$$
C_s^K
=
\omega_s^{\hat u}(k,L)(k+L).
$$

The owner consumption term uses the frozen continuation object $\omega_s^{\hat u}$. It is not recomputed during pointwise improvement.

---

## Regime-0 Poisson term

In regime $s=0$, the planner HJB contains a Poisson continuation term of the form

$$
\lambda
\left(
J_1(x)-J_0(x)
\right).
$$

At a fixed state $x$, this term is independent of the current control $u=(\tau,T,H)$.

Therefore Block 8 may include this term in HJB evaluation, but it does not affect the pointwise argmax:

$$
\arg\max_u
\left[
\mathcal H_0^{\hat u}(x,u;p)
+
\lambda(J_1(x)-J_0(x))
\right]
=
\arg\max_u
\mathcal H_0^{\hat u}(x,u;p).
$$

Thus the Poisson term is an HJB-evaluation object, not an active current-control first-order condition.

---

## Current admissible controls

The planner’s full current policy set is

$$
U_s^{full}(k,L)
=
\left\{
(\tau,T,H):
\tau\in[0,\bar\tau),
\quad
T\in[\underline T_s(k),\infty),
\quad
H\in[\max\{0,-L\},k]
\right\}.
$$

Block 8 works with the full current policy set. Numerically, the open upper bound on $\tau$ is represented by a closed approximation

$$
\tau\in[0,\bar\tau-\varepsilon_\tau].
$$

The transfer control remains semi-infinite:

$$
T\in[\underline T_s(k),\infty).
$$

The artificial transfer cap from the compactified policy set is not an economic upper bound. It is only a diagnostic.

The rule is:

$$
\boxed{
\text{Do not treat }T\text{ as an ordinary boxed control in the baseline pointwise solver.}
}
$$

---

## Why this is an active-set problem

The pointwise problem is not a coarse global control-grid maximisation.

It has:

- an open or nearly open tax upper bound;
- a semi-infinite transfer control;
- public ownership bounds;
- primitive state-boundary restrictions;
- oracle validity branches;
- portfolio-branch restrictions;
- viability-domain tangent restrictions;
- possible no-finite-maximiser transfer branches.

Therefore Block 8 is an active-set/KKT solver.

Newton or Nelder-Mead steps are allowed as local smooth-branch tools, but the mathematical object is an active-set problem.

The active-set logic is:

1. construct interior candidates;
2. construct control-bound candidates;
3. solve the semi-infinite transfer branch analytically;
4. evaluate the live oracle at each candidate;
5. filter by primitive feasibility and oracle validity;
6. filter by inward/tangent feasibility;
7. compare feasible branches by Hamiltonian value;
8. report KKT residuals and active-set diagnostics.

---

## Semi-infinite transfer subproblem

The transfer control requires special treatment.

Under the Mode-A transfer convention,

$$
\partial_T C_s^W=1,
$$

$$
\partial_T C_s^K=0,
$$

$$
\partial_T\dot k_s^{\hat u}=-1,
$$

and

$$
\partial_T\dot L_s^{\hat u}=1.
$$

Therefore the drift contribution to the Hamiltonian has linear transfer coefficient

$$
\beta
=
J_{s,L}-J_{s,k}.
$$

For fixed $(\tau,H)$, define the transfer floor

$$
T_{\min}
=
\underline T_s(k),
$$

and worker consumption at the floor

$$
C_{\min}^W
=
w_s(k)+T_{\min}.
$$

The transfer subproblem is

$$
\max_{T\ge T_{\min}}
\left[
\alpha_W u_W(w_s(k)+T)
+
\beta T
\right],
$$

up to terms independent of $T$.

For CRRA worker utility with coefficient $\gamma_W$,

$$
u_W'(C)
=
C^{-\gamma_W}.
$$

The transfer derivative is

$$
\partial_T \mathcal H
=
\alpha_W
\left(w_s(k)+T\right)^{-\gamma_W}
+
\beta.
$$

Thus the interior transfer first-order condition is

$$
\alpha_W
\left(w_s(k)+T\right)^{-\gamma_W}
+
\beta
=
0.
$$

A finite interior transfer solution exists only if

$$
\beta<0.
$$

When $\beta<0$, the unconstrained interior worker consumption is

$$
C_\star^W
=
\left(
\frac{\alpha_W}{-\beta}
\right)^{1/\gamma_W},
$$

so

$$
T_\star
=
C_\star^W-w_s(k).
$$

The constrained optimum is:

- if $T_\star>T_{\min}$, use the finite interior branch;
- if $T_\star\le T_{\min}$, use the lower-bound branch;
- if $\beta\ge 0$, there is no finite transfer maximiser on the semi-infinite half-line.

This gives the branch classification:

$$
\texttt{finite\_interior},
$$

$$
\texttt{lower\_bound},
$$

$$
\texttt{flat\_no\_finite\_max},
$$

$$
\texttt{unbounded\_no\_finite\_max},
$$

and

$$
\texttt{invalid}.
$$

The branch

$$
\texttt{no\_finite\_maximizer}
$$

is not a numerical failure. It is an economically meaningful diagnostic: the Hamiltonian has no finite maximising transfer choice under the current costates.

---

## Reduced search over $(\tau,H)$

Because the transfer subproblem is solved analytically for each $(\tau,H)$, Block 8 reduces the numerical search to two dimensions:

$$
(\tau,H).
$$

For each candidate pair $(\tau,H)$, the solver:

1. computes the true transfer floor $T_{\min}$;
2. evaluates the live oracle at or near the transfer floor to obtain current wages and branch information;
3. solves the analytic transfer subproblem;
4. constructs the full control

$$
u=(\tau,T^\star,H);
$$

5. re-evaluates the live oracle at $u$;
6. computes the Hamiltonian;
7. applies feasibility and tangent filters.

This avoids treating the transfer cap as an economic restriction.

---

## Live oracle evaluation

Every candidate control is evaluated through the Block 6 oracle:

$$
\mathcal O_s(x,u;\mathcal G,\mathcal C[\hat u]).
$$

The oracle returns current objects such as:

$$
\pi^{mc},
\qquad
r_f,
\qquad
C_s^W,
\qquad
C_s^K,
\qquad
\dot k,
\qquad
\dot L,
\qquad
\dot W^K.
$$

The important point is that current prices and drifts are recomputed at every candidate control.

Block 8 must not reuse stale arrays for

$$
r_f,
\qquad
\dot k,
\qquad
\dot L.
$$

The rule is:

$$
\boxed{
\text{Current pricing and drifts are live inside the pointwise optimiser.}
}
$$

---

## Feasibility filters

A candidate is not accepted merely because it has a high Hamiltonian value.

It must pass the following filters.

### 1. Primitive state and control feasibility

The candidate must satisfy the primitive state and control restrictions:

$$
x\in S,
$$

and

$$
u\in U_s^{full}(x).
$$

### 2. Oracle validity

The oracle must return a valid drift.

In the baseline interior Merton implementation, a candidate with a portfolio-bound status is not accepted as a valid interior pricing candidate unless a binding-portfolio complementarity branch is later implemented.

Thus, in version 1,

$$
\texttt{portfolio\_bind}
$$

is a rejection branch for planner optimisation.

### 3. Primitive inward feasibility

At the primitive wall

$$
k=0,
$$

a candidate must satisfy

$$
\dot k\ge 0.
$$

At the primitive wall

$$
k+L=0,
$$

a candidate must satisfy

$$
\dot k+\dot L\ge 0.
$$

At the corner, both inequalities must hold.

### 4. Viability-domain tangent feasibility

If the pointwise solver is being used inside Howard on a fixed active domain, the candidate drift must also respect the local tangent condition for the relevant mask.

If $A_s$ is the current Howard active mask, then the drift should satisfy

$$
f_s^{\hat u}(x;u)\in T_{A_s}(x)
$$

in the discrete tangent sense.

This filter is numerical-domain logic. It must not redefine the pure viability set $V_s^{\hat u}$.

---

## Candidate hierarchy

The candidate hierarchy is:

$$
\text{warm start}
\to
\text{viability witness}
\to
\text{analytic candidates}
\to
\text{local solver}
\to
\text{boundary solvers}
\to
\text{tiny rescue grid}.
$$

The warm start is usually the previous policy at the same node.

The viability witness comes from Block 7 and is a useful feasible starting point, but it is not necessarily optimal.

Analytic candidates include combinations of:

$$
\tau=0,
\qquad
\tau=\bar\tau-\varepsilon_\tau,
\qquad
\tau=\text{midpoint},
$$

and

$$
H=H_{\min},
\qquad
H=H_{\max},
\qquad
H=\text{midpoint},
$$

plus model-specific candidates such as the value of $H$ that balances the risky and safe sides of the balance sheet when useful.

Local solvers refine smooth interior branches.

Boundary solvers explicitly search along active control faces such as:

$$
\tau=0,
\qquad
\tau=\bar\tau-\varepsilon_\tau,
\qquad
H=H_{\min},
\qquad
H=H_{\max}.
$$

The tiny rescue grid is for debugging only. It is not the main optimiser.

---

## Branch comparison

After candidate generation, Block 8 compares all feasible candidates by Hamiltonian value.

The selected branch is

$$
u^\star
\in
\arg\max_{u\in\mathcal A_s(x)}
\mathcal H_s^{\hat u}(x,u;p),
$$

where $\mathcal A_s(x)$ is the set of candidates that pass the oracle and feasibility filters.

If no feasible candidate exists, the solver returns

$$
\texttt{no\_feasible\_candidate}.
$$

If the semi-infinite transfer branch has no finite maximiser, the solver returns

$$
\texttt{no\_finite\_maximizer}.
$$

If a feasible maximising candidate exists, the solver returns

$$
\texttt{accepted}.
$$

---

## KKT diagnostics

Block 8 reports KKT diagnostics for the selected candidate.

For $\tau$ and $H$, the solver uses finite-difference derivatives of the reduced Hamiltonian after solving the transfer subproblem.

Let

$$
g_\tau
=
\partial_\tau \mathcal H,
\qquad
g_H
=
\partial_H \mathcal H.
$$

For a maximisation problem:

- if $\tau$ is interior, require

$$
g_\tau=0;
$$

- if $\tau$ is at its lower bound, feasible movement is upward, so require

$$
g_\tau\le 0;
$$

- if $\tau$ is at its upper numerical bound, feasible movement is downward, so require

$$
g_\tau\ge 0.
$$

Similarly, for $H$:

- if $H$ is interior, require

$$
g_H=0;
$$

- if $H=H_{\min}$, require

$$
g_H\le 0;
$$

- if $H=H_{\max}$, require

$$
g_H\ge 0.
$$

For the transfer branch:

- finite interior transfer requires

$$
\partial_T\mathcal H=0;
$$

- lower-bound transfer requires

$$
\partial_T\mathcal H\le 0;
$$

- no-finite-maximiser branches are reported as non-KKT finite optima.

The solver stores a maximum KKT violation:

$$
\texttt{max\_violation}
=
\max
\left\{
\text{tax violation},
\text{ownership violation},
\text{transfer violation}
\right\}.
$$

---

## Output objects

The main module is:

```text
planner_pointwise.py
```

It defines objects such as:

```text
PlannerPayoffParams
Costates
PointwiseSolverOptions
TransferOptimum
CandidateEvaluation
KKTDiagnostics
PointwiseSolution
```

The main solver entry point is:

```text
solve_pointwise_policy
```

The output contains:

```text
status
candidate
control
reason
kkt
n_candidates_evaluated
n_feasible_candidates
best_rejected
```

The accepted control is:

$$
u^\star=(\tau^\star,T^\star,H^\star).
$$

The accepted candidate also stores:

```text
hamiltonian
flow_payoff
drift_payoff
transfer_branch
oracle_status
primitive_inward
tangent_accepted
k_dot
L_dot
W_K_dot
active bound flags
compact-transfer-cap diagnostic
```

---

## Diagnostics

Important Block 8 diagnostics include:

```text
status
transfer_branch
finite_transfer_maximizer
n_candidates_evaluated
n_feasible_candidates
solution_hamiltonian
solution_tau
solution_T
solution_H
solution_kkt_max_violation
tau_lower_active
tau_upper_active
T_lower_active
H_lower_active
H_upper_active
exceeds_compact_T_cap
oracle_status
primitive_inward
tangent_accepted
```

The diagnostic

```text
exceeds_compact_T_cap
```

does not mean the economic policy is invalid. It means the full semi-infinite optimum lies beyond the numerical compactification cap. This should be reported so I can enlarge the compactification or diagnose a genuine semi-infinite-control issue.

---

## What Block 8 must not do

Block 8 should not:

- solve the private continuation problem;
- recompute $\Psi_s^{\hat u}$ or $\omega_s^{\hat u}$;
- construct or peel viability sets;
- redefine pure viability masks;
- solve the Howard linear HJB;
- run the outer Markov-perfect fixed point;
- freeze old arrays for $r_f$, $\dot k$, or $\dot L$;
- treat the artificial transfer cap as an economic upper bound;
- use a coarse global control grid as the main optimiser;
- treat a viability witness as an optimal policy;
- include the regime-0 Poisson term in the pointwise argmax as if it depended on $u$.

The key forbidden confusion is:

$$
\boxed{
\text{Do not confuse pointwise planner improvement with viability witness search.}
}
$$

Viability asks whether some control keeps the state feasible. Block 8 asks which feasible current control maximises the planner Hamiltonian.

---

## Validation checks

The Block 8 validation harness should check:

1. the finite interior transfer branch;
2. the lower-bound transfer branch;
3. the no-finite-maximiser transfer branch;
4. live oracle calls at candidate controls;
5. correct Hamiltonian construction;
6. primitive feasibility filtering;
7. oracle-validity filtering;
8. inward-feasibility filtering;
9. tangent-mask filtering when a mask is supplied;
10. active bound flags for $\tau$, $T$, and $H$;
11. KKT residual reporting;
12. rejection of invalid portfolio branches in version 1;
13. no use of the compact transfer cap as an economic restriction;
14. no recomputation of frozen continuation objects.

A useful staged validation is:

```text
finite interior T branch
lower-bound T branch
no-finite-maximizer T branch
manual candidate evaluation
full active-set solve
KKT diagnostic check
invalid-branch rejection
```

---

## One-line summary

Block 8 solves

$$
\boxed{
u_s^\star(x)
\in
\arg\max_{u\in U_s^{full}(x)}
\left[
\mathcal U_s^{\hat u}(x,u)
+
J_{s,k}\dot k_s^{\hat u}(x;u)
+
J_{s,L}\dot L_s^{\hat u}(x;u)
\right],
}
$$

subject to oracle validity, primitive feasibility, inward feasibility, and active-domain tangent feasibility.

It does this with active-set logic, live oracle evaluation, frozen private continuation objects, analytic treatment of the semi-infinite transfer control, branch comparison, and KKT diagnostics.

In [27]:
%%writefile planner_pointwise.py
from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Iterable, Literal, Optional, Sequence
import math
import numpy as np

try:
    from scipy.optimize import minimize
except Exception:  # pragma: no cover
    minimize = None

from automation_block import AutomationParams, RegimePrimitives, build_regime_primitives
from model.economy import (
    State,
    Control,
    PlannerEconomyParams,
    planner_flow_payoff,
)
import policy_sets
from policy_sets import PolicySetOptions
from asset_market import AssetMarketParams, make_infinite_asset_market_params
from continuation_block import ContinuationBundle, make_test_continuation_bundle
from equilibrium_oracle import OracleOptions, OracleEval, live_oracle
from state_constraints import (
    StateConstraintOptions,
    primitive_inward_diagnostics,
    state_constraint_diagnostics,
)


# ============================================================
# Block 8 contract: planner pointwise active-set / KKT solver
# ============================================================
#
# Given costates p = (J_k, J_L), solve the pointwise planner problem
#
#     max_u  U(C_W(x,u), C_K(x)) + p · f_s^{hat u}(x;u)
#
# over admissible current controls u = (tau,T,H), holding the frozen private
# continuation bundle C[hat u] fixed while evaluating current pricing and drifts
# live through the Block 6 oracle.
#
# Regime-0 Poisson continuation terms are HJB constants at fixed x. They are
# not included in the argmax because they do not depend on u.
#
# Forbidden responsibilities:
#   - no private continuation solve;
#   - no viability peeling;
#   - no Howard linear solve;
#   - no outer fixed point;
#   - no freezing of r_f, k_dot, or L_dot arrays from an old policy.
#
# Important convention:
#   T is semi-infinite. For each (tau,H), Block 8 solves the T subproblem
#   analytically using the Mode-A derivative, rather than treating T as an
#   ordinary boxed control. Compact transfer caps are diagnostics only.


TransferBranch = Literal[
    "finite_interior",
    "lower_bound",
    "flat_no_finite_max",
    "unbounded_no_finite_max",
    "invalid",
]

PointwiseStatus = Literal[
    "accepted",
    "no_finite_maximizer",
    "no_feasible_candidate",
    "invalid_state_or_inputs",
]

CandidateSource = Literal[
    "warm_start",
    "viability_witness",
    "analytic",
    "local_solver",
    "boundary_solver",
    "tiny_rescue_grid",
    "manual",
]


# ============================================================
# Options and parameter containers
# ============================================================

@dataclass(frozen=True)
class PlannerPayoffParams:
    """
    Planner flow payoff parameters.

    The default is deliberately neutral and can be replaced by the calibration
    notebook. gamma_owner should usually match AssetMarketParams.gamma.
    """
    gamma_worker: float = 1.0
    gamma_owner: float = 5.0
    weight_worker: float = 1.0
    weight_owner: float = 1.0

    def __post_init__(self) -> None:
        for name in ("gamma_worker", "gamma_owner"):
            val = float(getattr(self, name))
            if not math.isfinite(val) or val <= 0.0:
                raise ValueError(f"{name} must be positive and finite.")
            object.__setattr__(self, name, val)

        for name in ("weight_worker", "weight_owner"):
            val = float(getattr(self, name))
            if not math.isfinite(val) or val < 0.0:
                raise ValueError(f"{name} must be nonnegative and finite.")
            object.__setattr__(self, name, val)


@dataclass(frozen=True)
class Costates:
    J_k: float
    J_L: float

    def __post_init__(self) -> None:
        for name in ("J_k", "J_L"):
            val = float(getattr(self, name))
            if not math.isfinite(val):
                raise ValueError(f"{name} must be finite.")
            object.__setattr__(self, name, val)

    @property
    def transfer_drift_coefficient(self) -> float:
        """
        Coefficient on T in p · f under Mode-A transfer derivatives:

            d/dT [p · f] = J_L - J_k.
        """
        return self.J_L - self.J_k


@dataclass(frozen=True)
class PointwiseSolverOptions:
    """
    Numerical options for the active-set pointwise solver.

    The main optimizer is continuous in (tau,H) with the transfer subproblem
    solved analytically. The optional tiny rescue grid is for diagnostics only.
    """
    use_local_solver: bool = True
    use_boundary_solvers: bool = True
    local_solver_maxiter: int = 160
    objective_tol: float = 1.0e-9
    kkt_step: float = 1.0e-5
    kkt_tol: float = 1.0e-5
    accept_tangent_filter: bool = True
    tiny_rescue_grid_size: int = 0
    penalty_invalid: float = 1.0e12
    verbose: bool = False

    def __post_init__(self) -> None:
        if self.local_solver_maxiter < 1:
            raise ValueError("local_solver_maxiter must be at least 1.")
        if self.objective_tol < 0.0:
            raise ValueError("objective_tol must be nonnegative.")
        if self.kkt_step <= 0.0:
            raise ValueError("kkt_step must be positive.")
        if self.kkt_tol < 0.0:
            raise ValueError("kkt_tol must be nonnegative.")
        if self.tiny_rescue_grid_size < 0:
            raise ValueError("tiny_rescue_grid_size must be nonnegative.")
        if self.penalty_invalid <= 0.0:
            raise ValueError("penalty_invalid must be positive.")


@dataclass(frozen=True)
class TauHClosedBounds:
    tau_lower: float
    tau_upper_closed: float
    H_lower: float
    H_upper: float
    T_lower: float

    def __post_init__(self) -> None:
        for name in ("tau_lower", "tau_upper_closed", "H_lower", "H_upper", "T_lower"):
            val = float(getattr(self, name))
            if not math.isfinite(val):
                raise ValueError(f"{name} must be finite.")
            object.__setattr__(self, name, val)

        if self.tau_lower > self.tau_upper_closed:
            raise ValueError("tau bounds are inconsistent.")
        if self.H_lower > self.H_upper:
            raise ValueError("H bounds are inconsistent.")

    @property
    def tau_width(self) -> float:
        return max(0.0, self.tau_upper_closed - self.tau_lower)

    @property
    def H_width(self) -> float:
        return max(0.0, self.H_upper - self.H_lower)


# ============================================================
# Outputs
# ============================================================

@dataclass(frozen=True)
class TransferOptimum:
    branch: TransferBranch
    finite_maximizer: bool
    T: float
    C_W: float
    derivative_at_T: float
    derivative_at_floor: float
    beta: float
    reason: Optional[str]


@dataclass(frozen=True)
class CandidateEvaluation:
    feasible: bool
    control: Control
    source: CandidateSource

    hamiltonian: float
    flow_payoff: float
    drift_payoff: float

    transfer_branch: TransferBranch
    finite_transfer_maximizer: bool
    transfer_derivative: float

    oracle_status: Optional[str]
    oracle_reason: Optional[str]

    primitive_inward: bool
    tangent_accepted: bool
    inward_reason: Optional[str]
    tangent_reason: Optional[str]

    k_dot: float
    L_dot: float
    W_K_dot: float

    tau_lower_active: bool
    tau_upper_active: bool
    T_lower_active: bool
    H_lower_active: bool
    H_upper_active: bool
    exceeds_compact_T_cap: bool

    rejection_reason: Optional[str]


@dataclass(frozen=True)
class KKTDiagnostics:
    checked: bool
    max_violation: float
    grad_tau: float
    grad_H: float
    transfer_derivative: float
    tau_violation: float
    H_violation: float
    T_violation: float
    reason: Optional[str]


@dataclass(frozen=True)
class PointwiseSolution:
    status: PointwiseStatus
    candidate: Optional[CandidateEvaluation]
    control: Optional[Control]
    reason: Optional[str]
    kkt: KKTDiagnostics
    n_candidates_evaluated: int
    n_feasible_candidates: int
    best_rejected: Optional[CandidateEvaluation]

    @property
    def accepted(self) -> bool:
        return self.status == "accepted" and self.candidate is not None

    @property
    def hamiltonian(self) -> float:
        return -math.inf if self.candidate is None else self.candidate.hamiltonian


# ============================================================
# Basic helpers
# ============================================================

def _require_regime(s: int) -> int:
    if s not in (0, 1):
        raise ValueError("regime s must be 0 or 1.")
    return int(s)


def _oracle_full_options(options: Optional[OracleOptions]) -> OracleOptions:
    if options is None:
        return OracleOptions(control_set="full")
    return replace(options, control_set="full")


def _closed_tau_H_bounds(
    s: int,
    x: State,
    *,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
) -> TauHClosedBounds:
    bounds = policy_sets.full_policy_bounds(s, x, primitives, economy_params)
    tau_upper_closed = bounds.tau_upper - policy_options.tau_upper_margin

    if tau_upper_closed < bounds.tau_lower:
        raise ValueError("tau_upper_margin is too large for the primitive tau interval.")

    return TauHClosedBounds(
        tau_lower=bounds.tau_lower,
        tau_upper_closed=tau_upper_closed,
        H_lower=bounds.H_lower,
        H_upper=bounds.H_upper,
        T_lower=bounds.T_lower,
    )


def _z_to_tau_H(z: Sequence[float], bounds: TauHClosedBounds) -> tuple[float, float]:
    z = np.clip(np.asarray(z, dtype=float), 0.0, 1.0)

    tau = bounds.tau_lower + z[0] * bounds.tau_width
    H = bounds.H_lower + z[1] * bounds.H_width

    return float(tau), float(H)


def _tau_H_to_z(tau: float, H: float, bounds: TauHClosedBounds) -> np.ndarray:
    z = np.asarray([0.5, 0.5], dtype=float)

    if bounds.tau_width > 0.0:
        z[0] = (tau - bounds.tau_lower) / bounds.tau_width

    if bounds.H_width > 0.0:
        z[1] = (H - bounds.H_lower) / bounds.H_width

    return np.clip(z, 0.0, 1.0)


def _clamp(v: float, lo: float, hi: float) -> float:
    return min(max(float(v), float(lo)), float(hi))


def _dedupe_controls(
    items: Iterable[tuple[CandidateSource, float, float]],
    *,
    tol: float,
) -> list[tuple[CandidateSource, float, float]]:
    out: list[tuple[CandidateSource, float, float]] = []
    seen: set[tuple[int, int]] = set()
    scale = 1.0 / max(float(tol), 1.0e-16)

    for source, tau, H in items:
        key = (int(round(float(tau) * scale)), int(round(float(H) * scale)))
        if key not in seen:
            seen.add(key)
            out.append((source, float(tau), float(H)))

    return out


# ============================================================
# Hamiltonian and transfer subproblem
# ============================================================

def planner_flow_from_oracle(
    ev: OracleEval,
    payoff_params: PlannerPayoffParams,
) -> float:
    return planner_flow_payoff(
        ev.C_W,
        ev.C_K,
        gamma_worker=payoff_params.gamma_worker,
        gamma_owner=payoff_params.gamma_owner,
        weight_worker=payoff_params.weight_worker,
        weight_owner=payoff_params.weight_owner,
    )


def hamiltonian_from_oracle(
    ev: OracleEval,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    *,
    hjb_constant: float = 0.0,
) -> tuple[float, float, float]:
    """
    Return (Hamiltonian, flow payoff, drift payoff).
    """
    flow = planner_flow_from_oracle(ev, payoff_params)
    drift_payoff = costates.J_k * ev.k_dot + costates.J_L * ev.L_dot
    H = flow + drift_payoff + float(hjb_constant)

    if not math.isfinite(H):
        raise ValueError("Hamiltonian is non-finite.")

    return float(H), float(flow), float(drift_payoff)


def optimal_transfer_given_floor(
    *,
    w: float,
    T_lower: float,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
) -> TransferOptimum:
    """
    Solve the semi-infinite T subproblem exactly for fixed (tau,H).

    H(T) = weight_worker * u(w+T) + (J_L - J_k) T + constants.
    """
    w = float(w)
    T_lower = float(T_lower)

    if not (math.isfinite(w) and math.isfinite(T_lower)):
        return TransferOptimum(
            branch="invalid",
            finite_maximizer=False,
            T=math.nan,
            C_W=math.nan,
            derivative_at_T=math.nan,
            derivative_at_floor=math.nan,
            beta=math.nan,
            reason="non-finite wage or transfer floor",
        )

    C_floor = w + T_lower

    if C_floor <= 0.0 or not math.isfinite(C_floor):
        return TransferOptimum(
            branch="invalid",
            finite_maximizer=False,
            T=math.nan,
            C_W=C_floor,
            derivative_at_T=math.nan,
            derivative_at_floor=math.nan,
            beta=costates.transfer_drift_coefficient,
            reason="worker consumption at transfer floor is non-positive",
        )

    beta = costates.transfer_drift_coefficient
    weight = payoff_params.weight_worker
    gamma = payoff_params.gamma_worker

    if weight == 0.0:
        derivative_floor = beta

        if beta < 0.0:
            return TransferOptimum(
                branch="lower_bound",
                finite_maximizer=True,
                T=T_lower,
                C_W=C_floor,
                derivative_at_T=beta,
                derivative_at_floor=derivative_floor,
                beta=beta,
                reason="worker weight is zero and Hamiltonian decreases in T",
            )

        if beta == 0.0:
            return TransferOptimum(
                branch="lower_bound",
                finite_maximizer=True,
                T=T_lower,
                C_W=C_floor,
                derivative_at_T=0.0,
                derivative_at_floor=0.0,
                beta=beta,
                reason="worker weight is zero and Hamiltonian is flat in T; using lower bound",
            )

        return TransferOptimum(
            branch="unbounded_no_finite_max",
            finite_maximizer=False,
            T=math.inf,
            C_W=math.inf,
            derivative_at_T=beta,
            derivative_at_floor=derivative_floor,
            beta=beta,
            reason="worker weight is zero and Hamiltonian increases in T",
        )

    derivative_floor = weight * (C_floor ** (-gamma)) + beta

    if beta >= 0.0:
        branch: TransferBranch = (
            "flat_no_finite_max" if beta == 0.0 else "unbounded_no_finite_max"
        )
        return TransferOptimum(
            branch=branch,
            finite_maximizer=False,
            T=math.inf,
            C_W=math.inf,
            derivative_at_T=beta,
            derivative_at_floor=derivative_floor,
            beta=beta,
            reason="Hamiltonian has no finite maximizer in the semi-infinite transfer direction",
        )

    C_star = (weight / (-beta)) ** (1.0 / gamma)
    T_star = C_star - w

    if T_star <= T_lower:
        return TransferOptimum(
            branch="lower_bound",
            finite_maximizer=True,
            T=T_lower,
            C_W=C_floor,
            derivative_at_T=derivative_floor,
            derivative_at_floor=derivative_floor,
            beta=beta,
            reason="unconstrained transfer optimum lies below the lower bound",
        )

    return TransferOptimum(
        branch="finite_interior",
        finite_maximizer=True,
        T=float(T_star),
        C_W=float(C_star),
        derivative_at_T=0.0,
        derivative_at_floor=derivative_floor,
        beta=beta,
        reason=None,
    )


def transfer_optimum_for_tau_H(
    s: int,
    x: State,
    tau: float,
    H: float,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
) -> TransferOptimum:
    """
    Compute the analytic T optimum for a fixed (tau,H).

    The oracle at the transfer floor supplies the current wage via C_W - T.
    """
    bounds = _closed_tau_H_bounds(
        s,
        x,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    )

    u_floor = Control(tau=float(tau), T=bounds.T_lower, H=float(H))

    ev_floor = live_oracle(
        s=s,
        x=x,
        u=u_floor,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        options=oracle_options,
    )

    if not ev_floor.valid_for_drift:
        return TransferOptimum(
            branch="invalid",
            finite_maximizer=False,
            T=math.nan,
            C_W=math.nan,
            derivative_at_T=math.nan,
            derivative_at_floor=math.nan,
            beta=costates.transfer_drift_coefficient,
            reason=f"oracle invalid at transfer floor: {ev_floor.status}, {ev_floor.reason}",
        )

    w = ev_floor.C_W - bounds.T_lower

    return optimal_transfer_given_floor(
        w=w,
        T_lower=bounds.T_lower,
        costates=costates,
        payoff_params=payoff_params,
    )


# ============================================================
# Candidate evaluation and filters
# ============================================================

def _constraint_filter(
    ev: OracleEval,
    *,
    candidate_mask: Optional[np.ndarray],
    i: Optional[int],
    j: Optional[int],
    k_grid: Optional[Sequence[float]],
    L_grid: Optional[Sequence[float]],
    economy_params: PlannerEconomyParams,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
) -> tuple[bool, bool, Optional[str], Optional[str]]:
    primitive = primitive_inward_diagnostics(
        ev.state,
        ev.k_dot,
        ev.L_dot,
        economy_params=economy_params,
        options=state_options,
    )

    if not primitive.accepted:
        return False, False, primitive.reason, None

    if candidate_mask is None:
        return True, True, None, None

    if not solver_options.accept_tangent_filter:
        return True, True, None, None

    if i is None or j is None or k_grid is None or L_grid is None:
        raise ValueError(
            "candidate_mask filtering requires i, j, k_grid, and L_grid."
        )

    diag = state_constraint_diagnostics(
        ev,
        np.asarray(candidate_mask, dtype=bool),
        int(i),
        int(j),
        k_grid,
        L_grid,
        economy_params=economy_params,
        options=state_options,
    )

    return True, bool(diag.tangent.accepted), None, diag.tangent.reason


def evaluate_pointwise_candidate(
    s: int,
    x: State,
    tau: float,
    H: float,
    *,
    source: CandidateSource,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    candidate_mask: Optional[np.ndarray] = None,
    i: Optional[int] = None,
    j: Optional[int] = None,
    k_grid: Optional[Sequence[float]] = None,
    L_grid: Optional[Sequence[float]] = None,
    hjb_constant: float = 0.0,
) -> CandidateEvaluation:
    s = _require_regime(s)
    oracle_options = _oracle_full_options(oracle_options)

    try:
        bounds = _closed_tau_H_bounds(
            s,
            x,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    except Exception as exc:
        dummy = Control(float(tau), math.nan, float(H))
        return CandidateEvaluation(
            feasible=False,
            control=dummy,
            source=source,
            hamiltonian=-math.inf,
            flow_payoff=math.nan,
            drift_payoff=math.nan,
            transfer_branch="invalid",
            finite_transfer_maximizer=False,
            transfer_derivative=math.nan,
            oracle_status=None,
            oracle_reason=str(exc),
            primitive_inward=False,
            tangent_accepted=False,
            inward_reason=str(exc),
            tangent_reason=None,
            k_dot=math.nan,
            L_dot=math.nan,
            W_K_dot=math.nan,
            tau_lower_active=False,
            tau_upper_active=False,
            T_lower_active=False,
            H_lower_active=False,
            H_upper_active=False,
            exceeds_compact_T_cap=False,
            rejection_reason=f"failed to construct bounds: {exc}",
        )

    tau = _clamp(tau, bounds.tau_lower, bounds.tau_upper_closed)
    H = _clamp(H, bounds.H_lower, bounds.H_upper)

    transfer = transfer_optimum_for_tau_H(
        s=s,
        x=x,
        tau=tau,
        H=H,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        costates=costates,
        payoff_params=payoff_params,
    )

    if not transfer.finite_maximizer:
        control = Control(tau=tau, T=bounds.T_lower, H=H)
        return CandidateEvaluation(
            feasible=False,
            control=control,
            source=source,
            hamiltonian=-math.inf,
            flow_payoff=math.nan,
            drift_payoff=math.nan,
            transfer_branch=transfer.branch,
            finite_transfer_maximizer=False,
            transfer_derivative=transfer.derivative_at_T,
            oracle_status=None,
            oracle_reason=transfer.reason,
            primitive_inward=False,
            tangent_accepted=False,
            inward_reason=None,
            tangent_reason=None,
            k_dot=math.nan,
            L_dot=math.nan,
            W_K_dot=math.nan,
            tau_lower_active=abs(tau - bounds.tau_lower) <= policy_options.bound_tol,
            tau_upper_active=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
            T_lower_active=math.isfinite(transfer.T)
            and abs(transfer.T - bounds.T_lower) <= policy_options.bound_tol,
            H_lower_active=abs(H - bounds.H_lower) <= policy_options.bound_tol,
            H_upper_active=abs(H - bounds.H_upper) <= policy_options.bound_tol,
            exceeds_compact_T_cap=False,
            rejection_reason=transfer.reason,
        )

    u = Control(tau=tau, T=transfer.T, H=H)

    try:
        ev = live_oracle(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            options=oracle_options,
        )
    except Exception as exc:
        return CandidateEvaluation(
            feasible=False,
            control=u,
            source=source,
            hamiltonian=-math.inf,
            flow_payoff=math.nan,
            drift_payoff=math.nan,
            transfer_branch=transfer.branch,
            finite_transfer_maximizer=transfer.finite_maximizer,
            transfer_derivative=transfer.derivative_at_T,
            oracle_status=None,
            oracle_reason=str(exc),
            primitive_inward=False,
            tangent_accepted=False,
            inward_reason=None,
            tangent_reason=None,
            k_dot=math.nan,
            L_dot=math.nan,
            W_K_dot=math.nan,
            tau_lower_active=abs(tau - bounds.tau_lower) <= policy_options.bound_tol,
            tau_upper_active=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
            T_lower_active=abs(transfer.T - bounds.T_lower) <= policy_options.bound_tol,
            H_lower_active=abs(H - bounds.H_lower) <= policy_options.bound_tol,
            H_upper_active=abs(H - bounds.H_upper) <= policy_options.bound_tol,
            exceeds_compact_T_cap=False,
            rejection_reason=f"oracle exception: {exc}",
        )

    if not ev.valid_for_drift:
        return CandidateEvaluation(
            feasible=False,
            control=u,
            source=source,
            hamiltonian=-math.inf,
            flow_payoff=math.nan,
            drift_payoff=math.nan,
            transfer_branch=transfer.branch,
            finite_transfer_maximizer=transfer.finite_maximizer,
            transfer_derivative=transfer.derivative_at_T,
            oracle_status=ev.status,
            oracle_reason=ev.reason,
            primitive_inward=False,
            tangent_accepted=False,
            inward_reason=None,
            tangent_reason=None,
            k_dot=ev.k_dot,
            L_dot=ev.L_dot,
            W_K_dot=ev.W_K_dot,
            tau_lower_active=abs(tau - bounds.tau_lower) <= policy_options.bound_tol,
            tau_upper_active=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
            T_lower_active=abs(transfer.T - bounds.T_lower) <= policy_options.bound_tol,
            H_lower_active=abs(H - bounds.H_lower) <= policy_options.bound_tol,
            H_upper_active=abs(H - bounds.H_upper) <= policy_options.bound_tol,
            exceeds_compact_T_cap=False,
            rejection_reason=f"oracle invalid for drift: {ev.status}, {ev.reason}",
        )

    try:
        Hval, flow, drift_payoff = hamiltonian_from_oracle(
            ev,
            costates,
            payoff_params,
            hjb_constant=hjb_constant,
        )
    except Exception as exc:
        return CandidateEvaluation(
            feasible=False,
            control=u,
            source=source,
            hamiltonian=-math.inf,
            flow_payoff=math.nan,
            drift_payoff=math.nan,
            transfer_branch=transfer.branch,
            finite_transfer_maximizer=transfer.finite_maximizer,
            transfer_derivative=transfer.derivative_at_T,
            oracle_status=ev.status,
            oracle_reason=ev.reason,
            primitive_inward=False,
            tangent_accepted=False,
            inward_reason=str(exc),
            tangent_reason=None,
            k_dot=ev.k_dot,
            L_dot=ev.L_dot,
            W_K_dot=ev.W_K_dot,
            tau_lower_active=abs(tau - bounds.tau_lower) <= policy_options.bound_tol,
            tau_upper_active=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
            T_lower_active=abs(transfer.T - bounds.T_lower) <= policy_options.bound_tol,
            H_lower_active=abs(H - bounds.H_lower) <= policy_options.bound_tol,
            H_upper_active=abs(H - bounds.H_upper) <= policy_options.bound_tol,
            exceeds_compact_T_cap=False,
            rejection_reason=f"flow payoff / Hamiltonian failed: {exc}",
        )

    primitive_ok, tangent_ok, primitive_reason, tangent_reason = _constraint_filter(
        ev,
        candidate_mask=candidate_mask,
        i=i,
        j=j,
        k_grid=k_grid,
        L_grid=L_grid,
        economy_params=economy_params,
        state_options=state_options,
        solver_options=solver_options,
    )

    compact_bounds = policy_sets.compact_policy_bounds(
        s,
        x,
        primitives,
        economy_params,
        policy_options,
    )
    exceeds_cap = transfer.T > compact_bounds.T_upper + policy_options.bound_tol

    feasible = bool(primitive_ok and tangent_ok)
    rejection_reason = None

    if not primitive_ok:
        rejection_reason = primitive_reason
    elif not tangent_ok:
        rejection_reason = tangent_reason

    return CandidateEvaluation(
        feasible=feasible,
        control=u,
        source=source,
        hamiltonian=Hval if feasible else -math.inf,
        flow_payoff=flow,
        drift_payoff=drift_payoff,
        transfer_branch=transfer.branch,
        finite_transfer_maximizer=transfer.finite_maximizer,
        transfer_derivative=transfer.derivative_at_T,
        oracle_status=ev.status,
        oracle_reason=ev.reason,
        primitive_inward=bool(primitive_ok),
        tangent_accepted=bool(tangent_ok),
        inward_reason=primitive_reason,
        tangent_reason=tangent_reason,
        k_dot=ev.k_dot,
        L_dot=ev.L_dot,
        W_K_dot=ev.W_K_dot,
        tau_lower_active=abs(tau - bounds.tau_lower) <= policy_options.bound_tol,
        tau_upper_active=abs(tau - bounds.tau_upper_closed) <= policy_options.bound_tol,
        T_lower_active=abs(transfer.T - bounds.T_lower) <= policy_options.bound_tol,
        H_lower_active=abs(H - bounds.H_lower) <= policy_options.bound_tol,
        H_upper_active=abs(H - bounds.H_upper) <= policy_options.bound_tol,
        exceeds_compact_T_cap=bool(exceeds_cap),
        rejection_reason=rejection_reason,
    )


# ============================================================
# Candidate generation
# ============================================================

def analytic_tau_H_candidates(
    s: int,
    x: State,
    *,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
) -> list[tuple[CandidateSource, float, float]]:
    bounds = _closed_tau_H_bounds(
        s,
        x,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    )

    tau_vals = [
        bounds.tau_lower,
        0.5 * (bounds.tau_lower + bounds.tau_upper_closed),
        bounds.tau_upper_closed,
    ]

    H_vals = [
        bounds.H_lower,
        0.5 * (bounds.H_lower + bounds.H_upper),
        bounds.H_upper,
    ]

    if x.W_K > 0.0:
        H_xi = _clamp(0.5 * (x.k - x.L), bounds.H_lower, bounds.H_upper)
        H_vals.append(H_xi)

    return _dedupe_controls(
        [("analytic", tau, H) for tau in tau_vals for H in H_vals],
        tol=policy_options.bound_tol,
    )


def _tiny_tau_H_grid(
    bounds: TauHClosedBounds,
    n: int,
) -> list[tuple[CandidateSource, float, float]]:
    if n <= 0:
        return []

    if n == 1:
        tau, H = _z_to_tau_H((0.5, 0.5), bounds)
        return [("tiny_rescue_grid", tau, H)]

    zs = np.linspace(0.0, 1.0, n)
    return [
        ("tiny_rescue_grid", *_z_to_tau_H((zt, zh), bounds))
        for zt in zs
        for zh in zs
    ]


# ============================================================
# Reduced optimizer in tau/H
# ============================================================

def _candidate_objective_factory(
    s: int,
    x: State,
    bounds: TauHClosedBounds,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    candidate_mask: Optional[np.ndarray],
    i: Optional[int],
    j: Optional[int],
    k_grid: Optional[Sequence[float]],
    L_grid: Optional[Sequence[float]],
    hjb_constant: float,
    source: CandidateSource,
):
    def objective(z: Sequence[float]) -> float:
        tau, H = _z_to_tau_H(z, bounds)

        cand = evaluate_pointwise_candidate(
            s=s,
            x=x,
            tau=tau,
            H=H,
            source=source,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            solver_options=solver_options,
            costates=costates,
            payoff_params=payoff_params,
            candidate_mask=candidate_mask,
            i=i,
            j=j,
            k_grid=k_grid,
            L_grid=L_grid,
            hjb_constant=hjb_constant,
        )

        if cand.feasible and math.isfinite(cand.hamiltonian):
            return -cand.hamiltonian

        penalty = solver_options.penalty_invalid

        if cand.transfer_branch in ("unbounded_no_finite_max", "flat_no_finite_max"):
            return penalty * 0.1

        return penalty

    return objective


def _run_local_solver(
    starts: Sequence[np.ndarray],
    s: int,
    x: State,
    bounds: TauHClosedBounds,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    candidate_mask: Optional[np.ndarray],
    i: Optional[int],
    j: Optional[int],
    k_grid: Optional[Sequence[float]],
    L_grid: Optional[Sequence[float]],
    hjb_constant: float,
) -> list[CandidateEvaluation]:
    if minimize is None or not solver_options.use_local_solver:
        return []

    out: list[CandidateEvaluation] = []

    obj = _candidate_objective_factory(
        s,
        x,
        bounds,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        solver_options=solver_options,
        costates=costates,
        payoff_params=payoff_params,
        candidate_mask=candidate_mask,
        i=i,
        j=j,
        k_grid=k_grid,
        L_grid=L_grid,
        hjb_constant=hjb_constant,
        source="local_solver",
    )

    unique: list[np.ndarray] = []

    for z in starts:
        z = np.clip(np.asarray(z, dtype=float), 0.0, 1.0)
        if not any(np.allclose(z, old) for old in unique):
            unique.append(z)

    for z0 in unique:
        try:
            res = minimize(
                obj,
                z0,
                method="Nelder-Mead",
                options={
                    "maxiter": solver_options.local_solver_maxiter,
                    "xatol": solver_options.objective_tol,
                    "fatol": solver_options.objective_tol,
                    "disp": False,
                },
            )

            tau, H = _z_to_tau_H(res.x, bounds)

            out.append(
                evaluate_pointwise_candidate(
                    s=s,
                    x=x,
                    tau=tau,
                    H=H,
                    source="local_solver",
                    primitives=primitives,
                    continuation=continuation,
                    asset_params=asset_params,
                    economy_params=economy_params,
                    policy_options=policy_options,
                    oracle_options=oracle_options,
                    state_options=state_options,
                    solver_options=solver_options,
                    costates=costates,
                    payoff_params=payoff_params,
                    candidate_mask=candidate_mask,
                    i=i,
                    j=j,
                    k_grid=k_grid,
                    L_grid=L_grid,
                    hjb_constant=hjb_constant,
                )
            )

        except Exception:
            continue

    return out


def _run_boundary_solvers(
    starts: Sequence[np.ndarray],
    s: int,
    x: State,
    bounds: TauHClosedBounds,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    candidate_mask: Optional[np.ndarray],
    i: Optional[int],
    j: Optional[int],
    k_grid: Optional[Sequence[float]],
    L_grid: Optional[Sequence[float]],
    hjb_constant: float,
) -> list[CandidateEvaluation]:
    if minimize is None or not solver_options.use_boundary_solvers:
        return []

    out: list[CandidateEvaluation] = []

    boundary_specs = [
        ("tau_lower", 0, 0.0),
        ("tau_upper", 0, 1.0),
        ("H_lower", 1, 0.0),
        ("H_upper", 1, 1.0),
    ]

    for _, fixed_idx, fixed_val in boundary_specs:
        if fixed_idx == 0 and bounds.tau_width <= 0.0:
            continue

        if fixed_idx == 1 and bounds.H_width <= 0.0:
            continue

        free_idx = 1 - fixed_idx

        def one_dim_obj(y: Sequence[float]) -> float:
            z = np.asarray([0.5, 0.5], dtype=float)
            z[fixed_idx] = fixed_val
            z[free_idx] = np.clip(float(np.asarray(y).reshape(-1)[0]), 0.0, 1.0)

            tau, H = _z_to_tau_H(z, bounds)

            cand = evaluate_pointwise_candidate(
                s=s,
                x=x,
                tau=tau,
                H=H,
                source="boundary_solver",
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                solver_options=solver_options,
                costates=costates,
                payoff_params=payoff_params,
                candidate_mask=candidate_mask,
                i=i,
                j=j,
                k_grid=k_grid,
                L_grid=L_grid,
                hjb_constant=hjb_constant,
            )

            if cand.feasible and math.isfinite(cand.hamiltonian):
                return -cand.hamiltonian

            return solver_options.penalty_invalid

        one_dim_starts = sorted(
            set([0.0, 0.5, 1.0] + [float(z[free_idx]) for z in starts])
        )

        for y0 in one_dim_starts:
            try:
                res = minimize(
                    one_dim_obj,
                    np.asarray([y0], dtype=float),
                    method="Nelder-Mead",
                    options={
                        "maxiter": max(30, solver_options.local_solver_maxiter // 2),
                        "xatol": solver_options.objective_tol,
                        "fatol": solver_options.objective_tol,
                        "disp": False,
                    },
                )

                z = np.asarray([0.5, 0.5], dtype=float)
                z[fixed_idx] = fixed_val
                z[free_idx] = np.clip(float(res.x[0]), 0.0, 1.0)

                tau, H = _z_to_tau_H(z, bounds)

                out.append(
                    evaluate_pointwise_candidate(
                        s=s,
                        x=x,
                        tau=tau,
                        H=H,
                        source="boundary_solver",
                        primitives=primitives,
                        continuation=continuation,
                        asset_params=asset_params,
                        economy_params=economy_params,
                        policy_options=policy_options,
                        oracle_options=oracle_options,
                        state_options=state_options,
                        solver_options=solver_options,
                        costates=costates,
                        payoff_params=payoff_params,
                        candidate_mask=candidate_mask,
                        i=i,
                        j=j,
                        k_grid=k_grid,
                        L_grid=L_grid,
                        hjb_constant=hjb_constant,
                    )
                )

            except Exception:
                continue

    return out


# ============================================================
# KKT diagnostics
# ============================================================

def _reduced_hamiltonian_at_tau_H(
    tau: float,
    H: float,
    s: int,
    x: State,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    hjb_constant: float,
) -> float:
    cand = evaluate_pointwise_candidate(
        s=s,
        x=x,
        tau=tau,
        H=H,
        source="manual",
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        solver_options=replace(solver_options, accept_tangent_filter=False),
        costates=costates,
        payoff_params=payoff_params,
        candidate_mask=None,
        i=None,
        j=None,
        k_grid=None,
        L_grid=None,
        hjb_constant=hjb_constant,
    )

    return cand.hamiltonian if cand.feasible else -math.inf


def _finite_diff_grad_tau_H(
    cand: CandidateEvaluation,
    s: int,
    x: State,
    bounds: TauHClosedBounds,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    hjb_constant: float,
) -> tuple[float, float]:
    h = solver_options.kkt_step

    def fd_one(name: str, val: float, lo: float, hi: float) -> float:
        if hi - lo <= 0.0:
            return 0.0

        step = h * max(1.0, abs(val), hi - lo)
        plus = min(hi, val + step)
        minus = max(lo, val - step)

        if plus == minus:
            return 0.0

        if name == "tau":
            f_plus = _reduced_hamiltonian_at_tau_H(
                plus,
                cand.control.H,
                s,
                x,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                solver_options=solver_options,
                costates=costates,
                payoff_params=payoff_params,
                hjb_constant=hjb_constant,
            )
            f_minus = _reduced_hamiltonian_at_tau_H(
                minus,
                cand.control.H,
                s,
                x,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                solver_options=solver_options,
                costates=costates,
                payoff_params=payoff_params,
                hjb_constant=hjb_constant,
            )

        else:
            f_plus = _reduced_hamiltonian_at_tau_H(
                cand.control.tau,
                plus,
                s,
                x,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                solver_options=solver_options,
                costates=costates,
                payoff_params=payoff_params,
                hjb_constant=hjb_constant,
            )
            f_minus = _reduced_hamiltonian_at_tau_H(
                cand.control.tau,
                minus,
                s,
                x,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                solver_options=solver_options,
                costates=costates,
                payoff_params=payoff_params,
                hjb_constant=hjb_constant,
            )

        if not (math.isfinite(f_plus) and math.isfinite(f_minus)):
            return math.nan

        return float((f_plus - f_minus) / (plus - minus))

    grad_tau = fd_one("tau", cand.control.tau, bounds.tau_lower, bounds.tau_upper_closed)
    grad_H = fd_one("H", cand.control.H, bounds.H_lower, bounds.H_upper)

    return grad_tau, grad_H


def kkt_diagnostics(
    cand: Optional[CandidateEvaluation],
    s: int,
    x: State,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    solver_options: PointwiseSolverOptions,
    costates: Costates,
    payoff_params: PlannerPayoffParams,
    hjb_constant: float = 0.0,
) -> KKTDiagnostics:
    if cand is None or not cand.feasible:
        return KKTDiagnostics(
            checked=False,
            max_violation=math.inf,
            grad_tau=math.nan,
            grad_H=math.nan,
            transfer_derivative=math.nan,
            tau_violation=math.inf,
            H_violation=math.inf,
            T_violation=math.inf,
            reason="no feasible candidate",
        )

    bounds = _closed_tau_H_bounds(
        s,
        x,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    )

    grad_tau, grad_H = _finite_diff_grad_tau_H(
        cand,
        s,
        x,
        bounds,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        solver_options=solver_options,
        costates=costates,
        payoff_params=payoff_params,
        hjb_constant=hjb_constant,
    )

    def bound_violation(grad: float, val: float, lo: float, hi: float) -> float:
        if not math.isfinite(grad):
            return math.inf

        if hi - lo <= 0.0:
            return 0.0

        if abs(val - lo) <= policy_options.bound_tol:
            # At lower bound, feasible motion is upward; max requires grad <= 0.
            return max(0.0, grad)

        if abs(val - hi) <= policy_options.bound_tol:
            # At upper bound, feasible motion is downward; max requires grad >= 0.
            return max(0.0, -grad)

        return abs(grad)

    tau_v = bound_violation(
        grad_tau,
        cand.control.tau,
        bounds.tau_lower,
        bounds.tau_upper_closed,
    )
    H_v = bound_violation(
        grad_H,
        cand.control.H,
        bounds.H_lower,
        bounds.H_upper,
    )

    dT = cand.transfer_derivative

    if cand.transfer_branch == "finite_interior":
        T_v = abs(dT)
    elif cand.transfer_branch == "lower_bound":
        # At lower T bound, feasible motion is upward; max requires dH/dT <= 0.
        T_v = max(0.0, dT)
    else:
        T_v = math.inf

    max_v = max(float(tau_v), float(H_v), float(T_v))

    return KKTDiagnostics(
        checked=True,
        max_violation=float(max_v),
        grad_tau=float(grad_tau),
        grad_H=float(grad_H),
        transfer_derivative=float(dT),
        tau_violation=float(tau_v),
        H_violation=float(H_v),
        T_violation=float(T_v),
        reason=None if max_v <= solver_options.kkt_tol else "KKT residual above tolerance",
    )


# ============================================================
# Main pointwise solver
# ============================================================

def solve_pointwise_policy(
    s: int,
    x: State,
    costates: Costates,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    oracle_options: Optional[OracleOptions] = None,
    state_options: Optional[StateConstraintOptions] = None,
    solver_options: Optional[PointwiseSolverOptions] = None,
    payoff_params: Optional[PlannerPayoffParams] = None,
    warm_start: Optional[Control] = None,
    viability_witness: Optional[Control] = None,
    candidate_mask: Optional[np.ndarray] = None,
    i: Optional[int] = None,
    j: Optional[int] = None,
    k_grid: Optional[Sequence[float]] = None,
    L_grid: Optional[Sequence[float]] = None,
    hjb_constant: float = 0.0,
) -> PointwiseSolution:
    s = _require_regime(s)

    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if oracle_options is None:
        oracle_options = OracleOptions(control_set="full")

    oracle_options = _oracle_full_options(oracle_options)

    if state_options is None:
        state_options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    if solver_options is None:
        solver_options = PointwiseSolverOptions()

    if payoff_params is None:
        gamma_owner = asset_params.gamma if asset_params is not None else continuation.gamma
        payoff_params = PlannerPayoffParams(gamma_owner=gamma_owner)

    try:
        bounds = _closed_tau_H_bounds(
            s,
            x,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    except Exception as exc:
        return PointwiseSolution(
            status="invalid_state_or_inputs",
            candidate=None,
            control=None,
            reason=str(exc),
            kkt=KKTDiagnostics(
                False,
                math.inf,
                math.nan,
                math.nan,
                math.nan,
                math.inf,
                math.inf,
                math.inf,
                str(exc),
            ),
            n_candidates_evaluated=0,
            n_feasible_candidates=0,
            best_rejected=None,
        )

    raw_candidates: list[tuple[CandidateSource, float, float]] = []

    if warm_start is not None:
        raw_candidates.append(("warm_start", warm_start.tau, warm_start.H))

    if viability_witness is not None:
        raw_candidates.append(("viability_witness", viability_witness.tau, viability_witness.H))

    raw_candidates.extend(
        analytic_tau_H_candidates(
            s,
            x,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    )

    raw_candidates = [
        (
            src,
            _clamp(tau, bounds.tau_lower, bounds.tau_upper_closed),
            _clamp(H, bounds.H_lower, bounds.H_upper),
        )
        for src, tau, H in raw_candidates
    ]

    raw_candidates = _dedupe_controls(raw_candidates, tol=policy_options.bound_tol)

    evaluated: list[CandidateEvaluation] = []

    for source, tau, H in raw_candidates:
        evaluated.append(
            evaluate_pointwise_candidate(
                s=s,
                x=x,
                tau=tau,
                H=H,
                source=source,
                primitives=primitives,
                continuation=continuation,
                asset_params=asset_params,
                economy_params=economy_params,
                policy_options=policy_options,
                oracle_options=oracle_options,
                state_options=state_options,
                solver_options=solver_options,
                costates=costates,
                payoff_params=payoff_params,
                candidate_mask=candidate_mask,
                i=i,
                j=j,
                k_grid=k_grid,
                L_grid=L_grid,
                hjb_constant=hjb_constant,
            )
        )

    starts = [_tau_H_to_z(tau, H, bounds) for _, tau, H in raw_candidates]
    starts.append(np.asarray([0.5, 0.5], dtype=float))

    evaluated.extend(
        _run_local_solver(
            starts,
            s,
            x,
            bounds,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            solver_options=solver_options,
            costates=costates,
            payoff_params=payoff_params,
            candidate_mask=candidate_mask,
            i=i,
            j=j,
            k_grid=k_grid,
            L_grid=L_grid,
            hjb_constant=hjb_constant,
        )
    )

    evaluated.extend(
        _run_boundary_solvers(
            starts,
            s,
            x,
            bounds,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            solver_options=solver_options,
            costates=costates,
            payoff_params=payoff_params,
            candidate_mask=candidate_mask,
            i=i,
            j=j,
            k_grid=k_grid,
            L_grid=L_grid,
            hjb_constant=hjb_constant,
        )
    )

    if solver_options.tiny_rescue_grid_size > 0:
        for source, tau, H in _tiny_tau_H_grid(bounds, solver_options.tiny_rescue_grid_size):
            evaluated.append(
                evaluate_pointwise_candidate(
                    s=s,
                    x=x,
                    tau=tau,
                    H=H,
                    source=source,
                    primitives=primitives,
                    continuation=continuation,
                    asset_params=asset_params,
                    economy_params=economy_params,
                    policy_options=policy_options,
                    oracle_options=oracle_options,
                    state_options=state_options,
                    solver_options=solver_options,
                    costates=costates,
                    payoff_params=payoff_params,
                    candidate_mask=candidate_mask,
                    i=i,
                    j=j,
                    k_grid=k_grid,
                    L_grid=L_grid,
                    hjb_constant=hjb_constant,
                )
            )

    feasible = [c for c in evaluated if c.feasible and math.isfinite(c.hamiltonian)]

    best_rejected: Optional[CandidateEvaluation] = None
    rejected = [c for c in evaluated if not c.feasible]

    if rejected:
        best_rejected = max(
            rejected,
            key=lambda c: c.hamiltonian if math.isfinite(c.hamiltonian) else -math.inf,
        )

    if not feasible:
        no_finite = any(
            c.transfer_branch in ("unbounded_no_finite_max", "flat_no_finite_max")
            for c in evaluated
        )

        status: PointwiseStatus = (
            "no_finite_maximizer" if no_finite else "no_feasible_candidate"
        )
        reason = (
            "semi-infinite transfer branch has no finite maximizer"
            if no_finite
            else "no feasible active-set candidate found"
        )

        return PointwiseSolution(
            status=status,
            candidate=None,
            control=None,
            reason=reason,
            kkt=KKTDiagnostics(
                False,
                math.inf,
                math.nan,
                math.nan,
                math.nan,
                math.inf,
                math.inf,
                math.inf,
                reason,
            ),
            n_candidates_evaluated=len(evaluated),
            n_feasible_candidates=0,
            best_rejected=best_rejected,
        )

    best = max(feasible, key=lambda c: c.hamiltonian)

    kkt = kkt_diagnostics(
        best,
        s,
        x,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        solver_options=solver_options,
        costates=costates,
        payoff_params=payoff_params,
        hjb_constant=hjb_constant,
    )

    return PointwiseSolution(
        status="accepted",
        candidate=best,
        control=best.control,
        reason=None,
        kkt=kkt,
        n_candidates_evaluated=len(evaluated),
        n_feasible_candidates=len(feasible),
        best_rejected=best_rejected,
    )


# ============================================================
# Validation
# ============================================================

def validate_pointwise_layer(
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    oracle_options: Optional[OracleOptions] = None,
) -> dict[str, float]:
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if oracle_options is None:
        oracle_options = OracleOptions(control_set="full")

    oracle_options = _oracle_full_options(oracle_options)

    gamma_owner = asset_params.gamma if asset_params is not None else continuation.gamma
    payoff_params = PlannerPayoffParams(
        gamma_worker=2.0,
        gamma_owner=gamma_owner,
        weight_worker=1.0,
        weight_owner=1.0,
    )

    s = 0
    x = State(k=1.0, L=0.5)
    tau = 0.25
    H = 0.50

    bounds = _closed_tau_H_bounds(
        s,
        x,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    )

    # Finite interior T branch.
    costates_interior = Costates(J_k=0.0, J_L=-0.25)
    t_int = transfer_optimum_for_tau_H(
        s=s,
        x=x,
        tau=tau,
        H=H,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        costates=costates_interior,
        payoff_params=payoff_params,
    )

    if t_int.branch != "finite_interior":
        raise RuntimeError(f"Expected finite interior transfer branch, got {t_int}.")

    # Lower-bound T branch.
    costates_lower = Costates(J_k=0.0, J_L=-10.0)
    t_lower = transfer_optimum_for_tau_H(
        s=s,
        x=x,
        tau=tau,
        H=H,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        costates=costates_lower,
        payoff_params=payoff_params,
    )

    if t_lower.branch != "lower_bound":
        raise RuntimeError(f"Expected lower-bound transfer branch, got {t_lower}.")

    # No-finite-maximizer T branch.
    costates_unbounded = Costates(J_k=0.0, J_L=0.10)
    t_unbounded = transfer_optimum_for_tau_H(
        s=s,
        x=x,
        tau=tau,
        H=H,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        costates=costates_unbounded,
        payoff_params=payoff_params,
    )

    if t_unbounded.branch != "unbounded_no_finite_max":
        raise RuntimeError(f"Expected no-finite transfer branch, got {t_unbounded}.")

    # Candidate evaluation should call live oracle and return finite Hamiltonian.
    cand = evaluate_pointwise_candidate(
        s=s,
        x=x,
        tau=tau,
        H=H,
        source="manual",
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=StateConstraintOptions(primitive_wall_tol=economy_params.state_tol),
        solver_options=PointwiseSolverOptions(
            use_local_solver=False,
            use_boundary_solvers=False,
        ),
        costates=costates_interior,
        payoff_params=payoff_params,
    )

    if not cand.feasible or not math.isfinite(cand.hamiltonian):
        raise RuntimeError(f"Expected feasible pointwise candidate, got {cand}.")

    # Full active-set solve.
    sol = solve_pointwise_policy(
        s=s,
        x=x,
        costates=costates_interior,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        solver_options=PointwiseSolverOptions(
            use_local_solver=True,
            use_boundary_solvers=True,
            tiny_rescue_grid_size=0,
        ),
        payoff_params=payoff_params,
    )

    if sol.status != "accepted" or sol.control is None:
        raise RuntimeError(f"Pointwise active-set solve failed: {sol}.")

    # Explicit no-finite-maximizer branch through the main solver.
    sol_unbounded = solve_pointwise_policy(
        s=s,
        x=x,
        costates=costates_unbounded,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        solver_options=PointwiseSolverOptions(
            use_local_solver=False,
            use_boundary_solvers=False,
            tiny_rescue_grid_size=0,
        ),
        payoff_params=payoff_params,
    )

    if sol_unbounded.status != "no_finite_maximizer":
        raise RuntimeError("Expected no_finite_maximizer status for positive transfer slope.")

    return {
        "finite_interior_T": float(t_int.T),
        "lower_bound_T": float(t_lower.T),
        "transfer_floor": float(bounds.T_lower),
        "manual_candidate_hamiltonian": float(cand.hamiltonian),
        "solution_hamiltonian": float(sol.hamiltonian),
        "solution_tau": float(sol.control.tau),
        "solution_T": float(sol.control.T),
        "solution_H": float(sol.control.H),
        "solution_kkt_max_violation": float(sol.kkt.max_violation),
        "n_candidates_evaluated": float(sol.n_candidates_evaluated),
        "n_feasible_candidates": float(sol.n_feasible_candidates),
        "no_finite_maximizer_status": 1.0,
    }


def module_smoke_test() -> dict[str, float]:
    automation_params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    primitives = build_regime_primitives(automation_params)

    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    continuation = make_test_continuation_bundle(
        asset_params=asset_params,
    )

    economy_params = PlannerEconomyParams(
        tau_upper=1.0,
        transfer_min=0.0,
        worker_consumption_eps=1.0e-8,
        state_tol=1.0e-10,
        control_tol=1.0e-12,
    )

    return validate_pointwise_layer(
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
    )


__all__ = [
    "TransferBranch",
    "PointwiseStatus",
    "CandidateSource",
    "PlannerPayoffParams",
    "Costates",
    "PointwiseSolverOptions",
    "TauHClosedBounds",
    "TransferOptimum",
    "CandidateEvaluation",
    "KKTDiagnostics",
    "PointwiseSolution",
    "planner_flow_from_oracle",
    "hamiltonian_from_oracle",
    "optimal_transfer_given_floor",
    "transfer_optimum_for_tau_H",
    "evaluate_pointwise_candidate",
    "analytic_tau_H_candidates",
    "kkt_diagnostics",
    "solve_pointwise_policy",
    "validate_pointwise_layer",
    "module_smoke_test",
]

Overwriting planner_pointwise.py


In [28]:
import importlib

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import state_constraints
import viability_sets
import planner_pointwise

importlib.reload(automation_block)
importlib.reload(economy)
importlib.reload(policy_sets)
importlib.reload(asset_market)
importlib.reload(continuation_block)
importlib.reload(equilibrium_oracle)
importlib.reload(state_constraints)
importlib.reload(viability_sets)
importlib.reload(planner_pointwise)

block8_report = planner_pointwise.module_smoke_test()

print("Block 8 validation passed.")
print(block8_report)

Block 8 validation passed.
{'finite_interior_T': 0.8239209774753264, 'lower_bound_T': 0.0, 'transfer_floor': 0.0, 'manual_candidate_hamiltonian': -7764.424235037154, 'solution_hamiltonian': -7764.2258686606465, 'solution_tau': 0.99999999, 'solution_T': 0.8239209774753264, 'solution_H': 0.0, 'solution_kkt_max_violation': 0.0, 'n_candidates_evaluated': 38.0, 'n_feasible_candidates': 38.0, 'no_finite_maximizer_status': 1.0}


In [29]:
import importlib
import numpy as np

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import planner_pointwise

importlib.reload(planner_pointwise)

automation_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(automation_params)

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

C_hat = continuation_block.make_test_continuation_bundle(
    asset_params=asset_params,
)

economy_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

policy_options = policy_sets.PolicySetOptions()

s = 0
x = economy.State(k=1.0, L=0.5)

costates = planner_pointwise.Costates(
    J_k=0.0,
    J_L=-0.25,
)

payoff_params = planner_pointwise.PlannerPayoffParams(
    gamma_worker=2.0,
    gamma_owner=5.0,
    weight_worker=1.0,
    weight_owner=1.0,
)

solution = planner_pointwise.solve_pointwise_policy(
    s=s,
    x=x,
    costates=costates,
    primitives=G,
    continuation=C_hat,
    asset_params=asset_params,
    economy_params=economy_params,
    policy_options=policy_options,
    oracle_options=equilibrium_oracle.OracleOptions(control_set="full"),
    solver_options=planner_pointwise.PointwiseSolverOptions(
        use_local_solver=True,
        use_boundary_solvers=True,
        tiny_rescue_grid_size=0,
    ),
    payoff_params=payoff_params,
)

print("Pointwise solution:")
print(solution)

print("\nControl:")
print(solution.control)

print("\nKKT diagnostics:")
print(solution.kkt)

Pointwise solution:
PointwiseSolution(status='accepted', candidate=CandidateEvaluation(feasible=True, control=Control(tau=0.99999999, T=0.8239209774753264, H=0.0), source='analytic', hamiltonian=-7764.2258686606465, flow_payoff=-7764.200901583983, drift_payoff=-0.02496707666325021, transfer_branch='finite_interior', finite_transfer_maximizer=True, transfer_derivative=0.0, oracle_status='interior', oracle_reason=None, primitive_inward=True, tangent_accepted=True, inward_reason=None, tangent_reason=None, k_dot=-0.19519826731115025, L_dot=0.09986830665300084, W_K_dot=-0.09532996065814942, tau_lower_active=False, tau_upper_active=True, T_lower_active=False, H_lower_active=True, H_upper_active=False, exceeds_compact_T_cap=False, rejection_reason=None), control=Control(tau=0.99999999, T=0.8239209774753264, H=0.0), reason=None, kkt=KKTDiagnostics(checked=True, max_violation=0.0, grad_tau=0.2715196387738239, grad_H=0.0, transfer_derivative=0.0, tau_violation=0.0, H_violation=0.0, T_violation=0

# Block 9 — Howard inner planner solver

Block 9 solves the planner HJB for a **fixed anticipated environment**.

At this stage the private continuation environment is already fixed:

$$
\mathcal C[\hat u]
=
\left\{
\Psi_s^{\hat u},
\omega_s^{\hat u},
\chi^{\hat u},
\lambda^{Q,\hat u},
\text{validity masks}
\right\}_{s=0,1}.
$$

The pure conditional viability sets are also fixed:

$$
V_1^{\hat u},
\qquad
V_0^{\hat u}.
$$

Block 9 then runs Howard iteration to compute the planner value functions and the planner best-response policy on those fixed domains.

The central rule is:

$$
\boxed{
\text{Howard may update numerical active masks, but it must not redefine the pure viability sets.}
}
$$

The pure viability sets $V_s^{\hat u}$ are economic domain objects from Block 7. The Howard active masks $A_s$ are numerical working domains used by the linear solver.

---

## Role of Block 9

For a fixed anticipated Markov rule $\hat u$, the inner planner problem is:

$$
\mathcal G,
\quad
\mathcal C[\hat u],
\quad
V_1^{\hat u},
\quad
V_0^{\hat u}
\quad
\longrightarrow
\quad
(J_1,J_0,u_1^\star,u_0^\star).
$$

Block 9 does not update $\hat u$. It computes the planner best response given the current frozen continuation environment.

The outer fixed point comes later in Block 10.

---

## Planner HJBs

The planner state is

$$
x=(k,L).
$$

The planner control is

$$
u=(\tau,T,H).
$$

For a frozen continuation environment, define the live current-control drift

$$
f_s^{\hat u}(x;u)
=
\left(
\dot k_s^{\hat u}(x;u),
\dot L_s^{\hat u}(x;u)
\right).
$$

The superscript $\hat u$ means that private continuation objects are frozen. The argument $u$ means that current policy is evaluated live through the oracle.

In regime $s=1$, which is absorbing, the state-constrained planner HJB is

$$
\rho J_1(x)
=
\sup_{u\in U_1^{in}(x)}
\left\{
\mathcal U_1^{\hat u}(x,u)
+
\nabla J_1(x)\cdot f_1^{\hat u}(x;u)
\right\},
\qquad
x\in V_1^{\hat u}.
$$

In regime $s=0$, the planner faces a Poisson switch to regime $1$, so the HJB is

$$
\rho J_0(x)
=
\sup_{u\in U_0^{in}(x)}
\left\{
\mathcal U_0^{\hat u}(x,u)
+
\nabla J_0(x)\cdot f_0^{\hat u}(x;u)
+
\lambda
\left(
J_1(x)-J_0(x)
\right)
\right\},
\qquad
x\in V_0^{\hat u}.
$$

The Poisson term matters for HJB evaluation in regime $0$, but it is independent of the current control at fixed $x$. Therefore it does not affect the pointwise argmax in Block 8.

---

## Inward-feasible controls

The planner solves a state-constrained HJB. At boundaries of the admissible domain, the admissible current controls are restricted to those whose induced drift is inward or tangent.

Formally, define

$$
U_s^{in}(x)
=
\left\{
u\in U_s^{full}(x):
f_s^{\hat u}(x;u)\in T_{V_s^{\hat u}}(x)
\right\},
$$

where $T_{V_s^{\hat u}}(x)$ is the tangent cone of the viability set at $x$.

On the numerical grid, Block 9 uses the Howard active mask $A_s\subseteq V_s^{\hat u}$ as the current working domain. During linear evaluation, the mask is frozen. Between Howard sweeps, the solver may update $A_s$ as a numerical active mask, but it must never mutate or recompute $V_s^{\hat u}$.

The distinction is:

$$
\boxed{
V_s^{\hat u}
=
\text{pure viability set},
\qquad
A_s
=
\text{Howard working mask}.
}
$$

---

## Linear policy evaluation

Given a fixed policy $u_s(x)$, the HJB becomes linear.

For regime $1$:

$$
\rho J_1(x)
=
\mathcal U_1^{\hat u}(x,u_1(x))
+
\nabla J_1(x)\cdot f_1^{\hat u}(x;u_1(x)).
$$

For regime $0$:

$$
(\rho+\lambda)J_0(x)
=
\mathcal U_0^{\hat u}(x,u_0(x))
+
\nabla J_0(x)\cdot f_0^{\hat u}(x;u_0(x))
+
\lambda J_1(x).
$$

Thus the regime-0 linear solve uses the already-evaluated $J_1$ array as a source term.

---

## Discrete generator representation

At each active grid node $x_i$, the live oracle gives a drift

$$
f_i
=
f_s^{\hat u}(x_i;u_i).
$$

Block 9 represents this drift using local active neighbours:

$$
f_i
\approx
\sum_{m}
q_{im}
\left(
x_m-x_i
\right),
\qquad
q_{im}\ge 0.
$$

This gives the monotone local approximation

$$
f_i\cdot \nabla J(x_i)
\approx
\sum_m
q_{im}
\left(
J(x_m)-J(x_i)
\right).
$$

This representation is consistent with the tangent-cone logic from Block 7. If the drift cannot be represented by active neighbours, the node is removed from the Howard active mask, not from the pure viability set.

---

## Howard active masks

The active masks satisfy

$$
A_1\subseteq V_1^{\hat u},
$$

$$
A_0\subseteq V_0^{\hat u},
$$

and

$$
A_0\subseteq A_1.
$$

The final condition reflects that pre-switch admissibility requires post-switch admissibility at the same state.

A mask update inside Howard is a numerical support update. It is not a new viability solve. The pure viability masks remain unchanged throughout Block 9.

The hard check is:

$$
\boxed{
\text{Block 9 must leave }V_1^{\hat u}\text{ and }V_0^{\hat u}\text{ bitwise unchanged.}
}
$$

---

## Policy improvement

After linear policy evaluation, Block 9 computes costates

$$
p_s(x)
=
\left(
J_{s,k}(x),
J_{s,L}(x)
\right)
$$

from the current value function.

It then calls the Block 8 pointwise active-set solver at each active node:

$$
u_s^{new}(x)
\in
\arg\max_{u\in U_s^{in}(x)}
\left\{
\mathcal U_s^{\hat u}(x,u)
+
J_{s,k}(x)\dot k_s^{\hat u}(x;u)
+
J_{s,L}(x)\dot L_s^{\hat u}(x;u)
\right\}.
$$

For regime $0$, the Poisson term

$$
\lambda(J_1(x)-J_0(x))
$$

may be included in the reported HJB value, but it does not affect the current-control argmax because it is independent of $u$.

Policy improvement must call the live oracle through Block 8. It must not reuse stale arrays for

$$
r_f,
\qquad
\dot k,
\qquad
\dot L.
$$

---

## Policy damping

Block 9 may damp policy updates:

$$
u_s^{next}
=
(1-\alpha)u_s^{old}
+
\alpha u_s^{new},
\qquad
\alpha\in(0,1].
$$

The default is usually

$$
\alpha=1.
$$

Damping is a numerical device. It does not change the definition of the planner best-response problem.

---

## Howard iteration

A Howard cycle is:

1. freeze $V_1^{\hat u}$ and $V_0^{\hat u}$;
2. freeze $A_1$ and $A_0$ during linear evaluation;
3. solve the linear HJB for $J_1$ under the current $u_1$;
4. solve the linear HJB for $J_0$ under the current $u_0$, using $J_1$ in the Poisson term;
5. compute costates from $J_1$ and $J_0$;
6. improve $u_1$ and $u_0$ node by node using Block 8;
7. optionally update Howard active masks between sweeps;
8. repeat until value, policy, residual, and active-mask diagnostics stabilise.

The flow is:

$$
(u_1,u_0,A_1,A_0)
\to
(J_1,J_0)
\to
(u_1',u_0',A_1',A_0')
\to
\cdots.
$$

---

## Inputs

Block 9 takes:

```text
viability
primitives
continuation
asset_params
economy_params
policy_options
oracle_options
state_options
pointwise_options
howard_options
hjb_params
payoff_params
warm_start
```

Economically, the important inputs are:

$$
\mathcal G,
\qquad
\mathcal C[\hat u],
\qquad
V_1^{\hat u},
\qquad
V_0^{\hat u}.
$$

Warm starts may include:

$$
J_1,
\quad
J_0,
\quad
u_1,
\quad
u_0,
\quad
A_1,
\quad
A_0.
$$

---

## Outputs

Block 9 returns:

```text
HowardResult
```

with:

```text
J1
J0
policy1
policy0
A1
A0
converged
n_iter
diagnostics
history
```

The policy arrays contain:

$$
u_s(x)
=
(\tau_s(x),T_s(x),H_s(x)).
$$

The active masks satisfy:

$$
A_s\subseteq V_s^{\hat u}.
$$

---

## Diagnostics

Important diagnostics include:

```text
converged
n_iter
n_V1
n_V0
n_A1
n_A0
A1_subset_V1
A0_subset_V0
A0_subset_A1
pure_viability_masks_unchanged
last_value_change
last_policy_change
last_active_mask_change
last_hjb_residual
last_kkt_violation
last_policy_fail_1
last_policy_fail_0
```

The most important diagnostic is:

```text
pure_viability_masks_unchanged
```

which should equal `1.0`.

---

## What Block 9 must not do

Block 9 should not:

- solve the private continuation problem;
- recompute $\Psi_s^{\hat u}$ or $\omega_s^{\hat u}$;
- recompute viability sets;
- run viability peeling;
- redefine $V_1^{\hat u}$ or $V_0^{\hat u}$;
- run the outer Markov-perfect fixed point;
- freeze old arrays for $r_f$, $\dot k$, or $\dot L$ during policy improvement;
- treat Howard active masks as economic viability sets;
- treat the artificial transfer cap as an economic primitive;
- use a coarse global control grid as the main policy optimiser.

The key forbidden confusion is:

$$
\boxed{
\text{Do not confuse Howard active masks with pure viability sets.}
}
$$

---

## Validation checks

The Block 9 validation harness should check:

1. $V_1^{\hat u}$ is not mutated;
2. $V_0^{\hat u}$ is not mutated;
3. $A_1\subseteq V_1^{\hat u}$;
4. $A_0\subseteq V_0^{\hat u}$;
5. $A_0\subseteq A_1$;
6. $J_1$ is finite on $A_1$;
7. $J_0$ is finite on $A_0$;
8. the linear HJB residual is reported;
9. policy improvement calls the Block 8 active-set solver;
10. live oracle evaluation is used for fixed-policy evaluation and policy improvement;
11. policy failures and no-finite-maximiser branches are reported rather than hidden.

---

## One-line summary

Block 9 computes the inner planner best response for a fixed anticipated environment:

$$
\boxed{
(\mathcal G,\mathcal C[\hat u],V_1^{\hat u},V_0^{\hat u})
\longrightarrow
(J_1,J_0,u_1^\star,u_0^\star).
}
$$

It does this by solving linear HJBs under fixed policies, improving policies node by node with the live oracle and the active-set solver, and preserving the pure viability sets unchanged throughout Howard iteration.

In [30]:
%%writefile planner_howard.py
from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Optional, Sequence
import math
import numpy as np

try:
    from scipy.sparse import lil_matrix
    from scipy.sparse.linalg import spsolve
except Exception:  # pragma: no cover
    lil_matrix = None
    spsolve = None

from automation_block import (
    AutomationParams,
    RegimePrimitives,
    build_regime_primitives,
)
from model.economy import (
    State,
    Control,
    PlannerEconomyParams,
)
import policy_sets
from policy_sets import PolicySetOptions
from asset_market import (
    AssetMarketParams,
    make_infinite_asset_market_params,
)
from continuation_block import (
    ContinuationBundle,
    make_test_continuation_bundle,
)
from equilibrium_oracle import (
    OracleOptions,
    OracleEval,
    live_oracle,
)
from state_constraints import (
    StateConstraintOptions,
    primitive_inward_diagnostics,
)
from viability_sets import (
    ViabilityGrid,
    ViabilityKernel,
    ConditionalViabilityResult,
    ViabilityOptions,
    compute_conditional_viability_sets,
)
from planner_pointwise import (
    Costates,
    PlannerPayoffParams,
    PointwiseSolverOptions,
    PointwiseSolution,
    solve_pointwise_policy,
    planner_flow_from_oracle,
)


# ============================================================
# Block 9 contract: Howard inner planner solver
# ============================================================
#
# For a fixed anticipated environment, Howard solves the planner HJB on
# fixed pure viability sets:
#
#     V_1^{hat u}, V_0^{hat u}.
#
# Inputs:
#   - G;
#   - C[hat u];
#   - V_1^{hat u}, V_0^{hat u};
#   - warm-start values, policies, and Howard active masks.
#
# Main loop:
#   1. freeze pure viability sets;
#   2. freeze Howard active masks during linear policy evaluation;
#   3. solve the linear HJB under the current policy;
#   4. improve policies nodewise using Block 8 and the live Block 6 oracle;
#   5. optionally update Howard active masks between sweeps only;
#   6. damp policy updates if requested;
#   7. repeat until policy/value/active-mask diagnostics stabilise.
#
# Forbidden responsibilities:
#   - no private continuation solve;
#   - no viability peeling;
#   - no outer MPE fixed point;
#   - no recomputation of omega_s^{hat u};
#   - no stale arrays for r_f, k_dot, or L_dot during policy improvement;
#   - no redefinition of pure viability sets.
#
# Important distinction:
#   V_s^{hat u} is an economic viability set from Block 7.
#   A_s is a Howard-only numerical working mask satisfying A_s subset V_s^{hat u}.


# ============================================================
# Dataclasses
# ============================================================

@dataclass(frozen=True)
class HowardHJBParams:
    """
    Discounting and regime-switching parameters used in the planner HJB.

    rho:
        Planner discount rate.

    lam:
        Physical Poisson arrival intensity. If None, the solver uses
        primitives.params.lam.
    """
    rho: float = 0.04
    lam: Optional[float] = None

    def __post_init__(self) -> None:
        rho = float(self.rho)
        if not math.isfinite(rho) or rho <= 0.0:
            raise ValueError("rho must be positive and finite.")
        object.__setattr__(self, "rho", rho)

        if self.lam is not None:
            lam = float(self.lam)
            if not math.isfinite(lam) or lam < 0.0:
                raise ValueError("lam must be nonnegative and finite.")
            object.__setattr__(self, "lam", lam)

    def lambda_value(self, primitives: RegimePrimitives) -> float:
        if self.lam is None:
            return float(primitives.params.lam)
        return float(self.lam)


@dataclass(frozen=True)
class HowardOptions:
    """
    Numerical options for the inner Howard planner solver.
    """
    max_iter: int = 40
    value_tol: float = 1.0e-7
    policy_tol: float = 1.0e-6
    residual_tol: float = 1.0e-7
    policy_damping: float = 1.0

    improve_policy: bool = True
    update_active_masks: bool = True
    active_prune_passes: int = 8

    gradient_radius: int = 1
    generator_radius: int = 1
    cone_residual_tol: float = 1.0e-7

    verbose: bool = False

    def __post_init__(self) -> None:
        if self.max_iter < 1:
            raise ValueError("max_iter must be at least 1.")
        if self.value_tol < 0.0:
            raise ValueError("value_tol must be nonnegative.")
        if self.policy_tol < 0.0:
            raise ValueError("policy_tol must be nonnegative.")
        if self.residual_tol < 0.0:
            raise ValueError("residual_tol must be nonnegative.")
        if not (0.0 < self.policy_damping <= 1.0):
            raise ValueError("policy_damping must lie in (0,1].")
        if self.active_prune_passes < 1:
            raise ValueError("active_prune_passes must be at least 1.")
        if self.gradient_radius < 1:
            raise ValueError("gradient_radius must be at least 1.")
        if self.generator_radius < 1:
            raise ValueError("generator_radius must be at least 1.")
        if self.cone_residual_tol < 0.0:
            raise ValueError("cone_residual_tol must be nonnegative.")


@dataclass(frozen=True)
class RegimePolicy:
    """
    Regime-wise policy arrays on a ViabilityGrid.

    Values are meaningful only on the associated active or viability mask.
    """
    tau: np.ndarray
    T: np.ndarray
    H: np.ndarray

    def copy(self) -> "RegimePolicy":
        return RegimePolicy(
            tau=np.array(self.tau, dtype=float, copy=True),
            T=np.array(self.T, dtype=float, copy=True),
            H=np.array(self.H, dtype=float, copy=True),
        )

    def control(self, i: int, j: int) -> Control:
        return Control(
            tau=float(self.tau[i, j]),
            T=float(self.T[i, j]),
            H=float(self.H[i, j]),
        )

    def with_control(self, i: int, j: int, u: Control) -> None:
        self.tau[i, j] = float(u.tau)
        self.T[i, j] = float(u.T)
        self.H[i, j] = float(u.H)


@dataclass(frozen=True)
class HowardWarmStart:
    """
    Optional warm start for Block 9.
    """
    J1: Optional[np.ndarray] = None
    J0: Optional[np.ndarray] = None
    policy1: Optional[RegimePolicy] = None
    policy0: Optional[RegimePolicy] = None
    A1: Optional[np.ndarray] = None
    A0: Optional[np.ndarray] = None


@dataclass(frozen=True)
class DriftWeights:
    """
    Local monotone generator representation:

        f(x_i) ≈ sum_m rate_m * (x_m - x_i).

    Then

        f · grad J ≈ sum_m rate_m * (J_m - J_i).
    """
    accepted: bool
    residual: float
    rates: tuple[float, ...]
    neighbors: tuple[tuple[int, int], ...]
    reason: Optional[str]

    @property
    def total_rate(self) -> float:
        return float(sum(self.rates))


@dataclass(frozen=True)
class FixedPolicyNodeEval:
    """
    Evaluation of a fixed policy at one grid node.
    """
    valid: bool
    reason: Optional[str]
    oracle: Optional[OracleEval]
    flow_payoff: float
    drift_weights: DriftWeights
    k_dot: float
    L_dot: float
    W_K_dot: float
    primitive_inward: bool
    primitive_reason: Optional[str]


@dataclass(frozen=True)
class RegimeLinearEvaluation:
    """
    Result of one linear HJB evaluation for a fixed regime policy.
    """
    regime: int
    J: np.ndarray
    active_mask: np.ndarray
    evaluation_mask: np.ndarray
    converged: bool
    reason: Optional[str]
    n_active_input: int
    n_active_eval: int
    n_pruned: int
    max_linear_residual: float
    node_evals: dict[tuple[int, int], FixedPolicyNodeEval]


@dataclass(frozen=True)
class RegimePolicyImprovement:
    """
    Result of one nodewise policy-improvement sweep.
    """
    regime: int
    policy: RegimePolicy
    n_attempted: int
    n_accepted: int
    n_failed: int
    max_policy_change: float
    max_kkt_violation: float
    n_no_finite_maximizer: int


@dataclass(frozen=True)
class HowardIterationDiagnostics:
    iteration: int
    value_change: float
    policy_change: float
    active_mask_change: int
    max_hjb_residual: float
    max_kkt_violation: float
    n_active_1: int
    n_active_0: int
    n_policy_fail_1: int
    n_policy_fail_0: int


@dataclass(frozen=True)
class HowardResult:
    """
    Output of Block 9.
    """
    grid: ViabilityGrid
    V1: ViabilityKernel
    V0: ViabilityKernel

    J1: np.ndarray
    J0: np.ndarray
    policy1: RegimePolicy
    policy0: RegimePolicy
    A1: np.ndarray
    A0: np.ndarray

    converged: bool
    n_iter: int
    diagnostics: dict[str, float]
    history: list[HowardIterationDiagnostics]


# ============================================================
# Basic helpers
# ============================================================

def _require_regime(s: int) -> int:
    if s not in (0, 1):
        raise ValueError("regime s must be 0 or 1.")
    return int(s)


def _empty_float(shape: tuple[int, int], fill: float = math.nan) -> np.ndarray:
    out = np.empty(shape, dtype=float)
    out.fill(float(fill))
    return out


def _as_bool_mask(mask: np.ndarray, shape: tuple[int, int], *, name: str) -> np.ndarray:
    arr = np.asarray(mask, dtype=bool)
    if arr.shape != shape:
        raise ValueError(f"{name} must have shape {shape}.")
    return arr


def _oracle_full_options(options: Optional[OracleOptions]) -> OracleOptions:
    if options is None:
        return OracleOptions(control_set="full")
    return replace(options, control_set="full")


def _finite_or_nan_max(values: Sequence[float]) -> float:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return math.inf
    return float(np.max(vals))


def _sup_norm_diff(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:
    mask = np.asarray(mask, dtype=bool)
    both = mask & np.isfinite(a) & np.isfinite(b)
    if not np.any(both):
        return math.inf
    return float(np.max(np.abs(a[both] - b[both])))


def _policy_sup_change(
    old: RegimePolicy,
    new: RegimePolicy,
    mask: np.ndarray,
) -> float:
    mask = np.asarray(mask, dtype=bool)
    if not np.any(mask):
        return 0.0

    changes = []

    for a, b in (
        (old.tau, new.tau),
        (old.T, new.T),
        (old.H, new.H),
    ):
        good = mask & np.isfinite(a) & np.isfinite(b)
        if np.any(good):
            denom = np.maximum(1.0, np.abs(a[good]))
            changes.append(np.max(np.abs(a[good] - b[good]) / denom))

    if not changes:
        return math.inf

    return float(max(changes))


def _damped_control(old: Control, new: Control, alpha: float) -> Control:
    return Control(
        tau=(1.0 - alpha) * old.tau + alpha * new.tau,
        T=(1.0 - alpha) * old.T + alpha * new.T,
        H=(1.0 - alpha) * old.H + alpha * new.H,
    )


# ============================================================
# Initial policies and warm starts
# ============================================================

def policy_from_viability_kernel(
    kernel: ViabilityKernel,
    *,
    primitives: RegimePrimitives,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
) -> RegimePolicy:
    """
    Initialise a Howard policy from Block 7 viability witnesses.

    If a viable node lacks a finite witness entry, fall back to the midpoint of
    the compactified policy set at that node. The fallback is numerical only and
    should be rare.
    """
    shape = kernel.mask.shape

    tau = _empty_float(shape)
    T = _empty_float(shape)
    H = _empty_float(shape)

    for i in range(shape[0]):
        for j in range(shape[1]):
            if not bool(kernel.mask[i, j]):
                continue

            if (
                math.isfinite(float(kernel.tau[i, j]))
                and math.isfinite(float(kernel.T[i, j]))
                and math.isfinite(float(kernel.H[i, j]))
            ):
                tau[i, j] = float(kernel.tau[i, j])
                T[i, j] = float(kernel.T[i, j])
                H[i, j] = float(kernel.H[i, j])
                continue

            x = kernel.grid.state(i, j)

            bounds = policy_sets.compact_policy_bounds(
                s=kernel.regime,
                x=x,
                primitives=primitives,
                economy_params=economy_params,
                options=policy_options,
            )

            u = policy_sets.midpoint_control(bounds)

            tau[i, j] = u.tau
            T[i, j] = u.T
            H[i, j] = u.H

    return RegimePolicy(tau=tau, T=T, H=H)


def _initial_value_array(
    mask: np.ndarray,
    warm: Optional[np.ndarray],
    shape: tuple[int, int],
) -> np.ndarray:
    if warm is not None:
        arr = np.asarray(warm, dtype=float)
        if arr.shape != shape:
            raise ValueError("warm value array has wrong shape.")
        out = np.array(arr, dtype=float, copy=True)
    else:
        out = _empty_float(shape)
        out[mask] = 0.0

    out[~mask] = np.nan
    return out


def _initial_active_mask(
    pure_mask: np.ndarray,
    warm: Optional[np.ndarray],
    shape: tuple[int, int],
    *,
    name: str,
) -> np.ndarray:
    if warm is None:
        return np.asarray(pure_mask, dtype=bool).copy()

    A = _as_bool_mask(warm, shape, name=name)
    return A & np.asarray(pure_mask, dtype=bool)


# ============================================================
# Local drift-generator representation
# ============================================================

def _local_active_neighbors(
    active_mask: np.ndarray,
    i: int,
    j: int,
    grid: ViabilityGrid,
    *,
    radius: int,
) -> tuple[list[tuple[int, int]], np.ndarray]:
    mask = np.asarray(active_mask, dtype=bool)
    k_grid = grid.k_grid
    L_grid = grid.L_grid

    neighbors: list[tuple[int, int]] = []
    generators: list[tuple[float, float]] = []

    n_k, n_L = mask.shape

    for ii in range(max(0, i - radius), min(n_k, i + radius + 1)):
        for jj in range(max(0, j - radius), min(n_L, j + radius + 1)):
            if ii == i and jj == j:
                continue
            if not bool(mask[ii, jj]):
                continue

            dk = float(k_grid[ii] - k_grid[i])
            dL = float(L_grid[jj] - L_grid[j])

            if dk == 0.0 and dL == 0.0:
                continue

            neighbors.append((ii, jj))
            generators.append((dk, dL))

    if not generators:
        return neighbors, np.zeros((0, 2), dtype=float)

    return neighbors, np.asarray(generators, dtype=float)


def _relative_residual(v: np.ndarray, approx: np.ndarray) -> float:
    scale = max(1.0, float(np.linalg.norm(v)))
    return float(np.linalg.norm(v - approx) / scale)


def _nonnegative_cone_weights_2d(
    target: np.ndarray,
    neighbors: list[tuple[int, int]],
    generators: np.ndarray,
    *,
    tol: float,
) -> DriftWeights:
    """
    Find nonnegative rates alpha such that

        target ≈ sum alpha_m generator_m.

    In two dimensions, it is enough to enumerate single rays and pairs.
    """
    f = np.asarray(target, dtype=float).reshape(2)
    f_norm = float(np.linalg.norm(f))

    if f_norm <= tol:
        return DriftWeights(
            accepted=True,
            residual=0.0,
            rates=tuple(),
            neighbors=tuple(),
            reason=None,
        )

    if generators.size == 0:
        return DriftWeights(
            accepted=False,
            residual=math.inf,
            rates=tuple(),
            neighbors=tuple(),
            reason="nonzero drift but no active neighbours",
        )

    if generators.ndim != 2 or generators.shape[1] != 2:
        raise ValueError("generators must have shape (n,2).")

    best_resid = math.inf
    best_rates: tuple[float, ...] = tuple()
    best_neighbors: tuple[tuple[int, int], ...] = tuple()

    n = generators.shape[0]

    # Single-ray candidates.
    for idx in range(n):
        g = generators[idx]
        gg = float(np.dot(g, g))

        if gg <= 0.0 or not math.isfinite(gg):
            continue

        a = float(np.dot(f, g) / gg)

        if a < -tol:
            continue

        a = max(0.0, a)
        approx = a * g
        resid = _relative_residual(f, approx)

        if resid < best_resid:
            best_resid = resid
            best_rates = (a,)
            best_neighbors = (neighbors[idx],)

    # Pair-cone candidates.
    for a_idx in range(n):
        for b_idx in range(a_idx + 1, n):
            G = np.column_stack((generators[a_idx], generators[b_idx]))
            det = float(np.linalg.det(G))

            if abs(det) <= 1.0e-14:
                continue

            coeff = np.linalg.solve(G, f)

            if np.any(coeff < -tol):
                continue

            coeff = np.maximum(coeff, 0.0)
            approx = G @ coeff
            resid = _relative_residual(f, approx)

            if resid < best_resid:
                best_resid = resid
                best_rates = (float(coeff[0]), float(coeff[1]))
                best_neighbors = (neighbors[a_idx], neighbors[b_idx])

    accepted = best_resid <= tol

    return DriftWeights(
        accepted=bool(accepted),
        residual=float(best_resid),
        rates=best_rates if accepted else tuple(),
        neighbors=best_neighbors if accepted else tuple(),
        reason=None if accepted else "drift outside local active-mask tangent cone",
    )


def drift_weights_for_node(
    active_mask: np.ndarray,
    i: int,
    j: int,
    grid: ViabilityGrid,
    k_dot: float,
    L_dot: float,
    *,
    options: HowardOptions,
) -> DriftWeights:
    neighbors, generators = _local_active_neighbors(
        active_mask,
        i,
        j,
        grid,
        radius=options.generator_radius,
    )

    return _nonnegative_cone_weights_2d(
        np.asarray([float(k_dot), float(L_dot)], dtype=float),
        neighbors,
        generators,
        tol=options.cone_residual_tol,
    )


# ============================================================
# Fixed-policy evaluation
# ============================================================

def evaluate_fixed_policy_node(
    s: int,
    i: int,
    j: int,
    policy: RegimePolicy,
    active_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    howard_options: HowardOptions,
    payoff_params: PlannerPayoffParams,
) -> FixedPolicyNodeEval:
    s = _require_regime(s)
    x = grid.state(i, j)
    u = policy.control(i, j)

    try:
        ev = live_oracle(
            s=s,
            x=x,
            u=u,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            options=oracle_options,
        )
    except Exception as exc:
        return FixedPolicyNodeEval(
            valid=False,
            reason=f"live oracle exception: {exc}",
            oracle=None,
            flow_payoff=math.nan,
            drift_weights=DriftWeights(False, math.inf, tuple(), tuple(), str(exc)),
            k_dot=math.nan,
            L_dot=math.nan,
            W_K_dot=math.nan,
            primitive_inward=False,
            primitive_reason=str(exc),
        )

    if not ev.valid_for_drift:
        return FixedPolicyNodeEval(
            valid=False,
            reason=f"oracle invalid for drift: {ev.status}, {ev.reason}",
            oracle=ev,
            flow_payoff=math.nan,
            drift_weights=DriftWeights(False, math.inf, tuple(), tuple(), ev.reason),
            k_dot=ev.k_dot,
            L_dot=ev.L_dot,
            W_K_dot=ev.W_K_dot,
            primitive_inward=False,
            primitive_reason=ev.reason,
        )

    primitive = primitive_inward_diagnostics(
        x,
        ev.k_dot,
        ev.L_dot,
        economy_params=economy_params,
        options=state_options,
    )

    if not primitive.accepted:
        return FixedPolicyNodeEval(
            valid=False,
            reason=primitive.reason,
            oracle=ev,
            flow_payoff=math.nan,
            drift_weights=DriftWeights(False, math.inf, tuple(), tuple(), primitive.reason),
            k_dot=ev.k_dot,
            L_dot=ev.L_dot,
            W_K_dot=ev.W_K_dot,
            primitive_inward=False,
            primitive_reason=primitive.reason,
        )

    weights = drift_weights_for_node(
        active_mask,
        i,
        j,
        grid,
        ev.k_dot,
        ev.L_dot,
        options=howard_options,
    )

    if not weights.accepted:
        return FixedPolicyNodeEval(
            valid=False,
            reason=weights.reason,
            oracle=ev,
            flow_payoff=math.nan,
            drift_weights=weights,
            k_dot=ev.k_dot,
            L_dot=ev.L_dot,
            W_K_dot=ev.W_K_dot,
            primitive_inward=True,
            primitive_reason=None,
        )

    try:
        flow = planner_flow_from_oracle(ev, payoff_params)
    except Exception as exc:
        return FixedPolicyNodeEval(
            valid=False,
            reason=f"flow payoff failed: {exc}",
            oracle=ev,
            flow_payoff=math.nan,
            drift_weights=weights,
            k_dot=ev.k_dot,
            L_dot=ev.L_dot,
            W_K_dot=ev.W_K_dot,
            primitive_inward=True,
            primitive_reason=None,
        )

    return FixedPolicyNodeEval(
        valid=True,
        reason=None,
        oracle=ev,
        flow_payoff=float(flow),
        drift_weights=weights,
        k_dot=ev.k_dot,
        L_dot=ev.L_dot,
        W_K_dot=ev.W_K_dot,
        primitive_inward=True,
        primitive_reason=None,
    )


def _stable_fixed_policy_evaluations(
    s: int,
    policy: RegimePolicy,
    active_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    howard_options: HowardOptions,
    payoff_params: PlannerPayoffParams,
) -> tuple[dict[tuple[int, int], FixedPolicyNodeEval], np.ndarray, int]:
    """
    Trim the Howard active mask until the fixed policy is valid on the
    evaluation mask.

    This updates only the Howard working mask. It never changes V_s^{hat u}.
    """
    working = np.asarray(active_mask, dtype=bool).copy()
    total_pruned = 0
    evals: dict[tuple[int, int], FixedPolicyNodeEval] = {}

    for _ in range(howard_options.active_prune_passes):
        evals = {}
        valid = np.zeros_like(working, dtype=bool)

        for i in range(working.shape[0]):
            for j in range(working.shape[1]):
                if not bool(working[i, j]):
                    continue

                ev = evaluate_fixed_policy_node(
                    s,
                    i,
                    j,
                    policy,
                    working,
                    grid,
                    primitives=primitives,
                    continuation=continuation,
                    asset_params=asset_params,
                    economy_params=economy_params,
                    policy_options=policy_options,
                    oracle_options=oracle_options,
                    state_options=state_options,
                    howard_options=howard_options,
                    payoff_params=payoff_params,
                )

                evals[(i, j)] = ev
                valid[i, j] = ev.valid

        removed = int(np.sum(working & ~valid))
        total_pruned += removed

        if removed == 0:
            return evals, working, total_pruned

        working = valid

    return evals, working, total_pruned


# ============================================================
# Linear HJB evaluation
# ============================================================

def solve_linear_hjb_for_regime(
    s: int,
    policy: RegimePolicy,
    active_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    howard_options: HowardOptions,
    hjb_params: HowardHJBParams,
    payoff_params: PlannerPayoffParams,
    J_switch: Optional[np.ndarray] = None,
) -> RegimeLinearEvaluation:
    """
    Solve the linear HJB under a fixed regime policy.

    Regime 1:
        rho J_1 = U_1 + L^{u_1} J_1.

    Regime 0:
        rho J_0 = U_0 + L^{u_0} J_0 + lambda (J_1 - J_0),

    equivalently

        (rho + lambda) J_0 - L^{u_0}J_0 = U_0 + lambda J_1.
    """
    s = _require_regime(s)
    shape = grid.shape
    oracle_options = _oracle_full_options(oracle_options)

    active = _as_bool_mask(active_mask, shape, name="active_mask").copy()

    if s == 0:
        if J_switch is None:
            raise ValueError("J_switch=J1 is required when solving regime 0.")
        J_switch = np.asarray(J_switch, dtype=float)
        if J_switch.shape != shape:
            raise ValueError("J_switch must have grid.shape.")
        active = active & np.isfinite(J_switch)

    evals, eval_mask, n_pruned = _stable_fixed_policy_evaluations(
        s,
        policy,
        active,
        grid,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=oracle_options,
        state_options=state_options,
        howard_options=howard_options,
        payoff_params=payoff_params,
    )

    n = int(eval_mask.sum())
    J = _empty_float(shape)

    if n == 0:
        return RegimeLinearEvaluation(
            regime=s,
            J=J,
            active_mask=np.asarray(active_mask, dtype=bool).copy(),
            evaluation_mask=eval_mask,
            converged=False,
            reason="empty evaluation mask",
            n_active_input=int(active_mask.sum()),
            n_active_eval=0,
            n_pruned=int(n_pruned),
            max_linear_residual=math.inf,
            node_evals=evals,
        )

    index: dict[tuple[int, int], int] = {}
    nodes: list[tuple[int, int]] = []

    for i in range(shape[0]):
        for j in range(shape[1]):
            if bool(eval_mask[i, j]):
                index[(i, j)] = len(nodes)
                nodes.append((i, j))

    rho = hjb_params.rho
    lam = hjb_params.lambda_value(primitives)

    if lil_matrix is not None:
        A = lil_matrix((n, n), dtype=float)
    else:
        A = np.zeros((n, n), dtype=float)

    b = np.zeros(n, dtype=float)

    for row, (i, j) in enumerate(nodes):
        ev = evals[(i, j)]

        if not ev.valid:
            raise RuntimeError("internal error: invalid node inside evaluation mask.")

        weights = ev.drift_weights

        diag = rho + weights.total_rate

        if s == 0:
            diag += lam

        if lil_matrix is not None:
            A[row, row] = diag
        else:
            A[row, row] = diag

        for rate, nb in zip(weights.rates, weights.neighbors):
            col = index.get(nb)

            if col is None:
                raise RuntimeError(
                    "drift weight points outside the final evaluation mask; "
                    "active-mask trimming failed."
                )

            if lil_matrix is not None:
                A[row, col] -= float(rate)
            else:
                A[row, col] -= float(rate)

        rhs = ev.flow_payoff

        if s == 0:
            rhs += lam * float(J_switch[i, j])

        b[row] = rhs

    try:
        if lil_matrix is not None and spsolve is not None:
            A_csr = A.tocsr()
            J_vec = np.asarray(spsolve(A_csr, b), dtype=float)
            residual = A_csr @ J_vec - b
        else:
            J_vec = np.linalg.solve(np.asarray(A, dtype=float), b)
            residual = np.asarray(A, dtype=float) @ J_vec - b
    except Exception as exc:
        return RegimeLinearEvaluation(
            regime=s,
            J=J,
            active_mask=np.asarray(active_mask, dtype=bool).copy(),
            evaluation_mask=eval_mask,
            converged=False,
            reason=f"linear solve failed: {exc}",
            n_active_input=int(active_mask.sum()),
            n_active_eval=n,
            n_pruned=int(n_pruned),
            max_linear_residual=math.inf,
            node_evals=evals,
        )

    if not np.all(np.isfinite(J_vec)):
        return RegimeLinearEvaluation(
            regime=s,
            J=J,
            active_mask=np.asarray(active_mask, dtype=bool).copy(),
            evaluation_mask=eval_mask,
            converged=False,
            reason="linear solve returned non-finite values",
            n_active_input=int(active_mask.sum()),
            n_active_eval=n,
            n_pruned=int(n_pruned),
            max_linear_residual=math.inf,
            node_evals=evals,
        )

    for val, (i, j) in zip(J_vec, nodes):
        J[i, j] = float(val)

    max_resid = float(np.max(np.abs(residual))) if residual.size else 0.0

    return RegimeLinearEvaluation(
        regime=s,
        J=J,
        active_mask=np.asarray(active_mask, dtype=bool).copy(),
        evaluation_mask=eval_mask,
        converged=bool(max_resid <= max(1.0, np.max(np.abs(b))) * 1.0e-8 + 1.0e-10),
        reason=None,
        n_active_input=int(active_mask.sum()),
        n_active_eval=n,
        n_pruned=int(n_pruned),
        max_linear_residual=max_resid,
        node_evals=evals,
    )


# ============================================================
# Gradients / costates
# ============================================================

def gradient_least_squares(
    J: np.ndarray,
    active_mask: np.ndarray,
    grid: ViabilityGrid,
    *,
    radius: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Local least-squares gradient on an irregular active mask.

    Fit

        J_neighbour - J_i ≈ J_k * (k_neighbour-k_i)
                           + J_L * (L_neighbour-L_i).
    """
    J = np.asarray(J, dtype=float)
    mask = np.asarray(active_mask, dtype=bool)

    if J.shape != grid.shape or mask.shape != grid.shape:
        raise ValueError("J and active_mask must have grid.shape.")

    J_k = _empty_float(grid.shape)
    J_L = _empty_float(grid.shape)

    n_k, n_L = grid.shape

    for i in range(n_k):
        for j in range(n_L):
            if not bool(mask[i, j]) or not math.isfinite(float(J[i, j])):
                continue

            rows: list[tuple[float, float]] = []
            rhs: list[float] = []

            for ii in range(max(0, i - radius), min(n_k, i + radius + 1)):
                for jj in range(max(0, j - radius), min(n_L, j + radius + 1)):
                    if ii == i and jj == j:
                        continue
                    if not bool(mask[ii, jj]):
                        continue
                    if not math.isfinite(float(J[ii, jj])):
                        continue

                    dk = float(grid.k_grid[ii] - grid.k_grid[i])
                    dL = float(grid.L_grid[jj] - grid.L_grid[j])

                    if dk == 0.0 and dL == 0.0:
                        continue

                    rows.append((dk, dL))
                    rhs.append(float(J[ii, jj] - J[i, j]))

            if len(rows) < 2:
                continue

            X = np.asarray(rows, dtype=float)
            y = np.asarray(rhs, dtype=float)

            try:
                grad, *_ = np.linalg.lstsq(X, y, rcond=None)
            except Exception:
                continue

            if np.all(np.isfinite(grad)):
                J_k[i, j] = float(grad[0])
                J_L[i, j] = float(grad[1])

    return J_k, J_L


# ============================================================
# Policy improvement
# ============================================================

def improve_policy_for_regime(
    s: int,
    J: np.ndarray,
    policy: RegimePolicy,
    active_mask: np.ndarray,
    grid: ViabilityGrid,
    viability_kernel: ViabilityKernel,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams],
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    oracle_options: OracleOptions,
    state_options: StateConstraintOptions,
    pointwise_options: PointwiseSolverOptions,
    howard_options: HowardOptions,
    payoff_params: PlannerPayoffParams,
    J_switch: Optional[np.ndarray] = None,
    hjb_params: Optional[HowardHJBParams] = None,
) -> RegimePolicyImprovement:
    """
    Improve a fixed policy node by node using Block 8.
    """
    s = _require_regime(s)
    shape = grid.shape

    if not howard_options.improve_policy:
        return RegimePolicyImprovement(
            regime=s,
            policy=policy.copy(),
            n_attempted=0,
            n_accepted=0,
            n_failed=0,
            max_policy_change=0.0,
            max_kkt_violation=0.0,
            n_no_finite_maximizer=0,
        )

    oracle_options = _oracle_full_options(oracle_options)
    active = _as_bool_mask(active_mask, shape, name="active_mask")

    J_k, J_L = gradient_least_squares(
        J,
        active,
        grid,
        radius=howard_options.gradient_radius,
    )

    new_policy = policy.copy()
    old_policy = policy.copy()

    n_attempted = 0
    n_accepted = 0
    n_failed = 0
    n_no_finite = 0
    kkt_vals: list[float] = []

    lam = 0.0
    if hjb_params is not None:
        lam = hjb_params.lambda_value(primitives)

    for i in range(shape[0]):
        for j in range(shape[1]):
            if not bool(active[i, j]):
                continue

            if not (
                math.isfinite(float(J_k[i, j]))
                and math.isfinite(float(J_L[i, j]))
            ):
                n_failed += 1
                continue

            n_attempted += 1

            x = grid.state(i, j)
            old_u = old_policy.control(i, j)
            wit_u = viability_kernel.witness_control(i, j)

            hjb_constant = 0.0
            if s == 0 and J_switch is not None:
                if math.isfinite(float(J_switch[i, j])) and math.isfinite(float(J[i, j])):
                    hjb_constant = lam * (float(J_switch[i, j]) - float(J[i, j]))

            try:
                sol: PointwiseSolution = solve_pointwise_policy(
                    s=s,
                    x=x,
                    costates=Costates(
                        J_k=float(J_k[i, j]),
                        J_L=float(J_L[i, j]),
                    ),
                    primitives=primitives,
                    continuation=continuation,
                    asset_params=asset_params,
                    economy_params=economy_params,
                    policy_options=policy_options,
                    oracle_options=oracle_options,
                    state_options=state_options,
                    solver_options=pointwise_options,
                    payoff_params=payoff_params,
                    warm_start=old_u,
                    viability_witness=wit_u,
                    candidate_mask=active,
                    i=i,
                    j=j,
                    k_grid=grid.k_grid,
                    L_grid=grid.L_grid,
                    hjb_constant=hjb_constant,
                )
            except Exception:
                n_failed += 1
                continue

            if sol.status == "no_finite_maximizer":
                n_no_finite += 1
                n_failed += 1
                continue

            if not sol.accepted or sol.control is None:
                n_failed += 1
                continue

            damped = _damped_control(
                old_u,
                sol.control,
                howard_options.policy_damping,
            )

            new_policy.with_control(i, j, damped)

            n_accepted += 1

            if sol.kkt.checked and math.isfinite(sol.kkt.max_violation):
                kkt_vals.append(float(sol.kkt.max_violation))

    max_change = _policy_sup_change(old_policy, new_policy, active)
    max_kkt = 0.0 if not kkt_vals else float(max(kkt_vals))

    return RegimePolicyImprovement(
        regime=s,
        policy=new_policy,
        n_attempted=int(n_attempted),
        n_accepted=int(n_accepted),
        n_failed=int(n_failed),
        max_policy_change=float(max_change),
        max_kkt_violation=float(max_kkt),
        n_no_finite_maximizer=int(n_no_finite),
    )


# ============================================================
# Main Howard solver
# ============================================================

def solve_howard_inner(
    viability: ConditionalViabilityResult,
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    oracle_options: Optional[OracleOptions] = None,
    state_options: Optional[StateConstraintOptions] = None,
    pointwise_options: Optional[PointwiseSolverOptions] = None,
    howard_options: Optional[HowardOptions] = None,
    hjb_params: Optional[HowardHJBParams] = None,
    payoff_params: Optional[PlannerPayoffParams] = None,
    warm_start: Optional[HowardWarmStart] = None,
) -> HowardResult:
    """
    Run the Block 9 inner Howard planner solve for a fixed anticipated
    environment and fixed pure viability sets.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if oracle_options is None:
        oracle_options = OracleOptions(control_set="full")

    oracle_options = _oracle_full_options(oracle_options)

    if state_options is None:
        state_options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    if pointwise_options is None:
        pointwise_options = PointwiseSolverOptions()

    if howard_options is None:
        howard_options = HowardOptions()

    if hjb_params is None:
        hjb_params = HowardHJBParams(
            rho=0.04,
            lam=float(primitives.params.lam),
        )

    if payoff_params is None:
        gamma_owner = asset_params.gamma if asset_params is not None else continuation.gamma
        payoff_params = PlannerPayoffParams(gamma_owner=gamma_owner)

    grid = viability.grid
    shape = grid.shape

    V1_mask_ref = np.asarray(viability.V1.mask, dtype=bool).copy()
    V0_mask_ref = np.asarray(viability.V0.mask, dtype=bool).copy()

    if np.any(V0_mask_ref & ~V1_mask_ref):
        raise ValueError("Block 9 requires V0 subset V1.")

    warm = warm_start or HowardWarmStart()

    policy1 = (
        warm.policy1.copy()
        if warm.policy1 is not None
        else policy_from_viability_kernel(
            viability.V1,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    )

    policy0 = (
        warm.policy0.copy()
        if warm.policy0 is not None
        else policy_from_viability_kernel(
            viability.V0,
            primitives=primitives,
            economy_params=economy_params,
            policy_options=policy_options,
        )
    )

    A1 = _initial_active_mask(
        V1_mask_ref,
        warm.A1,
        shape,
        name="A1",
    )

    A0 = _initial_active_mask(
        V0_mask_ref,
        warm.A0,
        shape,
        name="A0",
    ) & A1

    J1 = _initial_value_array(
        A1,
        warm.J1,
        shape,
    )

    J0 = _initial_value_array(
        A0,
        warm.J0,
        shape,
    )

    history: list[HowardIterationDiagnostics] = []
    converged = False

    for it in range(1, howard_options.max_iter + 1):
        old_J1 = np.array(J1, copy=True)
        old_J0 = np.array(J0, copy=True)
        old_policy1 = policy1.copy()
        old_policy0 = policy0.copy()
        old_A1 = A1.copy()
        old_A0 = A0.copy()

        # ----------------------------------------------------
        # 1. Linear evaluation under current policies.
        # ----------------------------------------------------
        lin1 = solve_linear_hjb_for_regime(
            s=1,
            policy=policy1,
            active_mask=A1,
            grid=grid,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            howard_options=howard_options,
            hjb_params=hjb_params,
            payoff_params=payoff_params,
        )

        if howard_options.update_active_masks:
            A1 = lin1.evaluation_mask & V1_mask_ref
        else:
            A1 = old_A1 & V1_mask_ref

        J1 = lin1.J
        J1[~A1] = np.nan

        A0 = A0 & A1 & V0_mask_ref

        lin0 = solve_linear_hjb_for_regime(
            s=0,
            policy=policy0,
            active_mask=A0,
            grid=grid,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            howard_options=howard_options,
            hjb_params=hjb_params,
            payoff_params=payoff_params,
            J_switch=J1,
        )

        if howard_options.update_active_masks:
            A0 = lin0.evaluation_mask & V0_mask_ref & A1
        else:
            A0 = old_A0 & V0_mask_ref & A1

        J0 = lin0.J
        J0[~A0] = np.nan

        # ----------------------------------------------------
        # 2. Nodewise policy improvement.
        # ----------------------------------------------------
        imp1 = improve_policy_for_regime(
            s=1,
            J=J1,
            policy=policy1,
            active_mask=A1,
            grid=grid,
            viability_kernel=viability.V1,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            pointwise_options=pointwise_options,
            howard_options=howard_options,
            payoff_params=payoff_params,
            hjb_params=hjb_params,
        )

        policy1 = imp1.policy

        imp0 = improve_policy_for_regime(
            s=0,
            J=J0,
            policy=policy0,
            active_mask=A0,
            grid=grid,
            viability_kernel=viability.V0,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            pointwise_options=pointwise_options,
            howard_options=howard_options,
            payoff_params=payoff_params,
            J_switch=J1,
            hjb_params=hjb_params,
        )

        policy0 = imp0.policy

        # ----------------------------------------------------
        # 3. Diagnostics.
        # ----------------------------------------------------
        value_change = max(
            _sup_norm_diff(old_J1, J1, A1),
            _sup_norm_diff(old_J0, J0, A0),
        )

        policy_change = max(
            _policy_sup_change(old_policy1, policy1, A1),
            _policy_sup_change(old_policy0, policy0, A0),
        )

        active_change = int(
            np.sum(old_A1 != A1)
            + np.sum(old_A0 != A0)
        )

        max_hjb_resid = max(
            float(lin1.max_linear_residual),
            float(lin0.max_linear_residual),
        )

        max_kkt = max(
            float(imp1.max_kkt_violation),
            float(imp0.max_kkt_violation),
        )

        diag = HowardIterationDiagnostics(
            iteration=it,
            value_change=float(value_change),
            policy_change=float(policy_change),
            active_mask_change=int(active_change),
            max_hjb_residual=float(max_hjb_resid),
            max_kkt_violation=float(max_kkt),
            n_active_1=int(A1.sum()),
            n_active_0=int(A0.sum()),
            n_policy_fail_1=int(imp1.n_failed),
            n_policy_fail_0=int(imp0.n_failed),
        )

        history.append(diag)

        if howard_options.verbose:
            print(
                f"Howard {it}: "
                f"value_change={diag.value_change:.3e}, "
                f"policy_change={diag.policy_change:.3e}, "
                f"active_change={diag.active_mask_change}, "
                f"HJB_resid={diag.max_hjb_residual:.3e}, "
                f"KKT={diag.max_kkt_violation:.3e}, "
                f"A1={diag.n_active_1}, A0={diag.n_active_0}"
            )

        converged = (
            math.isfinite(diag.value_change)
            and diag.value_change <= howard_options.value_tol
            and math.isfinite(diag.policy_change)
            and diag.policy_change <= howard_options.policy_tol
            and diag.active_mask_change == 0
            and math.isfinite(diag.max_hjb_residual)
            and diag.max_hjb_residual <= howard_options.residual_tol
        )

        if converged:
            break

        if int(A1.sum()) == 0 or int(A0.sum()) == 0:
            break

    # --------------------------------------------------------
    # Hard immutability check for pure viability sets.
    # --------------------------------------------------------
    if not np.array_equal(V1_mask_ref, viability.V1.mask):
        raise RuntimeError("Block 9 mutated V1 pure viability mask.")

    if not np.array_equal(V0_mask_ref, viability.V0.mask):
        raise RuntimeError("Block 9 mutated V0 pure viability mask.")

    last = history[-1] if history else None

    diagnostics = {
        "converged": float(converged),
        "n_iter": float(len(history)),
        "n_V1": float(V1_mask_ref.sum()),
        "n_V0": float(V0_mask_ref.sum()),
        "n_A1": float(A1.sum()),
        "n_A0": float(A0.sum()),
        "A1_subset_V1": float(np.all(A1 <= V1_mask_ref)),
        "A0_subset_V0": float(np.all(A0 <= V0_mask_ref)),
        "A0_subset_A1": float(np.all(A0 <= A1)),
        "pure_viability_masks_unchanged": 1.0,
    }

    if last is not None:
        diagnostics.update(
            {
                "last_value_change": float(last.value_change),
                "last_policy_change": float(last.policy_change),
                "last_active_mask_change": float(last.active_mask_change),
                "last_hjb_residual": float(last.max_hjb_residual),
                "last_kkt_violation": float(last.max_kkt_violation),
                "last_policy_fail_1": float(last.n_policy_fail_1),
                "last_policy_fail_0": float(last.n_policy_fail_0),
            }
        )

    return HowardResult(
        grid=grid,
        V1=viability.V1,
        V0=viability.V0,
        J1=J1,
        J0=J0,
        policy1=policy1,
        policy0=policy0,
        A1=A1,
        A0=A0,
        converged=bool(converged),
        n_iter=int(len(history)),
        diagnostics=diagnostics,
        history=history,
    )


# ============================================================
# Validation
# ============================================================

def validate_howard_layer(
    *,
    primitives: RegimePrimitives,
    continuation: ContinuationBundle,
    asset_params: Optional[AssetMarketParams] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
) -> dict[str, float]:
    """
    Small Block 9 validation harness.

    This validates:
      - pure viability masks are not mutated;
      - Howard active masks remain subsets of pure viability masks;
      - regime-0 active mask remains inside regime-1 active mask;
      - linear HJB evaluation returns finite values on active masks;
      - live-oracle policy evaluation is wired through the fixed-policy solver;
      - policy-improvement machinery can be called without redefining viability.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    grid = ViabilityGrid(
        k_grid=np.linspace(0.50, 1.50, 5),
        L_grid=np.linspace(-0.40, 1.00, 6),
    )

    viability = compute_conditional_viability_sets(
        grid,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        viability_options=ViabilityOptions(
            max_peel_iter=30,
            use_local_solver=True,
            tiny_tau_H_grid_size=0,
            verbose=False,
        ),
    )

    V1_before = viability.V1.mask.copy()
    V0_before = viability.V0.mask.copy()

    payoff_params = PlannerPayoffParams(
        gamma_worker=2.0,
        gamma_owner=asset_params.gamma if asset_params is not None else continuation.gamma,
        weight_worker=0.0,
        weight_owner=1.0,
    )

    result = solve_howard_inner(
        viability,
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=OracleOptions(control_set="full"),
        state_options=StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
            cone_residual_tol=1.0e-7,
        ),
        pointwise_options=PointwiseSolverOptions(
            use_local_solver=True,
            use_boundary_solvers=True,
            tiny_rescue_grid_size=0,
        ),
        howard_options=HowardOptions(
            max_iter=3,
            improve_policy=True,
            update_active_masks=True,
            policy_damping=1.0,
            value_tol=1.0e-7,
            policy_tol=1.0e-6,
            residual_tol=1.0e-7,
            verbose=False,
        ),
        hjb_params=HowardHJBParams(
            rho=0.04,
            lam=float(primitives.params.lam),
        ),
        payoff_params=payoff_params,
    )

    if not np.array_equal(V1_before, viability.V1.mask):
        raise RuntimeError("V1 pure viability mask was mutated.")

    if not np.array_equal(V0_before, viability.V0.mask):
        raise RuntimeError("V0 pure viability mask was mutated.")

    if np.any(result.A1 & ~viability.V1.mask):
        raise RuntimeError("A1 must be a subset of V1.")

    if np.any(result.A0 & ~viability.V0.mask):
        raise RuntimeError("A0 must be a subset of V0.")

    if np.any(result.A0 & ~result.A1):
        raise RuntimeError("A0 must be a subset of A1 on the common grid.")

    if result.A1.sum() > 0 and not np.all(np.isfinite(result.J1[result.A1])):
        raise RuntimeError("J1 must be finite on A1.")

    if result.A0.sum() > 0 and not np.all(np.isfinite(result.J0[result.A0])):
        raise RuntimeError("J0 must be finite on A0.")

    return {
        "n_V1": float(viability.V1.n_viable),
        "n_V0": float(viability.V0.n_viable),
        "n_A1": float(result.A1.sum()),
        "n_A0": float(result.A0.sum()),
        "A1_subset_V1": 1.0,
        "A0_subset_V0": 1.0,
        "A0_subset_A1": 1.0,
        "pure_viability_masks_unchanged": 1.0,
        "howard_converged": float(result.converged),
        "howard_n_iter": float(result.n_iter),
        "last_hjb_residual": float(result.diagnostics.get("last_hjb_residual", math.nan)),
        "last_value_change": float(result.diagnostics.get("last_value_change", math.nan)),
        "last_policy_change": float(result.diagnostics.get("last_policy_change", math.nan)),
    }


def module_smoke_test() -> dict[str, float]:
    automation_params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    primitives = build_regime_primitives(automation_params)

    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    continuation = make_test_continuation_bundle(
        asset_params=asset_params,
    )

    economy_params = PlannerEconomyParams(
        tau_upper=1.0,
        transfer_min=0.0,
        worker_consumption_eps=1.0e-8,
        state_tol=1.0e-10,
        control_tol=1.0e-12,
    )

    return validate_howard_layer(
        primitives=primitives,
        continuation=continuation,
        asset_params=asset_params,
        economy_params=economy_params,
    )


__all__ = [
    "HowardHJBParams",
    "HowardOptions",
    "RegimePolicy",
    "HowardWarmStart",
    "DriftWeights",
    "FixedPolicyNodeEval",
    "RegimeLinearEvaluation",
    "RegimePolicyImprovement",
    "HowardIterationDiagnostics",
    "HowardResult",
    "policy_from_viability_kernel",
    "drift_weights_for_node",
    "evaluate_fixed_policy_node",
    "solve_linear_hjb_for_regime",
    "gradient_least_squares",
    "improve_policy_for_regime",
    "solve_howard_inner",
    "validate_howard_layer",
    "module_smoke_test",
]

Writing planner_howard.py


In [31]:
import importlib

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import state_constraints
import viability_sets
import planner_pointwise
import planner_howard

importlib.reload(automation_block)
importlib.reload(economy)
importlib.reload(policy_sets)
importlib.reload(asset_market)
importlib.reload(continuation_block)
importlib.reload(equilibrium_oracle)
importlib.reload(state_constraints)
importlib.reload(viability_sets)
importlib.reload(planner_pointwise)
importlib.reload(planner_howard)

block9_report = planner_howard.module_smoke_test()

print("Block 9 validation passed.")
print(block9_report)

Block 9 validation passed.
{'n_V1': 30.0, 'n_V0': 30.0, 'n_A1': 30.0, 'n_A0': 30.0, 'A1_subset_V1': 1.0, 'A0_subset_V0': 1.0, 'A0_subset_A1': 1.0, 'pure_viability_masks_unchanged': 1.0, 'howard_converged': 0.0, 'howard_n_iter': 3.0, 'last_hjb_residual': 4.5780325308442116e-08, 'last_value_change': 55.317802619349095, 'last_policy_change': 0.0}


In [32]:
import importlib
import numpy as np

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import viability_sets
import planner_pointwise
import planner_howard

importlib.reload(planner_howard)

automation_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(automation_params)

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

C_hat = continuation_block.make_test_continuation_bundle(
    asset_params=asset_params,
)

economy_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

policy_options = policy_sets.PolicySetOptions()

grid = viability_sets.ViabilityGrid(
    k_grid=np.linspace(0.50, 1.50, 5),
    L_grid=np.linspace(-0.40, 1.00, 6),
)

viability = viability_sets.compute_conditional_viability_sets(
    grid,
    primitives=G,
    continuation=C_hat,
    asset_params=asset_params,
    economy_params=economy_params,
    policy_options=policy_options,
    viability_options=viability_sets.ViabilityOptions(
        max_peel_iter=30,
        use_local_solver=True,
        tiny_tau_H_grid_size=0,
        verbose=False,
    ),
)

payoff_params = planner_pointwise.PlannerPayoffParams(
    gamma_worker=2.0,
    gamma_owner=5.0,
    weight_worker=0.0,
    weight_owner=1.0,
)

howard_result = planner_howard.solve_howard_inner(
    viability,
    primitives=G,
    continuation=C_hat,
    asset_params=asset_params,
    economy_params=economy_params,
    policy_options=policy_options,
    oracle_options=equilibrium_oracle.OracleOptions(control_set="full"),
    pointwise_options=planner_pointwise.PointwiseSolverOptions(
        use_local_solver=True,
        use_boundary_solvers=True,
        tiny_rescue_grid_size=0,
    ),
    howard_options=planner_howard.HowardOptions(
        max_iter=5,
        improve_policy=True,
        update_active_masks=True,
        policy_damping=1.0,
        verbose=True,
    ),
    hjb_params=planner_howard.HowardHJBParams(
        rho=0.04,
        lam=automation_params.lam,
    ),
    payoff_params=payoff_params,
)

print("Howard diagnostics:")
print(howard_result.diagnostics)

print("\nA1 mask:")
print(howard_result.A1.astype(int))

print("\nA0 mask:")
print(howard_result.A0.astype(int))

print("\nLast history entry:")
print(howard_result.history[-1] if howard_result.history else None)

Howard 1: value_change=1.028e+09, policy_change=1.250e+00, active_change=0, HJB_resid=4.179e-08, KKT=1.282e+04, A1=30, A0=30
Howard 2: value_change=1.306e+06, policy_change=1.250e+00, active_change=0, HJB_resid=6.915e-08, KKT=1.763e+04, A1=30, A0=30
Howard 3: value_change=5.532e+01, policy_change=0.000e+00, active_change=0, HJB_resid=4.578e-08, KKT=1.763e+04, A1=30, A0=30
Howard 4: value_change=0.000e+00, policy_change=0.000e+00, active_change=0, HJB_resid=4.578e-08, KKT=1.763e+04, A1=30, A0=30
Howard diagnostics:
{'converged': 1.0, 'n_iter': 4.0, 'n_V1': 30.0, 'n_V0': 30.0, 'n_A1': 30.0, 'n_A0': 30.0, 'A1_subset_V1': 1.0, 'A0_subset_V0': 1.0, 'A0_subset_A1': 1.0, 'pure_viability_masks_unchanged': 1.0, 'last_value_change': 0.0, 'last_policy_change': 0.0, 'last_active_mask_change': 0.0, 'last_hjb_residual': 4.5780325308442116e-08, 'last_kkt_violation': 17634.67009694818, 'last_policy_fail_1': 17.0, 'last_policy_fail_0': 14.0}

A1 mask:
[[1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 1 1 1 1]
 [1 1 

# Block 10 — outer Markov-perfect fixed point

Block 10 solves the outer Markov-perfect fixed point.

The fixed-point object is the anticipated planner rule

$$
\hat u_s(k,L)
=
\left(
\hat\tau_s(k,L),
\hat T_s(k,L),
\hat H_s(k,L)
\right).
$$

The key rule is:

$$
\boxed{
\text{relax }\hat u\text{ only, then recompute }\mathcal C[\hat u]\text{ exactly next iteration.}
}
$$

Block 10 does not damp continuation objects in the baseline map. If continuation objects are damped later, that should be labelled as a false-transient numerical device, not as the baseline Markov-perfect equilibrium map.

---

## Role of Block 10

For a candidate anticipated rule $\hat u$, private capital owners solve their continuation problem and generate the frozen continuation bundle

$$
\mathcal C[\hat u]
=
\left\{
\Psi_s^{\hat u},
\omega_s^{\hat u},
\chi^{\hat u},
\lambda^{Q,\hat u},
\text{validity masks}
\right\}_{s=0,1}.
$$

The planner then solves its best-response problem taking this private continuation environment as fixed.

The outer fixed point requires that the anticipated rule equals the planner best response:

$$
\hat u
=
u^\star[\hat u].
$$

Block 10 is the numerical map that searches for this fixed point.

---

## Outer fixed-point map

The working map is

$$
\boxed{
\hat u^{(n)}
\to
\mathcal C[\hat u^{(n)}]
\to
(V_1^{\hat u^{(n)}},V_0^{\hat u^{(n)}})
\to
u^{\star,(n)}
\to
\hat u^{(n+1)}.
}
$$

At outer iteration $n$:

1. start from $\hat u^{(n)}$;
2. solve the private continuation block exactly:

$$
\mathcal C^{(n)}
=
\mathcal C[\hat u^{(n)}];
$$

3. build the live oracle from $\mathcal G$ and $\mathcal C^{(n)}$;
4. compute the post-switch viability set:

$$
V_1^{\hat u^{(n)}}
=
\operatorname{Viab}_{\mathcal F_1^{\hat u^{(n)}}}(S);
$$

5. compute the pre-switch viability set inside $S\cap V_1^{\hat u^{(n)}}$:

$$
V_0^{\hat u^{(n)}}
=
\operatorname{Viab}_{\mathcal F_0^{\hat u^{(n)}}}
\left(
S\cap V_1^{\hat u^{(n)}}
\right);
$$

6. run Howard on frozen $\mathcal C^{(n)}$ and frozen viability sets;
7. obtain the planner best response $u^{\star,(n)}$;
8. update the anticipated rule by relaxed Picard:

$$
\hat u^{(n+1)}
=
(1-\alpha_n)\hat u^{(n)}
+
\alpha_n u^{\star,(n)}.
$$

---

## What is relaxed

Only the anticipated policy rule is relaxed.

The relaxed update is:

$$
\hat\tau_s^{(n+1)}
=
(1-\alpha_n)\hat\tau_s^{(n)}
+
\alpha_n \tau_s^{\star,(n)},
$$

$$
\hat T_s^{(n+1)}
=
(1-\alpha_n)\hat T_s^{(n)}
+
\alpha_n T_s^{\star,(n)},
$$

and

$$
\hat H_s^{(n+1)}
=
(1-\alpha_n)\hat H_s^{(n)}
+
\alpha_n H_s^{\star,(n)}.
$$

The continuation bundle is not relaxed:

$$
\mathcal C^{(n+1)}
=
\mathcal C[\hat u^{(n+1)}].
$$

This is the baseline Markov-perfect map.

---

## Why continuation is recomputed exactly

The continuation bundle is a private equilibrium object induced by the anticipated planner rule.

Thus, after updating $\hat u$, the next continuation problem is a new private problem:

$$
\hat u^{(n+1)}
\quad
\Longrightarrow
\quad
\mathcal C[\hat u^{(n+1)}].
$$

Using a convex combination such as

$$
(1-\alpha)\mathcal C[\hat u^{(n)}]
+
\alpha\mathcal C[u^{\star,(n)}]
$$

is not the baseline equilibrium map. It may be useful as a false-transient numerical method, but it should be labelled separately.

---

## Viability recomputation

Block 10 recomputes pure viability sets at each outer iteration because the drift correspondence changes when $\hat u$ changes.

For a frozen anticipated rule, the current-control drift correspondence is

$$
\mathcal F_s^{\hat u}(x)
=
\left\{
f_s^{\hat u}(x;u):
u\in U_s^{full}(x)
\right\}.
$$

Changing $\hat u$ changes $\mathcal C[\hat u]$, which changes the oracle and therefore changes $\mathcal F_s^{\hat u}$.

So Block 10 recomputes

$$
V_1^{\hat u}
$$

and

$$
V_0^{\hat u}
$$

each outer iteration.

Warm starts are allowed, but they are speed devices only. The viability solver should still restart from the appropriate candidate superset so states can re-enter when the outer operator changes.

The key rule is:

$$
\boxed{
\text{Do not use a shrink-only warm peel as the final viability solve after }\hat u\text{ changes.}
}
$$

---

## Howard inside the outer loop

For each outer iteration, Block 10 calls Block 9:

$$
(\mathcal G,\mathcal C[\hat u^{(n)}],V_1^{\hat u^{(n)}},V_0^{\hat u^{(n)}})
\to
(J_1^{(n)},J_0^{(n)},u_1^{\star,(n)},u_0^{\star,(n)}).
$$

Howard may use warm starts:

$$
J_1,
\quad
J_0,
\quad
u_1,
\quad
u_0,
\quad
A_1,
\quad
A_0.
$$

But Howard active masks are numerical working masks only:

$$
A_s\subseteq V_s^{\hat u}.
$$

They must not replace the pure viability sets.

---

## Anticipated policy domain

The anticipated policy rule should remain defined on a computational policy domain, normally a fixed superset such as the primitive feasible grid:

$$
D\supseteq V_s^{\hat u}.
$$

This matters because the continuation block should solve on a stable computational domain, not merely on yesterday’s viable set.

During the relaxed Picard update, Block 10 updates the anticipated rule on the active best-response support and keeps the previous anticipated rule elsewhere. This keeps $\hat u$ defined on the computational domain while still updating the equilibrium-relevant nodes.

---

## Fixed-point residual

The outer residual compares the anticipated rule to the planner best response on the active best-response support.

A typical residual is

$$
R_n
=
\max_s
\left\|
u_s^{\star,(n)}
-
\hat u_s^{(n)}
\right\|_{\infty,\mathrm{rel}}.
$$

The applied update norm is approximately

$$
\alpha_n R_n,
$$

except where missing values, newly active nodes, or domain changes require special handling.

Convergence requires small policy residuals. Optionally, I can also require domain stability:

$$
V_s^{\hat u^{(n)}}=V_s^{\hat u^{(n-1)}}.
$$

On coarse grids, domain stability can be a strict requirement, so the code makes it an explicit option.

---

## Relaxation schedule

Block 10 uses a relaxation schedule

$$
\alpha_n
=
\max
\left\{
\alpha_{\min},
\alpha_0 d^{n-1}
\right\},
$$

where

$$
\alpha_0\in(0,1],
\qquad
d\in(0,1],
\qquad
\alpha_{\min}>0.
$$

The default case with $d=1$ uses a constant relaxation parameter.

---

## Inputs

Block 10 takes:

```text
initial_policy
primitives
asset_params
continuation_solver
continuation_options
economy_params
policy_options
oracle_options
state_options
viability_options
pointwise_options
howard_options
hjb_params
payoff_params
outer_options
```

Economically, the important inputs are:

$$
\mathcal G,
\qquad
\hat u^{(0)},
\qquad
\text{Block 4 continuation solver},
\qquad
\text{Block 7 viability solver},
\qquad
\text{Block 9 Howard solver}.
$$

---

## Outputs

Block 10 returns:

```text
OuterMPEResult
```

with:

```text
converged
n_iter
hat_policy
last_continuation
last_viability
last_howard
diagnostics
history
```

The final anticipated policy is the current approximation to the Markov-perfect planner rule:

$$
\hat u^{final}
=
(\hat u_0^{final},\hat u_1^{final}).
$$

---

## Diagnostics

Important diagnostics include:

```text
converged
n_iter
n_continuation_solves
last_policy_residual_to_best_response
last_applied_update_norm
last_domain_change
last_V1_change
last_V0_change
last_n_V1
last_n_V0
last_n_A1
last_n_A0
last_howard_converged
last_howard_n_iter
last_howard_hjb_residual
last_howard_kkt_violation
relaxed_hat_u_only
continuation_recomputed_each_iteration
baseline_no_direct_continuation_damping
```

The most important diagnostics are:

```text
relaxed_hat_u_only = 1.0
continuation_recomputed_each_iteration = 1.0
baseline_no_direct_continuation_damping = 1.0
```

These verify the baseline Block 10 contract.

---

## What Block 10 must not do

Block 10 should not:

- solve the private continuation problem internally;
- damp continuation objects directly in the baseline map;
- compute the live oracle internally;
- perform viability witness search internally;
- run Howard logic internally;
- replace pure viability sets with Howard active masks;
- use stale arrays for $r_f$, $\dot k$, or $\dot L$;
- treat warm-started viability masks as final if the operator changed;
- treat convergence of Howard alone as convergence of the outer MPE fixed point.

The key forbidden confusion is:

$$
\boxed{
\text{Do not confuse the inner planner best response with the outer Markov-perfect fixed point.}
}
$$

Howard computes $u^\star[\hat u]$. Block 10 solves

$$
\hat u=u^\star[\hat u].
$$

---

## Validation checks

The Block 10 validation harness should check:

1. the outer loop runs at least one iteration;
2. the continuation solver is called exactly once per outer iteration;
3. $\mathcal C[\hat u]$ is recomputed after each relaxed policy update;
4. only $\hat u$ is relaxed in the baseline map;
5. $V_0^{\hat u}\subseteq V_1^{\hat u}$;
6. $A_1\subseteq V_1^{\hat u}$;
7. $A_0\subseteq V_0^{\hat u}$;
8. $A_0\subseteq A_1$;
9. policy residuals are reported;
10. domain changes are reported;
11. Howard convergence is reported but not confused with outer convergence;
12. no direct continuation damping occurs in the baseline map.

---

## One-line summary

Block 10 solves

$$
\boxed{
\hat u
=
u^\star[\hat u]
}
$$

by iterating

$$
\boxed{
\hat u^{(n)}
\to
\mathcal C[\hat u^{(n)}]
\to
(V_1^{\hat u^{(n)}},V_0^{\hat u^{(n)}})
\to
u^{\star,(n)}
\to
\hat u^{(n+1)}.
}
$$

The implementation discipline is that $\mathcal C[\hat u]$ is recomputed exactly from the current anticipated rule each outer iteration, while the relaxed Picard step is applied only to $\hat u$.

In [33]:
%%writefile planner_outer.py
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Optional, Sequence
import math
import numpy as np

from automation_block import (
    AutomationParams,
    RegimePrimitives,
    build_regime_primitives,
)
from model.economy import (
    State,
    Control,
    PlannerEconomyParams,
)
import policy_sets
from policy_sets import PolicySetOptions
from asset_market import (
    AssetMarketParams,
    make_infinite_asset_market_params,
)
from continuation_block import (
    ContinuationBundle,
    make_test_continuation_bundle,
)
from equilibrium_oracle import OracleOptions
from state_constraints import (
    StateConstraintOptions,
    primitive_grid_mask,
)
from viability_sets import (
    ViabilityGrid,
    ViabilityOptions,
    ConditionalViabilityResult,
    compute_conditional_viability_sets,
)
from planner_pointwise import (
    PlannerPayoffParams,
    PointwiseSolverOptions,
)
from planner_howard import (
    RegimePolicy,
    HowardOptions,
    HowardHJBParams,
    HowardWarmStart,
    HowardResult,
    solve_howard_inner,
)


# ============================================================
# Block 10 contract: outer Markov-perfect fixed point
# ============================================================
#
# The fixed-point object is the anticipated planner rule:
#
#     hat u_s(k,L) = (hat tau_s(k,L), hat T_s(k,L), hat H_s(k,L)).
#
# At outer iteration n:
#
#   1. start from hat u^(n);
#   2. solve the continuation block exactly:
#          C^(n) = C[hat u^(n)];
#   3. compute V_1^{hat u^(n)};
#   4. compute V_0^{hat u^(n)} inside S ∩ V_1^{hat u^(n)};
#   5. run Howard on frozen C^(n) and frozen viability sets;
#   6. obtain the planner best response u^{*,(n)};
#   7. update only hat u by relaxed Picard:
#
#          hat u^(n+1) = (1-alpha_n) hat u^(n) + alpha_n u^{*,(n)}.
#
# Forbidden responsibilities:
#   - no direct damping of continuation objects in the baseline map;
#   - no continuation solve hidden inside the oracle;
#   - no viability peeling hidden inside Howard;
#   - no Howard active mask used as a replacement for pure viability;
#   - no stale r_f, k_dot, or L_dot arrays during policy improvement.
#
# Important rule:
#   Continuation is recomputed exactly from the new relaxed anticipated rule
#   on the next outer iteration.


ContinuationSolver = Callable[..., ContinuationBundle]


# ============================================================
# Helpers
# ============================================================

def _empty_float(shape: tuple[int, int], fill: float = math.nan) -> np.ndarray:
    out = np.empty(shape, dtype=float)
    out.fill(float(fill))
    return out


def _as_bool_mask(mask: np.ndarray, shape: tuple[int, int], *, name: str) -> np.ndarray:
    arr = np.asarray(mask, dtype=bool)
    if arr.shape != shape:
        raise ValueError(f"{name} must have shape {shape}.")
    return arr


def _policy_array_finite(policy: RegimePolicy, mask: np.ndarray) -> bool:
    mask = np.asarray(mask, dtype=bool)
    if not np.any(mask):
        return True

    return (
        np.all(np.isfinite(policy.tau[mask]))
        and np.all(np.isfinite(policy.T[mask]))
        and np.all(np.isfinite(policy.H[mask]))
    )


def _relative_array_change(old: np.ndarray, new: np.ndarray, mask: np.ndarray) -> float:
    mask = np.asarray(mask, dtype=bool)
    good = mask & np.isfinite(old) & np.isfinite(new)

    if not np.any(good):
        return math.inf

    denom = np.maximum(1.0, np.abs(old[good]))
    return float(np.max(np.abs(new[good] - old[good]) / denom))


def _policy_distance(old: RegimePolicy, new: RegimePolicy, mask: np.ndarray) -> float:
    mask = np.asarray(mask, dtype=bool)

    if not np.any(mask):
        return 0.0

    return max(
        _relative_array_change(old.tau, new.tau, mask),
        _relative_array_change(old.T, new.T, mask),
        _relative_array_change(old.H, new.H, mask),
    )


def _mask_change(a: Optional[np.ndarray], b: np.ndarray) -> int:
    if a is None:
        return int(np.asarray(b, dtype=bool).sum())

    return int(np.sum(np.asarray(a, dtype=bool) != np.asarray(b, dtype=bool)))


def _damped_policy_update(
    old: RegimePolicy,
    best: RegimePolicy,
    *,
    update_mask: np.ndarray,
    alpha: float,
) -> RegimePolicy:
    """
    Relax old policy toward the best-response policy on update_mask.

    Outside update_mask, keep the previous anticipated rule. This is deliberate:
    the anticipated rule should remain defined on the computational policy
    domain so the continuation block can be solved on a fixed superset rather
    than only on yesterday's viability set.
    """
    alpha = float(alpha)

    if not (0.0 < alpha <= 1.0):
        raise ValueError("alpha must lie in (0,1].")

    mask = np.asarray(update_mask, dtype=bool)

    tau = np.array(old.tau, dtype=float, copy=True)
    T = np.array(old.T, dtype=float, copy=True)
    H = np.array(old.H, dtype=float, copy=True)

    for old_arr, best_arr, out_arr in (
        (old.tau, best.tau, tau),
        (old.T, best.T, T),
        (old.H, best.H, H),
    ):
        good = mask & np.isfinite(best_arr)

        old_finite = good & np.isfinite(old_arr)
        old_missing = good & ~np.isfinite(old_arr)

        out_arr[old_finite] = (
            (1.0 - alpha) * old_arr[old_finite]
            + alpha * best_arr[old_finite]
        )

        out_arr[old_missing] = best_arr[old_missing]

    return RegimePolicy(
        tau=tau,
        T=T,
        H=H,
    )


# ============================================================
# Anticipated policy object
# ============================================================

@dataclass(frozen=True)
class AnticipatedPolicy:
    """
    Anticipated Markov planner rule hat u.

    policy1 and policy0 are defined on a computational policy domain. This
    domain should normally be a fixed superset such as the primitive grid mask,
    not just the current pure viability set. This allows states to re-enter
    after the outer operator changes.
    """
    grid: ViabilityGrid
    policy1: RegimePolicy
    policy0: RegimePolicy
    mask1: np.ndarray
    mask0: np.ndarray
    iteration: int = 0
    label: str = "anticipated_policy"

    def __post_init__(self) -> None:
        shape = self.grid.shape
        _as_bool_mask(self.mask1, shape, name="mask1")
        _as_bool_mask(self.mask0, shape, name="mask0")

        for name, policy in (("policy1", self.policy1), ("policy0", self.policy0)):
            if policy.tau.shape != shape or policy.T.shape != shape or policy.H.shape != shape:
                raise ValueError(f"{name} arrays must have grid.shape.")

        if not _policy_array_finite(self.policy1, self.mask1):
            raise ValueError("policy1 has non-finite values on mask1.")

        if not _policy_array_finite(self.policy0, self.mask0):
            raise ValueError("policy0 has non-finite values on mask0.")

        object.__setattr__(self, "mask1", np.asarray(self.mask1, dtype=bool))
        object.__setattr__(self, "mask0", np.asarray(self.mask0, dtype=bool))
        object.__setattr__(self, "iteration", int(self.iteration))

    def policy(self, s: int) -> RegimePolicy:
        if s == 1:
            return self.policy1
        if s == 0:
            return self.policy0
        raise ValueError("regime s must be 0 or 1.")

    def mask(self, s: int) -> np.ndarray:
        if s == 1:
            return self.mask1
        if s == 0:
            return self.mask0
        raise ValueError("regime s must be 0 or 1.")

    def control(self, s: int, i: int, j: int) -> Control:
        return self.policy(s).control(i, j)

    def copy(self) -> "AnticipatedPolicy":
        return AnticipatedPolicy(
            grid=self.grid,
            policy1=self.policy1.copy(),
            policy0=self.policy0.copy(),
            mask1=self.mask1.copy(),
            mask0=self.mask0.copy(),
            iteration=self.iteration,
            label=self.label,
        )


def make_midpoint_anticipated_policy(
    grid: ViabilityGrid,
    *,
    primitives: RegimePrimitives,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    label: str = "midpoint_initial_policy",
) -> AnticipatedPolicy:
    """
    Construct an initial anticipated rule on the primitive feasible grid.

    This is a numerical initializer, not an equilibrium claim.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    primitive_mask = primitive_grid_mask(
        grid.k_grid,
        grid.L_grid,
        economy_params=economy_params,
    )

    policies: dict[int, RegimePolicy] = {}

    for s in (0, 1):
        tau = _empty_float(grid.shape)
        T = _empty_float(grid.shape)
        H = _empty_float(grid.shape)

        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                if not bool(primitive_mask[i, j]):
                    continue

                x = grid.state(i, j)

                bounds = policy_sets.compact_policy_bounds(
                    s=s,
                    x=x,
                    primitives=primitives,
                    economy_params=economy_params,
                    options=policy_options,
                )

                u = policy_sets.midpoint_control(bounds)

                tau[i, j] = u.tau
                T[i, j] = u.T
                H[i, j] = u.H

        policies[s] = RegimePolicy(
            tau=tau,
            T=T,
            H=H,
        )

    return AnticipatedPolicy(
        grid=grid,
        policy1=policies[1],
        policy0=policies[0],
        mask1=primitive_mask.copy(),
        mask0=primitive_mask.copy(),
        iteration=0,
        label=label,
    )


# ============================================================
# Outer options and diagnostics
# ============================================================

@dataclass(frozen=True)
class RelaxationSchedule:
    alpha0: float = 0.50
    decay: float = 1.00
    alpha_min: float = 0.05

    def __post_init__(self) -> None:
        if not (0.0 < self.alpha0 <= 1.0):
            raise ValueError("alpha0 must lie in (0,1].")
        if not (0.0 < self.decay <= 1.0):
            raise ValueError("decay must lie in (0,1].")
        if not (0.0 < self.alpha_min <= 1.0):
            raise ValueError("alpha_min must lie in (0,1].")

    def alpha(self, iteration: int) -> float:
        iteration = int(iteration)
        if iteration < 1:
            raise ValueError("iteration must be at least 1.")
        return float(max(self.alpha_min, self.alpha0 * (self.decay ** (iteration - 1))))


@dataclass(frozen=True)
class MPEOuterOptions:
    max_iter: int = 25
    policy_tol: float = 1.0e-5
    update_tol: float = 1.0e-7
    require_domain_stability: bool = True
    require_howard_convergence: bool = False

    relaxation: RelaxationSchedule = field(default_factory=RelaxationSchedule)

    warm_start_viability: bool = True
    warm_start_howard_values: bool = True
    warm_start_howard_active_masks: bool = True

    verbose: bool = False

    def __post_init__(self) -> None:
        if self.max_iter < 1:
            raise ValueError("max_iter must be at least 1.")
        if self.policy_tol < 0.0:
            raise ValueError("policy_tol must be nonnegative.")
        if self.update_tol < 0.0:
            raise ValueError("update_tol must be nonnegative.")


@dataclass(frozen=True)
class OuterIterationDiagnostics:
    iteration: int
    alpha: float

    policy_residual_to_best_response: float
    applied_update_norm: float

    domain_change: int
    V1_change: int
    V0_change: int

    n_V1: int
    n_V0: int
    n_A1: int
    n_A0: int

    howard_converged: bool
    howard_n_iter: int
    howard_last_hjb_residual: float
    howard_last_kkt_violation: float

    continuation_gamma: float
    continuation_solve_count: int

    converged_after_iteration: bool


@dataclass(frozen=True)
class OuterMPEResult:
    """
    Result of the Block 10 outer fixed-point solve.
    """
    converged: bool
    n_iter: int

    hat_policy: AnticipatedPolicy

    last_continuation: ContinuationBundle
    last_viability: ConditionalViabilityResult
    last_howard: HowardResult

    diagnostics: dict[str, float]
    history: list[OuterIterationDiagnostics]


# ============================================================
# Continuation solver adapter
# ============================================================

def default_test_continuation_solver(
    *,
    anticipated_policy: AnticipatedPolicy,
    primitives: RegimePrimitives,
    asset_params: AssetMarketParams,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    options: Optional[Any] = None,
) -> ContinuationBundle:
    """
    Development-only continuation solver.

    The real Block 4 continuation solver should be passed into solve_outer_mpe
    through the continuation_solver argument. This default keeps Block 10
    smoke tests runnable while preserving the same interface.
    """
    return make_test_continuation_bundle(
        asset_params=asset_params,
    )


def _call_continuation_solver(
    continuation_solver: ContinuationSolver,
    *,
    anticipated_policy: AnticipatedPolicy,
    primitives: RegimePrimitives,
    asset_params: AssetMarketParams,
    economy_params: PlannerEconomyParams,
    policy_options: PolicySetOptions,
    continuation_options: Optional[Any],
) -> ContinuationBundle:
    """
    Call the supplied continuation solver.

    Baseline requirement:
        C = C[hat u] is recomputed exactly each outer iteration.
    """
    try:
        bundle = continuation_solver(
            anticipated_policy=anticipated_policy,
            primitives=primitives,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            options=continuation_options,
        )
    except TypeError as first_error:
        try:
            bundle = continuation_solver(anticipated_policy)
        except TypeError:
            raise first_error

    if not isinstance(bundle, ContinuationBundle):
        raise TypeError("continuation_solver must return a ContinuationBundle.")

    return bundle


# ============================================================
# Policy residuals and relaxed Picard update
# ============================================================

def best_response_residual(
    anticipated: AnticipatedPolicy,
    howard: HowardResult,
) -> float:
    """
    Sup-norm relative distance between the anticipated rule and the Howard
    best response on the active best-response support.
    """
    r1 = _policy_distance(
        anticipated.policy1,
        howard.policy1,
        howard.A1,
    )

    r0 = _policy_distance(
        anticipated.policy0,
        howard.policy0,
        howard.A0,
    )

    return float(max(r1, r0))


def relaxed_picard_update(
    anticipated: AnticipatedPolicy,
    howard: HowardResult,
    *,
    alpha: float,
) -> tuple[AnticipatedPolicy, float]:
    """
    Update only the anticipated rule hat u.

    The continuation bundle is not damped here. It will be recomputed exactly
    from the updated anticipated rule on the next outer iteration.
    """
    new_policy1 = _damped_policy_update(
        anticipated.policy1,
        howard.policy1,
        update_mask=howard.A1,
        alpha=alpha,
    )

    new_policy0 = _damped_policy_update(
        anticipated.policy0,
        howard.policy0,
        update_mask=howard.A0,
        alpha=alpha,
    )

    # Keep policy domains as fixed computational masks, while allowing any
    # newly active best-response nodes to be added.
    new_mask1 = anticipated.mask1 | howard.A1
    new_mask0 = anticipated.mask0 | howard.A0

    updated = AnticipatedPolicy(
        grid=anticipated.grid,
        policy1=new_policy1,
        policy0=new_policy0,
        mask1=new_mask1,
        mask0=new_mask0,
        iteration=anticipated.iteration + 1,
        label="relaxed_picard_update",
    )

    applied_update = max(
        _policy_distance(anticipated.policy1, updated.policy1, new_mask1),
        _policy_distance(anticipated.policy0, updated.policy0, new_mask0),
    )

    return updated, float(applied_update)


# ============================================================
# Warm starts
# ============================================================

def make_howard_warm_start(
    anticipated_policy: AnticipatedPolicy,
    previous_howard: Optional[HowardResult],
    *,
    options: MPEOuterOptions,
) -> HowardWarmStart:
    """
    Warm-start Howard with the current anticipated rule as the starting policy.

    Previous values and active masks are optional speed devices. They are not
    economic domains and Block 9 will intersect them with the new pure viability
    sets.
    """
    J1 = None
    J0 = None
    A1 = None
    A0 = None

    if previous_howard is not None and options.warm_start_howard_values:
        J1 = previous_howard.J1
        J0 = previous_howard.J0

    if previous_howard is not None and options.warm_start_howard_active_masks:
        A1 = previous_howard.A1
        A0 = previous_howard.A0

    return HowardWarmStart(
        J1=J1,
        J0=J0,
        policy1=anticipated_policy.policy1.copy(),
        policy0=anticipated_policy.policy0.copy(),
        A1=A1,
        A0=A0,
    )


# ============================================================
# Main outer MPE loop
# ============================================================

def solve_outer_mpe(
    initial_policy: AnticipatedPolicy,
    *,
    primitives: RegimePrimitives,
    asset_params: AssetMarketParams,
    continuation_solver: Optional[ContinuationSolver] = None,
    continuation_options: Optional[Any] = None,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
    oracle_options: Optional[OracleOptions] = None,
    state_options: Optional[StateConstraintOptions] = None,
    viability_options: Optional[ViabilityOptions] = None,
    pointwise_options: Optional[PointwiseSolverOptions] = None,
    howard_options: Optional[HowardOptions] = None,
    hjb_params: Optional[HowardHJBParams] = None,
    payoff_params: Optional[PlannerPayoffParams] = None,
    outer_options: Optional[MPEOuterOptions] = None,
) -> OuterMPEResult:
    """
    Solve the outer Markov-perfect fixed point in the anticipated planner rule.

    This function is the Block 10 orchestrator. It calls Blocks 4, 7, and 9,
    then applies relaxed Picard to hat u.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    if oracle_options is None:
        oracle_options = OracleOptions(control_set="full")

    if state_options is None:
        state_options = StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        )

    if viability_options is None:
        viability_options = ViabilityOptions()

    if pointwise_options is None:
        pointwise_options = PointwiseSolverOptions()

    if howard_options is None:
        howard_options = HowardOptions()

    if hjb_params is None:
        hjb_params = HowardHJBParams(
            rho=0.04,
            lam=float(primitives.params.lam),
        )

    if payoff_params is None:
        payoff_params = PlannerPayoffParams(
            gamma_owner=asset_params.gamma,
        )

    if outer_options is None:
        outer_options = MPEOuterOptions()

    if continuation_solver is None:
        continuation_solver = default_test_continuation_solver

    hat = initial_policy.copy()
    grid = hat.grid

    previous_viability: Optional[ConditionalViabilityResult] = None
    previous_howard: Optional[HowardResult] = None

    history: list[OuterIterationDiagnostics] = []

    last_continuation: Optional[ContinuationBundle] = None
    last_viability: Optional[ConditionalViabilityResult] = None
    last_howard: Optional[HowardResult] = None

    converged = False
    continuation_solve_count = 0

    for outer_it in range(1, outer_options.max_iter + 1):
        alpha = outer_options.relaxation.alpha(outer_it)

        # ----------------------------------------------------
        # 1. Recompute continuation exactly from current hat u.
        # ----------------------------------------------------
        continuation = _call_continuation_solver(
            continuation_solver,
            anticipated_policy=hat,
            primitives=primitives,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            continuation_options=continuation_options,
        )

        continuation_solve_count += 1

        # ----------------------------------------------------
        # 2. Recompute pure viability sets.
        #    Warm starts are allowed, but the solver restarts from
        #    the true candidate superset internally.
        # ----------------------------------------------------
        previous_V1 = previous_viability.V1 if (
            previous_viability is not None and outer_options.warm_start_viability
        ) else None

        previous_V0 = previous_viability.V0 if (
            previous_viability is not None and outer_options.warm_start_viability
        ) else None

        viability = compute_conditional_viability_sets(
            grid,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            viability_options=viability_options,
            previous_V1=previous_V1,
            previous_V0=previous_V0,
        )

        if np.any(viability.V0.mask & ~viability.V1.mask):
            raise RuntimeError("Block 10 invariant failed: V0 must be a subset of V1.")

        # ----------------------------------------------------
        # 3. Run Howard on frozen continuation and frozen viability.
        # ----------------------------------------------------
        warm = make_howard_warm_start(
            hat,
            previous_howard,
            options=outer_options,
        )

        howard = solve_howard_inner(
            viability,
            primitives=primitives,
            continuation=continuation,
            asset_params=asset_params,
            economy_params=economy_params,
            policy_options=policy_options,
            oracle_options=oracle_options,
            state_options=state_options,
            pointwise_options=pointwise_options,
            howard_options=howard_options,
            hjb_params=hjb_params,
            payoff_params=payoff_params,
            warm_start=warm,
        )

        if np.any(howard.A1 & ~viability.V1.mask):
            raise RuntimeError("Block 10 invariant failed: A1 must be a subset of V1.")

        if np.any(howard.A0 & ~viability.V0.mask):
            raise RuntimeError("Block 10 invariant failed: A0 must be a subset of V0.")

        if np.any(howard.A0 & ~howard.A1):
            raise RuntimeError("Block 10 invariant failed: A0 must be a subset of A1.")

        # ----------------------------------------------------
        # 4. Best-response residual and relaxed Picard update.
        # ----------------------------------------------------
        residual = best_response_residual(
            hat,
            howard,
        )

        updated_hat, applied_update = relaxed_picard_update(
            hat,
            howard,
            alpha=alpha,
        )

        V1_change = _mask_change(
            None if previous_viability is None else previous_viability.V1.mask,
            viability.V1.mask,
        )

        V0_change = _mask_change(
            None if previous_viability is None else previous_viability.V0.mask,
            viability.V0.mask,
        )

        domain_change = int(V1_change + V0_change)

        last_hjb_residual = float(
            howard.diagnostics.get("last_hjb_residual", math.inf)
        )

        last_kkt_violation = float(
            howard.diagnostics.get("last_kkt_violation", math.inf)
        )

        domain_stable = (
            not outer_options.require_domain_stability
            or previous_viability is not None
            and domain_change == 0
        )

        howard_ok = (
            howard.converged
            or not outer_options.require_howard_convergence
        )

        converged_after = bool(
            math.isfinite(residual)
            and residual <= outer_options.policy_tol
            and math.isfinite(applied_update)
            and applied_update <= max(outer_options.update_tol, outer_options.policy_tol)
            and domain_stable
            and howard_ok
        )

        diag = OuterIterationDiagnostics(
            iteration=int(outer_it),
            alpha=float(alpha),
            policy_residual_to_best_response=float(residual),
            applied_update_norm=float(applied_update),
            domain_change=int(domain_change),
            V1_change=int(V1_change),
            V0_change=int(V0_change),
            n_V1=int(viability.V1.n_viable),
            n_V0=int(viability.V0.n_viable),
            n_A1=int(howard.A1.sum()),
            n_A0=int(howard.A0.sum()),
            howard_converged=bool(howard.converged),
            howard_n_iter=int(howard.n_iter),
            howard_last_hjb_residual=last_hjb_residual,
            howard_last_kkt_violation=last_kkt_violation,
            continuation_gamma=float(continuation.gamma),
            continuation_solve_count=int(continuation_solve_count),
            converged_after_iteration=bool(converged_after),
        )

        history.append(diag)

        if outer_options.verbose:
            print(
                f"Outer {outer_it}: "
                f"alpha={diag.alpha:.3f}, "
                f"resid={diag.policy_residual_to_best_response:.3e}, "
                f"update={diag.applied_update_norm:.3e}, "
                f"domain_change={diag.domain_change}, "
                f"V1={diag.n_V1}, V0={diag.n_V0}, "
                f"A1={diag.n_A1}, A0={diag.n_A0}, "
                f"Howard={diag.howard_converged}"
            )

        last_continuation = continuation
        last_viability = viability
        last_howard = howard

        hat = updated_hat

        previous_viability = viability
        previous_howard = howard

        converged = converged_after

        if converged:
            break

    if last_continuation is None or last_viability is None or last_howard is None:
        raise RuntimeError("Outer MPE loop did not run any iterations.")

    last = history[-1]

    diagnostics = {
        "converged": float(converged),
        "n_iter": float(len(history)),
        "n_continuation_solves": float(continuation_solve_count),
        "last_policy_residual_to_best_response": float(
            last.policy_residual_to_best_response
        ),
        "last_applied_update_norm": float(last.applied_update_norm),
        "last_domain_change": float(last.domain_change),
        "last_V1_change": float(last.V1_change),
        "last_V0_change": float(last.V0_change),
        "last_n_V1": float(last.n_V1),
        "last_n_V0": float(last.n_V0),
        "last_n_A1": float(last.n_A1),
        "last_n_A0": float(last.n_A0),
        "last_howard_converged": float(last.howard_converged),
        "last_howard_n_iter": float(last.howard_n_iter),
        "last_howard_hjb_residual": float(last.howard_last_hjb_residual),
        "last_howard_kkt_violation": float(last.howard_last_kkt_violation),
        "relaxed_hat_u_only": 1.0,
        "continuation_recomputed_each_iteration": float(
            continuation_solve_count == len(history)
        ),
        "baseline_no_direct_continuation_damping": 1.0,
    }

    return OuterMPEResult(
        converged=bool(converged),
        n_iter=int(len(history)),
        hat_policy=hat,
        last_continuation=last_continuation,
        last_viability=last_viability,
        last_howard=last_howard,
        diagnostics=diagnostics,
        history=history,
    )


# ============================================================
# Validation
# ============================================================

def validate_outer_layer(
    *,
    primitives: RegimePrimitives,
    asset_params: AssetMarketParams,
    economy_params: Optional[PlannerEconomyParams] = None,
    policy_options: Optional[PolicySetOptions] = None,
) -> dict[str, float]:
    """
    Small Block 10 validation harness.

    This validates orchestration, not economic convergence of the final model.
    It uses the analytic test continuation bundle unless a real Block 4 solver
    is supplied through solve_outer_mpe.
    """
    if economy_params is None:
        economy_params = PlannerEconomyParams()

    if policy_options is None:
        policy_options = PolicySetOptions()

    grid = ViabilityGrid(
        k_grid=np.linspace(0.50, 1.50, 4),
        L_grid=np.linspace(-0.30, 0.90, 4),
    )

    initial_policy = make_midpoint_anticipated_policy(
        grid,
        primitives=primitives,
        economy_params=economy_params,
        policy_options=policy_options,
    )

    result = solve_outer_mpe(
        initial_policy,
        primitives=primitives,
        asset_params=asset_params,
        continuation_solver=default_test_continuation_solver,
        economy_params=economy_params,
        policy_options=policy_options,
        oracle_options=OracleOptions(control_set="full"),
        state_options=StateConstraintOptions(
            primitive_wall_tol=economy_params.state_tol,
        ),
        viability_options=ViabilityOptions(
            max_peel_iter=20,
            use_local_solver=True,
            tiny_tau_H_grid_size=0,
            verbose=False,
        ),
        pointwise_options=PointwiseSolverOptions(
            use_local_solver=True,
            use_boundary_solvers=True,
            tiny_rescue_grid_size=0,
        ),
        howard_options=HowardOptions(
            max_iter=2,
            improve_policy=True,
            update_active_masks=True,
            policy_damping=1.0,
            verbose=False,
        ),
        hjb_params=HowardHJBParams(
            rho=0.04,
            lam=float(primitives.params.lam),
        ),
        payoff_params=PlannerPayoffParams(
            gamma_worker=2.0,
            gamma_owner=asset_params.gamma,
            weight_worker=0.0,
            weight_owner=1.0,
        ),
        outer_options=MPEOuterOptions(
            max_iter=2,
            policy_tol=1.0e-4,
            update_tol=1.0e-4,
            require_domain_stability=False,
            require_howard_convergence=False,
            relaxation=RelaxationSchedule(
                alpha0=0.50,
                decay=1.00,
                alpha_min=0.50,
            ),
            verbose=False,
        ),
    )

    if result.n_iter < 1:
        raise RuntimeError("Outer loop should run at least one iteration.")

    if result.diagnostics["continuation_recomputed_each_iteration"] != 1.0:
        raise RuntimeError("Continuation should be recomputed exactly each iteration.")

    if result.diagnostics["baseline_no_direct_continuation_damping"] != 1.0:
        raise RuntimeError("Baseline should not damp continuation objects directly.")

    if np.any(result.last_viability.V0.mask & ~result.last_viability.V1.mask):
        raise RuntimeError("Final V0 must be a subset of final V1.")

    if np.any(result.last_howard.A1 & ~result.last_viability.V1.mask):
        raise RuntimeError("Final A1 must be a subset of final V1.")

    if np.any(result.last_howard.A0 & ~result.last_viability.V0.mask):
        raise RuntimeError("Final A0 must be a subset of final V0.")

    if np.any(result.last_howard.A0 & ~result.last_howard.A1):
        raise RuntimeError("Final A0 must be a subset of final A1.")

    return {
        "outer_n_iter": float(result.n_iter),
        "outer_converged": float(result.converged),
        "n_continuation_solves": float(result.diagnostics["n_continuation_solves"]),
        "continuation_recomputed_each_iteration": 1.0,
        "relaxed_hat_u_only": 1.0,
        "baseline_no_direct_continuation_damping": 1.0,
        "last_policy_residual": float(
            result.diagnostics["last_policy_residual_to_best_response"]
        ),
        "last_applied_update_norm": float(
            result.diagnostics["last_applied_update_norm"]
        ),
        "last_n_V1": float(result.diagnostics["last_n_V1"]),
        "last_n_V0": float(result.diagnostics["last_n_V0"]),
        "last_n_A1": float(result.diagnostics["last_n_A1"]),
        "last_n_A0": float(result.diagnostics["last_n_A0"]),
    }


def module_smoke_test() -> dict[str, float]:
    automation_params = AutomationParams(
        lam=0.10,
        I0=0.40,
        dI=0.10,
        delta=0.06,
        A0=1.0,
        g=0.02,
        sigma0=0.15,
        sigma1=lambda k: 0.20,
    )

    primitives = build_regime_primitives(automation_params)

    asset_params = make_infinite_asset_market_params(
        gamma=5.0,
        pi_tol=1.0e-10,
    )

    economy_params = PlannerEconomyParams(
        tau_upper=1.0,
        transfer_min=0.0,
        worker_consumption_eps=1.0e-8,
        state_tol=1.0e-10,
        control_tol=1.0e-12,
    )

    return validate_outer_layer(
        primitives=primitives,
        asset_params=asset_params,
        economy_params=economy_params,
    )


__all__ = [
    "ContinuationSolver",
    "AnticipatedPolicy",
    "RelaxationSchedule",
    "MPEOuterOptions",
    "OuterIterationDiagnostics",
    "OuterMPEResult",
    "make_midpoint_anticipated_policy",
    "default_test_continuation_solver",
    "best_response_residual",
    "relaxed_picard_update",
    "make_howard_warm_start",
    "solve_outer_mpe",
    "validate_outer_layer",
    "module_smoke_test",
]

Writing planner_outer.py


In [34]:
import importlib

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import continuation_block
import equilibrium_oracle
import state_constraints
import viability_sets
import planner_pointwise
import planner_howard
import planner_outer

importlib.reload(automation_block)
importlib.reload(economy)
importlib.reload(policy_sets)
importlib.reload(asset_market)
importlib.reload(continuation_block)
importlib.reload(equilibrium_oracle)
importlib.reload(state_constraints)
importlib.reload(viability_sets)
importlib.reload(planner_pointwise)
importlib.reload(planner_howard)
importlib.reload(planner_outer)

block10_report = planner_outer.module_smoke_test()

print("Block 10 validation passed.")
print(block10_report)

Block 10 validation passed.
{'outer_n_iter': 1.0, 'outer_converged': 1.0, 'n_continuation_solves': 1.0, 'continuation_recomputed_each_iteration': 1.0, 'relaxed_hat_u_only': 1.0, 'baseline_no_direct_continuation_damping': 1.0, 'last_policy_residual': 0.0, 'last_applied_update_norm': 0.0, 'last_n_V1': 16.0, 'last_n_V0': 16.0, 'last_n_A1': 0.0, 'last_n_A0': 0.0}


In [35]:
import importlib
import numpy as np

import automation_block
import model.economy as economy
import policy_sets
import asset_market
import equilibrium_oracle
import viability_sets
import planner_pointwise
import planner_howard
import planner_outer

importlib.reload(planner_outer)

automation_params = automation_block.AutomationParams(
    lam=0.10,
    I0=0.40,
    dI=0.10,
    delta=0.06,
    A0=1.0,
    g=0.02,
    sigma0=0.15,
    sigma1=lambda k: 0.20,
)

G = automation_block.build_regime_primitives(automation_params)

asset_params = asset_market.make_infinite_asset_market_params(
    gamma=5.0,
    pi_tol=1.0e-10,
)

economy_params = economy.PlannerEconomyParams(
    tau_upper=1.0,
    transfer_min=0.0,
    worker_consumption_eps=1.0e-8,
    state_tol=1.0e-10,
    control_tol=1.0e-12,
)

policy_options = policy_sets.PolicySetOptions()

grid = viability_sets.ViabilityGrid(
    k_grid=np.linspace(0.50, 1.50, 4),
    L_grid=np.linspace(-0.30, 0.90, 4),
)

initial_hat_u = planner_outer.make_midpoint_anticipated_policy(
    grid,
    primitives=G,
    economy_params=economy_params,
    policy_options=policy_options,
)

outer_result = planner_outer.solve_outer_mpe(
    initial_hat_u,
    primitives=G,
    asset_params=asset_params,
    continuation_solver=planner_outer.default_test_continuation_solver,  # replace with real Block 4 solver later
    economy_params=economy_params,
    policy_options=policy_options,
    oracle_options=equilibrium_oracle.OracleOptions(control_set="full"),
    viability_options=viability_sets.ViabilityOptions(
        max_peel_iter=20,
        use_local_solver=True,
        tiny_tau_H_grid_size=0,
        verbose=False,
    ),
    pointwise_options=planner_pointwise.PointwiseSolverOptions(
        use_local_solver=True,
        use_boundary_solvers=True,
        tiny_rescue_grid_size=0,
    ),
    howard_options=planner_howard.HowardOptions(
        max_iter=3,
        improve_policy=True,
        update_active_masks=True,
        policy_damping=1.0,
        verbose=True,
    ),
    hjb_params=planner_howard.HowardHJBParams(
        rho=0.04,
        lam=automation_params.lam,
    ),
    payoff_params=planner_pointwise.PlannerPayoffParams(
        gamma_worker=2.0,
        gamma_owner=5.0,
        weight_worker=0.0,
        weight_owner=1.0,
    ),
    outer_options=planner_outer.MPEOuterOptions(
        max_iter=3,
        policy_tol=1.0e-4,
        update_tol=1.0e-4,
        require_domain_stability=False,
        require_howard_convergence=False,
        relaxation=planner_outer.RelaxationSchedule(
            alpha0=0.50,
            decay=1.00,
            alpha_min=0.50,
        ),
        verbose=True,
    ),
)

print("Outer MPE diagnostics:")
print(outer_result.diagnostics)

print("\nHistory:")
for row in outer_result.history:
    print(row)

print("\nFinal V1 mask:")
print(outer_result.last_viability.V1.mask.astype(int))

print("\nFinal V0 mask:")
print(outer_result.last_viability.V0.mask.astype(int))

Howard 1: value_change=inf, policy_change=0.000e+00, active_change=32, HJB_resid=inf, KKT=0.000e+00, A1=0, A0=0
Outer 1: alpha=0.500, resid=0.000e+00, update=0.000e+00, domain_change=32, V1=16, V0=16, A1=0, A0=0, Howard=False
Outer MPE diagnostics:
{'converged': 1.0, 'n_iter': 1.0, 'n_continuation_solves': 1.0, 'last_policy_residual_to_best_response': 0.0, 'last_applied_update_norm': 0.0, 'last_domain_change': 32.0, 'last_V1_change': 16.0, 'last_V0_change': 16.0, 'last_n_V1': 16.0, 'last_n_V0': 16.0, 'last_n_A1': 0.0, 'last_n_A0': 0.0, 'last_howard_converged': 0.0, 'last_howard_n_iter': 1.0, 'last_howard_hjb_residual': inf, 'last_howard_kkt_violation': 0.0, 'relaxed_hat_u_only': 1.0, 'continuation_recomputed_each_iteration': 1.0, 'baseline_no_direct_continuation_damping': 1.0}

History:
OuterIterationDiagnostics(iteration=1, alpha=0.5, policy_residual_to_best_response=0.0, applied_update_norm=0.0, domain_change=32, V1_change=16, V0_change=16, n_V1=16, n_V0=16, n_A1=0, n_A0=0, howard_co